# NeuroRobust — Research Development Notebook

**Public GitHub edition.** Execution outputs, figures and traceback history have been removed to avoid publishing accidentally sensitive runtime data and to keep the repository lightweight. The original experimental notebook remains in Google Colab.

**How to run:** This is a chronological research log, not a fully automated one-click pipeline. Some later cells are session-restoration or continuation steps and depend on files saved by earlier stages. Before executing, acquire the BraTS2021 dataset under its terms, configure your own Google Colab / Drive paths, and provide any needed Kaggle credentials via Colab Secrets (not source code). See the repository README for validated results and limitations.


In [ ]:
!pip install -q nibabel

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import nibabel as nib

print("NeuroRobust environment is ready!")

In [ ]:
!pip install -q kaggle

In [ ]:
from google.colab import userdata
import os

token = userdata.get("KAGGLE_API_TOKEN")
os.environ["KAGGLE_API_TOKEN"] = token

print("Kaggle token loaded securely!")

In [ ]:
!kaggle datasets list -s "BraTS 2021" | head

In [ ]:
!kaggle datasets files dschettler8845/brats-2021-task1 | head -30

In [ ]:
!mkdir -p /content/brats_sample

!kaggle datasets download \
    -d dschettler8845/brats-2021-task1 \
    -f BraTS2021_00495.tar \
    -p /content/brats_sample

In [ ]:
!mkdir -p /content/brats_sample/patient_00495

!tar -xf /content/brats_sample/BraTS2021_00495.tar \
    -C /content/brats_sample/patient_00495

print("Dosya açıldı!")

In [ ]:
import os

folder = "/content/brats_sample/patient_00495"

for root, dirs, files in os.walk(folder):
    for file in files:
        print(os.path.join(root, file))

In [ ]:
import nibabel as nib
import numpy as np

# FLAIR MRI dosyasının yolu
flair_path = "/content/brats_sample/patient_00495/BraTS2021_00495_flair.nii.gz"

# MRI dosyasını yükle
flair_img = nib.load(flair_path)

# Görüntüyü NumPy dizisine çevir
flair_data = flair_img.get_fdata()

print("MRI shape:", flair_data.shape)
print("Minimum pixel value:", flair_data.min())
print("Maximum pixel value:", flair_data.max())

In [ ]:
import matplotlib.pyplot as plt

# Ortadaki MRI kesitini seç
slice_number = flair_data.shape[2] // 2

plt.figure(figsize=(6, 6))

plt.imshow(
    flair_data[:, :, slice_number],
    cmap="gray"
)

plt.title(f"FLAIR MRI - Slice {slice_number}")
plt.axis("off")

plt.show()

In [ ]:
seg_path = "/content/brats_sample/patient_00495/BraTS2021_00495_seg.nii.gz"

seg_img = nib.load(seg_path)
seg_data = seg_img.get_fdata()

print("Segmentation shape:", seg_data.shape)
print("Mask labels:", np.unique(seg_data))

In [ ]:
# Daha önce baktığımız 77. MRI kesitini seçiyoruz
slice_number = 77

# Yan yana iki görüntü gösterebilmek için alan oluşturuyoruz
plt.figure(figsize=(12, 5))


# -------------------------
# 1. GÖRÜNTÜ: FLAIR MRI
# -------------------------

# 1 satır, 2 sütunluk alanın ilk kısmını seç
plt.subplot(1, 2, 1)

# 3 boyutlu MRI'dan 77. kesiti göster
plt.imshow(
    flair_data[:, :, slice_number],
    cmap="gray"
)

plt.title("FLAIR MRI")
plt.axis("off")


# -------------------------
# 2. GÖRÜNTÜ: TÜMÖR MASKESİ
# -------------------------

# 1 satır, 2 sütunluk alanın ikinci kısmını seç
plt.subplot(1, 2, 2)

# Aynı hastanın aynı 77. kesitindeki
# gerçek tümör anotasyonunu göster
plt.imshow(
    seg_data[:, :, slice_number],
    cmap="viridis"
)

plt.title("Ground Truth Tumor Mask")
plt.axis("off")


# İki görüntüyü ekrana getir
plt.show()

In [ ]:
# İncelemek istediğimiz MRI kesiti
slice_number = 77


# 77. FLAIR MRI kesitini al
mri_slice = flair_data[:, :, slice_number]


# Aynı kesite ait gerçek tümör maskesini al
seg_slice = seg_data[:, :, slice_number]


# Görüntünün boyutunu belirle
plt.figure(figsize=(6, 6))


# -----------------------------
# 1. Önce normal MRI'ı çiz
# -----------------------------

plt.imshow(
    mri_slice,
    cmap="gray"
)


# ----------------------------------------
# 2. Tümör olmayan bölgeleri gizle
# ----------------------------------------

# Maskede 0 olan yerler tümör değildir.
# Bu bölgeleri göstermiyoruz.
#
# Böylece yalnızca tümör anotasyonu
# MRI'ın üzerinde görünecek.

tumor_mask = np.ma.masked_where(
    seg_slice == 0,
    seg_slice
)


# ----------------------------------------
# 3. Tümör maskesini MRI'ın üzerine koy
# ----------------------------------------

plt.imshow(
    tumor_mask,
    cmap="autumn",

    # alpha görüntünün saydamlığını belirler.
    # 0   = tamamen görünmez
    # 1   = tamamen opak
    # 0.55 = yarı saydam
    alpha=0.55
)


plt.title("FLAIR MRI + Ground Truth Tumor")

# Grafik eksenlerine ihtiyacımız olmadığı için kapatıyoruz
plt.axis("off")

# Sonucu göster
plt.show()

In [ ]:
# ---------------------------------------------------------
# AMAÇ:
# Aynı hastaya ait 4 farklı MRI sekansını Python'a yüklemek.
#
# Bunlar:
# 1. T1
# 2. T1ce
# 3. T2
# 4. FLAIR
# ---------------------------------------------------------

# Hastanın dosyalarının bulunduğu klasör
patient_folder = "/content/brats_sample/patient_00495"


# Her MRI sekansının dosya yolunu oluşturuyoruz
t1_path = patient_folder + "/BraTS2021_00495_t1.nii.gz"

t1ce_path = patient_folder + "/BraTS2021_00495_t1ce.nii.gz"

t2_path = patient_folder + "/BraTS2021_00495_t2.nii.gz"

flair_path = patient_folder + "/BraTS2021_00495_flair.nii.gz"


# ---------------------------------------------------------
# nibabel ile MRI dosyalarını açıyoruz.
#
# get_fdata() ile MRI görüntüsünü
# NumPy dizisine çeviriyoruz.
# ---------------------------------------------------------

t1_data = nib.load(t1_path).get_fdata()

t1ce_data = nib.load(t1ce_path).get_fdata()

t2_data = nib.load(t2_path).get_fdata()

flair_data = nib.load(flair_path).get_fdata()


# Boyutlarını kontrol ediyoruz.
# Dört görüntünün de aynı boyutta olmasını bekliyoruz.

print("T1 shape:   ", t1_data.shape)
print("T1ce shape: ", t1ce_data.shape)
print("T2 shape:   ", t2_data.shape)
print("FLAIR shape:", flair_data.shape)

In [ ]:
# ---------------------------------------------------------
# AMAÇ:
# Aynı beynin aynı kesitini
# 4 farklı MRI sekansında yan yana görmek.
# ---------------------------------------------------------

# Daha önce tümörü gördüğümüz kesiti kullanıyoruz
slice_number = 77


# Büyük bir grafik alanı oluşturuyoruz
plt.figure(figsize=(16, 4))


# --------------------
# 1. T1
# --------------------

plt.subplot(1, 4, 1)

plt.imshow(
    t1_data[:, :, slice_number],
    cmap="gray"
)

plt.title("T1")
plt.axis("off")


# --------------------
# 2. T1ce
# --------------------

plt.subplot(1, 4, 2)

plt.imshow(
    t1ce_data[:, :, slice_number],
    cmap="gray"
)

plt.title("T1ce")
plt.axis("off")


# --------------------
# 3. T2
# --------------------

plt.subplot(1, 4, 3)

plt.imshow(
    t2_data[:, :, slice_number],
    cmap="gray"
)

plt.title("T2")
plt.axis("off")


# --------------------
# 4. FLAIR
# --------------------

plt.subplot(1, 4, 4)

plt.imshow(
    flair_data[:, :, slice_number],
    cmap="gray"
)

plt.title("FLAIR")
plt.axis("off")


# Grafiklerin birbirine girmemesini sağlar
plt.tight_layout()

# Sonucu göster
plt.show()

In [ ]:
# ---------------------------------------------------------
# AMAÇ:
# MRI görüntülerinin bilgisayar tarafından hangi sayısal
# değerlerle temsil edildiğini incelemek.
#
# Model resmi bizim gibi "beyin" olarak görmez.
# Her voxel için bir sayı görür.
# ---------------------------------------------------------

# Dört MRI sekansı için temel istatistikleri yazdırıyoruz.

print("T1")
print("Min:", t1_data.min())
print("Max:", t1_data.max())
print("Mean:", t1_data.mean())

print("\nT1ce")
print("Min:", t1ce_data.min())
print("Max:", t1ce_data.max())
print("Mean:", t1ce_data.mean())

print("\nT2")
print("Min:", t2_data.min())
print("Max:", t2_data.max())
print("Mean:", t2_data.mean())

print("\nFLAIR")
print("Min:", flair_data.min())
print("Max:", flair_data.max())
print("Mean:", flair_data.mean())

In [ ]:
# ---------------------------------------------------------
# AMAÇ:
# MRI intensity değerlerini normalize etmek.
#
# NEDEN?
# Farklı MRI sekanslarında ve hastalarda intensity
# değerleri farklı ölçeklerde olabilir.
#
# Z-score normalization:
#
#       voxel - ortalama
# z = -----------------------
#       standart sapma
#
# ÖNEMLİ:
# Hesabı siyah arka plan üzerinde değil,
# yalnızca 0'dan farklı voxel'lar üzerinde yapıyoruz.
# ---------------------------------------------------------


def normalize_mri(image):

    # -----------------------------------------------------
    # 1. Beyin/doku bulunan alanları seçiyoruz.
    #
    # BraTS görüntülerinde arka plan genellikle 0'dır.
    # image > 0 bize arka plan dışındaki voxel'ları verir.
    # -----------------------------------------------------

    mask = image > 0


    # -----------------------------------------------------
    # 2. Sadece bu voxel'ların ortalamasını hesaplıyoruz.
    # -----------------------------------------------------

    mean = image[mask].mean()


    # -----------------------------------------------------
    # 3. Standart sapmayı hesaplıyoruz.
    # -----------------------------------------------------

    std = image[mask].std()


    # -----------------------------------------------------
    # 4. Orijinal görüntü ile aynı boyutta,
    # başlangıçta tamamen 0 olan yeni bir array oluşturuyoruz.
    # -----------------------------------------------------

    normalized = np.zeros_like(image, dtype=np.float32)


    # -----------------------------------------------------
    # 5. Sadece beyin/doku alanlarını normalize ediyoruz.
    # -----------------------------------------------------

    normalized[mask] = (
        image[mask] - mean
    ) / std


    # Normalize edilmiş görüntüyü döndürüyoruz.
    return normalized

In [ ]:
# ---------------------------------------------------------
# AMAÇ:
# Oluşturduğumuz normalize_mri() fonksiyonunu
# FLAIR görüntüsüne uygulamak.
# ---------------------------------------------------------


# Orijinal FLAIR görüntüsünü normalize ediyoruz.
flair_normalized = normalize_mri(flair_data)


# ---------------------------------------------------------
# Normalize edilmiş görüntüde,
# yalnızca beyin/doku alanlarının istatistiklerine bakıyoruz.
# ---------------------------------------------------------

brain_mask = flair_data > 0


print("NORMALIZATION ÖNCESİ")
print("Mean:", flair_data[brain_mask].mean())
print("Std :", flair_data[brain_mask].std())


print("\nNORMALIZATION SONRASI")
print("Mean:", flair_normalized[brain_mask].mean())
print("Std :", flair_normalized[brain_mask].std())

In [ ]:
# ---------------------------------------------------------
# AMAÇ:
# Normalization öncesi ve sonrası MRI görüntüsünü
# yan yana karşılaştırmak.
#
# Böylece normalization işleminin anatomik yapıyı
# bozmadığını kontrol ediyoruz.
# ---------------------------------------------------------


slice_number = 77


plt.figure(figsize=(12, 5))


# ---------------------------------------------------------
# SOL: Orijinal FLAIR
# ---------------------------------------------------------

plt.subplot(1, 2, 1)

plt.imshow(
    flair_data[:, :, slice_number],
    cmap="gray"
)

plt.title("Original FLAIR")
plt.axis("off")


# ---------------------------------------------------------
# SAĞ: Normalize edilmiş FLAIR
# ---------------------------------------------------------

plt.subplot(1, 2, 2)

plt.imshow(
    flair_normalized[:, :, slice_number],
    cmap="gray"
)

plt.title("Normalized FLAIR")
plt.axis("off")


plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------
# AMAÇ:
# T1, T1ce, T2 ve FLAIR sekanslarının HER BİRİNİ
# ayrı ayrı normalize etmek.
#
# NEDEN AYRI AYRI?
#
# Çünkü az önce gördüğümüz gibi her MRI sekansının
# intensity dağılımı birbirinden farklı.
#
# Örneğin:
# T1ce max  = 12299
# FLAIR max = 2289
#
# Bu nedenle her sekans kendi ortalama ve standart
# sapmasına göre normalize ediliyor.
# ---------------------------------------------------------


# T1'i normalize et
t1_normalized = normalize_mri(t1_data)


# T1ce'yi normalize et
t1ce_normalized = normalize_mri(t1ce_data)


# T2'yi normalize et
t2_normalized = normalize_mri(t2_data)


# FLAIR'ı normalize et
flair_normalized = normalize_mri(flair_data)


print("Dört MRI sekansı normalize edildi.")

In [ ]:
# ---------------------------------------------------------
# AMAÇ:
#
# 4 ayrı MRI sekansını tek bir
# 4-kanallı MRI verisine dönüştürmek.
#
#
# Önce:
#
# T1     -> (240, 240, 155)
# T1ce   -> (240, 240, 155)
# T2     -> (240, 240, 155)
# FLAIR  -> (240, 240, 155)
#
#
# Sonra:
#
# MRI -> (240, 240, 155, 4)
#
#
# Son boyuttaki "4":
#
# Channel 0 = T1
# Channel 1 = T1ce
# Channel 2 = T2
# Channel 3 = FLAIR
#
# ---------------------------------------------------------


multi_modal_mri = np.stack(
    [
        t1_normalized,
        t1ce_normalized,
        t2_normalized,
        flair_normalized
    ],

    # axis=-1 demek:
    # Yeni kanal boyutunu EN SONA ekle.
    axis=-1
)


# Oluşturduğumuz yeni verinin boyutunu kontrol ediyoruz.
print("Multi-modal MRI shape:", multi_modal_mri.shape)

In [ ]:
# ---------------------------------------------------------
# AMAÇ:
#
# Tek bir voxel'ın 4 MRI kanalında
# hangi değerlere sahip olduğunu görmek.
# ---------------------------------------------------------


# Örnek bir koordinat seçiyoruz.
x = 150
y = 100
z = 77


# Bu koordinattaki dört kanalın değerlerini alıyoruz.
voxel_values = multi_modal_mri[x, y, z, :]


print("Voxel coordinate:", (x, y, z))

print("T1 value:   ", voxel_values[0])
print("T1ce value: ", voxel_values[1])
print("T2 value:   ", voxel_values[2])
print("FLAIR value:", voxel_values[3])

In [ ]:
# ---------------------------------------------------------
# AMAÇ:
# Google Drive hesabımızı Colab'a bağlamak.
#
# NEDEN?
# /content klasörü geçicidir.
# Colab oturumu kapanırsa buradaki dosyalar kaybolabilir.
#
# Google Drive'a kaydettiğimiz dataset ise kalıcı olur.
# ---------------------------------------------------------

from google.colab import drive

# Google Drive'ı Colab'a bağla
drive.mount("/content/drive")

In [ ]:
# ---------------------------------------------------------
# AMAÇ:
# Google Drive içinde NeuroRobust projesi için
# düzenli bir klasör yapısı oluşturmak.
# ---------------------------------------------------------

import os


# Projemizin ana klasörü
project_dir = "/content/drive/MyDrive/NeuroRobust"


# Dataset'i saklayacağımız klasör
dataset_dir = project_dir + "/dataset"


# Daha sonra eğittiğimiz modelleri saklayacağımız klasör
models_dir = project_dir + "/models"


# Sonuçlarımızı saklayacağımız klasör
results_dir = project_dir + "/results"


# Klasörleri oluştur.
# exist_ok=True:
# klasör zaten varsa hata verme demektir.
os.makedirs(dataset_dir, exist_ok=True)
os.makedirs(models_dir, exist_ok=True)
os.makedirs(results_dir, exist_ok=True)


print("Project folder:", project_dir)
print("Dataset folder:", dataset_dir)
print("Models folder:", models_dir)
print("Results folder:", results_dir)

In [ ]:
# ---------------------------------------------------------
# AMAÇ:
# Büyük BraTS dataset'ini indirmeden önce
# Google Drive'da ne kadar boş alan olduğunu kontrol etmek.
# ---------------------------------------------------------

import shutil


# Google Drive'ın disk kullanım bilgisini al
total, used, free = shutil.disk_usage("/content/drive")


# Byte değerlerini GB'a çeviriyoruz
total_gb = total / (1024 ** 3)
used_gb = used / (1024 ** 3)
free_gb = free / (1024 ** 3)


print(f"Total space: {total_gb:.2f} GB")
print(f"Used space : {used_gb:.2f} GB")
print(f"Free space : {free_gb:.2f} GB")

In [ ]:
# ---------------------------------------------------------
# AMAÇ:
# BraTS 2021'in ana training dataset'ini
# Google Drive'daki NeuroRobust/dataset klasörüne indirmek.
#
# Daha önce sadece 1 hastayı (~10 MB) indirmiştik.
# Şimdi model eğitimi için çok sayıda hastanın bulunduğu
# ana training arşivini indiriyoruz.
#
# NOT:
# Bu dosya büyük olduğu için indirme biraz sürebilir.
# ---------------------------------------------------------


# Kaggle'dan sadece Training Data arşivini indiriyoruz.
#
# -d : hangi Kaggle dataset'i?
# -f : dataset içindeki hangi dosya?
# -p : dosya nereye kaydedilecek?

!kaggle datasets download \
    -d dschettler8845/brats-2021-task1 \
    -f BraTS2021_Training_Data.tar \
    -p "/content/drive/MyDrive/NeuroRobust/dataset"

In [ ]:
# ---------------------------------------------------------
# AMAÇ:
# Büyük training dosyasının Google Drive'a
# başarıyla kaydedildiğini kontrol etmek.
# ---------------------------------------------------------

import os


# İndirdiğimiz dosyanın tam yolu
training_tar = (
    "/content/drive/MyDrive/NeuroRobust/dataset/"
    "BraTS2021_Training_Data.tar"
)


# Dosya gerçekten var mı?
if os.path.exists(training_tar):

    # Dosya boyutunu byte'tan GB'a çeviriyoruz
    file_size_gb = os.path.getsize(training_tar) / (1024 ** 3)

    print("Dataset bulundu! ✅")
    print(f"Dosya boyutu: {file_size_gb:.2f} GB")

else:

    print("Dataset bulunamadı ❌")

In [ ]:
# ---------------------------------------------------------
# AMAÇ:
# 12.48 GB'lık .tar arşivini açmadan önce
# içinde hangi dosya ve klasörlerin bulunduğunu görmek.
#
# NEDEN?
# Dataset'in klasör yapısını anlamadan bütün arşivi
# çıkarmak istemiyoruz.
#
# Bu kod SADECE ilk 30 dosya/klasör adını gösterir.
# Dosyaları henüz Drive'a çıkartmaz.
# ---------------------------------------------------------

import tarfile


# Daha önce tanımladığımız training_tar:
# /content/drive/MyDrive/NeuroRobust/dataset/
# BraTS2021_Training_Data.tar


# TAR arşivini sadece okuma modunda açıyoruz.
with tarfile.open(training_tar, "r") as tar:

    # Arşivde bulunan bütün öğelerin listesini alıyoruz.
    members = tar.getmembers()

    # Toplam kaç öğe olduğunu yazdırıyoruz.
    print("Arşivdeki toplam öğe sayısı:", len(members))

    print("\nİlk 30 öğe:")
    print("-" * 60)

    # Sadece ilk 30 tanesinin ismini gösteriyoruz.
    for member in members[:30]:
        print(member.name)

In [ ]:
# ---------------------------------------------------------
# AMAÇ:
# BraTS training dataset'inde kaç farklı hasta olduğunu
# TAR dosyasını çıkarmadan hesaplamak.
#
# ÖNEMLİ:
# Bizim arşivimizde dosya yolları şöyle başlıyor:
#
# ./BraTS2021_00000/...
#
# Bu nedenle baştaki "./" kısmını temizleyeceğiz.
# ---------------------------------------------------------

import tarfile


# Hasta ID'lerini burada tutacağız.
#
# "set" kullanmamızın nedeni:
# Aynı hastaya ait 5 farklı dosya olsa bile
# hasta adını sadece 1 kere saklamak.
patient_ids = set()


with tarfile.open(training_tar, "r") as tar:

    # Arşivdeki bütün dosya ve klasörleri sırayla inceliyoruz.
    for member in tar:

        # Örneğin:
        #
        # ./BraTS2021_00000/BraTS2021_00000_flair.nii.gz
        #
        # Baştaki "./" karakterlerini temizliyoruz.
        clean_name = member.name.lstrip("./")


        # Dosya yolunu "/" işaretinden bölüyoruz.
        #
        # Sonuç yaklaşık şöyle olur:
        #
        # [
        #   "BraTS2021_00000",
        #   "BraTS2021_00000_flair.nii.gz"
        # ]
        parts = clean_name.split("/")


        # İlk bölüm hasta ID'sidir.
        if len(parts) > 0:

            patient_id = parts[0]


            # Sadece gerçek BraTS hasta klasörlerini al.
            #
            # Böylece .DS_Store gibi gereksiz dosyalar
            # listeye girmez.
            if patient_id.startswith("BraTS2021_"):

                patient_ids.add(patient_id)


# Hasta isimlerini alfabetik olarak sıralıyoruz.
patient_ids = sorted(patient_ids)


# Toplam hasta sayısını göster.
print("Toplam hasta sayısı:", len(patient_ids))


# Kontrol amacıyla ilk 10 hastayı göster.
print("\nİlk 10 hasta:")

for patient in patient_ids[:10]:
    print(patient)

In [ ]:
# ---------------------------------------------------------
# AMAÇ:
# 1251 hastanın HER BİRİNDE gerekli 5 dosyanın
# bulunup bulunmadığını kontrol etmek.
#
# Beklediğimiz dosyalar:
#
# 1. T1
# 2. T1ce
# 3. T2
# 4. FLAIR
# 5. SEG (gerçek tümör maskesi)
#
# Henüz TAR dosyasını çıkarmıyoruz.
# Sadece içindeki DOSYA İSİMLERİNİ kontrol ediyoruz.
# ---------------------------------------------------------

import tarfile


# ---------------------------------------------------------
# Her hastada bulunmasını beklediğimiz MRI türleri
# ---------------------------------------------------------

required_modalities = [
    "t1",
    "t1ce",
    "t2",
    "flair",
    "seg"
]


# Eksik dosyası bulunan hastaları burada saklayacağız.
missing_patients = []


# ---------------------------------------------------------
# TAR dosyasını aç
# ---------------------------------------------------------

with tarfile.open(training_tar, "r") as tar:

    # TAR içindeki bütün dosya isimlerini alıyoruz.
    #
    # Baştaki "./" kısmını da temizliyoruz.
    file_names = {
        member.name.lstrip("./")
        for member in tar.getmembers()
        if member.isfile()
    }


# ---------------------------------------------------------
# Şimdi 1251 hastayı tek tek kontrol ediyoruz.
# ---------------------------------------------------------

for patient_id in patient_ids:

    # Bu hastada eksik olan modality'leri
    # geçici olarak burada tutacağız.
    missing = []

    for modality in required_modalities:

        # Örneğin patient_id:
        #
        # BraTS2021_00000
        #
        # modality:
        #
        # flair
        #
        # Beklenen dosya:
        #
        # BraTS2021_00000/
        # BraTS2021_00000_flair.nii.gz

        expected_file = (
            f"{patient_id}/"
            f"{patient_id}_{modality}.nii.gz"
        )


        # Dosya TAR içinde yoksa:
        if expected_file not in file_names:

            missing.append(modality)


    # Bu hastada en az bir dosya eksikse
    # listeye ekliyoruz.
    if len(missing) > 0:

        missing_patients.append(
            (patient_id, missing)
        )


# ---------------------------------------------------------
# SONUÇ
# ---------------------------------------------------------

print("Toplam hasta:", len(patient_ids))

print(
    "Tüm 5 dosyası eksiksiz hasta:",
    len(patient_ids) - len(missing_patients)
)

print(
    "Eksik dosyası bulunan hasta:",
    len(missing_patients)
)


# Eğer eksik hasta varsa ilk 10 tanesini göster.
if len(missing_patients) > 0:

    print("\nEksik dosyası olan ilk hastalar:")

    for patient, missing in missing_patients[:10]:

        print(
            patient,
            "-> Eksik:",
            missing
        )

else:

    print("\nTüm hastalarda gerekli 5 dosya mevcut! ✅")

In [ ]:
# ---------------------------------------------------------
# AMAÇ:
# 1251 hastayı hasta bazında
# TRAIN / VALIDATION / TEST olarak ayırmak.
#
# ORANLAR:
#
# Train      = %70
# Validation = %15
# Test       = %15
#
# ÖNEMLİ:
# Burada MRI slice'larını DEĞİL,
# HASTALARI ayırıyoruz.
#
# Böylece aynı hastanın görüntülerinin
# hem train hem test grubuna girmesini engelliyoruz.
# ---------------------------------------------------------

from sklearn.model_selection import train_test_split


# ---------------------------------------------------------
# 1. Önce hastaların %70'ini TRAIN olarak ayır.
#
# Geriye kalan %30'u geçici olarak "temp" grubuna koy.
# ---------------------------------------------------------

train_patients, temp_patients = train_test_split(

    patient_ids,

    # %30 validation + test için ayrılacak
    test_size=0.30,

    # Aynı kodu tekrar çalıştırdığımızda
    # AYNI hastaların aynı gruplara düşmesini sağlar.
    random_state=42
)


# ---------------------------------------------------------
# 2. Kalan %30'u iki eşit parçaya ayır.
#
# %15 Validation
# %15 Test
# ---------------------------------------------------------

val_patients, test_patients = train_test_split(

    temp_patients,

    # temp'in yarısı test olur.
    test_size=0.50,

    # Yine sonucu tekrarlanabilir yapıyoruz.
    random_state=42
)


# ---------------------------------------------------------
# SONUÇLARI GÖSTER
# ---------------------------------------------------------

print("Toplam hasta :", len(patient_ids))

print("\nTrain hasta sayısı     :", len(train_patients))
print("Validation hasta sayısı:", len(val_patients))
print("Test hasta sayısı      :", len(test_patients))


# Yüzdelerini de kontrol edelim.

print("\nORANLAR")

print(
    "Train:",
    round(len(train_patients) / len(patient_ids) * 100, 2),
    "%"
)

print(
    "Validation:",
    round(len(val_patients) / len(patient_ids) * 100, 2),
    "%"
)

print(
    "Test:",
    round(len(test_patients) / len(patient_ids) * 100, 2),
    "%"
)

In [ ]:
# ---------------------------------------------------------
# AMAÇ:
# Train, Validation ve Test grupları arasında
# aynı hastanın bulunmadığını doğrulamak.
#
# Eğer aynı hasta iki grupta bulunursa:
#
# DATA LEAKAGE oluşabilir.
#
# Bu nedenle grupların kesişimini kontrol ediyoruz.
# ---------------------------------------------------------


# Listeleri set'e çeviriyoruz.
train_set = set(train_patients)
val_set = set(val_patients)
test_set = set(test_patients)


# ---------------------------------------------------------
# Train ile Validation arasında ortak hasta var mı?
# ---------------------------------------------------------

train_val_overlap = train_set.intersection(val_set)


# ---------------------------------------------------------
# Train ile Test arasında ortak hasta var mı?
# ---------------------------------------------------------

train_test_overlap = train_set.intersection(test_set)


# ---------------------------------------------------------
# Validation ile Test arasında ortak hasta var mı?
# ---------------------------------------------------------

val_test_overlap = val_set.intersection(test_set)


print(
    "Train - Validation ortak hasta:",
    len(train_val_overlap)
)

print(
    "Train - Test ortak hasta:",
    len(train_test_overlap)
)

print(
    "Validation - Test ortak hasta:",
    len(val_test_overlap)
)


# ---------------------------------------------------------
# Eğer hepsi 0 ise split doğru yapılmıştır.
# ---------------------------------------------------------

if (
    len(train_val_overlap) == 0
    and len(train_test_overlap) == 0
    and len(val_test_overlap) == 0
):

    print("\nData leakage yok! ✅")

else:

    print("\nDİKKAT: Gruplar arasında ortak hasta var! ❌")

In [ ]:
# ---------------------------------------------------------
# AMAÇ:
# Train / Validation / Test hasta listelerini
# Google Drive'a kalıcı olarak kaydetmek.
#
# Böylece Colab kapansa bile hangi hastanın
# hangi grupta olduğunu kaybetmeyeceğiz.
# ---------------------------------------------------------

import json
import os


# Split dosyasının kaydedileceği yer
split_file = os.path.join(
    project_dir,
    "patient_split.json"
)


# Kaydetmek istediğimiz bilgileri
# Python dictionary olarak hazırlıyoruz.
split_data = {

    "train": train_patients,

    "validation": val_patients,

    "test": test_patients
}


# JSON dosyasına yazıyoruz.
with open(split_file, "w") as file:

    json.dump(
        split_data,
        file,

        # Dosyanın insan tarafından da
        # rahat okunabilmesini sağlar.
        indent=4
    )


print("Patient split kaydedildi! ✅")
print("Dosya yolu:")
print(split_file)

In [ ]:
# ---------------------------------------------------------
# 35. HÜCRE
#
# AMAÇ:
# BraTS2021_Training_Data.tar dosyasını açarak
# hastaların MRI dosyalarını ayrı klasörler halinde
# Google Drive'a çıkarmak.
#
#
# ŞU ANDA:
#
# dataset/
# └── BraTS2021_Training_Data.tar
#
#
# İŞLEM SONRASINDA:
#
# dataset/
# ├── BraTS2021_Training_Data.tar
# │
# └── BraTS2021_Training_Data/
#       │
#       ├── BraTS2021_00000/
#       │      ├── ..._t1.nii.gz
#       │      ├── ..._t1ce.nii.gz
#       │      ├── ..._t2.nii.gz
#       │      ├── ..._flair.nii.gz
#       │      └── ..._seg.nii.gz
#       │
#       ├── BraTS2021_00002/
#       └── ...
#
# ---------------------------------------------------------

import os
import tarfile


# Arşivin bulunduğu dosya
training_tar = (
    "/content/drive/MyDrive/NeuroRobust/dataset/"
    "BraTS2021_Training_Data.tar"
)


# Arşivin çıkarılacağı klasör
extracted_dir = (
    "/content/drive/MyDrive/NeuroRobust/dataset/"
    "BraTS2021_Training_Data"
)


# Klasör yoksa oluştur.
os.makedirs(
    extracted_dir,
    exist_ok=True
)


print("Dataset çıkarılıyor...")
print("Bu işlem biraz sürebilir.\n")


# TAR dosyasını okuma modunda açıyoruz.
with tarfile.open(training_tar, "r") as tar:

    # Arşivdeki bütün dosyaları
    # belirlediğimiz klasöre çıkarıyoruz.
    tar.extractall(path=extracted_dir)


print("Dataset başarıyla çıkarıldı! ✅")

print("\nDataset klasörü:")
print(extracted_dir)

In [ ]:
# ---------------------------------------------------------
# 36. HÜCRE
#
# AMAÇ:
# TAR arşivinden çıkarılan hasta klasörlerinin
# sayısını kontrol etmek.
# ---------------------------------------------------------

import os


# Dataset klasöründeki bütün öğeleri al.
items = os.listdir(extracted_dir)


# Sadece BraTS hasta klasörlerini seç.
extracted_patients = [

    item for item in items

    if (
        item.startswith("BraTS2021_")
        and
        os.path.isdir(
            os.path.join(extracted_dir, item)
        )
    )
]


# Hasta ID'lerini sırala.
extracted_patients = sorted(extracted_patients)


print(
    "Çıkarılan toplam hasta sayısı:",
    len(extracted_patients)
)


print("\nİlk 10 hasta:")

for patient in extracted_patients[:10]:

    print(patient)

In [ ]:
# ---------------------------------------------------------
# 37. HÜCRE
#
# AMAÇ:
# Çıkarılan dataset içerisindeki bir hastanın
# dosyalarını kontrol etmek.
#
# Böylece Python'ın ileride MRI dosyalarına
# hangi yollarla ulaşacağını görmüş olacağız.
# ---------------------------------------------------------


# İlk hastayı seçiyoruz.
example_patient = extracted_patients[0]


# Hastanın klasör yolunu oluşturuyoruz.
example_patient_dir = os.path.join(
    extracted_dir,
    example_patient
)


print("Seçilen hasta:")
print(example_patient)


print("\nHasta klasörü:")
print(example_patient_dir)


print("\nDosyalar:")
print("-" * 50)


# Hastanın klasöründeki dosyaları sırayla göster.
for file_name in sorted(
    os.listdir(example_patient_dir)
):

    print(file_name)

In [ ]:
# ---------------------------------------------------------
# 38. HÜCRE
#
# AMAÇ:
# Verdiğimiz bir hasta ID'sine göre:
#
# 1. T1 görüntüsünü bul
# 2. T1ce görüntüsünü bul
# 3. T2 görüntüsünü bul
# 4. FLAIR görüntüsünü bul
# 5. SEG maskesini bul
# 6. MRI dosyalarını yükle
# 7. Her MRI sekansını ayrı ayrı normalize et
# 8. 4 sekansı tek bir 4-kanallı görüntü haline getir
#
# SONUÇ:
#
# image -> (240, 240, 155, 4)
# mask  -> (240, 240, 155)
#
# ---------------------------------------------------------

import os
import numpy as np
import nibabel as nib


def load_patient(patient_id):

    # -----------------------------------------------------
    # 1. Hastanın klasörünü oluşturuyoruz.
    #
    # Örneğin:
    #
    # .../BraTS2021_Training_Data/BraTS2021_00000
    # -----------------------------------------------------

    patient_dir = os.path.join(
        extracted_dir,
        patient_id
    )


    # -----------------------------------------------------
    # 2. Hastanın 5 dosyasının yolunu oluşturuyoruz.
    # -----------------------------------------------------

    t1_path = os.path.join(
        patient_dir,
        f"{patient_id}_t1.nii.gz"
    )

    t1ce_path = os.path.join(
        patient_dir,
        f"{patient_id}_t1ce.nii.gz"
    )

    t2_path = os.path.join(
        patient_dir,
        f"{patient_id}_t2.nii.gz"
    )

    flair_path = os.path.join(
        patient_dir,
        f"{patient_id}_flair.nii.gz"
    )

    seg_path = os.path.join(
        patient_dir,
        f"{patient_id}_seg.nii.gz"
    )


    # -----------------------------------------------------
    # 3. NIfTI MRI dosyalarını belleğe yüklüyoruz.
    #
    # get_fdata() bize görüntüyü NumPy array olarak verir.
    # -----------------------------------------------------

    t1 = nib.load(t1_path).get_fdata()

    t1ce = nib.load(t1ce_path).get_fdata()

    t2 = nib.load(t2_path).get_fdata()

    flair = nib.load(flair_path).get_fdata()


    # -----------------------------------------------------
    # SEG dosyasını da yüklüyoruz.
    #
    # Bu bizim Ground Truth'umuz.
    # Yani modelin öğrenmeye çalışacağı gerçek cevap.
    # -----------------------------------------------------

    mask = nib.load(seg_path).get_fdata()


    # -----------------------------------------------------
    # 4. Her MRI sekansını AYRI AYRI normalize ediyoruz.
    #
    # Daha önce yazdığımız normalize_mri()
    # fonksiyonunu burada tekrar kullanıyoruz.
    # -----------------------------------------------------

    t1 = normalize_mri(t1)

    t1ce = normalize_mri(t1ce)

    t2 = normalize_mri(t2)

    flair = normalize_mri(flair)


    # -----------------------------------------------------
    # 5. Dört MRI sekansını tek görüntüde birleştiriyoruz.
    #
    # Önce:
    #
    # T1     -> (240, 240, 155)
    # T1ce   -> (240, 240, 155)
    # T2     -> (240, 240, 155)
    # FLAIR  -> (240, 240, 155)
    #
    # Sonra:
    #
    # image -> (240, 240, 155, 4)
    # -----------------------------------------------------

    image = np.stack(
        [
            t1,
            t1ce,
            t2,
            flair
        ],
        axis=-1
    )


    # -----------------------------------------------------
    # Bellek kullanımını azaltmak için image'ı
    # float32 olarak tutuyoruz.
    # -----------------------------------------------------

    image = image.astype(np.float32)


    # -----------------------------------------------------
    # Maskede sınıf numaraları bulunduğu için
    # float kullanmamıza gerek yok.
    #
    # int16 yeterlidir.
    # -----------------------------------------------------

    mask = mask.astype(np.int16)


    # -----------------------------------------------------
    # Fonksiyon iki sonuç döndürüyor:
    #
    # image = modelin girdisi
    # mask  = modelin öğrenmesi gereken doğru cevap
    # -----------------------------------------------------

    return image, mask

In [ ]:
# ---------------------------------------------------------
# 39. HÜCRE
#
# AMAÇ:
# load_patient() fonksiyonunun gerçekten çalışıp
# çalışmadığını ilk hastada test etmek.
# ---------------------------------------------------------


# İlk hastayı seçiyoruz.
test_patient_id = extracted_patients[0]


print("Yüklenen hasta:", test_patient_id)


# ---------------------------------------------------------
# Bir önceki hücrede yazdığımız fonksiyonu çağırıyoruz.
#
# Fonksiyon:
#
# image -> 4 MRI sekansını
# mask  -> gerçek tümör segmentasyonunu
#
# bize verecek.
# ---------------------------------------------------------

image, mask = load_patient(test_patient_id)


print("\nYükleme tamamlandı! ✅")


# Boyutları kontrol ediyoruz.
print("\nImage shape:", image.shape)

print("Mask shape :", mask.shape)


# Veri tiplerini de kontrol ediyoruz.
print("\nImage dtype:", image.dtype)

print("Mask dtype :", mask.dtype)

In [ ]:
# ---------------------------------------------------------
# 40. HÜCRE
#
# AMAÇ:
# Ground Truth segmentation maskesinde hangi
# sınıf numaralarının bulunduğunu görmek.
#
# np.unique():
# Bir array içerisinde bulunan farklı değerleri
# sadece bir kez gösterir.
#
# Örneğin maskede:
#
# 0 0 0 1 1 2 2 4 4
#
# varsa:
#
# np.unique(mask)
#
# bize:
#
# [0, 1, 2, 4]
#
# verir.
# ---------------------------------------------------------


unique_classes = np.unique(mask)


print("Maskede bulunan sınıflar:")

print(unique_classes)

In [ ]:
# ---------------------------------------------------------
# 41. HÜCRE
#
# AMAÇ:
# load_patient() fonksiyonundan gelen verinin
# görsel olarak doğru olduğunu kontrol etmek.
#
# Slice 77'yi kullanıyoruz.
#
# image[:, :, 77, 3]
#
# Buradaki:
#
# :   -> bütün x koordinatları
# :   -> bütün y koordinatları
# 77  -> seçtiğimiz slice
# 3   -> FLAIR kanalı
#
# Kanal sırası:
#
# 0 = T1
# 1 = T1ce
# 2 = T2
# 3 = FLAIR
# ---------------------------------------------------------

import matplotlib.pyplot as plt


slice_number = 77


plt.figure(figsize=(12, 5))


# ---------------------------------------------------------
# SOL TARAF:
# Normalize edilmiş FLAIR görüntüsü
# ---------------------------------------------------------

plt.subplot(1, 2, 1)

plt.imshow(
    image[:, :, slice_number, 3],
    cmap="gray"
)

plt.title(
    f"{test_patient_id} - Normalized FLAIR"
)

plt.axis("off")


# ---------------------------------------------------------
# SAĞ TARAF:
# Ground Truth segmentation maskesi
# ---------------------------------------------------------

plt.subplot(1, 2, 2)

plt.imshow(
    mask[:, :, slice_number]
)

plt.title(
    "Ground Truth Tumor Mask"
)

plt.axis("off")


plt.tight_layout()

plt.show()

## 🧠 Multimodal MRI Preprocessing & Ground-Truth Verification

Before training the segmentation model, the preprocessing pipeline was validated on an individual BraTS 2021 subject (`BraTS2021_00000`).

Each subject contains four co-registered MRI modalities:

- **T1-weighted (T1)** – provides structural anatomical information.
- **Contrast-enhanced T1 (T1ce)** – highlights contrast-enhancing tumor regions.
- **T2-weighted (T2)** – provides information about tissue and fluid-related abnormalities.
- **FLAIR** – suppresses free-fluid signal and helps visualize abnormal hyperintense regions surrounding the tumor.

Each MRI modality was independently normalized using **Z-score normalization over non-background voxels**:

\[
z = \frac{x-\mu}{\sigma}
\]

where $\mu$ and $\sigma$ are calculated only from non-zero voxels. This prevents the large zero-valued background region from dominating the normalization statistics.

The four normalized MRI volumes were then stacked to construct a multimodal input:

`(240, 240, 155, 4)`

where the final dimension corresponds to:

`[T1, T1ce, T2, FLAIR]`

### Ground-Truth Segmentation

The corresponding BraTS segmentation mask contains the labels:

`[0, 1, 2, 4]`

representing background and annotated tumor subregions.

The visualization above shows the **same axial slice** from the normalized FLAIR volume and its corresponding ground-truth segmentation mask. The spatial agreement between the visible abnormal region in the MRI and the annotated tumor region confirms that the image and segmentation mask remain aligned after preprocessing.

> **Important:** The colored segmentation image is the ground-truth annotation provided by the dataset, not a model prediction. It will later serve as the target against which the U-Net prediction is optimized and evaluated.

### Why this validation matters

Medical image segmentation pipelines are highly sensitive to preprocessing and image-mask alignment. Before model training, this step verifies that:

1. MRI volumes are loaded correctly.
2. All four modalities remain spatially aligned.
3. Normalization preserves anatomical structure.
4. The segmentation mask corresponds to the correct MRI slice.
5. The data pipeline is ready for patient-level model training.

This forms the preprocessing foundation for the next stage: **multi-modal brain tumor segmentation using U-Net**, followed by robustness experiments under corrupted or missing MRI modalities.

In [ ]:
# ---------------------------------------------------------
# 42. HÜCRE
#
# AMAÇ:
# FLAIR MRI görüntüsü ile Ground Truth tümör maskesini
# aynı görüntü üzerinde üst üste göstermek (overlay).
#
# NEDEN?
# Önceki görselde MRI ve maskeyi ayrı ayrı gördük.
#
# Bu görsel sayesinde:
#
# "Tümör anotasyonu MRI'daki anormal bölgeyle
# gerçekten aynı yerde mi?"
#
# sorusunu doğrudan kontrol edebiliriz.
# ---------------------------------------------------------

import matplotlib.pyplot as plt
import numpy as np


# ---------------------------------------------------------
# Görüntülemek istediğimiz MRI kesiti
# ---------------------------------------------------------

slice_number = 77


# ---------------------------------------------------------
# Aynı slice'ın FLAIR görüntüsünü alıyoruz.
#
# image boyutu:
# (240, 240, 155, 4)
#
# Son boyuttaki:
# 0 = T1
# 1 = T1ce
# 2 = T2
# 3 = FLAIR
# ---------------------------------------------------------

flair_slice = image[:, :, slice_number, 3]


# ---------------------------------------------------------
# Aynı slice'ın Ground Truth maskesini alıyoruz.
# ---------------------------------------------------------

mask_slice = mask[:, :, slice_number]


# ---------------------------------------------------------
# Görsel oluşturuyoruz.
# Üç görüntüyü yan yana göstereceğiz:
#
# 1. FLAIR
# 2. Ground Truth
# 3. FLAIR + Ground Truth Overlay
# ---------------------------------------------------------

plt.figure(figsize=(18, 6))


# =========================================================
# 1. NORMALIZED FLAIR
# =========================================================

plt.subplot(1, 3, 1)

plt.imshow(
    flair_slice,
    cmap="gray"
)

plt.title("Normalized FLAIR - Slice 77")

plt.axis("off")


# =========================================================
# 2. GROUND TRUTH MASK
# =========================================================

plt.subplot(1, 3, 2)

plt.imshow(
    mask_slice,
    cmap="viridis"
)

plt.title("Ground Truth Tumor Mask")

plt.axis("off")


# =========================================================
# 3. MRI + MASK OVERLAY
# =========================================================

plt.subplot(1, 3, 3)


# Önce MRI görüntüsünü çiziyoruz.
plt.imshow(
    flair_slice,
    cmap="gray"
)


# ---------------------------------------------------------
# Background'u overlay'den çıkarıyoruz.
#
# Maskedeki 0 değeri background.
#
# Eğer bunu çıkarmazsak tüm MRI'ın üzerine renk gelir.
#
# np.ma.masked_where:
# mask_slice == 0 olan yerleri görünmez yapar.
# ---------------------------------------------------------

tumor_overlay = np.ma.masked_where(
    mask_slice == 0,
    mask_slice
)


# ---------------------------------------------------------
# Şimdi yalnızca tümör bulunan bölgeleri
# MRI'ın üzerine yarı şeffaf olarak çiziyoruz.
#
# alpha=0.55:
# MRI'ın da alttan görünmesini sağlar.
# ---------------------------------------------------------

plt.imshow(
    tumor_overlay,
    cmap="viridis",
    alpha=0.55
)


plt.title("FLAIR + Ground Truth Overlay")

plt.axis("off")


# Görseller arasındaki boşlukları düzenle.
plt.tight_layout()


# Görseli ekrana getir.
plt.show()

In [ ]:
# ---------------------------------------------------------
# 43. HÜCRE
#
# AMAÇ:
# BraTS'in çok sınıflı Ground Truth maskesini
# ilk U-Net baseline modelimiz için binary maskeye çevirmek.
#
# ORİJİNAL:
#
# 0 = Background
# 1 = Tumor subregion
# 2 = Tumor subregion
# 4 = Tumor subregion
#
# BINARY:
#
# 0 = Background
# 1 = Tumor
#
# Yani:
# Maskedeki 0'dan büyük HER ŞEY tümör kabul edilecek.
# ---------------------------------------------------------

import numpy as np


# ---------------------------------------------------------
# mask > 0 işlemi:
#
# mask = 0 ise  -> False
# mask = 1 ise  -> True
# mask = 2 ise  -> True
# mask = 4 ise  -> True
#
# Daha sonra astype(np.uint8) ile:
#
# False -> 0
# True  -> 1
#
# yapıyoruz.
# ---------------------------------------------------------

binary_mask = (mask > 0).astype(np.uint8)


# ---------------------------------------------------------
# Kontrol:
# Artık sadece 0 ve 1 bulunmasını bekliyoruz.
# ---------------------------------------------------------

print("Original mask classes:")
print(np.unique(mask))

print("\nBinary mask classes:")
print(np.unique(binary_mask))


# Boyut değişti mi kontrol ediyoruz.
print("\nOriginal mask shape:", mask.shape)
print("Binary mask shape  :", binary_mask.shape)

In [ ]:
# ---------------------------------------------------------
# 44. HÜCRE
#
# AMAÇ:
# Orijinal çok sınıflı maskeyi ve oluşturduğumuz
# binary maskeyi yan yana karşılaştırmak.
#
# Böylece 1, 2 ve 4 etiketlerinin tek bir
# "Tumor = 1" sınıfında birleştiğini görebiliriz.
# ---------------------------------------------------------

import matplotlib.pyplot as plt


slice_number = 77


# Aynı slice'ın maskelerini alıyoruz.
original_mask_slice = mask[:, :, slice_number]

binary_mask_slice = binary_mask[:, :, slice_number]


plt.figure(figsize=(12, 5))


# ---------------------------------------------------------
# SOL:
# Orijinal BraTS maskesi
# ---------------------------------------------------------

plt.subplot(1, 2, 1)

plt.imshow(
    original_mask_slice,
    cmap="viridis"
)

plt.title("Original BraTS Mask [0, 1, 2, 4]")

plt.axis("off")


# ---------------------------------------------------------
# SAĞ:
# Binary mask
# ---------------------------------------------------------

plt.subplot(1, 2, 2)

plt.imshow(
    binary_mask_slice,
    cmap="gray"
)

plt.title("Binary Tumor Mask [0, 1]")

plt.axis("off")


plt.tight_layout()
plt.show()

### Binary Tumor Mask

For the initial segmentation baseline, the original BraTS labels `[0, 1, 2, 4]` were converted into a binary mask.

- **0 → Background**
- **1 → Tumor**

All annotated tumor subregions (`1`, `2`, and `4`) were merged into a single tumor class while preserving their spatial location.

This simplifies the first U-Net task to **tumor vs. background segmentation**. After establishing a reliable baseline, the project can be extended to multi-class tumor subregion segmentation.

In [ ]:
# ---------------------------------------------------------
# 45. HÜCRE
#
# AMAÇ:
# Bu hastanın 155 slice'ının hangilerinde tümör
# bulunduğunu analiz etmek.
#
# binary_mask:
#
# 0 = Background
# 1 = Tumor
#
# Dolayısıyla bir slice içerisindeki 1'leri toplarsak
# o slice'taki toplam tumor pixel sayısını bulabiliriz.
# ---------------------------------------------------------

import numpy as np
import matplotlib.pyplot as plt


# ---------------------------------------------------------
# Her slice için tumor pixel sayısını saklayacağımız liste.
#
# Örneğin:
#
# slice 0  -> 0 tumor pixel
# slice 1  -> 0 tumor pixel
# ...
# slice 70 -> 2500 tumor pixel
# ---------------------------------------------------------

tumor_pixels_per_slice = []


# ---------------------------------------------------------
# binary_mask.shape[2] = 155
#
# Yani bütün axial slice'ları sırayla geziyoruz.
# ---------------------------------------------------------

for slice_index in range(binary_mask.shape[2]):

    # O anki slice'ı al.
    current_slice = binary_mask[:, :, slice_index]

    # Binary mask sadece 0 ve 1 içerdiği için
    # toplamı bize tumor pixel sayısını verir.
    tumor_pixel_count = np.sum(current_slice)

    # Sonucu listeye ekle.
    tumor_pixels_per_slice.append(tumor_pixel_count)


# NumPy array'e çeviriyoruz.
tumor_pixels_per_slice = np.array(
    tumor_pixels_per_slice
)


# ---------------------------------------------------------
# Tümör bulunan slice'ları buluyoruz.
#
# > 0 demek:
# O slice içerisinde en az 1 tumor pixel var.
# ---------------------------------------------------------

tumor_slices = np.where(
    tumor_pixels_per_slice > 0
)[0]


# ---------------------------------------------------------
# Sonuçları yazdırıyoruz.
# ---------------------------------------------------------

print("Toplam slice sayısı:", binary_mask.shape[2])

print(
    "Tümör bulunan slice sayısı:",
    len(tumor_slices)
)

print(
    "Tümör olmayan slice sayısı:",
    binary_mask.shape[2] - len(tumor_slices)
)


print("\nİlk tümör görülen slice:")
print(tumor_slices[0])


print("\nSon tümör görülen slice:")
print(tumor_slices[-1])


# ---------------------------------------------------------
# En fazla tumor pixel bulunan slice'ı buluyoruz.
#
# np.argmax():
# listedeki en büyük değerin indeksini verir.
# ---------------------------------------------------------

max_tumor_slice = np.argmax(
    tumor_pixels_per_slice
)


print("\nEn fazla tümör bulunan slice:")
print(max_tumor_slice)


print("\nBu slice'taki tumor pixel sayısı:")
print(
    tumor_pixels_per_slice[max_tumor_slice]
)

In [ ]:
# ---------------------------------------------------------
# 46. HÜCRE
#
# AMAÇ:
# Tümörün MRI hacmi boyunca nasıl dağıldığını
# grafik üzerinde görmek.
#
# X ekseni = Slice numarası
# Y ekseni = O slice'taki tumor pixel sayısı
# ---------------------------------------------------------

import matplotlib.pyplot as plt


plt.figure(figsize=(12, 5))


# Her slice'ın tumor pixel sayısını çiziyoruz.
plt.plot(
    range(len(tumor_pixels_per_slice)),
    tumor_pixels_per_slice
)


# ---------------------------------------------------------
# En fazla tümör bulunan slice'ın yerini
# ayrıca işaretliyoruz.
# ---------------------------------------------------------

plt.scatter(
    max_tumor_slice,
    tumor_pixels_per_slice[max_tumor_slice]
)


plt.title(
    f"Tumor Distribution Across Axial Slices - {test_patient_id}"
)

plt.xlabel("Axial Slice Index")

plt.ylabel("Number of Tumor Pixels")


# Okumayı kolaylaştırmak için grid.
plt.grid(
    alpha=0.3
)


plt.tight_layout()

plt.show()

### 📊 Tumor Distribution Across MRI Slices

For `BraTS2021_00000`, tumor tissue is present in **47 of 155 axial slices**, spanning slices **49–95**.

The largest tumor cross-section occurs at **slice 74**, containing **1,978 tumor pixels**.

This analysis shows how the 3D tumor volume changes across consecutive 2D slices and helps identify informative slices for segmentation model training.

In [ ]:
# ---------------------------------------------------------
# 47. HÜCRE
#
# AMAÇ:
# En fazla tumor pixel içeren slice'ı otomatik olarak
# bulup MRI + binary mask şeklinde göstermek.
# ---------------------------------------------------------


# En fazla tümör bulunan slice.
best_slice = max_tumor_slice


# FLAIR görüntüsünü al.
flair_best_slice = image[:, :, best_slice, 3]


# Binary tumor maskesini al.
mask_best_slice = binary_mask[:, :, best_slice]


plt.figure(figsize=(15, 5))


# =========================================================
# 1. FLAIR
# =========================================================

plt.subplot(1, 3, 1)

plt.imshow(
    flair_best_slice,
    cmap="gray"
)

plt.title(
    f"FLAIR - Slice {best_slice}"
)

plt.axis("off")


# =========================================================
# 2. BINARY GROUND TRUTH
# =========================================================

plt.subplot(1, 3, 2)

plt.imshow(
    mask_best_slice,
    cmap="gray"
)

plt.title(
    "Binary Ground Truth"
)

plt.axis("off")


# =========================================================
# 3. OVERLAY
# =========================================================

plt.subplot(1, 3, 3)


# Önce MRI.
plt.imshow(
    flair_best_slice,
    cmap="gray"
)


# Background'u gizle.
tumor_only = np.ma.masked_where(
    mask_best_slice == 0,
    mask_best_slice
)


# Tümörü MRI'ın üzerine bindir.
plt.imshow(
    tumor_only,
    alpha=0.5
)


plt.title(
    f"Tumor Overlay - Slice {best_slice}"
)

plt.axis("off")


plt.tight_layout()

plt.show()

### 🎯 Automatic Selection of the Largest Tumor Slice

The preprocessing pipeline automatically identified **slice 74** as the axial slice containing the largest tumor area (**1,978 tumor pixels**) for `BraTS2021_00000`.

The figure compares the normalized **FLAIR MRI**, the corresponding **binary ground-truth mask**, and their **overlay**. The overlay confirms the spatial correspondence between the visible abnormal region and the tumor annotation.

Importantly, this slice was selected **automatically from all 155 slices**, rather than manually, providing a reproducible method for identifying highly informative tumor-containing slices.

In [ ]:
# ---------------------------------------------------------
# SPLIT DOSYASINI TEKRAR YÜKLE
#
# Daha önce oluşturduğumuz Train / Validation / Test
# ayrımını Google Drive'daki JSON dosyasından okuyoruz.
#
# Böylece Colab runtime yeniden başlasa bile
# aynı hasta ayrımını kullanabiliriz.
# ---------------------------------------------------------

import json

split_path = "/content/drive/MyDrive/NeuroRobust/patient_split.json"


# JSON dosyasını aç.
with open(split_path, "r") as file:
    patient_split = json.load(file)


# Önce JSON içindeki anahtarları görelim.
print("JSON içindeki gruplar:")
print(patient_split.keys())

In [ ]:
# ---------------------------------------------------------
# KAYDETTİĞİMİZ TRAIN / VALIDATION / TEST SPLIT'İNİ
# GOOGLE DRIVE'DAN TEKRAR YÜKLÜYORUZ.
# ---------------------------------------------------------

import json

# Daha önce kaydettiğimiz split dosyasının yolu
split_path = "/content/drive/MyDrive/NeuroRobust/patient_split.json"

# JSON dosyasını oku
with open(split_path, "r") as file:
    patient_split = json.load(file)

# Dosyanın içinde hangi isimleri kullandığımızı görelim
print("JSON içindeki anahtarlar:")
print(patient_split.keys())

In [ ]:
# ---------------------------------------------------------
# 48A. HÜCRE
#
# AMAÇ:
# patient_split.json dosyasından okuduğumuz
# Train / Validation / Test hasta listelerini
# tekrar Python değişkenlerine aktarmak.
# ---------------------------------------------------------

# Eğitim hastaları
train_ids = patient_split["train"]

# Validation hastaları
validation_ids = patient_split["validation"]

# Test hastaları
test_ids = patient_split["test"]


# ---------------------------------------------------------
# Hasta sayılarını kontrol ediyoruz.
# Daha önce oluşturduğumuz split ile aynı olmalı:
#
# Train      = 875
# Validation = 188
# Test       = 188
# ---------------------------------------------------------

print("Train hasta sayısı     :", len(train_ids))
print("Validation hasta sayısı:", len(validation_ids))
print("Test hasta sayısı      :", len(test_ids))

In [ ]:
# ============================================================
# 48. HÜCRE
#
# AMAÇ:
# Şimdiye kadar preprocessing işlemlerini ağırlıklı olarak
# BraTS2021_00000 hastasında kontrol ettik.
#
# Tek bir hasta yeterli olmadığı için TRAIN grubundan
# 5 farklı hasta seçip aynı kontrolleri otomatik yapacağız.
#
# Her hasta için:
#
# 1) T1, T1ce, T2, FLAIR ve SEG dosyaları yüklenir.
# 2) MRI görüntüleri load_patient() tarafından normalize edilir.
# 3) Orijinal [0, 1, 2, 4] maskesi binary maskeye çevrilir.
# 4) 155 slice'ın tamamındaki tumor pixel sayısı hesaplanır.
# 5) En fazla tümör içeren slice otomatik bulunur.
# 6) FLAIR, Binary Ground Truth ve Overlay gösterilir.
#
# NOT:
# Burada TEST hastalarını kullanmıyoruz.
# Sadece TRAIN grubundan örnekler inceliyoruz.
# ============================================================

import numpy as np
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# 1. TRAIN grubunun farklı noktalarından 5 hasta seçiyoruz.
#
# Sadece ilk 5 hastayı almak yerine listenin:
#
# başlangıcından
# %25 noktasından
# %50 noktasından
# %75 noktasından
# sonundan
#
# birer hasta alıyoruz.
# ------------------------------------------------------------

sample_indices = [
    0,
    len(train_ids) // 4,
    len(train_ids) // 2,
    3 * len(train_ids) // 4,
    len(train_ids) - 1
]


# Yukarıdaki indekslere karşılık gelen hasta ID'lerini al.
sample_patients = [
    train_ids[index]
    for index in sample_indices
]


# Hangi hastaları kontrol edeceğimizi görelim.
print("Kontrol edilecek TRAIN hastaları:")
print("-" * 45)

for patient_id in sample_patients:
    print(patient_id)


# ------------------------------------------------------------
# 2. Seçilen 5 hastayı sırayla işle.
# ------------------------------------------------------------

for patient_id in sample_patients:

    print("\n" + "=" * 60)
    print("Patient:", patient_id)
    print("=" * 60)


    # --------------------------------------------------------
    # 3. Hastayı yükle.
    #
    # load_patient() fonksiyonumuz:
    #
    # T1
    # T1ce
    # T2
    # FLAIR
    #
    # görüntülerini yükleyip normalize ediyor ve
    # 4 kanallı tek bir image oluşturuyor.
    #
    # patient_image shape:
    # (240, 240, 155, 4)
    #
    # patient_mask shape:
    # (240, 240, 155)
    # --------------------------------------------------------

    patient_image, patient_mask = load_patient(patient_id)


    # --------------------------------------------------------
    # 4. Orijinal BraTS maskesini binary hale getir.
    #
    # Orijinal:
    #
    # 0 = Background
    # 1 = Tumor subregion
    # 2 = Tumor subregion
    # 4 = Tumor subregion
    #
    # Binary:
    #
    # 0 = Background
    # 1 = Tumor
    # --------------------------------------------------------

    patient_binary_mask = (
        patient_mask > 0
    ).astype(np.uint8)


    # --------------------------------------------------------
    # 5. Her slice'taki tumor pixel sayısını hesapla.
    #
    # patient_binary_mask:
    #
    # (240, 240, 155)
    #
    # axis=(0, 1) ile ilk iki boyutu topluyoruz.
    #
    # Böylece sonuç:
    #
    # (155,)
    #
    # boyutunda oluyor.
    #
    # Yani her slice için bir tumor pixel sayımız oluyor.
    # --------------------------------------------------------

    tumor_counts = np.sum(
        patient_binary_mask,
        axis=(0, 1)
    )


    # --------------------------------------------------------
    # 6. Kaç slice'ta tümör olduğunu bul.
    #
    # tumor_counts > 0:
    # En az 1 tumor pixel bulunan slice'ları seçer.
    # --------------------------------------------------------

    number_of_tumor_slices = np.sum(
        tumor_counts > 0
    )


    # --------------------------------------------------------
    # 7. En fazla tümör bulunan slice'ı bul.
    #
    # np.argmax() en yüksek değerin indeksini verir.
    # --------------------------------------------------------

    best_slice = np.argmax(
        tumor_counts
    )


    # O slice'taki toplam tumor pixel sayısı.
    largest_tumor_pixel_count = tumor_counts[
        best_slice
    ]


    # --------------------------------------------------------
    # Sonuçları yazdır.
    # --------------------------------------------------------

    print(
        "Tumor-containing slices:",
        number_of_tumor_slices
    )

    print(
        "Largest tumor slice:",
        best_slice
    )

    print(
        "Tumor pixels on largest slice:",
        largest_tumor_pixel_count
    )


    # --------------------------------------------------------
    # 8. En fazla tümör bulunan slice'ın FLAIR görüntüsünü al.
    #
    # Kanal sıramız:
    #
    # 0 = T1
    # 1 = T1ce
    # 2 = T2
    # 3 = FLAIR
    #
    # Bu yüzden son indeks = 3.
    # --------------------------------------------------------

    flair_slice = patient_image[
        :, :, best_slice, 3
    ]


    # Aynı slice'ın binary ground truth maskesi.
    binary_slice = patient_binary_mask[
        :, :, best_slice
    ]


    # --------------------------------------------------------
    # 9. Overlay oluşturmak için background'u gizle.
    #
    # binary_slice == 0 olan yerler görünmez olacak.
    #
    # Böylece yalnızca tümör bölgesi MRI'ın üzerine
    # bindirilecek.
    # --------------------------------------------------------

    tumor_overlay = np.ma.masked_where(
        binary_slice == 0,
        binary_slice
    )


    # ========================================================
    # 10. GÖRSELLEŞTİRME
    #
    # Her hasta için üç görüntü:
    #
    # FLAIR | Binary Ground Truth | Overlay
    # ========================================================

    plt.figure(figsize=(15, 4))


    # --------------------------------------------------------
    # 1. PANEL — FLAIR MRI
    # --------------------------------------------------------

    plt.subplot(1, 3, 1)

    plt.imshow(
        flair_slice,
        cmap="gray"
    )

    plt.title(
        f"{patient_id}\nFLAIR - Slice {best_slice}"
    )

    plt.axis("off")


    # --------------------------------------------------------
    # 2. PANEL — BINARY GROUND TRUTH
    # --------------------------------------------------------

    plt.subplot(1, 3, 2)

    plt.imshow(
        binary_slice,
        cmap="gray"
    )

    plt.title(
        "Binary Ground Truth"
    )

    plt.axis("off")


    # --------------------------------------------------------
    # 3. PANEL — MRI + TUMOR OVERLAY
    # --------------------------------------------------------

    plt.subplot(1, 3, 3)


    # Önce FLAIR görüntüsünü çiz.
    plt.imshow(
        flair_slice,
        cmap="gray"
    )


    # Daha sonra tümör maskesini yarı şeffaf şekilde
    # MRI görüntüsünün üzerine ekle.
    plt.imshow(
        tumor_overlay,
        alpha=0.5
    )


    plt.title(
        "Tumor Overlay"
    )

    plt.axis("off")


    # Grafiklerin birbirine girmesini engelle.
    plt.tight_layout()

    # Görseli göster.
    plt.show()


    # --------------------------------------------------------
    # 11. BELLEK TEMİZLİĞİ
    #
    # MRI verileri büyük olduğu için bir hastayı
    # görüntüledikten sonra büyük değişkenleri siliyoruz.
    #
    # Böylece 5 hastanın tüm 3D görüntülerini aynı anda
    # RAM'de tutmamış oluyoruz.
    # --------------------------------------------------------

    del patient_image
    del patient_mask
    del patient_binary_mask
    del tumor_counts

## 🔍 Multi-Patient Visual Sanity Check

To verify that the preprocessing pipeline is not limited to a single subject, five patients were sampled from different positions of the training set.

For each patient, the pipeline automatically identified the axial slice containing the largest tumor area and visualized:

- **Normalized FLAIR MRI**
- **Binary ground-truth tumor mask**
- **FLAIR + tumor mask overlay**

| Patient | Tumor-Containing Slices | Largest Tumor Slice | Tumor Pixels |
|---|---:|---:|---:|
| BraTS2021_00100 | 68 | 90 | 2,348 |
| BraTS2021_01087 | 102 | 116 | 2,113 |
| BraTS2021_00491 | 65 | 92 | 2,282 |
| BraTS2021_01293 | 64 | 86 | 3,476 |
| BraTS2021_01542 | 75 | 83 | 3,362 |

### Observation

The examples show substantial variation in **tumor size, shape, location, and slice distribution** across patients. Despite this variability, the binary masks remain spatially aligned with the corresponding FLAIR abnormalities in the inspected slices.

This multi-subject sanity check supports the consistency of the preprocessing and mask-alignment pipeline before dataset-wide validation and model training.

In [ ]:
# ============================================================
# 49. HÜCRE — DATASET-WIDE INTEGRITY CHECK
#
# AMAÇ:
# 875 TRAIN hastasının tamamını otomatik olarak kontrol etmek.
#
# Her hasta için:
#
# 1) Gerekli 5 dosya mevcut mu?
#       - T1
#       - T1ce
#       - T2
#       - FLAIR
#       - SEG
#
# 2) Dosyalar NIfTI olarak açılabiliyor mu?
#
# 3) 5 dosyanın spatial shape'leri aynı mı?
#
# 4) Beklenen shape (240, 240, 155) mi?
#
# 5) Segmentation maskesinde yalnızca
#    beklenen BraTS label'ları [0, 1, 2, 4] var mı?
#
# 6) Maskede gerçekten tumor pixel var mı?
#
# Böylece model eğitiminden önce veri setindeki olası
# problemlerin büyük kısmını otomatik olarak yakalayabiliriz.
# ============================================================

import os
import nibabel as nib
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Dataset klasörü
# ------------------------------------------------------------

training_data_dir = (
    "/content/drive/MyDrive/NeuroRobust/"
    "dataset/BraTS2021_Training_Data"
)


# ------------------------------------------------------------
# BraTS datasetinde beklediğimiz spatial shape
# ------------------------------------------------------------

expected_shape = (240, 240, 155)


# ------------------------------------------------------------
# BraTS segmentation maskesinde beklenen sınıflar
#
# 0 = Background
# 1, 2, 4 = Tumor subregions
# ------------------------------------------------------------

expected_labels = {0, 1, 2, 4}


# ------------------------------------------------------------
# Kontrol sonuçlarını burada saklayacağız.
# ------------------------------------------------------------

validation_results = []


# Problemli hastaların ID'lerini ayrıca saklayacağız.
problem_patients = []


print("TRAIN dataset integrity check başlıyor...")
print("Kontrol edilecek hasta sayısı:", len(train_ids))
print("-" * 60)


# ============================================================
# BÜTÜN TRAIN HASTALARINI GEZ
# ============================================================

for patient_number, patient_id in enumerate(train_ids, start=1):

    # --------------------------------------------------------
    # Hastanın klasörü
    # --------------------------------------------------------

    patient_folder = os.path.join(
        training_data_dir,
        patient_id
    )


    # --------------------------------------------------------
    # Beklenen 5 dosyanın yollarını oluştur.
    # --------------------------------------------------------

    file_paths = {

        "t1": os.path.join(
            patient_folder,
            f"{patient_id}_t1.nii.gz"
        ),

        "t1ce": os.path.join(
            patient_folder,
            f"{patient_id}_t1ce.nii.gz"
        ),

        "t2": os.path.join(
            patient_folder,
            f"{patient_id}_t2.nii.gz"
        ),

        "flair": os.path.join(
            patient_folder,
            f"{patient_id}_flair.nii.gz"
        ),

        "seg": os.path.join(
            patient_folder,
            f"{patient_id}_seg.nii.gz"
        )
    }


    # --------------------------------------------------------
    # Bu hasta için başlangıçta her şey doğru kabul ediliyor.
    # Bir problem bulursak False yapacağız.
    # --------------------------------------------------------

    files_ok = True
    readable = True
    shapes_match = True
    expected_shape_ok = True
    labels_ok = True
    tumor_present = True

    problem_description = []


    # ========================================================
    # 1. DOSYALAR VAR MI?
    # ========================================================

    missing_files = []

    for modality, path in file_paths.items():

        if not os.path.exists(path):

            missing_files.append(modality)


    if len(missing_files) > 0:

        files_ok = False

        problem_description.append(
            "Missing: " + ", ".join(missing_files)
        )


    # ========================================================
    # Dosyalar eksikse aşağıdaki kontroller yapılamaz.
    # ========================================================

    if files_ok:

        try:

            # ------------------------------------------------
            # 2. NIfTI dosyalarını aç.
            #
            # nib.load() burada bütün MRI verisini RAM'e
            # yüklemez. Öncelikle dosya header/proxy bilgisine
            # erişmemizi sağlar.
            # ------------------------------------------------

            nii_files = {
                modality: nib.load(path)
                for modality, path in file_paths.items()
            }


            # ------------------------------------------------
            # Her dosyanın shape bilgisini al.
            # ------------------------------------------------

            shapes = {
                modality: nii.shape
                for modality, nii in nii_files.items()
            }


            # =================================================
            # 3. TÜM MODALITY VE MASK SHAPE'LERİ AYNI MI?
            # =================================================

            unique_shapes = set(
                shapes.values()
            )


            if len(unique_shapes) != 1:

                shapes_match = False

                problem_description.append(
                    "Shape mismatch"
                )


            # =================================================
            # 4. BEKLENEN SHAPE Mİ?
            # =================================================

            for modality, shape in shapes.items():

                if shape != expected_shape:

                    expected_shape_ok = False

                    problem_description.append(
                        f"{modality} shape={shape}"
                    )


            # =================================================
            # 5. SEGMENTATION MASKESİNİ OKU
            #
            # Burada yalnızca SEG dosyasının voxel değerlerine
            # ihtiyacımız var.
            # =================================================

            seg_data = np.asarray(
                nii_files["seg"].dataobj
            )


            # Maskede bulunan unique sınıfları bul.
            unique_labels = set(
                np.unique(seg_data).astype(int)
            )


            # ------------------------------------------------
            # Beklenmeyen bir label var mı?
            #
            # Örneğin 3 gibi bir değer çıkarsa bunu
            # problem olarak raporlayacağız.
            # ------------------------------------------------

            if not unique_labels.issubset(expected_labels):

                labels_ok = False

                problem_description.append(
                    f"Unexpected labels={sorted(unique_labels)}"
                )


            # =================================================
            # 6. HASTADA TÜMÖR VAR MI?
            #
            # Maskede 0 dışında en az bir voxel bulunmalı.
            # =================================================

            if not np.any(seg_data > 0):

                tumor_present = False

                problem_description.append(
                    "Empty tumor mask"
                )


            # Belleği boşalt.
            del seg_data
            del nii_files


        except Exception as error:

            readable = False

            problem_description.append(
                f"Read error: {str(error)}"
            )


    # ========================================================
    # HASTANIN GENEL DURUMU
    # ========================================================

    patient_ok = (
        files_ok
        and readable
        and shapes_match
        and expected_shape_ok
        and labels_ok
        and tumor_present
    )


    # Problem varsa listeye ekle.
    if not patient_ok:

        problem_patients.append(
            patient_id
        )


    # --------------------------------------------------------
    # Sonucu tablo için kaydet.
    # --------------------------------------------------------

    validation_results.append({

        "Patient": patient_id,

        "Files_OK": files_ok,

        "Readable": readable,

        "Shapes_Match": shapes_match,

        "Expected_Shape": expected_shape_ok,

        "Labels_OK": labels_ok,

        "Tumor_Present": tumor_present,

        "Overall_OK": patient_ok,

        "Problem": (
            "None"
            if patient_ok
            else " | ".join(problem_description)
        )
    })


    # --------------------------------------------------------
    # Her 100 hastada bir ilerlemeyi göster.
    # Böylece kodun çalıştığını görebiliriz.
    # --------------------------------------------------------

    if (
        patient_number % 100 == 0
        or patient_number == len(train_ids)
    ):

        print(
            f"Kontrol edildi: "
            f"{patient_number}/{len(train_ids)}"
        )


# ============================================================
# SONUÇLARI DATAFRAME'E ÇEVİR
# ============================================================

validation_df = pd.DataFrame(
    validation_results
)


# ============================================================
# ÖZET
# ============================================================

successful_patients = int(
    validation_df["Overall_OK"].sum()
)

failed_patients = len(
    validation_df
) - successful_patients


print("\n" + "=" * 60)
print("TRAIN DATASET VALIDATION SUMMARY")
print("=" * 60)

print(
    "Kontrol edilen hasta :",
    len(validation_df)
)

print(
    "Başarılı hasta       :",
    successful_patients
)

print(
    "Problemli hasta      :",
    failed_patients
)


# ============================================================
# PROBLEM VARSA GÖSTER
# ============================================================

if failed_patients == 0:

    print(
        "\nTüm TRAIN hastaları temel integrity "
        "kontrollerini geçti! ✅"
    )

else:

    print(
        "\nProblem bulunan hastalar:"
    )

    display(
        validation_df[
            validation_df["Overall_OK"] == False
        ]
    )

## ✅ Training Dataset Integrity Validation

Before model development, an automated integrity check was performed across all **875 training subjects**.

The validation verified:

- Presence and readability of all required MRI modalities and segmentation files
- Consistent spatial dimensions across T1, T1ce, T2, FLAIR, and segmentation masks
- Expected volume shape of **240 × 240 × 155**
- Valid BraTS segmentation labels (`0, 1, 2, 4`)
- Presence of annotated tumor voxels

### Results

| Validation Result | Patients |
|---|---:|
| Total checked | 875 |
| Passed | **875** |
| Failed | **0** |

**All 875 training subjects passed the implemented integrity checks.**

Together with the multi-patient visual sanity check, this provides a validated foundation for constructing the segmentation training pipeline.

In [ ]:
# ============================================================
# 50. HÜCRE — TRAIN SLICE DISTRIBUTION ANALYSIS
#
# AMAÇ:
# 875 TRAIN hastasının tamamında:
#
# - Toplam kaç axial slice var?
# - Kaç slice'ta tümör var?
# - Kaç slice'ta tümör yok?
# - Hasta başına tümörlü slice sayısı nasıl değişiyor?
# - Tümörlü slice'lardaki tumor alanları nasıl dağılıyor?
#
# Bunları ölçüyoruz.
#
# NEDEN?
# Model eğitiminde tumor / background dengesini
# tahmin ederek değil, gerçek dataset istatistiklerine
# bakarak belirlemek istiyoruz.
# ============================================================

import os
import nibabel as nib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# Dataset klasörü
# ------------------------------------------------------------

training_data_dir = (
    "/content/drive/MyDrive/NeuroRobust/"
    "dataset/BraTS2021_Training_Data"
)


# ------------------------------------------------------------
# Her hastanın istatistiklerini burada saklayacağız.
# ------------------------------------------------------------

patient_slice_stats = []


# ------------------------------------------------------------
# Dataset genelindeki tumor-containing slice'ların
# tumor pixel sayılarını burada biriktireceğiz.
#
# Böylece yalnızca:
#
# "Tumor var / yok"
#
# değil,
#
# "Tumor ne kadar büyük?"
#
# sorusunu da inceleyebileceğiz.
# ------------------------------------------------------------

all_tumor_slice_pixel_counts = []


print("TRAIN slice distribution analizi başlıyor...")
print("Hasta sayısı:", len(train_ids))
print("-" * 60)


# ============================================================
# 875 TRAIN HASTASINI GEZ
# ============================================================

for patient_number, patient_id in enumerate(
    train_ids,
    start=1
):

    # --------------------------------------------------------
    # Segmentation dosyasının yolu
    # --------------------------------------------------------

    seg_path = os.path.join(
        training_data_dir,
        patient_id,
        f"{patient_id}_seg.nii.gz"
    )


    # --------------------------------------------------------
    # Ground-truth segmentation maskesini yükle.
    # --------------------------------------------------------

    seg_nii = nib.load(seg_path)

    seg_data = np.asarray(
        seg_nii.dataobj
    )


    # --------------------------------------------------------
    # MULTICLASS -> BINARY
    #
    # 0       -> Background
    # 1,2,4   -> Tumor
    # --------------------------------------------------------

    binary_seg = (
        seg_data > 0
    ).astype(np.uint8)


    # --------------------------------------------------------
    # Her axial slice'taki tumor pixel sayısını hesapla.
    #
    # binary_seg shape:
    #
    # (240, 240, 155)
    #
    # Sonuç:
    #
    # (155,)
    #
    # Her eleman bir slice'taki tumor pixel sayısıdır.
    # --------------------------------------------------------

    tumor_pixels_per_slice = np.sum(
        binary_seg,
        axis=(0, 1)
    )


    # --------------------------------------------------------
    # Toplam slice sayısı
    # --------------------------------------------------------

    total_slices = binary_seg.shape[2]


    # --------------------------------------------------------
    # En az 1 tumor pixel içeren slice'lar
    # --------------------------------------------------------

    tumor_slice_mask = (
        tumor_pixels_per_slice > 0
    )


    tumor_slices = int(
        np.sum(tumor_slice_mask)
    )


    # --------------------------------------------------------
    # Tumor bulunmayan slice sayısı
    # --------------------------------------------------------

    background_slices = (
        total_slices - tumor_slices
    )


    # --------------------------------------------------------
    # O hastadaki en büyük 2D tumor alanı
    # --------------------------------------------------------

    max_tumor_pixels = int(
        np.max(tumor_pixels_per_slice)
    )


    # --------------------------------------------------------
    # Tumor-containing slice'lardaki pixel sayılarını
    # dataset-geneli listesine ekle.
    # --------------------------------------------------------

    tumor_pixel_values = (
        tumor_pixels_per_slice[
            tumor_slice_mask
        ]
    )

    all_tumor_slice_pixel_counts.extend(
        tumor_pixel_values.tolist()
    )


    # --------------------------------------------------------
    # Hasta istatistiklerini kaydet.
    # --------------------------------------------------------

    patient_slice_stats.append({

        "Patient": patient_id,

        "Total_Slices": total_slices,

        "Tumor_Slices": tumor_slices,

        "Background_Slices": background_slices,

        "Tumor_Slice_Ratio": (
            tumor_slices / total_slices
        ),

        "Max_Tumor_Pixels": max_tumor_pixels
    })


    # --------------------------------------------------------
    # Her 100 hastada ilerlemeyi göster.
    # --------------------------------------------------------

    if (
        patient_number % 100 == 0
        or patient_number == len(train_ids)
    ):

        print(
            f"Analiz edildi: "
            f"{patient_number}/{len(train_ids)}"
        )


    # --------------------------------------------------------
    # Büyük değişkenleri temizle.
    # --------------------------------------------------------

    del seg_data
    del binary_seg
    del tumor_pixels_per_slice


# ============================================================
# DATAFRAME
# ============================================================

slice_stats_df = pd.DataFrame(
    patient_slice_stats
)


# ============================================================
# DATASET GENELİ İSTATİSTİKLER
# ============================================================

total_train_slices = int(
    slice_stats_df["Total_Slices"].sum()
)

total_tumor_slices = int(
    slice_stats_df["Tumor_Slices"].sum()
)

total_background_slices = int(
    slice_stats_df["Background_Slices"].sum()
)


tumor_percentage = (
    total_tumor_slices
    / total_train_slices
    * 100
)


background_percentage = (
    total_background_slices
    / total_train_slices
    * 100
)


# ============================================================
# SONUÇLARI YAZDIR
# ============================================================

print("\n" + "=" * 60)
print("TRAIN SLICE DISTRIBUTION SUMMARY")
print("=" * 60)


print(
    "Toplam TRAIN hastası       :",
    len(slice_stats_df)
)

print(
    "Toplam axial slice         :",
    total_train_slices
)

print(
    "Tumor-containing slice     :",
    total_tumor_slices
)

print(
    "Background-only slice      :",
    total_background_slices
)


print(
    "\nTumor slice oranı          :",
    f"{tumor_percentage:.2f}%"
)

print(
    "Background slice oranı     :",
    f"{background_percentage:.2f}%"
)


print(
    "\nHasta başına ortalama tumor slice:",
    f"{slice_stats_df['Tumor_Slices'].mean():.2f}"
)

print(
    "Minimum tumor slice:",
    int(slice_stats_df["Tumor_Slices"].min())
)

print(
    "Maximum tumor slice:",
    int(slice_stats_df["Tumor_Slices"].max())
)


# ============================================================
# TUMOR ALANI İSTATİSTİKLERİ
# ============================================================

tumor_pixel_array = np.array(
    all_tumor_slice_pixel_counts
)


print("\n" + "-" * 60)
print("TUMOR AREA STATISTICS")
print("-" * 60)


print(
    "Median tumor pixels / tumor slice:",
    f"{np.median(tumor_pixel_array):.0f}"
)

print(
    "25th percentile:",
    f"{np.percentile(tumor_pixel_array, 25):.0f}"
)

print(
    "75th percentile:",
    f"{np.percentile(tumor_pixel_array, 75):.0f}"
)

print(
    "Maximum:",
    int(np.max(tumor_pixel_array))
)

### Training Slice Distribution Analysis

The training cohort contains **135,625 axial slices** from 875 subjects.

- **57,067 (42.08%)** slices contain tumor tissue.
- **78,558 (57.92%)** slices contain background only.
- Patients contain an average of **65.22 tumor-positive slices**, ranging from **11 to 109**.

Tumor size also varies substantially across positive slices. The median tumor area is **1,256 pixels**, with an interquartile range of **485–2,236 pixels** and a maximum of **6,444 pixels**.

These results show that the dataset has a moderate tumor/background slice imbalance rather than an extreme one. Therefore, the training strategy should preserve sufficient background examples while ensuring that tumor-positive slices, particularly those containing smaller tumor regions, remain well represented.

In [ ]:
# ============================================================
# 51. HÜCRE — TUMOR SLICE DISTRIBUTION VISUALIZATION
#
# AMAÇ:
# Her hastada kaç tane tümör içeren MRI slice'ı olduğunu
# grafik üzerinde görmek.
#
# 50. hücrede sayısal olarak bulduğumuz dağılımı
# şimdi görselleştiriyoruz.
# ============================================================

import matplotlib.pyplot as plt


# Grafik alanını oluşturuyoruz.
plt.figure(figsize=(10, 5))


# ------------------------------------------------------------
# Histogram:
#
# X ekseni -> Bir hastada kaç tane tümörlü slice var?
# Y ekseni -> Bu sayıya sahip kaç hasta var?
#
# Örneğin:
# X = 60 civarında sütun yüksekse,
# yaklaşık 60 tümörlü slice'a sahip çok sayıda hasta
# olduğunu gösterir.
# ------------------------------------------------------------

plt.hist(
    slice_stats_df["Tumor_Slices"],
    bins=30,
    edgecolor="black"
)


# X ekseninin açıklaması
plt.xlabel(
    "Number of Tumor-Containing Slices per Patient"
)


# Y ekseninin açıklaması
plt.ylabel(
    "Number of Patients"
)


# Grafiğin başlığı
plt.title(
    "Distribution of Tumor-Containing Slices "
    "Across Training Patients"
)


# Ortalama değeri dikey kesikli çizgiyle gösteriyoruz.
mean_tumor_slices = slice_stats_df["Tumor_Slices"].mean()

plt.axvline(
    mean_tumor_slices,
    linestyle="--",
    linewidth=2,
    label=f"Mean = {mean_tumor_slices:.2f}"
)


# Ortalama çizgisinin açıklamasını göster.
plt.legend()


# Grafiğin okunmasını kolaylaştır.
plt.grid(
    alpha=0.3
)


# Yerleşimi düzenle.
plt.tight_layout()


# Grafiği göster.
plt.show()

### Tumor Slice Distribution Across Patients

The number of tumor-containing slices varies substantially across the training cohort.

Most patients contain tumor annotations across approximately **50–85 axial slices**, with an average of **65.22 tumor-positive slices per patient**. However, the distribution also includes patients with much smaller and much larger tumor extents.

This patient-level variability highlights the heterogeneity of the BraTS cohort and will be considered when constructing the training samples.

In [ ]:
# ============================================================
# 52. HÜCRE — TRAINING SLICE INDEX CREATION
#
# AMAÇ:
# Model eğitiminde kullanılabilecek axial slice'ların
# listesini oluşturmak.
#
# Her satır bize şunu söyleyecek:
#
#   Hangi hasta?
#   Hangi slice?
#   O slice'ta tümör var mı?
#   Varsa kaç tumor pixel var?
#
# ÖNEMLİ:
# Bu hücre MRI görüntülerini tekrar kaydetmez.
# Sadece eğitim için bir "harita / index" oluşturur.
#
# Böylece:
# - Disk alanını gereksiz doldurmayız.
# - Train/validation/test ayrımını koruruz.
# - Daha sonra kontrollü sampling yapabiliriz.
# ============================================================

import os
import numpy as np
import pandas as pd
import nibabel as nib


# ------------------------------------------------------------
# TRAIN dataset klasörü
# ------------------------------------------------------------

training_data_dir = (
    "/content/drive/MyDrive/NeuroRobust/"
    "dataset/BraTS2021_Training_Data"
)


# ------------------------------------------------------------
# Her kullanılabilir slice hakkında bilgileri
# bu listede toplayacağız.
# ------------------------------------------------------------

train_slice_records = []


print("TRAIN slice index oluşturuluyor...")
print("Train hasta sayısı:", len(train_ids))
print("-" * 60)


# ============================================================
# TRAIN HASTALARINI GEZ
# ============================================================

for patient_number, patient_id in enumerate(
    train_ids,
    start=1
):

    # --------------------------------------------------------
    # Hastanın segmentation dosyasının yolu
    # --------------------------------------------------------

    seg_path = os.path.join(
        training_data_dir,
        patient_id,
        f"{patient_id}_seg.nii.gz"
    )


    # --------------------------------------------------------
    # Segmentation maskesini yükle.
    #
    # Burada yalnızca SEG dosyasını okuyoruz.
    # T1/T1ce/T2/FLAIR henüz yüklenmiyor.
    # --------------------------------------------------------

    seg_nii = nib.load(seg_path)

    seg_data = np.asarray(
        seg_nii.dataobj
    )


    # --------------------------------------------------------
    # BraTS multiclass mask:
    #
    # 0 = background
    # 1,2,4 = tumor bölgeleri
    #
    # Biz şimdilik binary segmentation yapacağımız için:
    #
    # 0     -> background
    # > 0   -> tumor
    # --------------------------------------------------------

    binary_seg = (
        seg_data > 0
    )


    # --------------------------------------------------------
    # Her axial slice'taki tumor pixel sayısını hesapla.
    #
    # Sonuç 155 elemanlı bir array olacak.
    # --------------------------------------------------------

    tumor_pixels_per_slice = np.sum(
        binary_seg,
        axis=(0, 1)
    )


    # --------------------------------------------------------
    # Hastanın 155 axial slice'ını tek tek gez.
    # --------------------------------------------------------

    for slice_index in range(
        binary_seg.shape[2]
    ):

        tumor_pixels = int(
            tumor_pixels_per_slice[slice_index]
        )


        # ----------------------------------------------------
        # Slice'ta en az 1 tumor pixel varsa:
        #
        # has_tumor = 1
        #
        # Yoksa:
        #
        # has_tumor = 0
        # ----------------------------------------------------

        has_tumor = int(
            tumor_pixels > 0
        )


        # ----------------------------------------------------
        # Bu slice'ın bilgilerini kaydet.
        # ----------------------------------------------------

        train_slice_records.append({

            "patient_id": patient_id,

            "slice_index": slice_index,

            "has_tumor": has_tumor,

            "tumor_pixels": tumor_pixels
        })


    # --------------------------------------------------------
    # Her 100 hastada ilerleme göster.
    # --------------------------------------------------------

    if (
        patient_number % 100 == 0
        or patient_number == len(train_ids)
    ):

        print(
            f"İşlendi: "
            f"{patient_number}/{len(train_ids)}"
        )


    # Büyük değişkenleri temizle.
    del seg_data
    del binary_seg
    del tumor_pixels_per_slice


# ============================================================
# LIST -> DATAFRAME
# ============================================================

train_slice_df = pd.DataFrame(
    train_slice_records
)


# ============================================================
# KONTROL
# ============================================================

print("\n" + "=" * 60)
print("TRAIN SLICE INDEX SUMMARY")
print("=" * 60)

print(
    "Toplam indexlenen slice :",
    len(train_slice_df)
)

print(
    "Tumor slice             :",
    int(train_slice_df["has_tumor"].sum())
)

print(
    "Background slice        :",
    int(
        (train_slice_df["has_tumor"] == 0).sum()
    )
)


# ------------------------------------------------------------
# İlk birkaç satırı göster.
# ------------------------------------------------------------

print("\nİlk 10 kayıt:")

display(
    train_slice_df.head(10)
)

In [ ]:
# ============================================================
# 53. HÜCRE — SAVE TRAIN SLICE INDEX
#
# AMAÇ:
# 52. hücrede oluşturduğumuz 135,625 satırlık
# train slice kataloğunu CSV olarak Google Drive'a kaydetmek.
#
# Böylece Colab kapanırsa bu analizi tekrar yapmak
# zorunda kalmayacağız.
# ============================================================

import os


# Kaydedilecek dosyanın yolu
train_index_path = os.path.join(
    project_dir,
    "train_slice_index.csv"
)


# DataFrame'i CSV dosyasına kaydet.
# index=False:
# Pandas'ın ekstra 0,1,2,3... sütunu oluşturmasını engeller.
train_slice_df.to_csv(
    train_index_path,
    index=False
)


# ------------------------------------------------------------
# Kaydın başarılı olduğunu kontrol et.
# ------------------------------------------------------------

print("TRAIN slice index kaydedildi! ✅")
print("Dosya yolu:")
print(train_index_path)

print(
    "\nKaydedilen kayıt sayısı:",
    len(train_slice_df)
)


# Dosya gerçekten oluşmuş mu?
print(
    "Dosya mevcut mu?:",
    os.path.exists(train_index_path)
)

In [ ]:
# ============================================================
# 54. HÜCRE — VALIDATION SLICE INDEX
#
# AMAÇ:
# 188 validation hastasının bütün axial slice'ları için
# bir katalog oluşturmak.
#
# Her kayıt:
#
# patient_id
# slice_index
# has_tumor
# tumor_pixels
#
# bilgisini içerecek.
#
# Bu kez işlemi bir FONKSİYON haline getiriyoruz.
# Böylece aynı kodu ileride TEST seti için de
# tekrar kullanabiliriz.
# ============================================================

import os
import numpy as np
import pandas as pd
import nibabel as nib


# ============================================================
# SLICE INDEX OLUŞTURAN FONKSİYON
# ============================================================

def create_slice_index(patient_ids, dataset_dir, split_name):

    # Bütün slice kayıtlarını burada tutacağız.
    records = []

    print(f"{split_name} slice index oluşturuluyor...")
    print("Hasta sayısı:", len(patient_ids))
    print("-" * 60)


    # --------------------------------------------------------
    # Hastaları tek tek dolaş.
    # --------------------------------------------------------

    for patient_number, patient_id in enumerate(
        patient_ids,
        start=1
    ):

        # Hastanın segmentation dosyasının yolu.
        seg_path = os.path.join(
            dataset_dir,
            patient_id,
            f"{patient_id}_seg.nii.gz"
        )


        # ----------------------------------------------------
        # Segmentation maskesini yükle.
        # ----------------------------------------------------

        seg_nii = nib.load(seg_path)

        seg_data = np.asarray(
            seg_nii.dataobj
        )


        # ----------------------------------------------------
        # Binary tumor mask oluştur.
        #
        # 0       -> background
        # 1,2,4   -> tumor
        #
        # > 0 dediğimiz için bütün tumor sınıfları
        # tek bir tumor sınıfına dönüşüyor.
        # ----------------------------------------------------

        binary_seg = (
            seg_data > 0
        )


        # ----------------------------------------------------
        # Her axial slice'taki tumor pixel sayısını hesapla.
        # ----------------------------------------------------

        tumor_pixels_per_slice = np.sum(
            binary_seg,
            axis=(0, 1)
        )


        # ----------------------------------------------------
        # Hastanın bütün axial slice'larını dolaş.
        # ----------------------------------------------------

        for slice_index in range(
            binary_seg.shape[2]
        ):

            # Bu slice'taki tumor pixel sayısı.
            tumor_pixels = int(
                tumor_pixels_per_slice[slice_index]
            )


            # Tumor varsa 1, yoksa 0.
            has_tumor = int(
                tumor_pixels > 0
            )


            # Slice bilgisini kaydet.
            records.append({
                "patient_id": patient_id,
                "slice_index": slice_index,
                "has_tumor": has_tumor,
                "tumor_pixels": tumor_pixels
            })


        # ----------------------------------------------------
        # İlerlemeyi göster.
        # ----------------------------------------------------

        if (
            patient_number % 50 == 0
            or patient_number == len(patient_ids)
        ):

            print(
                f"İşlendi: "
                f"{patient_number}/{len(patient_ids)}"
            )


        # Büyük değişkenleri bellekten kaldır.
        del seg_data
        del binary_seg
        del tumor_pixels_per_slice


    # --------------------------------------------------------
    # Listeyi DataFrame'e dönüştür.
    # --------------------------------------------------------

    df = pd.DataFrame(records)

    return df


# ============================================================
# VALIDATION SETİ İÇİN ÇALIŞTIR
# ============================================================

val_slice_df = create_slice_index(
    validation_ids,
    training_data_dir,
    "VALIDATION"
)


# ============================================================
# SONUÇLARI GÖSTER
# ============================================================

print("\n" + "=" * 60)
print("VALIDATION SLICE INDEX SUMMARY")
print("=" * 60)


print(
    "Toplam validation hastası :",
    len(validation_ids)
)

print(
    "Toplam slice              :",
    len(val_slice_df)
)

print(
    "Tumor slice               :",
    int(val_slice_df["has_tumor"].sum())
)

print(
    "Background slice          :",
    int(
        (val_slice_df["has_tumor"] == 0).sum()
    )
)


# Tumor yüzdesini hesapla.
val_tumor_percentage = (
    val_slice_df["has_tumor"].mean()
    * 100
)


print(
    "Tumor slice oranı         :",
    f"{val_tumor_percentage:.2f}%"
)


print("\nİlk 10 kayıt:")

display(
    val_slice_df.head(10)
)

### Validation Set Distribution

The validation cohort contains **29,140 axial slices from 188 patients**. Of these, **12,445 (42.71%)** contain tumor tissue and **16,695 (57.29%)** contain background only.

The tumor-positive slice proportion is very similar between the training (**42.08%**) and validation (**42.71%**) cohorts, indicating a comparable slice-level class distribution across the patient-level split.

In [ ]:
# ============================================================
# 55. HÜCRE — SAVE VALIDATION SLICE INDEX
#
# AMAÇ:
# 54. hücrede oluşturduğumuz validation slice kataloğunu
# Google Drive'a kaydetmek.
#
# Böylece Colab kapansa bile tekrar oluşturmak zorunda
# kalmayacağız.
# ============================================================

import os


# ------------------------------------------------------------
# Kaydedilecek dosyanın yolu
# ------------------------------------------------------------

val_index_path = os.path.join(
    project_dir,
    "validation_slice_index.csv"
)


# ------------------------------------------------------------
# Validation DataFrame'ini CSV olarak kaydet.
# ------------------------------------------------------------

val_slice_df.to_csv(
    val_index_path,
    index=False
)


# ------------------------------------------------------------
# Kaydı kontrol et.
# ------------------------------------------------------------

print("VALIDATION slice index kaydedildi! ✅")

print("\nDosya yolu:")
print(val_index_path)

print(
    "\nKaydedilen kayıt sayısı:",
    len(val_slice_df)
)

print(
    "Dosya mevcut mu?:",
    os.path.exists(val_index_path)
)

In [ ]:
# ============================================================
# 56. HÜCRE — MODEL INPUT PIPELINE
#
# AMAÇ:
# Bir hasta + slice numarası verildiğinde:
#
#   T1
#   T1ce
#   T2
#   FLAIR
#
# görüntülerini okuyup normalize ederek
# 4 kanallı model input'u oluşturmak.
#
# Ayrıca aynı slice'ın binary tumor maskesini hazırlamak.
#
# ÇIKTI:
#
# image -> (240, 240, 4)
# mask  -> (240, 240, 1)
#
# Henüz model eğitmiyoruz.
# Önce modelin alacağı veriyi doğru hazırladığımızdan
# emin oluyoruz.
# ============================================================

import os
import numpy as np
import nibabel as nib


# ============================================================
# NORMALIZATION FONKSİYONU
# ============================================================

def normalize_mri(volume):

    # --------------------------------------------------------
    # Background pikselleri genellikle 0'dır.
    #
    # Normalizasyon hesabına background'u katmak istemiyoruz.
    # Bu nedenle yalnızca sıfır olmayan brain voxellerini
    # kullanıyoruz.
    # --------------------------------------------------------

    brain_mask = volume > 0


    # Eğer geçerli voxel yoksa görüntüyü değiştirmeden döndür.
    if np.sum(brain_mask) == 0:
        return volume.astype(np.float32)


    # --------------------------------------------------------
    # Brain bölgesinin ortalama ve standart sapmasını hesapla.
    # --------------------------------------------------------

    mean = volume[brain_mask].mean()
    std = volume[brain_mask].std()


    # Sonucu float32 olarak oluştur.
    normalized = np.zeros_like(
        volume,
        dtype=np.float32
    )


    # --------------------------------------------------------
    # Z-score normalization:
    #
    # normalized = (pixel - mean) / std
    #
    # Sadece brain bölgesine uyguluyoruz.
    # --------------------------------------------------------

    if std > 0:

        normalized[brain_mask] = (
            volume[brain_mask] - mean
        ) / std


    return normalized


# ============================================================
# MODEL SAMPLE OLUŞTURAN FONKSİYON
# ============================================================

def load_model_sample(
    patient_id,
    slice_index,
    dataset_dir
):

    # BraTS'taki dört MRI modality.
    modalities = [
        "t1",
        "t1ce",
        "t2",
        "flair"
    ]


    # Her modality'nin hazırlanmış slice'ını
    # burada saklayacağız.
    modality_slices = []


    # ========================================================
    # 4 MRI MODALITY'SİNİ YÜKLE
    # ========================================================

    for modality in modalities:

        # Dosyanın yolunu oluştur.
        image_path = os.path.join(
            dataset_dir,
            patient_id,
            f"{patient_id}_{modality}.nii.gz"
        )


        # MRI volume'u yükle.
        nii = nib.load(image_path)

        volume = np.asarray(
            nii.dataobj,
            dtype=np.float32
        )


        # ----------------------------------------------------
        # ÖNEMLİ:
        #
        # Normalizasyonu TEK SLICE üzerinde değil,
        # hastanın 3D MRI volume'u üzerinde yapıyoruz.
        #
        # Böylece aynı hastanın slice'ları arasında
        # tutarlı intensity dönüşümü korunuyor.
        # ----------------------------------------------------

        normalized_volume = normalize_mri(
            volume
        )


        # İstenen axial slice'ı al.
        image_slice = normalized_volume[
            :,
            :,
            slice_index
        ]


        # Listeye ekle.
        modality_slices.append(
            image_slice
        )


    # ========================================================
    # 4 MODALITY -> 4 CHANNEL
    # ========================================================

    # Dört adet:
    #
    # (240,240)
    #
    # görüntüyü birleştiriyoruz.
    #
    # Sonuç:
    #
    # (240,240,4)
    #
    image = np.stack(
        modality_slices,
        axis=-1
    ).astype(np.float32)


    # ========================================================
    # SEGMENTATION MASK
    # ========================================================

    seg_path = os.path.join(
        dataset_dir,
        patient_id,
        f"{patient_id}_seg.nii.gz"
    )


    seg_nii = nib.load(
        seg_path
    )

    seg_volume = np.asarray(
        seg_nii.dataobj
    )


    # İstenen slice.
    mask = seg_volume[
        :,
        :,
        slice_index
    ]


    # --------------------------------------------------------
    # Multiclass BraTS mask:
    #
    # 0 = background
    # 1,2,4 = tumor
    #
    # Binary segmentation:
    #
    # 0 -> background
    # 1 -> tumor
    # --------------------------------------------------------

    mask = (
        mask > 0
    ).astype(np.float32)


    # --------------------------------------------------------
    # (240,240)
    #
    # yerine:
    #
    # (240,240,1)
    #
    # yapıyoruz.
    #
    # Bu U-Net output yapısıyla uyumlu olacak.
    # --------------------------------------------------------

    mask = np.expand_dims(
        mask,
        axis=-1
    )


    return image, mask

In [ ]:
# ============================================================
# 57. HÜCRE — MODEL INPUT PIPELINE TEST
#
# Train indexinden tümör içeren gerçek bir slice seçiyoruz
# ve 56. hücredeki fonksiyonu test ediyoruz.
# ============================================================


# ------------------------------------------------------------
# Tumor bulunan slice'lardan bir örnek seç.
#
# tumor_pixels değeri median'a yakın bir örnek seçiyoruz.
# Böylece ilk test için aşırı küçük veya aşırı büyük
# bir tumor örneğine bağımlı kalmıyoruz.
# ------------------------------------------------------------

tumor_samples = train_slice_df[
    train_slice_df["has_tumor"] == 1
].copy()


median_tumor_size = tumor_samples[
    "tumor_pixels"
].median()


# Median tumor büyüklüğüne en yakın slice'ı bul.
selected_row = tumor_samples.loc[
    (
        tumor_samples["tumor_pixels"]
        - median_tumor_size
    ).abs().idxmin()
]


selected_patient = selected_row[
    "patient_id"
]

selected_slice = int(
    selected_row["slice_index"]
)


print("Seçilen hasta :", selected_patient)
print("Seçilen slice :", selected_slice)
print(
    "Tumor pixels  :",
    int(selected_row["tumor_pixels"])
)


# ------------------------------------------------------------
# Gerçek model sample'ını oluştur.
# ------------------------------------------------------------

sample_image, sample_mask = load_model_sample(
    selected_patient,
    selected_slice,
    training_data_dir
)


# ============================================================
# KONTROLLER
# ============================================================

print("\nMODEL INPUT")
print("-" * 40)

print(
    "Image shape :",
    sample_image.shape
)

print(
    "Image dtype :",
    sample_image.dtype
)

print(
    "Image min   :",
    sample_image.min()
)

print(
    "Image max   :",
    sample_image.max()
)


print("\nGROUND TRUTH")
print("-" * 40)

print(
    "Mask shape  :",
    sample_mask.shape
)

print(
    "Mask dtype  :",
    sample_mask.dtype
)

print(
    "Mask classes:",
    np.unique(sample_mask)
)

In [ ]:
# ============================================================
# 58. HÜCRE — MODEL INPUT VISUAL SANITY CHECK
#
# AMAÇ:
# Modelin gerçekten göreceği 4 MRI kanalını ve
# öğrenmeye çalışacağı Ground Truth maskesini
# yan yana görsel olarak kontrol etmek.
#
# sample_image:
#   channel 0 -> T1
#   channel 1 -> T1ce
#   channel 2 -> T2
#   channel 3 -> FLAIR
#
# sample_mask:
#   0 -> background
#   1 -> tumor
# ============================================================

import matplotlib.pyplot as plt


# MRI kanallarının isimleri
modality_names = [
    "T1",
    "T1ce",
    "T2",
    "FLAIR"
]


# ------------------------------------------------------------
# 5 panel oluşturuyoruz:
#
# 1 -> T1
# 2 -> T1ce
# 3 -> T2
# 4 -> FLAIR
# 5 -> Ground Truth
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1,
    5,
    figsize=(20, 4)
)


# ============================================================
# 4 MRI KANALINI GÖSTER
# ============================================================

for channel in range(4):

    axes[channel].imshow(
        sample_image[:, :, channel],
        cmap="gray"
    )

    axes[channel].set_title(
        modality_names[channel]
    )

    axes[channel].axis("off")


# ============================================================
# GROUND TRUTH MASK
# ============================================================

axes[4].imshow(
    sample_mask[:, :, 0],
    cmap="gray"
)

axes[4].set_title(
    "Binary Ground Truth"
)

axes[4].axis("off")


# ------------------------------------------------------------
# Genel başlık
# ------------------------------------------------------------

plt.suptitle(
    f"{selected_patient} — Slice {selected_slice}",
    fontsize=14
)


plt.tight_layout()

plt.show()

### Multi-Modal MRI Input and Ground Truth Verification

The final model input pipeline combines four complementary MRI modalities (**T1, T1ce, T2, and FLAIR**) from the same patient and axial slice into a single multi-channel input with shape **(240, 240, 4)**.

The corresponding BraTS segmentation labels were converted into a **binary ground-truth mask**, where `0` represents background and `1` represents tumor tissue. The resulting target has shape **(240, 240, 1)**.

The visualization above provides a sanity check of the preprocessing pipeline. All four modalities represent the same anatomical slice, while the binary ground-truth mask spatially corresponds to the visible lesion region.

**Final sample representation:**

- **Input:** `(240, 240, 4)` → T1 + T1ce + T2 + FLAIR
- **Target:** `(240, 240, 1)` → Binary tumor segmentation mask

This confirms that the raw 3D BraTS MRI volumes can be transformed into aligned, normalized multi-modal 2D samples suitable for training a segmentation network.

In [ ]:
# ============================================================
# 59. HÜCRE — TRAINING PIPELINE HAZIRLIK KONTROLÜ
#
# AMAÇ:
# Deep learning eğitimine geçmeden önce:
#
# 1) TensorFlow'un çalıştığını kontrol etmek
# 2) Train / validation indexlerinin hazır olduğunu doğrulamak
# 3) Model input boyutlarını tanımlamak
# 4) Eğitimde kullanacağımız temel parametreleri belirlemek
#
# Bu hücre henüz modeli eğitmez.
# ============================================================

import tensorflow as tf
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# TensorFlow sürümünü kontrol ediyoruz.
# ------------------------------------------------------------

print("TensorFlow version:", tf.__version__)


# ------------------------------------------------------------
# GPU var mı kontrol ediyoruz.
#
# Model eğitimi sırasında GPU kullanmak,
# CPU'ya göre çok daha hızlı olacaktır.
# ------------------------------------------------------------

gpus = tf.config.list_physical_devices("GPU")

print("\nGPU sayısı:", len(gpus))

if len(gpus) > 0:
    print("GPU bulundu! ✅")
    print("GPU:", gpus[0])
else:
    print("GPU bulunamadı ⚠️")
    print("Model eğitmeden önce Colab GPU açacağız.")


# ============================================================
# DATASET INDEX KONTROLÜ
# ============================================================

print("\n" + "=" * 55)
print("DATASET INDEX CHECK")
print("=" * 55)


print(
    "Train slice sayısı      :",
    len(train_slice_df)
)

print(
    "Validation slice sayısı :",
    len(val_slice_df)
)


# ------------------------------------------------------------
# Tumor / background dağılımını tekrar kontrol ediyoruz.
# ------------------------------------------------------------

train_tumor = int(
    (train_slice_df["has_tumor"] == 1).sum()
)

train_background = int(
    (train_slice_df["has_tumor"] == 0).sum()
)


val_tumor = int(
    (val_slice_df["has_tumor"] == 1).sum()
)

val_background = int(
    (val_slice_df["has_tumor"] == 0).sum()
)


print("\nTRAIN")
print("-" * 30)

print("Tumor slices     :", train_tumor)
print("Background slices:", train_background)


print("\nVALIDATION")
print("-" * 30)

print("Tumor slices     :", val_tumor)
print("Background slices:", val_background)


# ============================================================
# MODEL INPUT PARAMETRELERİ
# ============================================================

IMAGE_HEIGHT = 240
IMAGE_WIDTH = 240
NUM_CHANNELS = 4

# Başlangıç için küçük batch kullanıyoruz.
# 4 MRI kanalı nedeniyle her örnek normal RGB görüntülerden
# daha fazla bellek kullanabilir.
BATCH_SIZE = 8


print("\n" + "=" * 55)
print("MODEL INPUT CONFIGURATION")
print("=" * 55)

print(
    "Input shape:",
    (
        IMAGE_HEIGHT,
        IMAGE_WIDTH,
        NUM_CHANNELS
    )
)

print(
    "Batch size:",
    BATCH_SIZE
)

In [ ]:
# ============================================================
# SESSION RESTORE — GPU OTURUMU
#
# AMAÇ:
# Runtime yeniden başladığı için RAM'den silinen gerekli
# değişkenleri Google Drive'daki kayıtlarımızdan geri yüklemek.
#
# Dataseti tekrar indirmiyoruz.
# Slice indexlerini tekrar oluşturmuyoruz.
# ============================================================

from google.colab import drive
import os
import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. GOOGLE DRIVE'I BAĞLA
# ------------------------------------------------------------

drive.mount("/content/drive")


# ------------------------------------------------------------
# 2. PROJE KLASÖRLERİNİ TEKRAR TANIMLA
# ------------------------------------------------------------

project_dir = "/content/drive/MyDrive/NeuroRobust"

dataset_dir = os.path.join(
    project_dir,
    "dataset"
)

training_data_dir = os.path.join(
    dataset_dir,
    "BraTS2021_Training_Data"
)

models_dir = os.path.join(
    project_dir,
    "models"
)

results_dir = os.path.join(
    project_dir,
    "results"
)


# ------------------------------------------------------------
# 3. DAHA ÖNCE KAYDETTİĞİMİZ SLICE INDEXLERİNİ YÜKLE
# ------------------------------------------------------------

train_index_path = os.path.join(
    project_dir,
    "train_slice_index.csv"
)

val_index_path = os.path.join(
    project_dir,
    "validation_slice_index.csv"
)


train_slice_df = pd.read_csv(
    train_index_path
)

val_slice_df = pd.read_csv(
    val_index_path
)


# ------------------------------------------------------------
# 4. MODEL PARAMETRELERİNİ GERİ TANIMLA
# ------------------------------------------------------------

IMAGE_HEIGHT = 240
IMAGE_WIDTH = 240
NUM_CHANNELS = 4

BATCH_SIZE = 8


# ------------------------------------------------------------
# 5. HER ŞEY GERİ GELDİ Mİ KONTROL ET
# ------------------------------------------------------------

print("\nSESSION RESTORED! ✅")
print("=" * 50)

print("Project folder:")
print(project_dir)

print("\nTraining dataset:")
print(training_data_dir)

print("\nTrain slice sayısı:")
print(len(train_slice_df))

print("\nValidation slice sayısı:")
print(len(val_slice_df))

print("\nDataset mevcut mu?:")
print(os.path.exists(training_data_dir))

print("\nTrain index mevcut mu?:")
print(os.path.exists(train_index_path))

print("\nValidation index mevcut mu?:")
print(os.path.exists(val_index_path))

In [ ]:
# ============================================================
# 60. HÜCRE — TRAINING-READY SLICE LOADER
#
# AMAÇ:
# Bir patient_id + slice_index verildiğinde:
#
#   T1
#   T1ce
#   T2
#   FLAIR
#
# görüntülerini okuyup model için hazırlamak.
#
# Ayrıca aynı slice'ın binary tumor maskesini üretmek.
#
# ÖNEMLİ:
# Orijinal MRI: 240 x 240
# Model input   : 128 x 128
#
# Datasetin kendisini değiştirmiyoruz.
# Resize yalnızca model input'u hazırlanırken yapılıyor.
# ============================================================

import os
import numpy as np
import nibabel as nib
import cv2


# ============================================================
# MODEL İÇİN KULLANACAĞIMIZ BOYUT
# ============================================================

MODEL_HEIGHT = 128
MODEL_WIDTH = 128
NUM_CHANNELS = 4


# ============================================================
# NORMALIZATION FONKSİYONU
# ============================================================

def normalize_volume(volume):

    # --------------------------------------------------------
    # MRI görüntülerinde 0 değerleri çoğunlukla
    # görüntünün dışındaki background bölgesidir.
    #
    # Ortalama ve standart sapmayı hesaplarken
    # sadece gerçek görüntü bölgesini kullanıyoruz.
    # --------------------------------------------------------

    brain_mask = volume > 0


    # Geçerli voxel yoksa sıfır volume döndür.
    if np.sum(brain_mask) == 0:

        return np.zeros_like(
            volume,
            dtype=np.float32
        )


    # Brain bölgesinin istatistiklerini hesapla.
    mean = volume[brain_mask].mean()
    std = volume[brain_mask].std()


    # Normalized volume için boş array.
    normalized = np.zeros_like(
        volume,
        dtype=np.float32
    )


    # --------------------------------------------------------
    # Z-score normalization
    #
    # x_normalized = (x - mean) / std
    # --------------------------------------------------------

    if std > 0:

        normalized[brain_mask] = (
            volume[brain_mask] - mean
        ) / std


    return normalized


# ============================================================
# TEK BİR TRAINING SAMPLE YÜKLE
# ============================================================

def load_training_sample(
    patient_id,
    slice_index
):

    # --------------------------------------------------------
    # Hastanın klasörü
    # --------------------------------------------------------

    patient_folder = os.path.join(
        training_data_dir,
        patient_id
    )


    # BraTS MRI modalities
    modalities = [
        "t1",
        "t1ce",
        "t2",
        "flair"
    ]


    # Her MRI kanalını burada biriktireceğiz.
    channels = []


    # ========================================================
    # 4 MRI MODALITY
    # ========================================================

    for modality in modalities:

        # Dosya yolu
        file_path = os.path.join(
            patient_folder,
            f"{patient_id}_{modality}.nii.gz"
        )


        # 3D MRI volume'u yükle.
        nii = nib.load(file_path)

        volume = np.asarray(
            nii.dataobj,
            dtype=np.float32
        )


        # ----------------------------------------------------
        # Volume-level normalization
        #
        # Önce tüm 3D volume normalize ediliyor.
        # Daha sonra istediğimiz slice alınıyor.
        # ----------------------------------------------------

        volume = normalize_volume(volume)


        # İstenen axial slice
        image_slice = volume[
            :,
            :,
            int(slice_index)
        ]


        # ----------------------------------------------------
        # 240 x 240 -> 128 x 128
        #
        # MRI görüntüsü sürekli değerlerden oluştuğu için
        # INTER_LINEAR interpolation kullanıyoruz.
        # ----------------------------------------------------

        image_slice = cv2.resize(
            image_slice,
            (MODEL_WIDTH, MODEL_HEIGHT),
            interpolation=cv2.INTER_LINEAR
        )


        channels.append(
            image_slice
        )


    # ========================================================
    # DÖRT MODALITY'Yİ TEK INPUT'TA BİRLEŞTİR
    # ========================================================

    image = np.stack(
        channels,
        axis=-1
    ).astype(np.float32)


    # ========================================================
    # GROUND TRUTH MASK
    # ========================================================

    mask_path = os.path.join(
        patient_folder,
        f"{patient_id}_seg.nii.gz"
    )


    mask_nii = nib.load(
        mask_path
    )

    mask_volume = np.asarray(
        mask_nii.dataobj
    )


    # Aynı axial slice
    mask = mask_volume[
        :,
        :,
        int(slice_index)
    ]


    # --------------------------------------------------------
    # BraTS multiclass mask:
    #
    # 0 = background
    # 1, 2, 4 = tumor regions
    #
    # Biz binary segmentation yaptığımız için:
    #
    # 0     -> 0
    # 1/2/4 -> 1
    # --------------------------------------------------------

    mask = (
        mask > 0
    ).astype(np.float32)


    # --------------------------------------------------------
    # MASK RESIZE
    #
    # Mask için INTER_NEAREST kullanmak ÇOK önemli.
    #
    # Çünkü maskede yalnızca 0 ve 1 kalmasını istiyoruz.
    # Linear interpolation kullansaydık:
    #
    # 0.23, 0.57, 0.81...
    #
    # gibi anlamsız ara sınıflar oluşabilirdi.
    # --------------------------------------------------------

    mask = cv2.resize(
        mask,
        (MODEL_WIDTH, MODEL_HEIGHT),
        interpolation=cv2.INTER_NEAREST
    )


    # (128,128) -> (128,128,1)
    mask = np.expand_dims(
        mask,
        axis=-1
    ).astype(np.float32)


    return image, mask

In [ ]:
# ============================================================
# 61. HÜCRE — TRAINING LOADER TEST
#
# AMAÇ:
# 60. hücrede oluşturduğumuz loader'ın gerçekten
# doğru image ve mask ürettiğini kontrol etmek.
# ============================================================


# ------------------------------------------------------------
# Train indexinden tumor bulunan bir örnek seçiyoruz.
# ------------------------------------------------------------

test_row = train_slice_df[
    train_slice_df["has_tumor"] == 1
].iloc[1000]


test_patient = test_row["patient_id"]

test_slice = int(
    test_row["slice_index"]
)


# ------------------------------------------------------------
# Loader'ı çalıştır.
# ------------------------------------------------------------

test_image, test_mask = load_training_sample(
    test_patient,
    test_slice
)


# ============================================================
# SONUÇ KONTROLLERİ
# ============================================================

print("Patient :", test_patient)
print("Slice   :", test_slice)

print("\nIMAGE")
print("-" * 35)

print("Shape :", test_image.shape)
print("dtype :", test_image.dtype)

print(
    "Min   :",
    test_image.min()
)

print(
    "Max   :",
    test_image.max()
)


print("\nMASK")
print("-" * 35)

print("Shape :", test_mask.shape)
print("dtype :", test_mask.dtype)

print(
    "Classes:",
    np.unique(test_mask)
)

print(
    "Tumor pixels:",
    int(test_mask.sum())
)

In [ ]:
# ============================================================
# 62. HÜCRE — KERAS BATCH DATA GENERATOR
#
# AMAÇ:
# train_slice_df içindeki kayıtları kullanarak modeli
# besleyebilecek batch'ler oluşturmak.
#
# Örneğin BATCH_SIZE = 8 ise:
#
# X batch -> (8, 128, 128, 4)
# Y batch -> (8, 128, 128, 1)
#
# X = MRI görüntüleri
# Y = Ground Truth tumor maskeleri
# ============================================================

import numpy as np
import tensorflow as tf


# ============================================================
# DATA GENERATOR CLASS
# ============================================================

class BraTSDataGenerator(tf.keras.utils.Sequence):

    def __init__(
        self,
        dataframe,
        batch_size=8,
        shuffle=True
    ):

        # DataFrame'i sıfırdan indexliyoruz.
        self.dataframe = dataframe.reset_index(drop=True)

        self.batch_size = batch_size
        self.shuffle = shuffle

        # ----------------------------------------------------
        # Her sample'ın sıra numarasını tutuyoruz.
        #
        # Örneğin:
        # [0, 1, 2, 3, ...]
        # ----------------------------------------------------

        self.indices = np.arange(
            len(self.dataframe)
        )

        # İlk epoch başlamadan önce gerekirse karıştır.
        self.on_epoch_end()


    # ========================================================
    # BİR EPOCH'TA KAÇ BATCH OLACAĞINI BELİRLER
    # ========================================================

    def __len__(self):

        return int(
            np.ceil(
                len(self.dataframe)
                / self.batch_size
            )
        )


    # ========================================================
    # BİR BATCH OLUŞTUR
    # ========================================================

    def __getitem__(self, batch_index):

        # Bu batch'in başlangıç noktası.
        start = (
            batch_index
            * self.batch_size
        )

        # Bitiş noktası.
        end = min(
            start + self.batch_size,
            len(self.dataframe)
        )


        # Bu batch'te kullanılacak kayıtların indexleri.
        batch_indices = self.indices[
            start:end
        ]


        # ----------------------------------------------------
        # Batch arraylerini önceden oluşturuyoruz.
        # ----------------------------------------------------

        current_batch_size = len(
            batch_indices
        )


        X = np.zeros(
            (
                current_batch_size,
                MODEL_HEIGHT,
                MODEL_WIDTH,
                NUM_CHANNELS
            ),
            dtype=np.float32
        )


        Y = np.zeros(
            (
                current_batch_size,
                MODEL_HEIGHT,
                MODEL_WIDTH,
                1
            ),
            dtype=np.float32
        )


        # ====================================================
        # BATCH İÇİNDEKİ SAMPLE'LARI YÜKLE
        # ====================================================

        for i, row_index in enumerate(
            batch_indices
        ):

            row = self.dataframe.iloc[
                row_index
            ]


            patient_id = row[
                "patient_id"
            ]

            slice_index = int(
                row["slice_index"]
            )


            # 60. hücrede oluşturduğumuz fonksiyon.
            image, mask = load_training_sample(
                patient_id,
                slice_index
            )


            X[i] = image
            Y[i] = mask


        return X, Y


    # ========================================================
    # EPOCH SONUNDA DATA'YI KARIŞTIR
    # ========================================================

    def on_epoch_end(self):

        if self.shuffle:

            np.random.shuffle(
                self.indices
            )

In [ ]:
# ============================================================
# 63. HÜCRE — GENERATOR TEST
#
# AMAÇ:
# Generator'ın gerçekten doğru batch ürettiğini
# küçük bir örnek üzerinde kontrol etmek.
#
# Henüz model eğitimi yapılmıyor.
# ============================================================


# ------------------------------------------------------------
# Test için 16 adet tumor-containing slice seç.
#
# Burada amaç yalnızca generator'ın teknik olarak
# çalıştığını doğrulamak.
# ------------------------------------------------------------

generator_test_df = (
    train_slice_df[
        train_slice_df["has_tumor"] == 1
    ]
    .sample(
        n=16,
        random_state=42
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Generator oluştur.
# ------------------------------------------------------------

test_generator = BraTSDataGenerator(
    dataframe=generator_test_df,
    batch_size=8,
    shuffle=False
)


print(
    "Generator içindeki sample sayısı:",
    len(generator_test_df)
)

print(
    "Batch sayısı:",
    len(test_generator)
)


# ------------------------------------------------------------
# İlk batch'i generator'dan al.
# ------------------------------------------------------------

X_batch, Y_batch = test_generator[0]


# ============================================================
# KONTROLLER
# ============================================================

print("\nFIRST BATCH")
print("=" * 45)

print(
    "X batch shape:",
    X_batch.shape
)

print(
    "Y batch shape:",
    Y_batch.shape
)

print(
    "X dtype:",
    X_batch.dtype
)

print(
    "Y dtype:",
    Y_batch.dtype
)

print(
    "Mask classes:",
    np.unique(Y_batch)
)


# Her sample'daki tumor pixel sayısını görelim.
print("\nTumor pixels in each sample:")

for i in range(len(Y_batch)):

    print(
        f"Sample {i}:",
        int(Y_batch[i].sum())
    )

In [ ]:
# ============================================================
# 64. HÜCRE — PATIENT-LEVEL LRU CACHE
#
# AMAÇ:
# Aynı hastanın MRI volume'larını her slice için tekrar tekrar
# Google Drive'dan okumayı önlemek.
#
# Örneğin bir hastadan 60 slice kullanıyorsak:
#
# ESKİ:
# slice 1  -> dosyaları tekrar oku
# slice 2  -> dosyaları tekrar oku
# slice 3  -> dosyaları tekrar oku
# ...
#
# YENİ:
# Hasta ilk kez geldi -> volume'ları oku ve normalize et
# Sonraki slice'lar   -> RAM'deki cache'ten kullan
#
# LRU cache sayesinde RAM'in sınırsız büyümesini de önlüyoruz.
# ============================================================

from functools import lru_cache
import os
import numpy as np
import nibabel as nib


# ------------------------------------------------------------
# Aynı anda en fazla 4 hastanın hazırlanmış volume'larını
# RAM'de tutacağız.
#
# 4 ile başlamak Colab RAM'i açısından daha güvenlidir.
# ------------------------------------------------------------

@lru_cache(maxsize=4)
def load_patient_volumes(patient_id):

    patient_folder = os.path.join(
        training_data_dir,
        patient_id
    )

    modalities = [
        "t1",
        "t1ce",
        "t2",
        "flair"
    ]

    normalized_volumes = []


    # ========================================================
    # 4 MRI MODALITY'SİNİ SADECE BİR KEZ YÜKLE
    # ========================================================

    for modality in modalities:

        file_path = os.path.join(
            patient_folder,
            f"{patient_id}_{modality}.nii.gz"
        )

        nii = nib.load(file_path)

        volume = np.asarray(
            nii.dataobj,
            dtype=np.float32
        )

        # Daha önce yazdığımız normalization fonksiyonu.
        volume = normalize_volume(
            volume
        )

        normalized_volumes.append(
            volume
        )


    # ========================================================
    # SEGMENTATION VOLUME
    # ========================================================

    mask_path = os.path.join(
        patient_folder,
        f"{patient_id}_seg.nii.gz"
    )

    mask_nii = nib.load(
        mask_path
    )

    mask_volume = np.asarray(
        mask_nii.dataobj
    )

    # Binary segmentation:
    # 0     -> background
    # 1/2/4 -> tumor
    mask_volume = (
        mask_volume > 0
    ).astype(np.uint8)


    # --------------------------------------------------------
    # Dört MRI volume + mask döndür.
    # --------------------------------------------------------

    return (
        normalized_volumes[0],  # T1
        normalized_volumes[1],  # T1ce
        normalized_volumes[2],  # T2
        normalized_volumes[3],  # FLAIR
        mask_volume
    )


print("Patient cache oluşturuldu! ✅")
print("Cache kapasitesi: 4 hasta")

In [ ]:
# ============================================================
# 65. HÜCRE — CACHED TRAINING SAMPLE LOADER
#
# AMAÇ:
# 64. hücredeki patient cache'i kullanarak
# model sample'larını daha verimli oluşturmak.
# ============================================================

import cv2
import numpy as np


def load_training_sample_cached(
    patient_id,
    slice_index
):

    # --------------------------------------------------------
    # Hastanın 4 MRI volume'u ve maskesi.
    #
    # Hasta cache'te varsa:
    # -> Drive'dan tekrar okunmaz.
    #
    # Yoksa:
    # -> bir kez okunur ve cache'e eklenir.
    # --------------------------------------------------------

    (
        t1_volume,
        t1ce_volume,
        t2_volume,
        flair_volume,
        mask_volume
    ) = load_patient_volumes(patient_id)


    volumes = [
        t1_volume,
        t1ce_volume,
        t2_volume,
        flair_volume
    ]


    channels = []


    # ========================================================
    # MRI SLICES
    # ========================================================

    for volume in volumes:

        # İstenen axial slice.
        image_slice = volume[
            :,
            :,
            int(slice_index)
        ]


        # 240x240 -> 128x128
        image_slice = cv2.resize(
            image_slice,
            (MODEL_WIDTH, MODEL_HEIGHT),
            interpolation=cv2.INTER_LINEAR
        )


        channels.append(
            image_slice
        )


    # 4 modality -> 4 channel
    image = np.stack(
        channels,
        axis=-1
    ).astype(np.float32)


    # ========================================================
    # MASK
    # ========================================================

    mask = mask_volume[
        :,
        :,
        int(slice_index)
    ]


    # Mask için nearest-neighbor kullanıyoruz.
    mask = cv2.resize(
        mask,
        (MODEL_WIDTH, MODEL_HEIGHT),
        interpolation=cv2.INTER_NEAREST
    )


    mask = np.expand_dims(
        mask,
        axis=-1
    ).astype(np.float32)


    return image, mask

In [ ]:
# ============================================================
# 66. HÜCRE — CACHE TEST
#
# AMAÇ:
# Aynı hastanın ikinci kez çağrıldığında cache'ten
# geldiğini doğrulamak.
# ============================================================

import time


# Cache'i temizleyerek adil bir test başlatıyoruz.
load_patient_volumes.cache_clear()


# Train setinden en az iki kaydı bulunan bir hasta seç.
cache_test_patient = train_slice_df.iloc[0]["patient_id"]

patient_rows = train_slice_df[
    train_slice_df["patient_id"] == cache_test_patient
]


slice_1 = int(
    patient_rows.iloc[0]["slice_index"]
)

slice_2 = int(
    patient_rows.iloc[1]["slice_index"]
)


print("Test patient:", cache_test_patient)
print("Slice 1:", slice_1)
print("Slice 2:", slice_2)


# ------------------------------------------------------------
# İLK ÇAĞRI
#
# Dosyalar Drive'dan okunacak.
# ------------------------------------------------------------

start = time.time()

image_1, mask_1 = load_training_sample_cached(
    cache_test_patient,
    slice_1
)

first_time = time.time() - start


# ------------------------------------------------------------
# İKİNCİ ÇAĞRI
#
# Aynı hasta olduğu için volume'lar cache'ten gelmeli.
# ------------------------------------------------------------

start = time.time()

image_2, mask_2 = load_training_sample_cached(
    cache_test_patient,
    slice_2
)

second_time = time.time() - start


# ============================================================
# SONUÇLAR
# ============================================================

print("\nFIRST LOAD")
print("-" * 35)

print(
    "Time:",
    round(first_time, 3),
    "seconds"
)


print("\nSECOND LOAD — SAME PATIENT")
print("-" * 35)

print(
    "Time:",
    round(second_time, 3),
    "seconds"
)


print("\nCACHE INFO")
print("-" * 35)

print(
    load_patient_volumes.cache_info()
)


print("\nOUTPUT SHAPES")
print("-" * 35)

print("Image:", image_2.shape)
print("Mask :", mask_2.shape)
print(
    "Mask classes:",
    np.unique(mask_2)
)

In [ ]:
# ============================================================
# 67. HÜCRE — CACHED DATA GENERATOR
#
# AMAÇ:
# Model eğitimi sırasında MRI slice'larını batch halinde
# üretirken yeni cache sistemimizi kullanmak.
#
# Böylece:
#
# Google Drive
#      ↓
# Patient Cache
#      ↓
# Slice çıkarma
#      ↓
# Resize
#      ↓
# Batch
#      ↓
# U-Net
#
# şeklinde daha verimli bir veri akışı elde ediyoruz.
# ============================================================

class CachedBraTSDataGenerator(tf.keras.utils.Sequence):

    def __init__(
        self,
        dataframe,
        batch_size=8,
        shuffle=True
    ):

        # DataFrame indexlerini yeniden düzenle.
        self.dataframe = dataframe.reset_index(drop=True)

        self.batch_size = batch_size
        self.shuffle = shuffle

        # Tüm sample'ların indexlerini oluştur.
        self.indices = np.arange(
            len(self.dataframe)
        )

        # İlk epoch öncesinde gerekiyorsa karıştır.
        self.on_epoch_end()


    # ========================================================
    # TOPLAM BATCH SAYISI
    # ========================================================

    def __len__(self):

        return int(
            np.ceil(
                len(self.dataframe)
                / self.batch_size
            )
        )


    # ========================================================
    # BİR BATCH OLUŞTUR
    # ========================================================

    def __getitem__(self, batch_index):

        # Bu batch'in başlangıç indexi.
        start = (
            batch_index
            * self.batch_size
        )

        # Bu batch'in bitiş indexi.
        end = min(
            start + self.batch_size,
            len(self.dataframe)
        )


        # Batch'te kullanılacak sample indexleri.
        batch_indices = self.indices[
            start:end
        ]


        current_batch_size = len(
            batch_indices
        )


        # ----------------------------------------------------
        # MRI batch
        #
        # 8 örnek varsa:
        # (8, 128, 128, 4)
        # ----------------------------------------------------

        X = np.zeros(
            (
                current_batch_size,
                MODEL_HEIGHT,
                MODEL_WIDTH,
                NUM_CHANNELS
            ),
            dtype=np.float32
        )


        # ----------------------------------------------------
        # Ground truth mask batch
        #
        # 8 örnek varsa:
        # (8, 128, 128, 1)
        # ----------------------------------------------------

        Y = np.zeros(
            (
                current_batch_size,
                MODEL_HEIGHT,
                MODEL_WIDTH,
                1
            ),
            dtype=np.float32
        )


        # ====================================================
        # BATCH İÇİNDEKİ SLICELARI YÜKLE
        # ====================================================

        for i, row_index in enumerate(
            batch_indices
        ):

            row = self.dataframe.iloc[
                row_index
            ]


            patient_id = row[
                "patient_id"
            ]

            slice_index = int(
                row["slice_index"]
            )


            # ------------------------------------------------
            # ÖNEMLİ:
            #
            # Artık eski load_training_sample() yerine
            # CACHE kullanan loader'ı çağırıyoruz.
            # ------------------------------------------------

            image, mask = load_training_sample_cached(
                patient_id,
                slice_index
            )


            X[i] = image
            Y[i] = mask


        return X, Y


    # ========================================================
    # EPOCH SONU
    # ========================================================

    def on_epoch_end(self):

        if self.shuffle:

            np.random.shuffle(
                self.indices
            )


print("Cached generator oluşturuldu! ✅")

In [ ]:
# ============================================================
# 68. HÜCRE — BALANCED TRAINING SAMPLING
#
# AMAÇ:
# Tumor-containing ve background-only slice'lar arasında
# dengeli bir training set oluşturmak.
#
# Tumor      -> has_tumor = 1
# Background -> has_tumor = 0
#
# Başlangıç stratejimiz:
#
#           %50 tumor
#           %50 background
#
# Validation setine DOKUNMUYORUZ.
# Validation gerçek dağılımını koruyacak.
# ============================================================


# ------------------------------------------------------------
# Tumor-containing slice'ları ayır.
# ------------------------------------------------------------

tumor_train_df = train_slice_df[
    train_slice_df["has_tumor"] == 1
].copy()


# ------------------------------------------------------------
# Background-only slice'ları ayır.
# ------------------------------------------------------------

background_train_df = train_slice_df[
    train_slice_df["has_tumor"] == 0
].copy()


print("Orijinal TRAIN")
print("=" * 45)

print(
    "Tumor slices:",
    len(tumor_train_df)
)

print(
    "Background slices:",
    len(background_train_df)
)


# ============================================================
# BALANCING
# ============================================================
#
# Tumor sayısı kadar background seçiyoruz.
#
# Tumor = 57,067 ise:
# Background'dan da 57,067 tane seçilecek.
# ============================================================

background_sampled_df = (
    background_train_df
    .sample(
        n=len(tumor_train_df),
        random_state=42
    )
)


# Tumor + seçilmiş background
balanced_train_df = pd.concat(
    [
        tumor_train_df,
        background_sampled_df
    ],
    ignore_index=True
)


# ------------------------------------------------------------
# Kayıtların sırasını karıştır.
# ------------------------------------------------------------

balanced_train_df = (
    balanced_train_df
    .sample(
        frac=1,
        random_state=42
    )
    .reset_index(drop=True)
)


# ============================================================
# SONUÇ
# ============================================================

print("\nBALANCED TRAIN")
print("=" * 45)

print(
    "Toplam slice:",
    len(balanced_train_df)
)

print(
    "Tumor slices:",
    (
        balanced_train_df["has_tumor"] == 1
    ).sum()
)

print(
    "Background slices:",
    (
        balanced_train_df["has_tumor"] == 0
    ).sum()
)


tumor_ratio = (
    balanced_train_df["has_tumor"].mean()
    * 100
)


print(
    "Tumor ratio:",
    f"{tumor_ratio:.2f}%"
)

In [ ]:
# ============================================================
# 69. HÜCRE — BALANCED GENERATOR TEST
#
# AMAÇ:
# 68. hücrede oluşturduğumuz balanced_train_df ile
# cache kullanan generator'ı çalıştırmak.
#
# Kontrol edeceğimiz şeyler:
#
# X -> MRI görüntüleri
# Y -> Ground Truth maskeleri
#
# Beklenen:
# X shape = (8, 128, 128, 4)
# Y shape = (8, 128, 128, 1)
# ============================================================


# ------------------------------------------------------------
# Balanced training generator oluştur.
# ------------------------------------------------------------

train_generator = CachedBraTSDataGenerator(
    dataframe=balanced_train_df,
    batch_size=8,
    shuffle=True
)


print("Train generator oluşturuldu! ✅")

print(
    "Generator içindeki toplam slice:",
    len(balanced_train_df)
)

print(
    "Toplam batch sayısı:",
    len(train_generator)
)


# ============================================================
# İLK BATCH'İ AL
# ============================================================

X_batch, Y_batch = train_generator[0]


print("\nFIRST BATCH")
print("=" * 45)

print(
    "X batch shape:",
    X_batch.shape
)

print(
    "Y batch shape:",
    Y_batch.shape
)

print(
    "X dtype:",
    X_batch.dtype
)

print(
    "Y dtype:",
    Y_batch.dtype
)

print(
    "Mask classes:",
    np.unique(Y_batch)
)


# ============================================================
# HER ÖRNEKTEKİ TÜMÖR PİKSEL SAYISI
# ============================================================

print("\nTumor pixels in each sample:")

for i in range(len(Y_batch)):

    tumor_pixels = int(
        Y_batch[i].sum()
    )

    print(
        f"Sample {i}: {tumor_pixels}"
    )

## Data Pipeline Validation

The preprocessing and data-loading pipeline was successfully validated before model training.

Each model input has a shape of **128 × 128 × 4**, where the four channels correspond to **T1, T1ce, T2, and FLAIR MRI modalities**. The corresponding segmentation target has a shape of **128 × 128 × 1** and contains a binary tumor mask.

The training slice pool was balanced to contain:

- **57,067 tumor-containing slices**
- **57,067 background-only slices**
- **114,134 training slices in total**

A patient-level cache was also introduced to reduce repeated loading of the same 3D MRI volumes from storage. In our cache test, loading another slice from an already cached patient decreased the loading time from approximately **6.94 s to 0.001 s**.

The final batch test confirmed:

- MRI batch shape: `(8, 128, 128, 4)`
- Mask batch shape: `(8, 128, 128, 1)`
- Data type: `float32`
- Binary mask classes: `[0, 1]`

These checks confirm that the preprocessing pipeline can now provide correctly formatted multi-modal MRI images and segmentation masks for model training.

In [ ]:
# ============================================================
# 70. HÜCRE — BATCH VISUAL SANITY CHECK
#
# AMAÇ:
# Generator'ın oluşturduğu MRI görüntüleri ile Ground Truth
# maskelerinin gerçekten doğru eşleştiğini görsel olarak kontrol etmek.
#
# Her satırda:
#
# 1) FLAIR MRI
# 2) Binary Ground Truth
# 3) FLAIR + Ground Truth Overlay
#
# gösterilecektir.
# ============================================================

import matplotlib.pyplot as plt
import numpy as np


# ------------------------------------------------------------
# 69. hücrede oluşturduğumuz batch'i kullanıyoruz.
#
# X_batch shape:
# (8, 128, 128, 4)
#
# Y_batch shape:
# (8, 128, 128, 1)
# ------------------------------------------------------------

num_samples_to_show = min(
    8,
    len(X_batch)
)


# ------------------------------------------------------------
# Her sample için 3 görsel göstereceğimiz için:
#
# satır    = sample sayısı
# sütun    = 3
# ------------------------------------------------------------

fig, axes = plt.subplots(
    num_samples_to_show,
    3,
    figsize=(12, 4 * num_samples_to_show)
)


# ============================================================
# HER SAMPLE'I GÖRSELLEŞTİR
# ============================================================

for i in range(num_samples_to_show):

    # --------------------------------------------------------
    # X'in kanal sırası:
    #
    # 0 -> T1
    # 1 -> T1ce
    # 2 -> T2
    # 3 -> FLAIR
    #
    # Görselleştirme için FLAIR kanalını kullanıyoruz.
    # --------------------------------------------------------

    flair = X_batch[
        i,
        :,
        :,
        3
    ]


    # Ground Truth maskesi.
    mask = Y_batch[
        i,
        :,
        :,
        0
    ]


    # Bu slice'taki tumor pixel sayısı.
    tumor_pixels = int(
        mask.sum()
    )


    # ========================================================
    # 1. FLAIR MRI
    # ========================================================

    axes[i, 0].imshow(
        flair,
        cmap="gray"
    )

    axes[i, 0].set_title(
        f"Sample {i} — FLAIR"
    )

    axes[i, 0].axis("off")


    # ========================================================
    # 2. BINARY GROUND TRUTH
    # ========================================================

    axes[i, 1].imshow(
        mask,
        cmap="gray",
        vmin=0,
        vmax=1
    )

    axes[i, 1].set_title(
        f"Ground Truth — Tumor Pixels: {tumor_pixels}"
    )

    axes[i, 1].axis("off")


    # ========================================================
    # 3. OVERLAY
    # ========================================================

    # Önce MRI görüntüsünü çiz.
    axes[i, 2].imshow(
        flair,
        cmap="gray"
    )


    # --------------------------------------------------------
    # Sadece tumor bölgesini overlay olarak göster.
    #
    # mask == 0 olan background piksellerini gizliyoruz.
    # Böylece yalnızca tumor alanı MRI'nın üzerinde görünür.
    # --------------------------------------------------------

    tumor_overlay = np.ma.masked_where(
        mask == 0,
        mask
    )


    axes[i, 2].imshow(
        tumor_overlay,
        cmap="autumn",
        alpha=0.55
    )


    # Slice'ta tumor olup olmadığını başlıkta göster.
    if tumor_pixels > 0:

        status = "TUMOR"

    else:

        status = "BACKGROUND"


    axes[i, 2].set_title(
        f"Overlay — {status}"
    )

    axes[i, 2].axis("off")


plt.suptitle(
    "Training Batch — Visual Sanity Check",
    fontsize=16
)

plt.tight_layout()

plt.show()

In [ ]:
# ============================================================
# 71. HÜCRE — BRAIN CONTENT CHECK
#
# AMAÇ:
# Batch içerisindeki her slice'ın ne kadar görüntü içeriğine
# sahip olduğunu ölçmek.
#
# Özellikle background slice'larda beynin çok küçük kaldığı
# uç kesitleri tespit etmek istiyoruz.
# ============================================================

print("BRAIN CONTENT CHECK")
print("=" * 55)


for i in range(len(X_batch)):

    # --------------------------------------------------------
    # FLAIR kanalını alıyoruz.
    #
    # Kanal sırası:
    # 0 = T1
    # 1 = T1ce
    # 2 = T2
    # 3 = FLAIR
    # --------------------------------------------------------

    flair = X_batch[i, :, :, 3]

    mask = Y_batch[i, :, :, 0]


    # --------------------------------------------------------
    # Normalize edilmiş MRI'da background yaklaşık 0 değildir:
    # önceki normalization yöntemimiz nedeniyle background
    # sabit bir değerde olabilir.
    #
    # Bu yüzden köşe pikselini background referansı olarak
    # kullanıyoruz.
    # --------------------------------------------------------

    background_value = flair[0, 0]


    # Background değerinden anlamlı şekilde farklı pikselleri
    # görüntü içeriği olarak kabul ediyoruz.
    brain_pixels = np.sum(
        np.abs(flair - background_value) > 0.05
    )


    total_pixels = flair.size


    brain_ratio = (
        brain_pixels
        / total_pixels
        * 100
    )


    tumor_pixels = int(
        mask.sum()
    )


    print(
        f"Sample {i} | "
        f"Brain content: {brain_ratio:6.2f}% | "
        f"Tumor pixels: {tumor_pixels}"
    )

In [ ]:
# ============================================================
# 72. HÜCRE — BACKGROUND BRAIN-CONTENT ANALYSIS
#
# AMAÇ:
# TRAIN setindeki background slice'ların ne kadar gerçek
# beyin dokusu içerdiğini analiz etmek.
#
# Böylece filtreleme eşiğini 8 örneğe bakarak tahmin etmek
# yerine tüm TRAIN dağılımına bakarak belirleyeceğiz.
# ============================================================

import numpy as np
import pandas as pd


print("Background brain-content analizi başlıyor...")
print("Background slice sayısı:", len(background_train_df))
print("-" * 55)


brain_content_records = []


# ------------------------------------------------------------
# Hasta bazında grupla.
#
# Bunun avantajı:
# Aynı hastanın FLAIR volume'unu her slice için tekrar
# yüklemek yerine bir kez yükleyip ilgili slice'ları inceliyoruz.
# ------------------------------------------------------------

grouped_background = background_train_df.groupby(
    "patient_id"
)


total_patients = len(grouped_background)


for patient_number, (patient_id, patient_rows) in enumerate(
    grouped_background,
    start=1
):

    # --------------------------------------------------------
    # Cache üzerinden hastanın volume'larını al.
    #
    # Sadece FLAIR'a ihtiyacımız var.
    # --------------------------------------------------------

    (
        _,
        _,
        _,
        flair_volume,
        _
    ) = load_patient_volumes(patient_id)


    # ========================================================
    # BU HASTANIN BACKGROUND SLICELARI
    # ========================================================

    for _, row in patient_rows.iterrows():

        slice_index = int(
            row["slice_index"]
        )


        flair_slice = flair_volume[
            :,
            :,
            slice_index
        ]


        # ----------------------------------------------------
        # Köşe pikselini background referansı kabul ediyoruz.
        # ----------------------------------------------------

        background_value = flair_slice[
            0,
            0
        ]


        # Background'dan farklı görüntü piksellerini bul.
        brain_pixels = np.sum(
            np.abs(
                flair_slice - background_value
            ) > 0.05
        )


        total_pixels = flair_slice.size


        brain_ratio = (
            brain_pixels
            / total_pixels
            * 100
        )


        brain_content_records.append(
            {
                "patient_id": patient_id,
                "slice_index": slice_index,
                "brain_content": brain_ratio
            }
        )


    # İlerlemeyi göster.
    if (
        patient_number % 100 == 0
        or patient_number == total_patients
    ):

        print(
            f"Analiz edildi: "
            f"{patient_number}/{total_patients} hasta"
        )


# ============================================================
# DATAFRAME
# ============================================================

background_brain_df = pd.DataFrame(
    brain_content_records
)


print("\n" + "=" * 55)
print("BACKGROUND BRAIN-CONTENT SUMMARY")
print("=" * 55)

print(
    "Toplam background slice:",
    len(background_brain_df)
)

print(
    "Minimum:",
    round(
        background_brain_df["brain_content"].min(),
        2
    ),
    "%"
)

print(
    "25th percentile:",
    round(
        background_brain_df["brain_content"].quantile(0.25),
        2
    ),
    "%"
)

print(
    "Median:",
    round(
        background_brain_df["brain_content"].median(),
        2
    ),
    "%"
)

print(
    "75th percentile:",
    round(
        background_brain_df["brain_content"].quantile(0.75),
        2
    ),
    "%"
)

print(
    "Maximum:",
    round(
        background_brain_df["brain_content"].max(),
        2
    ),
    "%"
)


# ============================================================
# OLASI EŞİKLERİ KARŞILAŞTIR
# ============================================================

print("\n" + "=" * 55)
print("POSSIBLE FILTER THRESHOLDS")
print("=" * 55)


for threshold in [1, 5, 10, 15, 20]:

    usable = (
        background_brain_df["brain_content"]
        >= threshold
    ).sum()

    removed = (
        background_brain_df["brain_content"]
        < threshold
    ).sum()


    print(
        f">= {threshold:2d}% brain content | "
        f"Keep: {usable:5d} | "
        f"Remove: {removed:5d}"
    )

In [ ]:
# ============================================================
# 73. HÜCRE — BACKGROUND ANALİZİNİ GOOGLE DRIVE'A KAYDET
#
# AMAÇ:
# Uzun süren 78,558 background slice analizinin sonuçlarını
# kalıcı olarak Google Drive'a kaydetmek.
#
# Böylece Colab runtime kapansa bile yarın bu analizi
# tekrar çalıştırmamıza gerek kalmayacak.
# ============================================================

import os


# ------------------------------------------------------------
# Dosyanın kaydedileceği yer
# ------------------------------------------------------------

background_analysis_path = os.path.join(
    project_dir,
    "background_brain_content.csv"
)


# ------------------------------------------------------------
# Analiz sonuçlarını CSV dosyasına kaydet
# ------------------------------------------------------------

background_brain_df.to_csv(
    background_analysis_path,
    index=False
)


# ============================================================
# KAYDIN BAŞARILI OLDUĞUNU KONTROL ET
# ============================================================

print("Background brain-content analizi kaydedildi! ✅")

print("\nDosya yolu:")
print(background_analysis_path)

print("\nKaydedilen kayıt sayısı:")
print(len(background_brain_df))

print("\nDosya mevcut mu?:")
print(os.path.exists(background_analysis_path))


# Dosyanın boyutunu da kontrol edelim
file_size_mb = os.path.getsize(
    background_analysis_path
) / (1024 ** 2)

print(
    f"\nDosya boyutu: {file_size_mb:.2f} MB"
)

In [ ]:
# ============================================================
# 74. HÜCRE — BACKGROUND ANALİZİNİ DRIVE'DAN GERİ YÜKLE
#
# AMAÇ:
# Dün uzun sürede hesapladığımız brain-content sonuçlarını
# tekrar hesaplamak yerine Google Drive'dan yüklemek.
# ============================================================

import os
import pandas as pd


background_analysis_path = os.path.join(
    project_dir,
    "background_brain_content.csv"
)


# ------------------------------------------------------------
# CSV dosyasını yükle
# ------------------------------------------------------------

background_brain_df = pd.read_csv(
    background_analysis_path
)


# ============================================================
# KONTROL
# ============================================================

print("Background analizi yüklendi! ✅")
print("=" * 50)

print(
    "Toplam kayıt:",
    len(background_brain_df)
)

print("\nSütunlar:")
print(
    background_brain_df.columns.tolist()
)

print("\nİlk 5 kayıt:")
display(
    background_brain_df.head()
)


print("\nBrain-content özeti:")
print(
    background_brain_df["brain_content"].describe()
)

In [ ]:
# ============================================================
# SESSION RESTORE
#
# Colab runtime kapandığında RAM'deki değişkenler silinir.
# Fakat Google Drive'a kaydettiğimiz dosyalar silinmez.
#
# Bu hücre:
# 1) Google Drive'ı yeniden bağlar.
# 2) Proje klasörünün yolunu yeniden tanımlar.
# 3) Dün kaydettiğimiz CSV'nin gerçekten var olduğunu kontrol eder.
# ============================================================

from google.colab import drive
import os
import pandas as pd


# Google Drive'ı Colab'a bağla
drive.mount("/content/drive")


# ------------------------------------------------------------
# PROJE KLASÖRÜ
# ------------------------------------------------------------

project_dir = "/content/drive/MyDrive/NeuroRobust"


# Dün kaydettiğimiz uzun analiz
background_analysis_path = os.path.join(
    project_dir,
    "background_brain_content.csv"
)


# ------------------------------------------------------------
# KONTROL
# ------------------------------------------------------------

print("\nSESSION RESTORED! ✅")
print("=" * 50)

print("Project folder:")
print(project_dir)

print("\nBackground analysis mevcut mu?:")
print(
    os.path.exists(background_analysis_path)
)

In [ ]:
# ============================================================
# 75A — TRAIN SLICE INDEX'İ DRIVE'DAN GERİ YÜKLE
#
# AMAÇ:
# Runtime yeniden başladığı için RAM'den silinen train_slice_df
# değişkenini, daha önce Drive'a kaydettiğimiz CSV dosyasından
# tekrar oluşturmak.
#
# Bu işlem slice analizini TEKRAR YAPMAZ.
# Sadece hazır CSV dosyasını belleğe yükler.
# ============================================================

import os
import pandas as pd


# Daha önce oluşturup Drive'a kaydettiğimiz TRAIN slice index
train_index_path = os.path.join(
    project_dir,
    "train_slice_index.csv"
)


# ------------------------------------------------------------
# Dosyanın gerçekten Drive'da olup olmadığını kontrol et
# ------------------------------------------------------------

print("Train index dosyası mevcut mu?:")
print(os.path.exists(train_index_path))


# ------------------------------------------------------------
# CSV dosyasını RAM'e yükle
# ------------------------------------------------------------

train_slice_df = pd.read_csv(
    train_index_path
)


# ============================================================
# KONTROLLER
# ============================================================

print("\nTRAIN slice index yüklendi! ✅")
print("=" * 50)

print(
    "Toplam slice:",
    len(train_slice_df)
)

print(
    "Tumor slice:",
    (train_slice_df["has_tumor"] == 1).sum()
)

print(
    "Background slice:",
    (train_slice_df["has_tumor"] == 0).sum()
)

print("\nSütunlar:")
print(
    train_slice_df.columns.tolist()
)

print("\nİlk 5 kayıt:")
display(
    train_slice_df.head()
)

In [ ]:
# ============================================================
# 75. HÜCRE — CLEAN TRAINING POOL
#
# AMAÇ:
# Çok az beyin dokusu içeren background slice'ları training
# setinden çıkarmak.
#
# Tumor-containing slice'lara dokunmuyoruz.
#
# Background için:
#
#       brain_content >= %5
#
# koşulunu kullanıyoruz.
# ============================================================


BRAIN_CONTENT_THRESHOLD = 5.0


# ============================================================
# 1. KULLANIŞLI BACKGROUND SLICELARI
# ============================================================

useful_background_df = background_brain_df[
    background_brain_df["brain_content"]
    >= BRAIN_CONTENT_THRESHOLD
].copy()


print("Background filtering")
print("=" * 55)

print(
    "Original background slices:",
    len(background_brain_df)
)

print(
    "Useful background slices:",
    len(useful_background_df)
)

print(
    "Removed low-content slices:",
    len(background_brain_df)
    - len(useful_background_df)
)


# ============================================================
# 2. GEREKLİ SÜTUNLARI KORU
#
# background_brain_df:
# patient_id | slice_index | brain_content
#
# Generator için ayrıca:
# has_tumor ve tumor_pixels
# sütunlarını ekliyoruz.
# ============================================================

useful_background_df["has_tumor"] = 0
useful_background_df["tumor_pixels"] = 0


# Generator'ın ihtiyaç duyduğu sütunlar
useful_background_df = useful_background_df[
    [
        "patient_id",
        "slice_index",
        "has_tumor",
        "tumor_pixels"
    ]
]


# ============================================================
# 3. TÜM TUMOR SLICELARI
# ============================================================

tumor_training_df = train_slice_df[
    train_slice_df["has_tumor"] == 1
][
    [
        "patient_id",
        "slice_index",
        "has_tumor",
        "tumor_pixels"
    ]
].copy()


# ============================================================
# 4. TUMOR + KALİTELİ BACKGROUND
# ============================================================

clean_train_df = pd.concat(
    [
        tumor_training_df,
        useful_background_df
    ],
    ignore_index=True
)


# Karıştır
clean_train_df = (
    clean_train_df
    .sample(
        frac=1,
        random_state=42
    )
    .reset_index(drop=True)
)


# ============================================================
# SONUÇ
# ============================================================

tumor_count = (
    clean_train_df["has_tumor"] == 1
).sum()

background_count = (
    clean_train_df["has_tumor"] == 0
).sum()


print("\nCLEAN TRAINING POOL")
print("=" * 55)

print(
    "Total slices:",
    len(clean_train_df)
)

print(
    "Tumor slices:",
    tumor_count
)

print(
    "Background slices:",
    background_count
)


print(
    "\nTumor ratio:",
    f"{tumor_count / len(clean_train_df) * 100:.2f}%"
)

print(
    "Background ratio:",
    f"{background_count / len(clean_train_df) * 100:.2f}%"
)

### Background Slice Filtering and Final Training Pool

To reduce uninformative training samples, background-only slices with less than **5% brain content** were removed from the training set.

- Original background slices: **78,558**
- Background slices retained: **49,977**
- Low-content slices removed: **28,581**
- Tumor-containing slices: **57,067**

After filtering, the final training pool contains **107,044 slices**, consisting of **53.31% tumor-containing** and **46.69% background-only** slices.

This filtering step removes nearly empty axial slices while preserving informative non-tumor brain regions. The resulting dataset provides a more meaningful and approximately balanced set of samples for segmentation model training.

In [ ]:
# ============================================================
# 75B — CLEAN TRAINING POOL'U YENİDEN OLUŞTUR
#
# Runtime yeniden başladığı için clean_train_df RAM'den silindi.
# Daha önce Drive'a kaydettiğimiz analiz sonuçlarını kullanarak
# aynı training pool'u tekrar oluşturuyoruz.
# ============================================================

import pandas as pd


# ------------------------------------------------------------
# 1. TRAIN slice index'i kontrol et
# ------------------------------------------------------------

print("TRAIN slice index:", len(train_slice_df))


# ------------------------------------------------------------
# 2. Tüm tumor slice'ları al
# ------------------------------------------------------------

tumor_training_df = train_slice_df[
    train_slice_df["has_tumor"] == 1
].copy()


# ------------------------------------------------------------
# 3. Background analizinden faydalı slice'ları seç
#
# En az %5 brain-content içeren background slice'ları tutuyoruz.
# ------------------------------------------------------------

useful_background_df = background_brain_df[
    background_brain_df["brain_content"] >= 5
].copy()


# ------------------------------------------------------------
# 4. TRAIN index bilgileriyle eşleştir
# ------------------------------------------------------------

useful_background_training_df = train_slice_df[
    train_slice_df["has_tumor"] == 0
].merge(
    useful_background_df[
        ["patient_id", "slice_index"]
    ],
    on=["patient_id", "slice_index"],
    how="inner"
)


# ------------------------------------------------------------
# 5. Tumor + kaliteli background örneklerini birleştir
# ------------------------------------------------------------

clean_train_df = pd.concat(
    [
        tumor_training_df,
        useful_background_training_df
    ],
    ignore_index=True
)


# ============================================================
# KONTROL
# ============================================================

tumor_count = (
    clean_train_df["has_tumor"] == 1
).sum()

background_count = (
    clean_train_df["has_tumor"] == 0
).sum()


print("\nCLEAN TRAINING POOL RESTORED ✅")
print("=" * 50)

print("Total slices      :", len(clean_train_df))
print("Tumor slices      :", tumor_count)
print("Background slices :", background_count)

print(
    "\nTumor ratio       :",
    f"{tumor_count / len(clean_train_df) * 100:.2f}%"
)

print(
    "Background ratio  :",
    f"{background_count / len(clean_train_df) * 100:.2f}%"
)

In [ ]:
# ============================================================
# SESSION RESTORE — TRAINING DATA
#
# Colab runtime yeniden başladığında gerekli DataFrame'leri
# Drive'daki kayıtlı CSV dosyalarından geri yükler.
#
# Ağır analizler TEKRAR yapılmaz.
# ============================================================

import os
import pandas as pd


# ------------------------------------------------------------
# 1. PROJE KLASÖRÜ
# ------------------------------------------------------------

project_dir = "/content/drive/MyDrive/NeuroRobust"


# ------------------------------------------------------------
# 2. KAYITLI CSV DOSYALARINI YÜKLE
# ------------------------------------------------------------

train_index_path = os.path.join(
    project_dir,
    "train_slice_index.csv"
)

background_analysis_path = os.path.join(
    project_dir,
    "background_brain_content.csv"
)


train_slice_df = pd.read_csv(
    train_index_path
)

background_brain_df = pd.read_csv(
    background_analysis_path
)


print("Kayıtlı analiz dosyaları yüklendi! ✅")


# ------------------------------------------------------------
# 3. TUMOR SLICE'LARI
# ------------------------------------------------------------

tumor_training_df = train_slice_df[
    train_slice_df["has_tumor"] == 1
].copy()


# ------------------------------------------------------------
# 4. FAYDALI BACKGROUND SLICE'LARI
#
# En az %5 brain content bulunan background görüntülerini tut.
# ------------------------------------------------------------

useful_background_df = background_brain_df[
    background_brain_df["brain_content"] >= 5
].copy()


# ------------------------------------------------------------
# 5. TRAIN INDEX İLE EŞLEŞTİR
# ------------------------------------------------------------

useful_background_training_df = train_slice_df[
    train_slice_df["has_tumor"] == 0
].merge(
    useful_background_df[
        ["patient_id", "slice_index"]
    ],
    on=["patient_id", "slice_index"],
    how="inner"
)


# ------------------------------------------------------------
# 6. CLEAN TRAINING POOL
# ------------------------------------------------------------

clean_train_df = pd.concat(
    [
        tumor_training_df,
        useful_background_training_df
    ],
    ignore_index=True
)


# ============================================================
# 7. SON KONTROL
# ============================================================

tumor_count = (
    clean_train_df["has_tumor"] == 1
).sum()

background_count = (
    clean_train_df["has_tumor"] == 0
).sum()


print("\n" + "=" * 55)
print("SESSION RESTORED ✅")
print("=" * 55)

print("Train index        :", len(train_slice_df))
print("Background analysis:", len(background_brain_df))

print("\nCLEAN TRAINING POOL")
print("-" * 55)

print("Total slices       :", len(clean_train_df))
print("Tumor slices       :", tumor_count)
print("Background slices  :", background_count)

print(
    "Tumor ratio        :",
    f"{100 * tumor_count / len(clean_train_df):.2f}%"
)

print(
    "Background ratio   :",
    f"{100 * background_count / len(clean_train_df):.2f}%"
)

In [ ]:
# 76 ÖNCESİ SESSION CHECK

required_items = [
    "clean_train_df",
    "load_patient_volumes",
    "normalize_volume"
]

print("76 öncesi kontrol")
print("=" * 40)

for item in required_items:
    if item in globals():
        print(f"✅ {item}")
    else:
        print(f"❌ {item}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

project_dir = "/content/drive/MyDrive/NeuroRobust"

print("Proje klasörü:", os.path.exists(project_dir))
print("\nDosyalar:")
print(os.listdir(project_dir))

In [ ]:
# ============================================================
# MRI FUNCTIONS RESTORE
# normalize_volume + load_patient_volumes
# ============================================================

import os
import numpy as np
import nibabel as nib

from functools import lru_cache


# ============================================================
# 1. DATASET PATH
# ============================================================

training_data_dir = os.path.join(
    project_dir,
    "dataset",
    "BraTS2021_Training_Data"
)

print("Dataset bulundu mu?:", os.path.exists(training_data_dir))


# ============================================================
# 2. NORMALIZATION FUNCTION
# ============================================================

def normalize_volume(volume):

    # MRI background = 0
    # Sadece beyin bölgesini normalization hesabına katıyoruz.
    brain_mask = volume > 0

    if np.sum(brain_mask) == 0:
        return np.zeros_like(
            volume,
            dtype=np.float32
        )

    mean = volume[brain_mask].mean()
    std = volume[brain_mask].std()

    normalized = np.zeros_like(
        volume,
        dtype=np.float32
    )

    if std > 0:
        normalized[brain_mask] = (
            volume[brain_mask] - mean
        ) / std

    return normalized


# ============================================================
# 3. PATIENT VOLUME LOADER
# ============================================================

@lru_cache(maxsize=4)
def load_patient_volumes(patient_id):

    patient_folder = os.path.join(
        training_data_dir,
        patient_id
    )

    # -------------------------
    # MRI modalities
    # -------------------------

    t1 = nib.load(
        os.path.join(
            patient_folder,
            f"{patient_id}_t1.nii.gz"
        )
    ).get_fdata().astype(np.float32)

    t1ce = nib.load(
        os.path.join(
            patient_folder,
            f"{patient_id}_t1ce.nii.gz"
        )
    ).get_fdata().astype(np.float32)

    t2 = nib.load(
        os.path.join(
            patient_folder,
            f"{patient_id}_t2.nii.gz"
        )
    ).get_fdata().astype(np.float32)

    flair = nib.load(
        os.path.join(
            patient_folder,
            f"{patient_id}_flair.nii.gz"
        )
    ).get_fdata().astype(np.float32)


    # -------------------------
    # Ground-truth segmentation
    # -------------------------

    mask = nib.load(
        os.path.join(
            patient_folder,
            f"{patient_id}_seg.nii.gz"
        )
    ).get_fdata().astype(np.int16)


    # -------------------------
    # Normalize MRI modalities
    # Mask normalize EDİLMEZ
    # -------------------------

    t1 = normalize_volume(t1)
    t1ce = normalize_volume(t1ce)
    t2 = normalize_volume(t2)
    flair = normalize_volume(flair)


    return (
        t1,
        t1ce,
        t2,
        flair,
        mask
    )


# ============================================================
# 4. FINAL CHECK
# ============================================================

print("\nMRI FUNCTIONS RESTORED ✅")
print("=" * 45)

print(
    "normalize_volume:",
    "✅" if "normalize_volume" in globals() else "❌"
)

print(
    "load_patient_volumes:",
    "✅" if "load_patient_volumes" in globals() else "❌"
)

print(
    "clean_train_df:",
    "✅" if "clean_train_df" in globals() else "❌"
)

In [ ]:
# ============================================================
# 76. HÜCRE — FILTERED TRAINING POOL VISUAL SANITY CHECK
#
# AMAÇ:
# Model eğitimine geçmeden önce temizlediğimiz TRAIN verisini
# son kez görsel olarak kontrol etmek.
#
# clean_train_df içerisinden:
#   - 4 tane TUMOR slice
#   - 4 tane BACKGROUND slice
#
# seçiyoruz.
#
# Her örnek için 3 görüntü göstereceğiz:
#   1) FLAIR MRI
#   2) Binary Ground Truth Mask
#   3) FLAIR + Mask Overlay
#
# Özellikle background görüntülerinin artık çok boş olmamasını
# kontrol edeceğiz.
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# 1. TUMOR ÖRNEKLERİNİ SEÇ
# ============================================================

# has_tumor = 1 olan slice'lar tümör içeren slice'lardır.
# Bunların arasından rastgele 4 örnek seçiyoruz.
#
# random_state=42 kullanmamızın nedeni:
# Kodu tekrar çalıştırdığımızda aynı örnekleri seçebilmek.

tumor_examples = clean_train_df[
    clean_train_df["has_tumor"] == 1
].sample(
    n=4,
    random_state=42
)


# ============================================================
# 2. BACKGROUND ÖRNEKLERİNİ SEÇ
# ============================================================

# has_tumor = 0 olan slice'lar tümör içermeyen slice'lardır.
#
# Bunlar artık %5 brain-content filtresinden geçmiş
# background örnekleridir.

background_examples = clean_train_df[
    clean_train_df["has_tumor"] == 0
].sample(
    n=4,
    random_state=42
)


# ============================================================
# 3. İKİ GRUBU BİRLEŞTİR
# ============================================================

# İlk 4 satır tumor,
# sonraki 4 satır background olacak.

visual_examples = pd.concat(
    [
        tumor_examples,
        background_examples
    ],
    ignore_index=True
)


print("Final visual sanity check başlıyor...")
print("=" * 55)

print("Tumor sample sayısı     :", len(tumor_examples))
print("Background sample sayısı:", len(background_examples))


# ============================================================
# 4. GRAFİK ALANINI OLUŞTUR
# ============================================================
#
# 8 örneğimiz olduğu için 8 satır oluşturuyoruz.
#
# Her satır:
#
# FLAIR | Ground Truth | Overlay
#
# şeklinde 3 sütundan oluşacak.
# ============================================================

fig, axes = plt.subplots(
    nrows=8,
    ncols=3,
    figsize=(12, 28)
)


# ============================================================
# 5. SEÇİLEN 8 ÖRNEĞİ TEK TEK İŞLE
# ============================================================

for i, row in visual_examples.iterrows():

    # Hastanın ID'sini al.
    patient_id = row["patient_id"]

    # Hangi axial slice olduğunu al.
    slice_index = int(row["slice_index"])

    # 1 = tumor
    # 0 = background
    has_tumor = int(row["has_tumor"])


    # ========================================================
    # HASTANIN MRI VOLUME'LARINI YÜKLE
    # ========================================================
    #
    # Daha önce oluşturduğumuz load_patient_volumes()
    # fonksiyonunu kullanıyoruz.
    #
    # Fonksiyon:
    # T1, T1ce, T2, FLAIR ve segmentation mask döndürüyor.
    # ========================================================

    (
        t1_volume,
        t1ce_volume,
        t2_volume,
        flair_volume,
        mask_volume
    ) = load_patient_volumes(patient_id)


    # ========================================================
    # İLGİLİ 2D SLICE'I AL
    # ========================================================

    flair_slice = flair_volume[
        :,
        :,
        slice_index
    ]

    mask_slice = mask_volume[
        :,
        :,
        slice_index
    ]


    # ========================================================
    # MULTI-CLASS MASK → BINARY MASK
    # ========================================================
    #
    # BraTS maskesinde:
    #
    # 0 = background
    # 1, 2, 4 = farklı tumor bölgeleri
    #
    # Biz şu anda binary segmentation yapacağımız için:
    #
    # 0       → 0
    # 1,2,4   → 1
    #
    # şeklinde dönüştürüyoruz.
    # ========================================================

    binary_mask = (
        mask_slice > 0
    ).astype(np.float32)


    # ========================================================
    # BRAIN CONTENT HESAPLA
    # ========================================================
    #
    # Bu değer özellikle background örnekleri için önemli.
    #
    # Filtrelemeden sonra background slice'ların
    # brain-content değerinin >= %5 olmasını bekliyoruz.
    # ========================================================

    background_value = flair_slice[0, 0]

    brain_pixels = np.sum(
        np.abs(
            flair_slice - background_value
        ) > 0.05
    )

    brain_content = (
        brain_pixels
        / flair_slice.size
        * 100
    )


    # ========================================================
    # TUMOR PIXEL SAYISI
    # ========================================================

    tumor_pixels = int(
        binary_mask.sum()
    )


    # ========================================================
    # 1. SÜTUN — FLAIR MRI
    # ========================================================

    axes[i, 0].imshow(
        flair_slice,
        cmap="gray"
    )

    axes[i, 0].set_title(
        f"{patient_id}\n"
        f"FLAIR — Slice {slice_index}\n"
        f"Brain Content: {brain_content:.2f}%"
    )

    axes[i, 0].axis("off")


    # ========================================================
    # 2. SÜTUN — BINARY GROUND TRUTH
    # ========================================================

    axes[i, 1].imshow(
        binary_mask,
        cmap="gray",
        vmin=0,
        vmax=1
    )

    axes[i, 1].set_title(
        f"Ground Truth\n"
        f"Tumor Pixels: {tumor_pixels}"
    )

    axes[i, 1].axis("off")


    # ========================================================
    # 3. SÜTUN — FLAIR + TUMOR OVERLAY
    # ========================================================

    # Önce MRI görüntüsünü çiziyoruz.
    axes[i, 2].imshow(
        flair_slice,
        cmap="gray"
    )


    # Maskenin 0 olan bölgelerini gizliyoruz.
    #
    # Böylece overlay sadece tumor bulunan bölgelerde
    # görünecek.
    tumor_overlay = np.ma.masked_where(
        binary_mask == 0,
        binary_mask
    )


    # Tumor varsa MRI üzerine yarı saydam şekilde ekle.
    axes[i, 2].imshow(
        tumor_overlay,
        cmap="plasma",
        alpha=0.45
    )


    # ========================================================
    # SLICE TÜRÜNÜ BELİRLE
    # ========================================================

    if has_tumor == 1:
        label = "TUMOR"

    else:
        label = "BACKGROUND"


    axes[i, 2].set_title(
        f"Overlay — {label}"
    )

    axes[i, 2].axis("off")


# ============================================================
# 6. GENEL BAŞLIK
# ============================================================

fig.suptitle(
    "Filtered Training Pool — Final Visual Sanity Check",
    fontsize=16,
    y=1.01
)


# Grafiklerin birbirine girmesini engeller.
plt.tight_layout()


# Sonucu ekrana getir.
plt.show()

### Final Visual Sanity Check

A final visual inspection was performed on samples from the filtered training pool before model development.

The selected tumor and background slices were compared using the **FLAIR image, binary ground-truth mask, and tumor overlay**. Background samples correctly contained **0 tumor pixels**, while still preserving meaningful brain tissue after the brain-content filtering step.

The inspected background examples had brain-content values above the **5% filtering threshold**, confirming that nearly empty and uninformative slices were successfully removed.

**Result:** The preprocessing and filtering pipeline produced a cleaner and more informative training dataset. The data is now ready for the segmentation model development stage.

In [ ]:
# ============================================================
# 77. HÜCRE — SEGMENTATION METRICS
#
# Binary brain tumor segmentation için:
#
# Dice Score:
# Tahmin edilen tümör alanı ile gerçek tümör alanının
# ne kadar örtüştüğünü ölçer.
#
# IoU (Intersection over Union):
# Tahmin ve gerçek maskenin kesişiminin birleşimlerine
# oranını ölçer.
#
# Değerler 0-1 arasındadır.
# 1'e yaklaştıkça tahmin daha iyidir.
# ============================================================

import tensorflow as tf


# ------------------------------------------------------------
# DICE COEFFICIENT
# ------------------------------------------------------------

def dice_coefficient(y_true, y_pred, smooth=1e-6):

    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    # Batch içerisindeki tüm pikseller üzerinden hesapla
    intersection = tf.reduce_sum(
        y_true * y_pred,
        axis=[1, 2, 3]
    )

    denominator = (
        tf.reduce_sum(y_true, axis=[1, 2, 3]) +
        tf.reduce_sum(y_pred, axis=[1, 2, 3])
    )

    dice = (
        (2.0 * intersection + smooth)
        /
        (denominator + smooth)
    )

    return tf.reduce_mean(dice)


# ------------------------------------------------------------
# IoU
# ------------------------------------------------------------

def iou_metric(y_true, y_pred, smooth=1e-6):

    y_true = tf.cast(y_true, tf.float32)

    # Metric için prediction'ı binary maskeye çevir
    y_pred = tf.cast(
        y_pred > 0.5,
        tf.float32
    )

    intersection = tf.reduce_sum(
        y_true * y_pred,
        axis=[1, 2, 3]
    )

    union = (
        tf.reduce_sum(y_true, axis=[1, 2, 3]) +
        tf.reduce_sum(y_pred, axis=[1, 2, 3]) -
        intersection
    )

    iou = (
        intersection + smooth
    ) / (
        union + smooth
    )

    return tf.reduce_mean(iou)


print("Segmentation metrics hazır! ✅")
print("• Dice coefficient")
print("• IoU metric")

In [ ]:
# ============================================================
# 78. HÜCRE — U-NET MODEL ARCHITECTURE
#
# INPUT:
# 128 x 128 x 4
# (T1, T1ce, T2, FLAIR)
#
# OUTPUT:
# 128 x 128 x 1
# Binary tumor segmentation mask
# ============================================================

import tensorflow as tf
from tensorflow.keras import layers, Model


# ------------------------------------------------------------
# CONVOLUTION BLOCK
# ------------------------------------------------------------

def conv_block(x, filters):

    x = layers.Conv2D(
        filters,
        3,
        padding="same",
        kernel_initializer="he_normal"
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.Conv2D(
        filters,
        3,
        padding="same",
        kernel_initializer="he_normal"
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    return x


# ------------------------------------------------------------
# U-NET
# ------------------------------------------------------------

def build_unet(input_shape=(128, 128, 4)):

    inputs = layers.Input(
        shape=input_shape,
        name="MRI_Input"
    )

    # ========================
    # ENCODER
    # ========================

    c1 = conv_block(inputs, 32)
    p1 = layers.MaxPooling2D((2, 2))(c1)

    c2 = conv_block(p1, 64)
    p2 = layers.MaxPooling2D((2, 2))(c2)

    c3 = conv_block(p2, 128)
    p3 = layers.MaxPooling2D((2, 2))(c3)

    c4 = conv_block(p3, 256)
    p4 = layers.MaxPooling2D((2, 2))(c4)


    # ========================
    # BOTTLENECK
    # ========================

    bottleneck = conv_block(p4, 512)


    # ========================
    # DECODER
    # ========================

    u4 = layers.Conv2DTranspose(
        256, 2, strides=2, padding="same"
    )(bottleneck)

    u4 = layers.Concatenate()([u4, c4])
    c5 = conv_block(u4, 256)


    u3 = layers.Conv2DTranspose(
        128, 2, strides=2, padding="same"
    )(c5)

    u3 = layers.Concatenate()([u3, c3])
    c6 = conv_block(u3, 128)


    u2 = layers.Conv2DTranspose(
        64, 2, strides=2, padding="same"
    )(c6)

    u2 = layers.Concatenate()([u2, c2])
    c7 = conv_block(u2, 64)


    u1 = layers.Conv2DTranspose(
        32, 2, strides=2, padding="same"
    )(c7)

    u1 = layers.Concatenate()([u1, c1])
    c8 = conv_block(u1, 32)


    # ========================
    # OUTPUT
    # ========================

    outputs = layers.Conv2D(
        1,
        kernel_size=1,
        activation="sigmoid",
        name="Tumor_Mask"
    )(c8)


    return Model(
        inputs=inputs,
        outputs=outputs,
        name="NeuroRobust_UNet"
    )


# ============================================================
# MODELİ OLUŞTUR
# ============================================================

model = build_unet()

print("U-Net başarıyla oluşturuldu! ✅")
print()

model.summary()

## U-Net Architecture for Brain Tumor Segmentation

A **2D U-Net** architecture was constructed for binary brain tumor segmentation using multi-modal MRI data.

### Model Configuration

- **Input:** `128 × 128 × 4`
  - T1
  - T1ce
  - T2
  - FLAIR
- **Output:** `128 × 128 × 1` binary tumor probability mask
- **Total parameters:** **7,772,161**
- **Trainable parameters:** **7,766,273**
- **Non-trainable parameters:** **5,888**

The encoder progressively extracts higher-level spatial features while reducing resolution from `128×128` to `8×8`. The decoder then reconstructs the segmentation mask using upsampling and **skip connections** from the encoder.

The final `1×1` convolution uses a **sigmoid activation**, producing a tumor probability for each pixel.

**Result:** The U-Net was successfully constructed with matching input/output dimensions and is ready for compilation and pipeline validation.

In [ ]:
# ============================================================
# 79. HÜCRE — MODEL COMPILE
# ============================================================

import tensorflow as tf


# ------------------------------------------------------------
# LOSS
#
# Binary segmentation yaptığımız için başlangıçta
# Binary Crossentropy kullanıyoruz.
#
# Dice ise ayrıca metric olarak takip edilecek.
# ------------------------------------------------------------

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4
    ),

    loss=tf.keras.losses.BinaryCrossentropy(),

    metrics=[
        dice_coefficient,
        iou_metric
    ]
)


print("Model compile edildi! ✅")
print("=" * 45)
print("Optimizer     : Adam")
print("Learning rate : 1e-4")
print("Loss          : Binary Crossentropy")
print("Metrics       : Dice + IoU")

In [ ]:
# ============================================================
# 79A — FINAL TRAINING GENERATOR
#
# AMAÇ:
# Filtreleme sonrası oluşturduğumuz clean_train_df üzerinden
# U-Net'e batch halinde veri sağlamak.
#
# INPUT:
#   (batch_size, 128, 128, 4)
#
# Kanallar:
#   0 -> T1
#   1 -> T1ce
#   2 -> T2
#   3 -> FLAIR
#
# OUTPUT:
#   (batch_size, 128, 128, 1)
#
# Output = binary tumor segmentation mask
# ============================================================

import numpy as np
import tensorflow as tf
import cv2


# ============================================================
# MODEL / GENERATOR AYARLARI
# ============================================================

MODEL_HEIGHT = 128
MODEL_WIDTH = 128
NUM_CHANNELS = 4
BATCH_SIZE = 8


# ============================================================
# FINAL DATA GENERATOR
# ============================================================

class FinalBraTSDataGenerator(tf.keras.utils.Sequence):

    def __init__(
        self,
        dataframe,
        batch_size=8,
        shuffle=True
    ):

        # DataFrame'in indexlerini temizle.
        self.dataframe = dataframe.reset_index(drop=True)

        self.batch_size = batch_size
        self.shuffle = shuffle

        # Her slice için bir index oluştur.
        self.indices = np.arange(
            len(self.dataframe)
        )

        # İlk epoch başlamadan önce gerekiyorsa karıştır.
        self.on_epoch_end()


    # ========================================================
    # TOPLAM BATCH SAYISI
    # ========================================================

    def __len__(self):

        return int(
            np.ceil(
                len(self.dataframe)
                / self.batch_size
            )
        )


    # ========================================================
    # BİR BATCH OLUŞTUR
    # ========================================================

    def __getitem__(self, batch_index):

        # Bu batch'in başlangıç noktası.
        start = (
            batch_index
            * self.batch_size
        )

        # Bu batch'in bitiş noktası.
        end = min(
            start + self.batch_size,
            len(self.dataframe)
        )

        # Bu batch'e ait DataFrame indexleri.
        batch_indices = self.indices[
            start:end
        ]

        current_batch_size = len(
            batch_indices
        )


        # ====================================================
        # INPUT ARRAY
        #
        # Örneğin batch_size=8 ise:
        #
        # X.shape = (8, 128, 128, 4)
        # ====================================================

        X = np.zeros(
            (
                current_batch_size,
                MODEL_HEIGHT,
                MODEL_WIDTH,
                NUM_CHANNELS
            ),
            dtype=np.float32
        )


        # ====================================================
        # MASK ARRAY
        #
        # Y.shape = (8, 128, 128, 1)
        # ====================================================

        Y = np.zeros(
            (
                current_batch_size,
                MODEL_HEIGHT,
                MODEL_WIDTH,
                1
            ),
            dtype=np.float32
        )


        # ====================================================
        # BATCH İÇİNDEKİ SLICELARI HAZIRLA
        # ====================================================

        for i, row_index in enumerate(batch_indices):

            row = self.dataframe.iloc[
                row_index
            ]

            patient_id = row[
                "patient_id"
            ]

            slice_index = int(
                row["slice_index"]
            )


            # ------------------------------------------------
            # Hastanın 3D MRI volume'larını yükle.
            #
            # load_patient_volumes() daha önce oluşturduğumuz
            # cache destekli fonksiyondur.
            # ------------------------------------------------

            (
                t1_volume,
                t1ce_volume,
                t2_volume,
                flair_volume,
                mask_volume
            ) = load_patient_volumes(
                patient_id
            )


            # =================================================
            # İLGİLİ 2D SLICE'LARI AL
            # =================================================

            t1_slice = t1_volume[
                :,
                :,
                slice_index
            ]

            t1ce_slice = t1ce_volume[
                :,
                :,
                slice_index
            ]

            t2_slice = t2_volume[
                :,
                :,
                slice_index
            ]

            flair_slice = flair_volume[
                :,
                :,
                slice_index
            ]

            mask_slice = mask_volume[
                :,
                :,
                slice_index
            ]


            # =================================================
            # 240x240 -> 128x128 RESIZE
            # =================================================
            #
            # MRI görüntülerinde bilinear interpolation.
            #
            # Maskede ise MUTLAKA nearest-neighbor kullanıyoruz.
            # Böylece maskede yeni ara sınıf değerleri oluşmaz.
            # =================================================

            t1_slice = cv2.resize(
                t1_slice,
                (MODEL_WIDTH, MODEL_HEIGHT),
                interpolation=cv2.INTER_LINEAR
            )

            t1ce_slice = cv2.resize(
                t1ce_slice,
                (MODEL_WIDTH, MODEL_HEIGHT),
                interpolation=cv2.INTER_LINEAR
            )

            t2_slice = cv2.resize(
                t2_slice,
                (MODEL_WIDTH, MODEL_HEIGHT),
                interpolation=cv2.INTER_LINEAR
            )

            flair_slice = cv2.resize(
                flair_slice,
                (MODEL_WIDTH, MODEL_HEIGHT),
                interpolation=cv2.INTER_LINEAR
            )


            # ------------------------------------------------
            # BraTS mask:
            #
            # 0 = background
            # 1,2,4 = tumor bölgeleri
            #
            # Binary segmentation yaptığımız için:
            #
            # > 0  -> 1
            # == 0 -> 0
            # ------------------------------------------------

            binary_mask = (
                mask_slice > 0
            ).astype(np.float32)


            binary_mask = cv2.resize(
                binary_mask,
                (MODEL_WIDTH, MODEL_HEIGHT),
                interpolation=cv2.INTER_NEAREST
            )


            # =================================================
            # 4 MRI MODALITY'SİNİ CHANNEL OLARAK BİRLEŞTİR
            # =================================================
            #
            # Sonuç:
            #
            # (128,128,4)
            # =================================================

            X[i] = np.stack(
                [
                    t1_slice,
                    t1ce_slice,
                    t2_slice,
                    flair_slice
                ],
                axis=-1
            )


            # Maskeye channel dimension ekle.
            #
            # (128,128)
            #       ↓
            # (128,128,1)

            Y[i, :, :, 0] = binary_mask


        return X, Y


    # ========================================================
    # EPOCH SONUNDA VERİYİ KARIŞTIR
    # ========================================================

    def on_epoch_end(self):

        if self.shuffle:

            np.random.shuffle(
                self.indices
            )


# ============================================================
# FINAL TRAIN GENERATOR OLUŞTUR
# ============================================================

train_generator = FinalBraTSDataGenerator(
    dataframe=clean_train_df,
    batch_size=BATCH_SIZE,
    shuffle=True
)


# ============================================================
# KONTROL
# ============================================================

print("Final train generator oluşturuldu! ✅")
print("=" * 55)

print(
    "Training slice sayısı:",
    len(clean_train_df)
)

print(
    "Batch size:",
    BATCH_SIZE
)

print(
    "Toplam batch sayısı:",
    len(train_generator)
)

In [ ]:
# ============================================================
# 80. HÜCRE — FORWARD PASS / PIPELINE TEST
#
# AMAÇ:
# Eğitime başlamadan önce gerçek bir batch'i U-Net'e vererek:
#
#   Generator → MRI Batch → U-Net → Prediction
#
# zincirinin hatasız çalıştığını doğrulamak.
#
# MODEL EĞİTİLMEZ.
# Sadece ileri yönlü tahmin yapılır.
# ============================================================

import numpy as np


# ------------------------------------------------------------
# 1. GENERATOR'DAN BİR BATCH AL
# ------------------------------------------------------------

X_batch, Y_batch = train_generator[0]


print("INPUT BATCH")
print("=" * 45)

print("X shape :", X_batch.shape)
print("Y shape :", Y_batch.shape)

print("X dtype :", X_batch.dtype)
print("Y dtype :", Y_batch.dtype)


# ------------------------------------------------------------
# 2. MODEL İLE FORWARD PASS
# ------------------------------------------------------------

predictions = model.predict(
    X_batch,
    verbose=0
)


# ------------------------------------------------------------
# 3. OUTPUT KONTROL
# ------------------------------------------------------------

print("\nMODEL OUTPUT")
print("=" * 45)

print(
    "Prediction shape :",
    predictions.shape
)

print(
    "Prediction dtype :",
    predictions.dtype
)

print(
    "Prediction min   :",
    float(predictions.min())
)

print(
    "Prediction max   :",
    float(predictions.max())
)


# ------------------------------------------------------------
# 4. SHAPE UYUMUNU KONTROL ET
# ------------------------------------------------------------

shape_match = (
    predictions.shape == Y_batch.shape
)


print("\nPIPELINE CHECK")
print("=" * 45)

print(
    "Prediction / mask shape match:",
    "✅" if shape_match else "❌"
)


# ------------------------------------------------------------
# 5. NaN / Inf KONTROLÜ
# ------------------------------------------------------------

has_nan = np.isnan(predictions).any()
has_inf = np.isinf(predictions).any()

print(
    "NaN detected:",
    "❌ YES" if has_nan else "✅ NO"
)

print(
    "Inf detected:",
    "❌ YES" if has_inf else "✅ NO"
)


if shape_match and not has_nan and not has_inf:

    print(
        "\nForward pass başarılı! "
        "Training pipeline hazır. ✅"
    )

In [ ]:
# ============================================================
# 82. HÜCRE — VALIDATION GENERATOR
#
# Validation verisi:
# - Train'den tamamen ayrı hastalardan gelir.
# - Shuffle yapılmaz.
# - Background/tumor dengesi değiştirilmez.
# - Modelin gerçek validation dağılımındaki performansını ölçer.
# ============================================================

import os
import pandas as pd


# ------------------------------------------------------------
# 1. VALIDATION INDEX'İ DRIVE'DAN YÜKLE
# ------------------------------------------------------------

validation_index_path = os.path.join(
    project_dir,
    "validation_slice_index.csv"
)

validation_slice_df = pd.read_csv(
    validation_index_path
)


# ------------------------------------------------------------
# 2. VALIDATION GENERATOR
#
# Train için yazdığımız generator sınıfının aynısını
# kullanabiliriz.
# ------------------------------------------------------------

validation_generator = FinalBraTSDataGenerator(
    dataframe=validation_slice_df,
    batch_size=BATCH_SIZE,
    shuffle=False
)


# ------------------------------------------------------------
# 3. GENEL KONTROL
# ------------------------------------------------------------

val_tumor = (
    validation_slice_df["has_tumor"] == 1
).sum()

val_background = (
    validation_slice_df["has_tumor"] == 0
).sum()


print("Validation generator oluşturuldu! ✅")
print("=" * 55)

print(
    "Validation slice sayısı:",
    len(validation_slice_df)
)

print(
    "Tumor slices:",
    val_tumor
)

print(
    "Background slices:",
    val_background
)

print(
    "Batch size:",
    BATCH_SIZE
)

print(
    "Toplam batch sayısı:",
    len(validation_generator)
)


# ------------------------------------------------------------
# 4. İLK BATCH TESTİ
# ------------------------------------------------------------

X_val, Y_val = validation_generator[0]

print("\nFIRST VALIDATION BATCH")
print("=" * 55)

print("X shape:", X_val.shape)
print("Y shape:", Y_val.shape)

print(
    "Mask classes:",
    np.unique(Y_val)
)


# ------------------------------------------------------------
# 5. TRAIN / VALIDATION PATIENT LEAKAGE CHECK
# ------------------------------------------------------------

train_patients = set(
    clean_train_df["patient_id"].unique()
)

val_patients = set(
    validation_slice_df["patient_id"].unique()
)

overlap = train_patients.intersection(
    val_patients
)

print("\nPATIENT LEAKAGE CHECK")
print("=" * 55)

print(
    "Train patients:",
    len(train_patients)
)

print(
    "Validation patients:",
    len(val_patients)
)

print(
    "Overlapping patients:",
    len(overlap)
)

if len(overlap) == 0:
    print("Patient-level split korunuyor. ✅")
else:
    print("UYARI: Patient leakage bulundu! ❌")

### End-to-End Pipeline Validation

Before training, a forward-pass test was performed using a real batch from the final filtered training dataset.

- **Input batch:** `(8, 128, 128, 4)`
- **Ground-truth masks:** `(8, 128, 128, 1)`
- **Model predictions:** `(8, 128, 128, 1)`
- Prediction and ground-truth dimensions matched successfully.
- No `NaN` or infinite values were detected in the model output.
- Predicted values remained within the expected sigmoid output range.

**Result:** The complete pipeline from multi-modal MRI loading and preprocessing to U-Net prediction is functioning correctly. The model is ready for the training stage.

In [ ]:
# ============================================================
# 81. HÜCRE — BCE + DICE LOSS
#
# Binary tumor segmentation için combined loss:
#
#   Binary Crossentropy
#          +
#      Dice Loss
#
# BCE  -> pixel-level classification
# Dice -> tumor-mask overlap
# ============================================================

import tensorflow as tf


# ------------------------------------------------------------
# SOFT DICE LOSS
# ------------------------------------------------------------

def dice_loss(y_true, y_pred, smooth=1e-6):

    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    intersection = tf.reduce_sum(
        y_true * y_pred,
        axis=[1, 2, 3]
    )

    denominator = (
        tf.reduce_sum(y_true, axis=[1, 2, 3]) +
        tf.reduce_sum(y_pred, axis=[1, 2, 3])
    )

    dice = (
        2.0 * intersection + smooth
    ) / (
        denominator + smooth
    )

    return 1.0 - tf.reduce_mean(dice)


# ------------------------------------------------------------
# BCE + DICE
# ------------------------------------------------------------

bce = tf.keras.losses.BinaryCrossentropy()


def bce_dice_loss(y_true, y_pred):

    bce_value = bce(
        y_true,
        y_pred
    )

    dice_value = dice_loss(
        y_true,
        y_pred
    )

    return bce_value + dice_value


# ------------------------------------------------------------
# MODELİ YENİ LOSS İLE COMPILE ET
# ------------------------------------------------------------

model.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4
    ),

    loss=bce_dice_loss,

    metrics=[
        dice_coefficient,
        iou_metric
    ]
)


print("Model BCE + Dice Loss ile compile edildi! ✅")
print("=" * 50)
print("Optimizer     : Adam")
print("Learning rate : 1e-4")
print("Loss          : BCE + Dice Loss")
print("Metrics       : Dice + IoU")

In [ ]:
# ============================================================
# 82. HÜCRE — VALIDATION GENERATOR
#
# Validation verisi:
# - Train'den tamamen ayrı hastalardan gelir.
# - Shuffle yapılmaz.
# - Background/tumor dengesi değiştirilmez.
# - Modelin gerçek validation dağılımındaki performansını ölçer.
# ============================================================

import os
import pandas as pd


# ------------------------------------------------------------
# 1. VALIDATION INDEX'İ DRIVE'DAN YÜKLE
# ------------------------------------------------------------

validation_index_path = os.path.join(
    project_dir,
    "validation_slice_index.csv"
)

validation_slice_df = pd.read_csv(
    validation_index_path
)


# ------------------------------------------------------------
# 2. VALIDATION GENERATOR
#
# Train için yazdığımız generator sınıfının aynısını
# kullanabiliriz.
# ------------------------------------------------------------

validation_generator = FinalBraTSDataGenerator(
    dataframe=validation_slice_df,
    batch_size=BATCH_SIZE,
    shuffle=False
)


# ------------------------------------------------------------
# 3. GENEL KONTROL
# ------------------------------------------------------------

val_tumor = (
    validation_slice_df["has_tumor"] == 1
).sum()

val_background = (
    validation_slice_df["has_tumor"] == 0
).sum()


print("Validation generator oluşturuldu! ✅")
print("=" * 55)

print(
    "Validation slice sayısı:",
    len(validation_slice_df)
)

print(
    "Tumor slices:",
    val_tumor
)

print(
    "Background slices:",
    val_background
)

print(
    "Batch size:",
    BATCH_SIZE
)

print(
    "Toplam batch sayısı:",
    len(validation_generator)
)


# ------------------------------------------------------------
# 4. İLK BATCH TESTİ
# ------------------------------------------------------------

X_val, Y_val = validation_generator[0]

print("\nFIRST VALIDATION BATCH")
print("=" * 55)

print("X shape:", X_val.shape)
print("Y shape:", Y_val.shape)

print(
    "Mask classes:",
    np.unique(Y_val)
)


# ------------------------------------------------------------
# 5. TRAIN / VALIDATION PATIENT LEAKAGE CHECK
# ------------------------------------------------------------

train_patients = set(
    clean_train_df["patient_id"].unique()
)

val_patients = set(
    validation_slice_df["patient_id"].unique()
)

overlap = train_patients.intersection(
    val_patients
)

print("\nPATIENT LEAKAGE CHECK")
print("=" * 55)

print(
    "Train patients:",
    len(train_patients)
)

print(
    "Validation patients:",
    len(val_patients)
)

print(
    "Overlapping patients:",
    len(overlap)
)

if len(overlap) == 0:
    print("Patient-level split korunuyor. ✅")
else:
    print("UYARI: Patient leakage bulundu! ❌")

In [ ]:
# ============================================================
# 83. HÜCRE — MINI TRAINING SMOKE TEST
#
# Amaç:
# Gerçek eğitime başlamadan önce training pipeline'ın
# backpropagation sırasında da sorunsuz çalıştığını kontrol etmek.
#
# SADECE:
#   5 train batch
#   3 validation batch
#   1 epoch
#
# kullanılır.
# ============================================================

print("Mini training smoke test başlıyor...")
print("=" * 55)

history_smoke = model.fit(
    train_generator,

    validation_data=validation_generator,

    epochs=1,

    steps_per_epoch=5,

    validation_steps=3,

    verbose=1
)

print("\n" + "=" * 55)
print("SMOKE TEST TAMAMLANDI ✅")
print("=" * 55)

print(
    "Train loss:",
    history_smoke.history["loss"][-1]
)

print(
    "Train Dice:",
    history_smoke.history["dice_coefficient"][-1]
)

print(
    "Train IoU:",
    history_smoke.history["iou_metric"][-1]
)

print(
    "Validation loss:",
    history_smoke.history["val_loss"][-1]
)

print(
    "Validation Dice:",
    history_smoke.history["val_dice_coefficient"][-1]
)

print(
    "Validation IoU:",
    history_smoke.history["val_iou_metric"][-1]
)

In [ ]:
# ============================================================
# 84. HÜCRE — PERFORMANCE BOTTLENECK TEST
#
# Amaç:
# Yavaşlığın nereden geldiğini ölçmek:
#
# 1) Generator / NIfTI data loading
# 2) Model forward pass
# 3) Tek training step
#
# Bu hücre gerçek training başlatmaz.
# ============================================================

import time
import numpy as np


print("PERFORMANCE BENCHMARK")
print("=" * 55)


# ============================================================
# TEST 1 — DATA LOADING
# ============================================================

data_times = []

print("\n1) DATA LOADING TEST")
print("-" * 55)

for i in range(3):

    start = time.time()

    X_test, Y_test = train_generator[i]

    elapsed = time.time() - start
    data_times.append(elapsed)

    print(
        f"Batch {i+1}: "
        f"{elapsed:.2f} seconds"
    )


avg_data_time = np.mean(data_times)

print(
    "\nAverage data loading time:",
    f"{avg_data_time:.2f} seconds / batch"
)


# ============================================================
# TEST 2 — MODEL FORWARD PASS
# ============================================================

print("\n2) MODEL FORWARD PASS TEST")
print("-" * 55)

# İlk çağrı TensorFlow warm-up olabilir.
_ = model.predict(
    X_test,
    verbose=0
)

forward_times = []

for i in range(3):

    start = time.time()

    _ = model.predict(
        X_test,
        verbose=0
    )

    elapsed = time.time() - start
    forward_times.append(elapsed)

    print(
        f"Forward pass {i+1}: "
        f"{elapsed:.2f} seconds"
    )


avg_forward_time = np.mean(
    forward_times
)

print(
    "\nAverage forward pass:",
    f"{avg_forward_time:.2f} seconds / batch"
)


# ============================================================
# TEST 3 — SINGLE TRAINING STEP
# ============================================================

print("\n3) TRAINING STEP TEST")
print("-" * 55)

start = time.time()

train_result = model.train_on_batch(
    X_test,
    Y_test,
    return_dict=True
)

train_time = time.time() - start


print(
    "Training step time:",
    f"{train_time:.2f} seconds"
)

print("\nTraining result:")

for key, value in train_result.items():
    print(
        f"{key}: {float(value):.4f}"
    )


# ============================================================
# SUMMARY
# ============================================================

print("\n" + "=" * 55)
print("BENCHMARK SUMMARY")
print("=" * 55)

print(
    "Average data loading :",
    f"{avg_data_time:.2f} sec/batch"
)

print(
    "Average forward pass :",
    f"{avg_forward_time:.2f} sec/batch"
)

print(
    "Training step        :",
    f"{train_time:.2f} sec/batch"
)


# ============================================================
# SIMPLE BOTTLENECK INTERPRETATION
# ============================================================

print("\nBOTTLENECK CHECK")
print("-" * 55)

if avg_data_time > avg_forward_time * 2:

    print(
        "⚠️ Data loading appears to be the main bottleneck."
    )

elif avg_forward_time > avg_data_time * 2:

    print(
        "⚠️ Model computation appears to be the main bottleneck."
    )

else:

    print(
        "⚠️ Both data loading and model computation "
        "contribute significantly."
    )

In [ ]:
# ============================================================
# SESSION STATUS CHECK
# ============================================================

items = [
    "project_dir",
    "train_slice_df",
    "background_brain_df",
    "clean_train_df",
    "normalize_volume",
    "load_patient_volumes",
    "FinalBraTSDataGenerator",
    "train_generator",
    "validation_generator",
    "dice_coefficient",
    "iou_metric",
    "bce_dice_loss",
    "model"
]

print("CURRENT SESSION STATUS")
print("=" * 45)

for item in items:
    print(
        f"{'✅' if item in globals() else '❌'} {item}"
    )

In [ ]:
# ============================================================
# SESSION RESTORE — STEP 1
# Drive + Project Files Check
# ============================================================

from google.colab import drive
import os
import pandas as pd
import numpy as np

# Google Drive'ı bağla
drive.mount("/content/drive")

# Proje klasörü
project_dir = "/content/drive/MyDrive/NeuroRobust"

print("\nPROJECT RESTORE")
print("=" * 55)
print("Project directory:", project_dir)
print("Project exists   :", os.path.exists(project_dir))


# ------------------------------------------------------------
# KAYDETTİĞİMİZ KRİTİK DOSYALARI KONTROL ET
# ------------------------------------------------------------

required_files = [
    "train_slice_index.csv",
    "validation_slice_index.csv",
    "background_brain_content.csv",
    "patient_split.json"
]

print("\nSAVED FILES")
print("-" * 55)

all_ok = True

for filename in required_files:

    path = os.path.join(
        project_dir,
        filename
    )

    exists = os.path.exists(path)

    print(
        "✅" if exists else "❌",
        filename
    )

    if not exists:
        all_ok = False


print("\n" + "=" * 55)

if all_ok:
    print("RESTORE STEP 1 READY ✅")
else:
    print("Eksik dosya bulundu ❌")

In [ ]:
# ============================================================
# SESSION RESTORE — STEP 2
# Saved indexes -> Final Clean Training Pool
# ============================================================

import os
import pandas as pd


# ------------------------------------------------------------
# DOSYALARI YÜKLE
# ------------------------------------------------------------

train_slice_df = pd.read_csv(
    os.path.join(
        project_dir,
        "train_slice_index.csv"
    )
)

validation_slice_df = pd.read_csv(
    os.path.join(
        project_dir,
        "validation_slice_index.csv"
    )
)

background_brain_df = pd.read_csv(
    os.path.join(
        project_dir,
        "background_brain_content.csv"
    )
)


print("Saved analysis files loaded! ✅")
print("=" * 55)

print(
    "Train index:",
    len(train_slice_df)
)

print(
    "Validation index:",
    len(validation_slice_df)
)

print(
    "Background analysis:",
    len(background_brain_df)
)


# ============================================================
# FINAL TRAINING POOL
# ============================================================
#
# Tumor slice'ların HEPSİNİ tutuyoruz.
#
# Background slice'larda ise:
# brain_content >= 5%
#
# filtresini kullanıyoruz.
# ============================================================

tumor_training_df = train_slice_df[
    train_slice_df["has_tumor"] == 1
].copy()


useful_background_df = background_brain_df[
    background_brain_df["brain_content"] >= 5.0
].copy()


# Background kayıtlarını train index formatına getir.
useful_background_df = useful_background_df[
    [
        "patient_id",
        "slice_index"
    ]
].copy()

useful_background_df["has_tumor"] = 0
useful_background_df["tumor_pixels"] = 0


# Kolon sırasını train index ile aynı yap.
useful_background_df = useful_background_df[
    train_slice_df.columns
]


# Tumor + useful background
clean_train_df = pd.concat(
    [
        tumor_training_df,
        useful_background_df
    ],
    ignore_index=True
)


# Karıştır
clean_train_df = clean_train_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)


# ============================================================
# KONTROL
# ============================================================

tumor_count = int(
    (clean_train_df["has_tumor"] == 1).sum()
)

background_count = int(
    (clean_train_df["has_tumor"] == 0).sum()
)


print("\nFINAL CLEAN TRAINING POOL")
print("=" * 55)

print(
    "Total slices      :",
    len(clean_train_df)
)

print(
    "Tumor slices      :",
    tumor_count
)

print(
    "Background slices :",
    background_count
)

print(
    "Tumor ratio       :",
    f"{100 * tumor_count / len(clean_train_df):.2f}%"
)

print(
    "Background ratio  :",
    f"{100 * background_count / len(clean_train_df):.2f}%"
)


# ------------------------------------------------------------
# BEKLENEN DEĞERLERİ OTOMATİK KONTROL ET
# ------------------------------------------------------------

expected = (
    len(clean_train_df) == 107044
    and tumor_count == 57067
    and background_count == 49977
)

print("\nRESTORE VALIDATION")
print("=" * 55)

if expected:
    print("Final training pool birebir restore edildi. ✅")
else:
    print("Beklenen sayılarla uyuşmuyor. ❌")
    print("Burada dur ve sonucu kontrol et.")

In [ ]:
# ============================================================
# SESSION RESTORE — STEP 3
# MRI Loader + Normalization + Patient Cache
# ============================================================

import os
import numpy as np
import nibabel as nib

from functools import lru_cache


# ============================================================
# TRAINING DATA DIRECTORY
# ============================================================

training_data_dir = os.path.join(
    project_dir,
    "dataset/BraTS2021_Training_Data"
)

print("Training dataset:")
print(training_data_dir)

print(
    "Dataset exists:",
    os.path.exists(training_data_dir)
)


# ============================================================
# NORMALIZATION
# ============================================================

def normalize_volume(volume):

    volume = volume.astype(np.float32)

    # Brain bölgesi:
    # BraTS görüntülerinde beyin dışı alan genellikle 0'dır.
    brain_mask = volume != 0

    # Tamamen boş volume güvenliği
    if np.sum(brain_mask) == 0:
        return np.zeros_like(
            volume,
            dtype=np.float32
        )

    mean = volume[brain_mask].mean()
    std = volume[brain_mask].std()

    normalized = np.zeros_like(
        volume,
        dtype=np.float32
    )

    if std > 0:

        normalized[brain_mask] = (
            volume[brain_mask] - mean
        ) / std

    return normalized


# ============================================================
# PATIENT-LEVEL LRU CACHE
#
# Aynı hastanın farklı slice'ları kullanıldığında NIfTI
# dosyalarının Drive'dan tekrar tekrar okunmasını azaltır.
# ============================================================

@lru_cache(maxsize=4)
def load_patient_volumes(patient_id):

    patient_folder = os.path.join(
        training_data_dir,
        patient_id
    )

    # --------------------------------------------------------
    # BraTS file paths
    # --------------------------------------------------------

    t1_path = os.path.join(
        patient_folder,
        f"{patient_id}_t1.nii.gz"
    )

    t1ce_path = os.path.join(
        patient_folder,
        f"{patient_id}_t1ce.nii.gz"
    )

    t2_path = os.path.join(
        patient_folder,
        f"{patient_id}_t2.nii.gz"
    )

    flair_path = os.path.join(
        patient_folder,
        f"{patient_id}_flair.nii.gz"
    )

    seg_path = os.path.join(
        patient_folder,
        f"{patient_id}_seg.nii.gz"
    )


    # --------------------------------------------------------
    # LOAD NIFTI
    # --------------------------------------------------------

    t1 = nib.load(t1_path).get_fdata(
        dtype=np.float32
    )

    t1ce = nib.load(t1ce_path).get_fdata(
        dtype=np.float32
    )

    t2 = nib.load(t2_path).get_fdata(
        dtype=np.float32
    )

    flair = nib.load(flair_path).get_fdata(
        dtype=np.float32
    )

    seg = nib.load(seg_path).get_fdata(
        dtype=np.float32
    )


    # --------------------------------------------------------
    # NORMALIZE MRI MODALITIES
    # --------------------------------------------------------

    t1 = normalize_volume(t1)
    t1ce = normalize_volume(t1ce)
    t2 = normalize_volume(t2)
    flair = normalize_volume(flair)


    return (
        t1,
        t1ce,
        t2,
        flair,
        seg
    )


# ============================================================
# QUICK TEST
# ============================================================

test_patient = clean_train_df.iloc[0][
    "patient_id"
]

print("\nLOADER TEST")
print("=" * 55)

print(
    "Test patient:",
    test_patient
)


try:

    volumes = load_patient_volumes(
        test_patient
    )

    print("Patient successfully loaded. ✅")

    print(
        "T1 shape    :",
        volumes[0].shape
    )

    print(
        "T1ce shape  :",
        volumes[1].shape
    )

    print(
        "T2 shape    :",
        volumes[2].shape
    )

    print(
        "FLAIR shape :",
        volumes[3].shape
    )

    print(
        "Mask shape  :",
        volumes[4].shape
    )

    print("\nCache:")
    print(
        load_patient_volumes.cache_info()
    )

except Exception as e:

    print("Loader test failed. ❌")
    print(type(e).__name__, ":", e)

In [ ]:
# ============================================================
# SESSION RESTORE — STEP 4
# Final Train + Validation Generators
# ============================================================

import numpy as np
import tensorflow as tf
import cv2


MODEL_HEIGHT = 128
MODEL_WIDTH = 128
NUM_CHANNELS = 4
BATCH_SIZE = 8


# ============================================================
# GENERATOR
# ============================================================

class FinalBraTSDataGenerator(tf.keras.utils.Sequence):

    def __init__(
        self,
        dataframe,
        batch_size=8,
        shuffle=True,
        **kwargs
    ):

        super().__init__(**kwargs)

        self.dataframe = dataframe.reset_index(
            drop=True
        )

        self.batch_size = batch_size
        self.shuffle = shuffle

        self.indices = np.arange(
            len(self.dataframe)
        )

        self.on_epoch_end()


    def __len__(self):

        return int(
            np.ceil(
                len(self.dataframe)
                / self.batch_size
            )
        )


    def __getitem__(self, batch_index):

        start = (
            batch_index
            * self.batch_size
        )

        end = min(
            start + self.batch_size,
            len(self.dataframe)
        )

        batch_indices = self.indices[
            start:end
        ]

        current_batch_size = len(
            batch_indices
        )


        X = np.zeros(
            (
                current_batch_size,
                MODEL_HEIGHT,
                MODEL_WIDTH,
                NUM_CHANNELS
            ),
            dtype=np.float32
        )


        Y = np.zeros(
            (
                current_batch_size,
                MODEL_HEIGHT,
                MODEL_WIDTH,
                1
            ),
            dtype=np.float32
        )


        for i, row_index in enumerate(
            batch_indices
        ):

            row = self.dataframe.iloc[
                row_index
            ]

            patient_id = row[
                "patient_id"
            ]

            slice_index = int(
                row["slice_index"]
            )


            # -----------------------------------------------
            # Load patient
            # -----------------------------------------------

            (
                t1,
                t1ce,
                t2,
                flair,
                mask
            ) = load_patient_volumes(
                patient_id
            )


            # -----------------------------------------------
            # Extract axial slice
            # -----------------------------------------------

            t1_slice = t1[
                :, :, slice_index
            ]

            t1ce_slice = t1ce[
                :, :, slice_index
            ]

            t2_slice = t2[
                :, :, slice_index
            ]

            flair_slice = flair[
                :, :, slice_index
            ]

            mask_slice = mask[
                :, :, slice_index
            ]


            # -----------------------------------------------
            # Resize MRI
            # -----------------------------------------------

            t1_slice = cv2.resize(
                t1_slice,
                (MODEL_WIDTH, MODEL_HEIGHT),
                interpolation=cv2.INTER_LINEAR
            )

            t1ce_slice = cv2.resize(
                t1ce_slice,
                (MODEL_WIDTH, MODEL_HEIGHT),
                interpolation=cv2.INTER_LINEAR
            )

            t2_slice = cv2.resize(
                t2_slice,
                (MODEL_WIDTH, MODEL_HEIGHT),
                interpolation=cv2.INTER_LINEAR
            )

            flair_slice = cv2.resize(
                flair_slice,
                (MODEL_WIDTH, MODEL_HEIGHT),
                interpolation=cv2.INTER_LINEAR
            )


            # -----------------------------------------------
            # Binary tumor mask
            # -----------------------------------------------

            binary_mask = (
                mask_slice > 0
            ).astype(np.float32)


            binary_mask = cv2.resize(
                binary_mask,
                (MODEL_WIDTH, MODEL_HEIGHT),
                interpolation=cv2.INTER_NEAREST
            )


            # -----------------------------------------------
            # 4 MRI modalities
            # -----------------------------------------------

            X[i] = np.stack(
                [
                    t1_slice,
                    t1ce_slice,
                    t2_slice,
                    flair_slice
                ],
                axis=-1
            )

            Y[i, :, :, 0] = binary_mask


        return X, Y


    def on_epoch_end(self):

        if self.shuffle:
            np.random.shuffle(
                self.indices
            )


# ============================================================
# TRAIN GENERATOR
# ============================================================

train_generator = FinalBraTSDataGenerator(
    dataframe=clean_train_df,
    batch_size=BATCH_SIZE,
    shuffle=True
)


# ============================================================
# VALIDATION GENERATOR
# ============================================================

validation_generator = FinalBraTSDataGenerator(
    dataframe=validation_slice_df,
    batch_size=BATCH_SIZE,
    shuffle=False
)


# ============================================================
# CHECK
# ============================================================

print("GENERATORS RESTORED ✅")
print("=" * 55)

print(
    "Train slices      :",
    len(clean_train_df)
)

print(
    "Train batches     :",
    len(train_generator)
)

print(
    "Validation slices :",
    len(validation_slice_df)
)

print(
    "Validation batches:",
    len(validation_generator)
)


# ============================================================
# ONE-BATCH SANITY CHECK
# ============================================================

X_check, Y_check = train_generator[0]

print("\nBATCH CHECK")
print("=" * 55)

print("X shape:", X_check.shape)
print("Y shape:", Y_check.shape)

print(
    "Mask classes:",
    np.unique(Y_check)
)

print("\nCache:")
print(
    load_patient_volumes.cache_info()
)


if (
    X_check.shape[1:] == (128, 128, 4)
    and
    Y_check.shape[1:] == (128, 128, 1)
):

    print("\nGenerator restore başarılı. ✅")

else:

    print("\nGenerator shape problemi var. ❌")

In [ ]:
# ============================================================
# SESSION RESTORE — STEP 5
# U-Net + Dice/IoU + BCE-Dice Loss + Compile
# ============================================================

import tensorflow as tf
from tensorflow.keras import layers, Model


# ============================================================
# 1. METRICS
# ============================================================

def dice_coefficient(y_true, y_pred, smooth=1e-6):

    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    y_true = tf.reshape(y_true, [-1])
    y_pred = tf.reshape(y_pred, [-1])

    intersection = tf.reduce_sum(
        y_true * y_pred
    )

    return (
        2.0 * intersection + smooth
    ) / (
        tf.reduce_sum(y_true)
        + tf.reduce_sum(y_pred)
        + smooth
    )


def iou_metric(y_true, y_pred, smooth=1e-6):

    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    y_true = tf.reshape(y_true, [-1])
    y_pred = tf.reshape(y_pred, [-1])

    intersection = tf.reduce_sum(
        y_true * y_pred
    )

    union = (
        tf.reduce_sum(y_true)
        + tf.reduce_sum(y_pred)
        - intersection
    )

    return (
        intersection + smooth
    ) / (
        union + smooth
    )


# ============================================================
# 2. LOSS
# ============================================================

def dice_loss(y_true, y_pred, smooth=1e-6):

    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    intersection = tf.reduce_sum(
        y_true * y_pred,
        axis=[1, 2, 3]
    )

    denominator = (
        tf.reduce_sum(
            y_true,
            axis=[1, 2, 3]
        )
        +
        tf.reduce_sum(
            y_pred,
            axis=[1, 2, 3]
        )
    )

    dice = (
        2.0 * intersection + smooth
    ) / (
        denominator + smooth
    )

    return 1.0 - tf.reduce_mean(dice)


bce = tf.keras.losses.BinaryCrossentropy()


def bce_dice_loss(y_true, y_pred):

    return (
        bce(y_true, y_pred)
        +
        dice_loss(y_true, y_pred)
    )


# ============================================================
# 3. CONVOLUTION BLOCK
# ============================================================

def conv_block(x, filters):

    x = layers.Conv2D(
        filters,
        3,
        padding="same"
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.Conv2D(
        filters,
        3,
        padding="same"
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    return x


# ============================================================
# 4. U-NET
# ============================================================

def build_unet(
    input_shape=(128, 128, 4)
):

    inputs = layers.Input(
        shape=input_shape,
        name="MRI_Input"
    )


    # --------------------------------------------------------
    # ENCODER 1
    # 128x128
    # --------------------------------------------------------

    c1 = conv_block(
        inputs,
        32
    )

    p1 = layers.MaxPooling2D(
        (2, 2)
    )(c1)


    # --------------------------------------------------------
    # ENCODER 2
    # 64x64
    # --------------------------------------------------------

    c2 = conv_block(
        p1,
        64
    )

    p2 = layers.MaxPooling2D(
        (2, 2)
    )(c2)


    # --------------------------------------------------------
    # ENCODER 3
    # 32x32
    # --------------------------------------------------------

    c3 = conv_block(
        p2,
        128
    )

    p3 = layers.MaxPooling2D(
        (2, 2)
    )(c3)


    # --------------------------------------------------------
    # ENCODER 4
    # 16x16
    # --------------------------------------------------------

    c4 = conv_block(
        p3,
        256
    )

    p4 = layers.MaxPooling2D(
        (2, 2)
    )(c4)


    # --------------------------------------------------------
    # BOTTLENECK
    # 8x8
    # --------------------------------------------------------

    bottleneck = conv_block(
        p4,
        512
    )


    # --------------------------------------------------------
    # DECODER 1
    # 8 -> 16
    # --------------------------------------------------------

    u1 = layers.Conv2DTranspose(
        256,
        2,
        strides=2,
        padding="same"
    )(bottleneck)

    u1 = layers.Concatenate()(
        [u1, c4]
    )

    c5 = conv_block(
        u1,
        256
    )


    # --------------------------------------------------------
    # DECODER 2
    # 16 -> 32
    # --------------------------------------------------------

    u2 = layers.Conv2DTranspose(
        128,
        2,
        strides=2,
        padding="same"
    )(c5)

    u2 = layers.Concatenate()(
        [u2, c3]
    )

    c6 = conv_block(
        u2,
        128
    )


    # --------------------------------------------------------
    # DECODER 3
    # 32 -> 64
    # --------------------------------------------------------

    u3 = layers.Conv2DTranspose(
        64,
        2,
        strides=2,
        padding="same"
    )(c6)

    u3 = layers.Concatenate()(
        [u3, c2]
    )

    c7 = conv_block(
        u3,
        64
    )


    # --------------------------------------------------------
    # DECODER 4
    # 64 -> 128
    # --------------------------------------------------------

    u4 = layers.Conv2DTranspose(
        32,
        2,
        strides=2,
        padding="same"
    )(c7)

    u4 = layers.Concatenate()(
        [u4, c1]
    )

    c8 = conv_block(
        u4,
        32
    )


    # --------------------------------------------------------
    # BINARY TUMOR MASK
    # --------------------------------------------------------

    outputs = layers.Conv2D(
        1,
        1,
        activation="sigmoid",
        name="Tumor_Mask"
    )(c8)


    return Model(
        inputs,
        outputs,
        name="NeuroRobust_UNet"
    )


# ============================================================
# 5. BUILD
# ============================================================

model = build_unet()


# ============================================================
# 6. COMPILE
# ============================================================

model.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4
    ),

    loss=bce_dice_loss,

    metrics=[
        dice_coefficient,
        iou_metric
    ]
)


# ============================================================
# 7. RESTORE CHECK
# ============================================================

print("MODEL RESTORED ✅")
print("=" * 55)

print(
    "Input shape :",
    model.input_shape
)

print(
    "Output shape:",
    model.output_shape
)

print(
    "Parameters  :",
    f"{model.count_params():,}"
)

print("\nOptimizer     : Adam")
print("Learning rate : 1e-4")
print("Loss          : BCE + Dice")
print("Metrics       : Dice + IoU")


# ============================================================
# 8. FINAL SESSION STATUS
# ============================================================

print("\nSESSION STATUS")
print("=" * 55)

required_objects = [
    "project_dir",
    "train_slice_df",
    "background_brain_df",
    "clean_train_df",
    "normalize_volume",
    "load_patient_volumes",
    "FinalBraTSDataGenerator",
    "train_generator",
    "validation_generator",
    "dice_coefficient",
    "iou_metric",
    "bce_dice_loss",
    "model"
]

all_restored = True

for item in required_objects:

    exists = item in globals()

    print(
        "✅" if exists else "❌",
        item
    )

    if not exists:
        all_restored = False


print("\n" + "=" * 55)

if all_restored:
    print("SESSION FULLY RESTORED ✅")
else:
    print("Restore tamamlanmadı. ❌")

## Session Restoration

The Colab session was successfully restored using the files saved in Google Drive.

- **Training slices:** 107,044
- **Validation slices:** 29,140
- **Input:** `(128, 128, 4)` — T1, T1ce, T2, FLAIR
- **Output:** `(128, 128, 1)` — binary tumor mask
- **U-Net parameters:** 7,772,161
- **Loss:** BCE + Dice
- **Metrics:** Dice and IoU

All preprocessing, data generators, metrics, and the U-Net architecture were successfully restored.

**Result:** The pipeline is ready for performance testing and model training.

In [ ]:
# ============================================================
# 84. HÜCRE — PERFORMANCE BOTTLENECK TEST
# ============================================================

import time
import numpy as np

print("PERFORMANCE BENCHMARK")
print("=" * 55)


# 1 — DATA LOADING
print("\n1) DATA LOADING TEST")
print("-" * 55)

data_times = []

for i in range(3):

    start = time.time()

    X_test, Y_test = train_generator[i]

    elapsed = time.time() - start
    data_times.append(elapsed)

    print(f"Batch {i+1}: {elapsed:.2f} sec")

avg_data_time = np.mean(data_times)

print(
    f"\nAverage data loading: "
    f"{avg_data_time:.2f} sec/batch"
)


# 2 — MODEL FORWARD PASS
print("\n2) MODEL FORWARD PASS TEST")
print("-" * 55)

# Warm-up
_ = model.predict(X_test, verbose=0)

forward_times = []

for i in range(3):

    start = time.time()

    _ = model.predict(
        X_test,
        verbose=0
    )

    elapsed = time.time() - start
    forward_times.append(elapsed)

    print(
        f"Forward {i+1}: "
        f"{elapsed:.2f} sec"
    )

avg_forward_time = np.mean(
    forward_times
)


# 3 — TRAINING STEP
print("\n3) TRAINING STEP TEST")
print("-" * 55)

start = time.time()

result = model.train_on_batch(
    X_test,
    Y_test,
    return_dict=True
)

train_time = time.time() - start


# SUMMARY
print("\n" + "=" * 55)
print("BENCHMARK SUMMARY")
print("=" * 55)

print(
    f"Data loading : {avg_data_time:.2f} sec/batch"
)

print(
    f"Forward pass : {avg_forward_time:.2f} sec/batch"
)

print(
    f"Training step: {train_time:.2f} sec/batch"
)

print("\nTrain result:")

for key, value in result.items():
    print(
        f"{key}: {float(value):.4f}"
    )


print("\nBOTTLENECK")
print("-" * 55)

if avg_data_time > avg_forward_time * 2:
    print("⚠️ Main bottleneck: DATA LOADING")

elif avg_forward_time > avg_data_time * 2:
    print("⚠️ Main bottleneck: MODEL COMPUTATION")

else:
    print("⚠️ Both components contribute significantly")

In [ ]:
# ============================================================
# 85. HÜCRE — PILOT PREPROCESSED CACHE
#
# AMAÇ:
# NIfTI dosyalarını training sırasında tekrar tekrar okumak
# yerine, küçük bir hasta grubu için önceden hazırlanmış
# 128x128 MRI + mask dosyaları oluşturmak.
#
# Şimdilik SADECE 10 HASTA -> pilot test
# ============================================================

import os
import time
import numpy as np
import cv2


# ------------------------------------------------------------
# LOCAL CACHE DIRECTORY
# ------------------------------------------------------------

cache_dir = "/content/NeuroRobust_cache"

os.makedirs(
    cache_dir,
    exist_ok=True
)


# ------------------------------------------------------------
# 10 TEST PATIENT
# ------------------------------------------------------------

pilot_patients = (
    clean_train_df["patient_id"]
    .drop_duplicates()
    .head(10)
    .tolist()
)

print("PILOT CACHE")
print("=" * 55)
print("Patient count:", len(pilot_patients))
print("Cache directory:", cache_dir)


# ============================================================
# PATIENT CACHE CREATION
# ============================================================

start_total = time.time()

created = 0


for number, patient_id in enumerate(
    pilot_patients,
    start=1
):

    patient_start = time.time()

    # --------------------------------------------------------
    # Load normalized patient volumes
    # --------------------------------------------------------

    (
        t1,
        t1ce,
        t2,
        flair,
        mask
    ) = load_patient_volumes(
        patient_id
    )


    # --------------------------------------------------------
    # Bu hastanın final training pool'daki slice'ları
    # --------------------------------------------------------

    patient_rows = clean_train_df[
        clean_train_df["patient_id"]
        == patient_id
    ].sort_values(
        "slice_index"
    )


    slice_indices = (
        patient_rows["slice_index"]
        .astype(np.int16)
        .to_numpy()
    )


    n_slices = len(slice_indices)


    # --------------------------------------------------------
    # Preprocessed arrays
    # --------------------------------------------------------

    X_patient = np.zeros(
        (
            n_slices,
            128,
            128,
            4
        ),
        dtype=np.float32
    )

    Y_patient = np.zeros(
        (
            n_slices,
            128,
            128,
            1
        ),
        dtype=np.uint8
    )


    # --------------------------------------------------------
    # Slice preprocessing
    # --------------------------------------------------------

    for i, slice_index in enumerate(
        slice_indices
    ):

        modalities = [
            t1[:, :, slice_index],
            t1ce[:, :, slice_index],
            t2[:, :, slice_index],
            flair[:, :, slice_index]
        ]


        for channel, image in enumerate(
            modalities
        ):

            X_patient[
                i, :, :, channel
            ] = cv2.resize(
                image,
                (128, 128),
                interpolation=cv2.INTER_LINEAR
            )


        binary_mask = (
            mask[:, :, slice_index] > 0
        ).astype(np.uint8)


        Y_patient[
            i, :, :, 0
        ] = cv2.resize(
            binary_mask,
            (128, 128),
            interpolation=cv2.INTER_NEAREST
        )


    # --------------------------------------------------------
    # Save patient cache
    # --------------------------------------------------------

    output_path = os.path.join(
        cache_dir,
        f"{patient_id}.npz"
    )

    np.savez(
        output_path,
        X=X_patient,
        Y=Y_patient,
        slice_indices=slice_indices
    )


    elapsed = (
        time.time()
        - patient_start
    )

    created += 1

    print(
        f"{number:02d}/10 | "
        f"{patient_id} | "
        f"{n_slices} slices | "
        f"{elapsed:.2f} sec"
    )


# ============================================================
# SUMMARY
# ============================================================

total_time = (
    time.time()
    - start_total
)

print("\n" + "=" * 55)
print("PILOT CACHE COMPLETE ✅")
print("=" * 55)

print(
    "Patients cached:",
    created
)

print(
    "Total time:",
    f"{total_time:.2f} sec"
)

print(
    "Cache files:",
    len(os.listdir(cache_dir))
)

In [ ]:
# ============================================================
# 86. HÜCRE — CACHE SPEED TEST
#
# Amaç:
# Hazırladığımız .npz cache dosyalarından 8 slice okumanın
# ne kadar sürdüğünü ölçmek.
# ============================================================

import os
import time
import numpy as np


print("CACHE SPEED TEST")
print("=" * 55)

cache_files = sorted(
    [
        f for f in os.listdir(cache_dir)
        if f.endswith(".npz")
    ]
)

print("Available cache files:", len(cache_files))


# ============================================================
# TEST — 3 DIFFERENT PATIENT CACHE FILES
# ============================================================

cache_times = []

for i in range(3):

    file_path = os.path.join(
        cache_dir,
        cache_files[i]
    )

    start = time.time()

    with np.load(file_path) as data:

        # 8 slice = bizim batch size
        X_cache = data["X"][:8]
        Y_cache = data["Y"][:8]

    elapsed = time.time() - start

    cache_times.append(elapsed)

    print(
        f"Batch {i+1}: "
        f"{elapsed:.4f} sec"
    )


# ============================================================
# SUMMARY
# ============================================================

avg_cache_time = np.mean(
    cache_times
)

old_time = 28.31

speedup = (
    old_time / avg_cache_time
    if avg_cache_time > 0
    else float("inf")
)


print("\n" + "=" * 55)
print("CACHE BENCHMARK SUMMARY")
print("=" * 55)

print(
    f"Original pipeline : {old_time:.2f} sec/batch"
)

print(
    f"Cached pipeline   : {avg_cache_time:.4f} sec/batch"
)

print(
    f"Estimated speedup : {speedup:.1f}x"
)


if avg_cache_time < 1:
    print("\nCache performance excellent. ✅")

elif avg_cache_time < 5:
    print("\nCache provides a strong improvement. ✅")

else:
    print("\nCache needs further optimization. ⚠️")

## Cache Performance Test

Preprocessed MRI caching significantly reduced data-loading time.

- **Original loading:** 28.31 s/batch
- **Cached loading:** 0.0205 s/batch
- **I/O benchmark speedup:** ~1379×  

The result confirms that repeated NIfTI loading and preprocessing was the main data pipeline bottleneck.

**Decision:** Use preprocessed cached data for full model training.

In [ ]:
# ============================================================
# 87. HÜCRE — CACHE STORAGE ANALYSIS
# ============================================================

import os
import numpy as np

print("CACHE STORAGE ANALYSIS")
print("=" * 55)

cache_files = [
    os.path.join(cache_dir, f)
    for f in os.listdir(cache_dir)
    if f.endswith(".npz")
]

# ------------------------------------------------------------
# Pilot cache toplam boyutu
# ------------------------------------------------------------

total_bytes = sum(
    os.path.getsize(f)
    for f in cache_files
)

total_mb = total_bytes / (1024 ** 2)
total_gb = total_bytes / (1024 ** 3)

print("Pilot patients :", len(cache_files))
print(f"Pilot size     : {total_mb:.2f} MB")


# ------------------------------------------------------------
# Pilot cache'teki toplam slice sayısı
# ------------------------------------------------------------

total_slices = 0

for file_path in cache_files:

    with np.load(file_path) as data:
        total_slices += len(data["X"])

print("Pilot slices   :", total_slices)


# ------------------------------------------------------------
# Slice başına ortalama depolama
# ------------------------------------------------------------

mb_per_slice = (
    total_mb / total_slices
)

print(
    f"MB per slice   : {mb_per_slice:.3f} MB"
)


# ------------------------------------------------------------
# Full training pool tahmini
# ------------------------------------------------------------

full_slices = len(clean_train_df)

estimated_mb = (
    mb_per_slice * full_slices
)

estimated_gb = (
    estimated_mb / 1024
)

print("\nFULL DATASET ESTIMATE")
print("=" * 55)

print(
    "Training slices :",
    full_slices
)

print(
    f"Estimated cache : {estimated_gb:.2f} GB"
)


# ------------------------------------------------------------
# float16 yaklaşık tahmini
# ------------------------------------------------------------

float16_estimated_gb = (
    estimated_gb / 2
)

print(
    f"Float16 estimate: "
    f"{float16_estimated_gb:.2f} GB"
)


print("\n" + "=" * 55)

if estimated_gb > 20:

    print(
        "⚠️ Float32 full cache is very large."
    )

elif estimated_gb > 10:

    print(
        "⚠️ Float32 cache requires significant storage."
    )

else:

    print(
        "✅ Float32 cache size is manageable."
    )

## Cache Storage Analysis

The pilot cache was used to estimate storage requirements for the full training dataset.

- **Float32 cache:** ~27.77 GB
- **Float16 estimate:** ~13.88 GB

Although caching greatly improves loading speed, a full Float32 cache is too large.

**Decision:** Use a more storage-efficient cached data format before full training.

In [ ]:
# ============================================================
# 88. HÜCRE — FLOAT16 + COMPRESSED CACHE PILOT
# ============================================================

import os
import time
import numpy as np

compressed_cache_dir = "/content/NeuroRobust_cache_compressed"
os.makedirs(compressed_cache_dir, exist_ok=True)

print("COMPRESSED FLOAT16 CACHE TEST")
print("=" * 55)

cache_files = sorted([
    f for f in os.listdir(cache_dir)
    if f.endswith(".npz")
])

conversion_start = time.time()

# ------------------------------------------------------------
# 1) Mevcut 10 pilot dosyayı float16 + compressed yap
# ------------------------------------------------------------

for i, filename in enumerate(cache_files, start=1):

    source_path = os.path.join(
        cache_dir,
        filename
    )

    destination_path = os.path.join(
        compressed_cache_dir,
        filename
    )

    with np.load(source_path) as data:

        X = data["X"].astype(np.float16)
        Y = data["Y"].astype(np.uint8)
        slice_indices = data["slice_indices"]

    np.savez_compressed(
        destination_path,
        X=X,
        Y=Y,
        slice_indices=slice_indices
    )

    print(
        f"{i:02d}/{len(cache_files)} | {filename}"
    )


conversion_time = time.time() - conversion_start


# ------------------------------------------------------------
# 2) BOYUT KARŞILAŞTIRMASI
# ------------------------------------------------------------

original_bytes = sum(
    os.path.getsize(
        os.path.join(cache_dir, f)
    )
    for f in cache_files
)

compressed_bytes = sum(
    os.path.getsize(
        os.path.join(compressed_cache_dir, f)
    )
    for f in cache_files
)

original_mb = original_bytes / (1024 ** 2)
compressed_mb = compressed_bytes / (1024 ** 2)

reduction = (
    100 * (1 - compressed_mb / original_mb)
)


print("\n" + "=" * 55)
print("STORAGE COMPARISON")
print("=" * 55)

print(
    f"Original Float32 : {original_mb:.2f} MB"
)

print(
    f"Compressed Float16: {compressed_mb:.2f} MB"
)

print(
    f"Storage reduction : {reduction:.2f}%"
)

print(
    f"Conversion time   : {conversion_time:.2f} sec"
)


# ------------------------------------------------------------
# 3) FULL DATASET BOYUT TAHMİNİ
# ------------------------------------------------------------

pilot_slices = 0

for filename in cache_files:

    path = os.path.join(
        compressed_cache_dir,
        filename
    )

    with np.load(path) as data:
        pilot_slices += len(data["X"])


mb_per_slice = (
    compressed_mb / pilot_slices
)

estimated_full_gb = (
    mb_per_slice
    * len(clean_train_df)
    / 1024
)


print("\nFULL DATASET ESTIMATE")
print("=" * 55)

print(
    f"Estimated full cache: "
    f"{estimated_full_gb:.2f} GB"
)


# ------------------------------------------------------------
# 4) OKUMA HIZI TESTİ
# ------------------------------------------------------------

print("\nREAD SPEED TEST")
print("=" * 55)

read_times = []

for i in range(3):

    path = os.path.join(
        compressed_cache_dir,
        cache_files[i]
    )

    start = time.time()

    with np.load(path) as data:

        X_test = data["X"][:8].astype(
            np.float32
        )

        Y_test = data["Y"][:8].astype(
            np.float32
        )

    elapsed = time.time() - start

    read_times.append(elapsed)

    print(
        f"Batch {i+1}: {elapsed:.4f} sec"
    )


average_read = np.mean(read_times)

print("\n" + "=" * 55)
print("FINAL PILOT RESULT")
print("=" * 55)

print(
    f"Average read time : "
    f"{average_read:.4f} sec/batch"
)

print(
    f"Estimated size    : "
    f"{estimated_full_gb:.2f} GB"
)

print(
    f"vs original loader: "
    f"{28.31 / average_read:.1f}x faster"
)


if (
    estimated_full_gb < 10
    and average_read < 1
):

    print(
        "\nCompressed cache strategy looks suitable. ✅"
    )

else:

    print(
        "\nFurther optimization should be considered. ⚠️"
    )

## Optimized Cache Strategy

Float16 compression significantly reduced storage while maintaining fast data access.

- **Storage reduction:** 90.31%
- **Estimated full training cache:** 2.69 GB
- **Cached loading:** 0.0633 s/batch
- **Original loading:** 28.31 s/batch

**Decision:** Use compressed Float16 cached MRI data for model training.

In [ ]:
# ============================================================
# 89. HÜCRE — FULL TRAINING CACHE
# Float16 + Compressed NPZ + Resume Support
# ============================================================

import os
import time
import numpy as np
import cv2

# ------------------------------------------------------------
# Kalıcı cache -> GOOGLE DRIVE
# ------------------------------------------------------------

full_cache_dir = os.path.join(
    project_dir,
    "cache_float16_train"
)

os.makedirs(full_cache_dir, exist_ok=True)

print("FULL TRAINING CACHE")
print("=" * 55)
print("Cache directory:")
print(full_cache_dir)


# ------------------------------------------------------------
# Training hastaları
# ------------------------------------------------------------

train_patients = (
    clean_train_df["patient_id"]
    .drop_duplicates()
    .tolist()
)

total_patients = len(train_patients)

print("\nTraining patients:", total_patients)


# ------------------------------------------------------------
# RESUME CHECK
# Daha önce tamamlanan .npz dosyalarını bul
# ------------------------------------------------------------

existing_files = {
    f.replace(".npz", "")
    for f in os.listdir(full_cache_dir)
    if f.endswith(".npz")
}

remaining_patients = [
    p for p in train_patients
    if p not in existing_files
]

print("Already cached   :", len(existing_files))
print("Remaining        :", len(remaining_patients))

if len(remaining_patients) == 0:
    print("\nAll training patients are already cached. ✅")


# ============================================================
# CACHE CREATION
# ============================================================

start_total = time.time()

created_now = 0
failed = []


for number, patient_id in enumerate(
    remaining_patients,
    start=1
):

    patient_start = time.time()

    try:

        # ----------------------------------------------------
        # Hasta volume'larını yükle
        # ----------------------------------------------------

        (
            t1,
            t1ce,
            t2,
            flair,
            mask
        ) = load_patient_volumes(patient_id)


        # ----------------------------------------------------
        # Bu hastaya ait kullanılacak training slice'ları
        # ----------------------------------------------------

        patient_rows = clean_train_df[
            clean_train_df["patient_id"] == patient_id
        ].sort_values("slice_index")

        slice_indices = (
            patient_rows["slice_index"]
            .astype(np.int16)
            .to_numpy()
        )

        n_slices = len(slice_indices)


        # ----------------------------------------------------
        # FLOAT16 MRI
        # ----------------------------------------------------

        X_patient = np.zeros(
            (
                n_slices,
                128,
                128,
                4
            ),
            dtype=np.float16
        )

        # Binary mask -> uint8
        Y_patient = np.zeros(
            (
                n_slices,
                128,
                128,
                1
            ),
            dtype=np.uint8
        )


        # ----------------------------------------------------
        # Slice preprocessing
        # ----------------------------------------------------

        for i, slice_index in enumerate(slice_indices):

            modalities = (
                t1[:, :, slice_index],
                t1ce[:, :, slice_index],
                t2[:, :, slice_index],
                flair[:, :, slice_index]
            )

            for channel, image in enumerate(modalities):

                resized = cv2.resize(
                    image,
                    (128, 128),
                    interpolation=cv2.INTER_LINEAR
                )

                X_patient[
                    i, :, :, channel
                ] = resized.astype(np.float16)


            binary_mask = (
                mask[:, :, slice_index] > 0
            ).astype(np.uint8)

            Y_patient[
                i, :, :, 0
            ] = cv2.resize(
                binary_mask,
                (128, 128),
                interpolation=cv2.INTER_NEAREST
            )


        # ----------------------------------------------------
        # COMPRESSED SAVE
        # Önce geçici dosya -> sonra rename.
        # Böylece bağlantı/yazma hatasında yarım .npz dosyasını
        # tamamlanmış sanmayız.
        # ----------------------------------------------------

        final_path = os.path.join(
            full_cache_dir,
            f"{patient_id}.npz"
        )

        temp_path = os.path.join(
            full_cache_dir,
            f"{patient_id}.tmp.npz"
        )

        np.savez_compressed(
            temp_path,
            X=X_patient,
            Y=Y_patient,
            slice_indices=slice_indices
        )

        os.replace(
            temp_path,
            final_path
        )

        created_now += 1

        elapsed = (
            time.time()
            - patient_start
        )

        completed_total = (
            len(existing_files)
            + created_now
        )

        print(
            f"{completed_total:03d}/{total_patients} | "
            f"{patient_id} | "
            f"{n_slices} slices | "
            f"{elapsed:.2f} sec"
        )


    except Exception as e:

        failed.append(
            (
                patient_id,
                str(e)
            )
        )

        print(
            f"❌ {patient_id} | "
            f"{type(e).__name__}: {e}"
        )


# ============================================================
# FINAL SUMMARY
# ============================================================

total_elapsed = (
    time.time()
    - start_total
)

final_files = [
    f for f in os.listdir(full_cache_dir)
    if f.endswith(".npz")
]

total_bytes = sum(
    os.path.getsize(
        os.path.join(full_cache_dir, f)
    )
    for f in final_files
)

total_gb = total_bytes / (1024 ** 3)


print("\n" + "=" * 55)
print("FULL CACHE SUMMARY")
print("=" * 55)

print(
    "Total patients :",
    total_patients
)

print(
    "Cached patients:",
    len(final_files)
)

print(
    "Created now    :",
    created_now
)

print(
    "Failed         :",
    len(failed)
)

print(
    f"Cache size     : {total_gb:.2f} GB"
)

print(
    f"Run time       : {total_elapsed / 60:.1f} min"
)


if (
    len(final_files) == total_patients
    and len(failed) == 0
):

    print("\nFULL TRAINING CACHE COMPLETE ✅")

else:

    print(
        "\nCache is incomplete. "
        "The cell can safely be run again to resume. ⚠️"
    )

    if failed:

        print("\nFailed patients:")

        for patient_id, error in failed:
            print(patient_id, "->", error)

## Full Training Cache Completed

The complete training dataset was successfully preprocessed and cached.

- **Training patients:** 875
- **Cached patients:** 875
- **Failed patients:** 0
- **Training slices:** 107,044
- **Cache format:** Compressed Float16
- **Final cache size:** 2.63 GB

The optimized cache eliminates repeated NIfTI loading and preprocessing during training.

**Status:** Full training cache completed successfully.

In [ ]:
# ============================================================
# 90. HÜCRE — CACHED TRAINING GENERATOR
# ============================================================

import os
import numpy as np
import tensorflow as tf
from functools import lru_cache


# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

CACHE_DIR = os.path.join(
    project_dir,
    "cache_float16_train"
)

BATCH_SIZE = 8


# ============================================================
# PATIENT CACHE LOADER
# ============================================================

@lru_cache(maxsize=8)
def load_cached_patient(patient_id):

    file_path = os.path.join(
        CACHE_DIR,
        f"{patient_id}.npz"
    )

    if not os.path.exists(file_path):
        raise FileNotFoundError(
            f"Cache not found: {file_path}"
        )

    with np.load(file_path) as data:

        X = data["X"]
        Y = data["Y"]
        slice_indices = data["slice_indices"]

    # slice_index -> cache position
    slice_map = {
        int(slice_idx): i
        for i, slice_idx in enumerate(slice_indices)
    }

    return X, Y, slice_map


# ============================================================
# CACHED GENERATOR
# ============================================================

class CachedBraTSGenerator(tf.keras.utils.Sequence):

    def __init__(
        self,
        dataframe,
        batch_size=8,
        shuffle=True
    ):

        super().__init__()

        self.df = dataframe.reset_index(drop=True).copy()

        self.batch_size = batch_size
        self.shuffle = shuffle

        self.indices = np.arange(
            len(self.df)
        )

        self.on_epoch_end()


    def __len__(self):

        return int(
            np.ceil(
                len(self.df)
                / self.batch_size
            )
        )


    def __getitem__(self, batch_index):

        start = (
            batch_index
            * self.batch_size
        )

        end = min(
            start + self.batch_size,
            len(self.df)
        )

        batch_indices = self.indices[
            start:end
        ]

        batch_rows = self.df.iloc[
            batch_indices
        ]


        X_batch = np.zeros(
            (
                len(batch_rows),
                128,
                128,
                4
            ),
            dtype=np.float32
        )

        Y_batch = np.zeros(
            (
                len(batch_rows),
                128,
                128,
                1
            ),
            dtype=np.float32
        )


        for i, (_, row) in enumerate(
            batch_rows.iterrows()
        ):

            patient_id = row[
                "patient_id"
            ]

            slice_index = int(
                row["slice_index"]
            )


            X_patient, Y_patient, slice_map = (
                load_cached_patient(
                    patient_id
                )
            )


            if slice_index not in slice_map:

                raise KeyError(
                    f"{patient_id} | "
                    f"slice {slice_index} "
                    f"not found in cache."
                )


            cache_position = slice_map[
                slice_index
            ]


            X_batch[i] = (
                X_patient[
                    cache_position
                ].astype(np.float32)
            )

            Y_batch[i] = (
                Y_patient[
                    cache_position
                ].astype(np.float32)
            )


        return X_batch, Y_batch


    def on_epoch_end(self):

        if self.shuffle:

            np.random.shuffle(
                self.indices
            )


# ============================================================
# CREATE TRAIN GENERATOR
# ============================================================

cached_train_generator = CachedBraTSGenerator(
    dataframe=clean_train_df,
    batch_size=BATCH_SIZE,
    shuffle=True
)


print("CACHED TRAIN GENERATOR CREATED ✅")
print("=" * 55)

print(
    "Training slices :",
    len(clean_train_df)
)

print(
    "Batch size      :",
    BATCH_SIZE
)

print(
    "Total batches   :",
    len(cached_train_generator)
)


# ============================================================
# FIRST BATCH CHECK
# ============================================================

X_cache_test, Y_cache_test = (
    cached_train_generator[0]
)

print("\nFIRST CACHED BATCH")
print("=" * 55)

print(
    "X shape:",
    X_cache_test.shape
)

print(
    "Y shape:",
    Y_cache_test.shape
)

print(
    "X dtype:",
    X_cache_test.dtype
)

print(
    "Y dtype:",
    Y_cache_test.dtype
)

print(
    "Mask classes:",
    np.unique(Y_cache_test)
)

print(
    "NaN detected:",
    np.isnan(X_cache_test).any()
)

print(
    "Inf detected:",
    np.isinf(X_cache_test).any()
)


print("\nCached generator ready. ✅")

In [ ]:
# ============================================================
# 91. HÜCRE — REAL CACHED GENERATOR SPEED TEST
#
# AMAÇ:
# 90. hücrede oluşturduğumuz cached_train_generator'ın
# gerçek kullanımda bir batch'i ne kadar hızlı hazırladığını
# ölçmek.
#
# NEDEN YAPIYORUZ?
# 84. hücrede eski veri pipeline'ının ortalama
# 28.31 saniye / batch sürdüğünü bulmuştuk.
#
# Daha sonra cache'in tek başına çok hızlı olduğunu gördük.
# Şimdi ise cache'i kullanan GERÇEK generator'ın hızını
# ölçerek optimizasyonun gerçekten işe yaradığını doğruluyoruz.
#
# BU HÜCRE:
# - Modeli eğitmez.
# - Model ağırlıklarını değiştirmez.
# - Cache dosyalarını değiştirmez.
# - Sadece veri yükleme süresini ölçer.
# ============================================================

import time
import numpy as np


print("REAL CACHED GENERATOR BENCHMARK")
print("=" * 55)


# ------------------------------------------------------------
# 1. WARM-UP
#
# İlk erişimde dosya sistemi/cache nedeniyle süre farklı
# olabilir. Bu nedenle bir batch'i önce bir kez okuyoruz.
# Bu süre benchmark hesabına dahil edilmeyecek.
# ------------------------------------------------------------

_ = cached_train_generator[0]

print("Warm-up completed. ✅")


# ------------------------------------------------------------
# 2. 10 FARKLI BATCH'İN YÜKLEME SÜRESİNİ ÖLÇ
#
# Her batch 8 MRI slice içeriyor.
# ------------------------------------------------------------

batch_times = []

test_batches = 10


for i in range(test_batches):

    start_time = time.time()

    X_test, Y_test = cached_train_generator[i]

    elapsed = time.time() - start_time

    batch_times.append(elapsed)

    print(
        f"Batch {i+1:02d}: "
        f"{elapsed:.4f} sec"
    )


# ------------------------------------------------------------
# 3. ORTALAMA YÜKLEME SÜRESİ
# ------------------------------------------------------------

average_time = np.mean(batch_times)

median_time = np.median(batch_times)


# Eski pipeline'da 84. hücrede ölçtüğümüz değer
old_pipeline_time = 28.31


# Eski sisteme göre yaklaşık veri yükleme hızlanması
speedup = old_pipeline_time / average_time


# ------------------------------------------------------------
# 4. SONUÇLAR
# ------------------------------------------------------------

print("\n" + "=" * 55)
print("GENERATOR SPEED SUMMARY")
print("=" * 55)

print(
    f"Old generator     : "
    f"{old_pipeline_time:.2f} sec/batch"
)

print(
    f"Cached generator  : "
    f"{average_time:.4f} sec/batch"
)

print(
    f"Median cached time: "
    f"{median_time:.4f} sec/batch"
)

print(
    f"Data speedup      : "
    f"{speedup:.1f}x"
)


# ------------------------------------------------------------
# 5. BASİT PERFORMANS DEĞERLENDİRMESİ
# ------------------------------------------------------------

if average_time < 1:

    print(
        "\n✅ Cached generator performance is excellent."
    )

elif average_time < 5:

    print(
        "\n✅ Cached generator provides a strong improvement."
    )

else:

    print(
        "\n⚠️ Cached generator may need further optimization."
    )


print("\n91. hücre tamamlandı. ✅")

## Cached Generator Benchmark

The cached pipeline improved real batch loading performance compared with direct NIfTI preprocessing.

- **Original generator:** 28.31 s/batch
- **Cached generator:** 5.04 s/batch
- **Observed speedup:** 5.6×
- **Median cached loading:** 5.64 s/batch

Although caching improved performance, random patient access still causes repeated compressed file loading from Google Drive.

**Decision:** Further optimize the cached data pipeline before full model training.

In [ ]:
# ============================================================
# 92. HÜCRE — PATIENT-GROUPED CACHED GENERATOR
#
# AMAÇ:
# Cache dosyalarını daha verimli kullanarak veri yükleme
# süresini azaltmak.
#
# NEDEN?
# 91. hücrede cached generator yaklaşık 5.04 sn/batch sürdü.
# Bunun önemli nedeni random slice erişimi sırasında farklı
# hastaların .npz dosyalarının tekrar tekrar açılmasıydı.
#
# ÇÖZÜM:
# Slice'ları hasta bazında gruplayarak bir batch'i mümkün
# olduğunca aynı hastanın cache dosyasından oluşturuyoruz.
#
# BU HÜCRE:
# - Modeli eğitmez.
# - Cache dosyalarını değiştirmez.
# - Sadece daha hızlı bir training generator oluşturur.
# ============================================================

import os
import time
import numpy as np
import tensorflow as tf


# ------------------------------------------------------------
# AYARLAR
# ------------------------------------------------------------

CACHE_DIR = os.path.join(
    project_dir,
    "cache_float16_train"
)

BATCH_SIZE = 8


# ============================================================
# PATIENT-GROUPED GENERATOR
# ============================================================

class PatientGroupedCachedGenerator(tf.keras.utils.Sequence):

    def __init__(
        self,
        dataframe,
        cache_dir,
        batch_size=8,
        shuffle=True
    ):

        super().__init__()

        self.df = dataframe.copy()
        self.cache_dir = cache_dir
        self.batch_size = batch_size
        self.shuffle = shuffle

        # Her hastanın kullanılacak slice indexlerini sakla.
        self.patient_slices = {}

        for patient_id, group in self.df.groupby("patient_id"):

            self.patient_slices[patient_id] = (
                group["slice_index"]
                .astype(int)
                .tolist()
            )

        self.patient_ids = list(
            self.patient_slices.keys()
        )

        # Eğitim sırasında kullanılacak batch planını oluştur.
        self.batch_plan = []

        self._build_batch_plan()


    def _build_batch_plan(self):

        self.batch_plan = []

        patient_order = self.patient_ids.copy()

        # Hasta sırasını karıştır.
        if self.shuffle:
            np.random.shuffle(patient_order)


        for patient_id in patient_order:

            slices = self.patient_slices[
                patient_id
            ].copy()

            # Aynı hastanın slice sırasını da karıştır.
            if self.shuffle:
                np.random.shuffle(slices)


            # Slice'ları batch_size kadar gruplara ayır.
            for start in range(
                0,
                len(slices),
                self.batch_size
            ):

                batch_slices = slices[
                    start:
                    start + self.batch_size
                ]

                self.batch_plan.append(
                    (
                        patient_id,
                        batch_slices
                    )
                )


        # Batch sırasını da karıştır.
        if self.shuffle:
            np.random.shuffle(
                self.batch_plan
            )


    def __len__(self):

        return len(
            self.batch_plan
        )


    def __getitem__(self, batch_index):

        # Bu batch'in hasta ve slice bilgilerini al.
        patient_id, requested_slices = (
            self.batch_plan[batch_index]
        )


        file_path = os.path.join(
            self.cache_dir,
            f"{patient_id}.npz"
        )


        if not os.path.exists(file_path):

            raise FileNotFoundError(
                f"Cache file not found: {file_path}"
            )


        # Hastanın cache dosyasını yalnızca bir kez aç.
        with np.load(file_path) as data:

            X_patient = data["X"]
            Y_patient = data["Y"]

            available_slices = (
                data["slice_indices"]
                .astype(int)
            )


        # Gerçek slice numarası -> cache içindeki konum
        slice_map = {
            int(slice_idx): position
            for position, slice_idx
            in enumerate(available_slices)
        }


        positions = []

        for slice_idx in requested_slices:

            if slice_idx not in slice_map:

                raise KeyError(
                    f"{patient_id} | "
                    f"slice {slice_idx} "
                    f"not found in cache."
                )

            positions.append(
                slice_map[slice_idx]
            )


        # Cache Float16.
        # Eğitim için Float32'ye çeviriyoruz.
        X_batch = (
            X_patient[positions]
            .astype(np.float32)
        )

        Y_batch = (
            Y_patient[positions]
            .astype(np.float32)
        )


        return X_batch, Y_batch


    def on_epoch_end(self):

        # Her epoch sonunda batch sırasını yeniden oluştur.
        if self.shuffle:
            self._build_batch_plan()


# ============================================================
# GENERATOR OLUŞTUR
# ============================================================

grouped_train_generator = (
    PatientGroupedCachedGenerator(
        dataframe=clean_train_df,
        cache_dir=CACHE_DIR,
        batch_size=BATCH_SIZE,
        shuffle=True
    )
)


print("PATIENT-GROUPED GENERATOR CREATED ✅")
print("=" * 55)

print(
    "Training patients:",
    len(grouped_train_generator.patient_ids)
)

print(
    "Training slices  :",
    len(clean_train_df)
)

print(
    "Batch size       :",
    BATCH_SIZE
)

print(
    "Total batches    :",
    len(grouped_train_generator)
)


# ============================================================
# İLK BATCH KONTROLÜ
# ============================================================

X_grouped, Y_grouped = (
    grouped_train_generator[0]
)

print("\nFIRST BATCH CHECK")
print("=" * 55)

print("X shape:", X_grouped.shape)
print("Y shape:", Y_grouped.shape)

print("X dtype:", X_grouped.dtype)
print("Y dtype:", Y_grouped.dtype)

print(
    "Mask classes:",
    np.unique(Y_grouped)
)

print(
    "NaN:",
    np.isnan(X_grouped).any()
)

print(
    "Inf:",
    np.isinf(X_grouped).any()
)


# ============================================================
# HIZ TESTİ
#
# 10 batch okuyarak gerçek veri yükleme hızını ölçüyoruz.
# ============================================================

print("\nSPEED TEST")
print("=" * 55)

test_times = []

test_count = min(
    10,
    len(grouped_train_generator)
)


for i in range(test_count):

    start_time = time.time()

    X_test, Y_test = (
        grouped_train_generator[i]
    )

    elapsed = (
        time.time()
        - start_time
    )

    test_times.append(elapsed)

    print(
        f"Batch {i+1:02d}: "
        f"{elapsed:.4f} sec"
    )


# ============================================================
# BENCHMARK SONUCU
# ============================================================

average_time = np.mean(test_times)
median_time = np.median(test_times)

old_nifti_time = 28.31
previous_cache_time = 5.0419


print("\n" + "=" * 55)
print("PATIENT-GROUPED BENCHMARK")
print("=" * 55)

print(
    f"Original NIfTI generator : "
    f"{old_nifti_time:.2f} sec/batch"
)

print(
    f"Previous cache generator : "
    f"{previous_cache_time:.2f} sec/batch"
)

print(
    f"Patient-grouped generator: "
    f"{average_time:.4f} sec/batch"
)

print(
    f"Median                   : "
    f"{median_time:.4f} sec/batch"
)

print(
    f"\nSpeedup vs original: "
    f"{old_nifti_time / average_time:.1f}x"
)

print(
    f"Speedup vs previous cache: "
    f"{previous_cache_time / average_time:.1f}x"
)


if average_time < 1:

    print(
        "\n✅ Performance is suitable for training."
    )

elif average_time < 3:

    print(
        "\n✅ Strong improvement."
    )

else:

    print(
        "\n⚠️ Google Drive I/O is still a bottleneck."
    )


print("\n92. hücre tamamlandı. ✅")

## Patient-Grouped Cache Optimization

The cached pipeline was optimized by grouping training slices by patient.

- **Original NIfTI generator:** 28.31 s/batch
- **Initial cached generator:** 5.04 s/batch
- **Patient-grouped generator:** 0.71 s/batch
- **Speedup vs. original:** 39.7×
- **Speedup vs. previous cache:** 7.1×

Patient-based grouping significantly reduces repeated Google Drive access and NPZ decompression.

**Result:** The optimized pipeline is suitable for model training.

In [ ]:
# ============================================================
# 93. HÜCRE — TRAINING DATA AUGMENTATION
#
# AMAÇ:
# Training sırasında MRI görüntülerine veri artırma
# (data augmentation) uygulamak.
#
# NEDEN?
# Modelin training görüntülerini ezberlemesini azaltmak ve
# daha önce görmediği hastalara daha iyi genellemesini sağlamak.
#
# UYGULANAN İŞLEMLER:
# - %50 ihtimalle horizontal flip
# - %30 ihtimalle 90°, 180° veya 270° rotation
#
# ÇOK ÖNEMLİ:
# MRI görüntüsüne uygulanan geometrik dönüşüm,
# Ground Truth maskesine de AYNI şekilde uygulanır.
#
# BU HÜCRE:
# - Modeli eğitmez.
# - Cache oluşturmaz.
# - Generator oluşturmaz.
# - Sadece augmentation sistemini tanımlar.
#
# Local training generator daha sonra 102. hücrede
# oluşturulacaktır.
# ============================================================

import numpy as np
import tensorflow as tf


# ============================================================
# 1. AUGMENTATION FONKSİYONU
# ============================================================

def augment_mri_batch(X, Y):

    # Orijinal batch'i değiştirmemek için kopyala.
    X_aug = X.copy()
    Y_aug = Y.copy()


    for i in range(len(X_aug)):

        # ----------------------------------------------------
        # HORIZONTAL FLIP
        # ----------------------------------------------------
        # %50 ihtimalle MRI ve maskeyi birlikte
        # sağ-sol çevir.
        # ----------------------------------------------------

        if np.random.rand() < 0.50:

            X_aug[i] = np.flip(
                X_aug[i],
                axis=1
            )

            Y_aug[i] = np.flip(
                Y_aug[i],
                axis=1
            )


        # ----------------------------------------------------
        # ROTATION
        # ----------------------------------------------------
        # %30 ihtimalle:
        #
        # k = 1 → 90 derece
        # k = 2 → 180 derece
        # k = 3 → 270 derece
        #
        # MRI ve maskeye aynı dönüşüm uygulanır.
        #
        # np.rot90 interpolation kullanmadığı için
        # maskenin binary 0/1 yapısı korunur.
        # ----------------------------------------------------

        if np.random.rand() < 0.30:

            k = np.random.choice(
                [1, 2, 3]
            )

            X_aug[i] = np.rot90(
                X_aug[i],
                k=k,
                axes=(0, 1)
            )

            Y_aug[i] = np.rot90(
                Y_aug[i],
                k=k,
                axes=(0, 1)
            )


    # TensorFlow training pipeline ile uyumlu olması için
    # float32 döndürüyoruz.

    return (
        X_aug.astype(np.float32),
        Y_aug.astype(np.float32)
    )


# ============================================================
# 2. AUGMENTATION GENERATOR CLASS
#
# Bu class başka bir generator'ın üzerine augmentation
# katmanı ekler.
#
# Örneğin:
#
# LOCAL CACHE
#      ↓
# PatientGroupedCachedGenerator
#      ↓
# AugmentedTrainingGenerator
#      ↓
# U-Net
#
# Asıl local generator 102. hücrede oluşturulacaktır.
# ============================================================

class AugmentedTrainingGenerator(
    tf.keras.utils.Sequence
):

    def __init__(self, base_generator):

        super().__init__()

        self.base_generator = base_generator


    def __len__(self):

        return len(
            self.base_generator
        )


    def __getitem__(self, index):

        # Base generator'dan MRI + mask batch al.
        X, Y = self.base_generator[index]


        # MRI ve maskeye aynı augmentation işlemlerini uygula.
        X_aug, Y_aug = augment_mri_batch(
            X,
            Y
        )


        return X_aug, Y_aug


    def on_epoch_end(self):

        # Base generator epoch sonunda shuffle destekliyorsa
        # onu da çalıştır.
        if hasattr(
            self.base_generator,
            "on_epoch_end"
        ):

            self.base_generator.on_epoch_end()


# ============================================================
# 3. CLASS TANIMLAMA KONTROLÜ
# ============================================================

augmentation_function_ready = (
    "augment_mri_batch" in globals()
)

augmentation_class_ready = (
    "AugmentedTrainingGenerator" in globals()
)


print("AUGMENTATION SYSTEM")
print("=" * 55)

print(
    "augment_mri_batch:",
    "✅" if augmentation_function_ready else "❌"
)

print(
    "AugmentedTrainingGenerator:",
    "✅" if augmentation_class_ready else "❌"
)


# ============================================================
# FINAL STATUS
# ============================================================

if (
    augmentation_function_ready
    and augmentation_class_ready
):

    print(
        "\n✅ AUGMENTATION SYSTEM READY"
    )

    print(
        "Local augmented generator will be created "
        "in cell 102."
    )

else:

    raise RuntimeError(
        "Augmentation system could not be initialized."
    )


print(
    "\n93. hücre tamamlandı. ✅"
)

## Training Data Augmentation

Controlled data augmentation was added to improve model generalization.

The training pipeline applies:

- Random horizontal flipping
- Random 90°, 180°, or 270° rotations
- Identical geometric transformations to MRI images and their masks

The original cached dataset remains unchanged because augmentation is performed dynamically during training.

**Result:** Augmented batches preserve the expected image/mask dimensions and binary mask structure.

In [ ]:
# ============================================================
# 94. HÜCRE — AUGMENTATION VISUAL SANITY CHECK
#
# AMAÇ:
# 93. hücrede eklediğimiz augmentation işlemlerinin
# MRI görüntüsü ile Ground Truth tümör maskesini
# birlikte ve doğru şekilde değiştirdiğini görsel olarak kontrol etmek.
#
# NEDEN?
# Shape kontrolünün doğru olması tek başına yeterli değildir.
# Örneğin MRI döndürülüp maskesi döndürülmezse model yanlış
# MRI-mask eşleşmelerinden öğrenir.
#
# BU HÜCRE:
# - Modeli eğitmez.
# - Cache'i değiştirmez.
# - Verileri kaydetmez/değiştirmez.
# - Sadece augmentation öncesi ve sonrası görüntüleri gösterir.
# ============================================================

import numpy as np
import matplotlib.pyplot as plt


# ============================================================
# 1. AYNI NORMAL BATCH'İ AL
# ============================================================

# 92'de oluşturduğumuz generator'dan augmentation uygulanmamış
# normal bir batch alıyoruz.
X_original, Y_original = grouped_train_generator[0]


# ============================================================
# 2. BU BATCH'E AUGMENTATION UYGULA
# ============================================================

# Aynı MRI ve maskelere 93'te yazdığımız augmentation
# fonksiyonunu uyguluyoruz.
X_augmented, Y_augmented = augment_mri_batch(
    X_original,
    Y_original
)


# ============================================================
# 3. GÖRSELLEŞTİRİLECEK ÖRNEK SAYISI
# ============================================================

# İlk 3 örneği incelemek yeterli.
num_examples = min(3, len(X_original))


# ============================================================
# 4. ORIGINAL vs AUGMENTED KARŞILAŞTIRMASI
#
# Her satırda:
#
# Original FLAIR | Original Mask | Augmented FLAIR | Augmented Mask
#
# gösterilecek.
#
# FLAIR = X[..., 3]
# Çünkü kanallarımız:
# 0 = T1
# 1 = T1ce
# 2 = T2
# 3 = FLAIR
# ============================================================

fig, axes = plt.subplots(
    num_examples,
    4,
    figsize=(14, 4 * num_examples)
)


# Tek örnek olması durumunda axes yapısını koru.
if num_examples == 1:
    axes = np.expand_dims(axes, axis=0)


for i in range(num_examples):

    # --------------------------------------------------------
    # ORIGINAL FLAIR
    # --------------------------------------------------------

    axes[i, 0].imshow(
        X_original[i, :, :, 3],
        cmap="gray"
    )

    axes[i, 0].set_title(
        "Original FLAIR"
    )

    axes[i, 0].axis("off")


    # --------------------------------------------------------
    # ORIGINAL GROUND TRUTH MASK
    # --------------------------------------------------------

    axes[i, 1].imshow(
        Y_original[i, :, :, 0],
        cmap="gray"
    )

    axes[i, 1].set_title(
        "Original Mask"
    )

    axes[i, 1].axis("off")


    # --------------------------------------------------------
    # AUGMENTED FLAIR
    # --------------------------------------------------------

    axes[i, 2].imshow(
        X_augmented[i, :, :, 3],
        cmap="gray"
    )

    axes[i, 2].set_title(
        "Augmented FLAIR"
    )

    axes[i, 2].axis("off")


    # --------------------------------------------------------
    # AUGMENTED GROUND TRUTH MASK
    # --------------------------------------------------------

    axes[i, 3].imshow(
        Y_augmented[i, :, :, 0],
        cmap="gray"
    )

    axes[i, 3].set_title(
        "Augmented Mask"
    )

    axes[i, 3].axis("off")


plt.tight_layout()
plt.show()


# ============================================================
# 5. SAYISAL KONTROL
# ============================================================

print("AUGMENTATION VISUAL CHECK")
print("=" * 55)

print(
    "Original X shape :",
    X_original.shape
)

print(
    "Augmented X shape:",
    X_augmented.shape
)

print(
    "Original Y shape :",
    Y_original.shape
)

print(
    "Augmented Y shape:",
    Y_augmented.shape
)

print(
    "Mask classes:",
    np.unique(Y_augmented)
)

print("\n94. hücre tamamlandı. ✅")
print(
    "MRI ve maskelerin birlikte hareket edip "
    "etmediğini yukarıdaki görüntülerden kontrol et."
)

## Augmentation Visual Validation

The augmentation pipeline was visually validated using FLAIR images and their corresponding tumor masks.

- MRI images and masks undergo identical transformations.
- Spatial alignment between the tumor region and ground-truth mask is preserved.
- Image and mask dimensions remain unchanged.
- Binary mask values remain `[0, 1]`.

**Result:** The augmentation pipeline is suitable for model training.

In [ ]:
# ============================================================
# 95. HÜCRE — TRAINING CALLBACKS
#
# AMAÇ:
# Gerçek model eğitimi sırasında:
#
# 1) En iyi modeli otomatik kaydetmek
# 2) Model gelişmeyi bırakırsa eğitimi durdurmak
# 3) Öğrenme tıkanırsa learning rate'i azaltmak
# 4) Epoch sonuçlarını Drive'a kaydetmek
#
# NEDEN?
# Eğitim uzun sürebilir ve Colab bağlantısı kesilebilir.
# Bu nedenle en iyi modelin ve training sonuçlarının
# kaybolmasını istemiyoruz.
#
# BU HÜCRE:
# - Modeli eğitmez.
# - Model ağırlıklarını değiştirmez.
# - Sadece gerçek training için güvenlik ve optimizasyon
#   mekanizmalarını hazırlar.
# ============================================================

import os
import tensorflow as tf


# ============================================================
# 1. MODEL / LOG KLASÖRLERİ
# ============================================================

model_dir = os.path.join(
    project_dir,
    "models"
)

results_dir = os.path.join(
    project_dir,
    "results"
)

os.makedirs(
    model_dir,
    exist_ok=True
)

os.makedirs(
    results_dir,
    exist_ok=True
)


# ============================================================
# 2. KAYDEDİLECEK DOSYALAR
# ============================================================

best_model_path = os.path.join(
    model_dir,
    "best_unet.keras"
)

training_log_path = os.path.join(
    results_dir,
    "training_log.csv"
)


# ============================================================
# 3. MODEL CHECKPOINT
#
# Validation Dice en yüksek olduğunda modeli Drive'a kaydeder.
#
# Böylece sonraki epoch'larda performans kötüleşse bile
# elimizde en iyi model kalır.
# ============================================================

checkpoint_callback = (
    tf.keras.callbacks.ModelCheckpoint(
        filepath=best_model_path,

        monitor="val_dice_coefficient",

        mode="max",

        save_best_only=True,

        verbose=1
    )
)


# ============================================================
# 4. EARLY STOPPING
#
# Validation Dice 5 epoch boyunca iyileşmezse training durur.
#
# restore_best_weights=True:
# Training sonunda modelin RAM'deki ağırlıklarını da
# en iyi epoch'a geri döndürür.
# ============================================================

early_stopping_callback = (
    tf.keras.callbacks.EarlyStopping(
        monitor="val_dice_coefficient",

        mode="max",

        patience=5,

        restore_best_weights=True,

        verbose=1
    )
)


# ============================================================
# 5. REDUCE LEARNING RATE
#
# Validation Dice 2 epoch boyunca gelişmezse learning rate'i
# yarıya indirir.
#
# Örneğin:
#
# 1e-4
#   ↓
# 5e-5
#   ↓
# 2.5e-5
#
# Böylece model optimum noktaya daha kontrollü yaklaşabilir.
# ============================================================

reduce_lr_callback = (
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_dice_coefficient",

        mode="max",

        factor=0.5,

        patience=2,

        min_lr=1e-7,

        verbose=1
    )
)


# ============================================================
# 6. CSV LOGGER
#
# Her epoch sonunda:
#
# loss
# dice
# IoU
# val_loss
# val_dice
# val_IoU
#
# gibi sonuçları Drive'daki CSV dosyasına kaydeder.
#
# Böylece daha sonra learning curve grafiklerini
# oluşturabileceğiz.
# ============================================================

csv_logger_callback = (
    tf.keras.callbacks.CSVLogger(
        training_log_path,
        append=False
    )
)


# ============================================================
# 7. CALLBACK LİSTESİ
#
# Gerçek training sırasında model.fit(...)
# içerisine bu listeyi vereceğiz.
# ============================================================

training_callbacks = [

    checkpoint_callback,

    early_stopping_callback,

    reduce_lr_callback,

    csv_logger_callback
]


# ============================================================
# 8. KONTROL
# ============================================================

print("TRAINING CALLBACKS READY ✅")
print("=" * 55)

print(
    "Best model path :",
    best_model_path
)

print(
    "Training log    :",
    training_log_path
)

print(
    "Monitor metric  : val_dice_coefficient"
)

print(
    "Early stopping  : 5 epochs"
)

print(
    "LR patience     : 2 epochs"
)

print(
    "Callbacks       :",
    len(training_callbacks)
)

print("\n95. hücre tamamlandı. ✅")

In [ ]:
# ============================================================
# 96. HÜCRE — VALIDATION PIPELINE BENCHMARK
#
# AMAÇ:
# Gerçek training'e başlamadan önce validation generator'ın
# hızını ölçmek.
#
# NEDEN?
# Training generator'ı yaklaşık:
#     28.31 sec/batch → 0.71 sec/batch
# seviyesine kadar hızlandırdık.
#
# Fakat her epoch sonunda model validation verisini de
# işleyecek. Validation yavaşsa training yine çok uzun sürer.
#
# BU HÜCRE:
# - Modeli eğitmez.
# - Model ağırlıklarını değiştirmez.
# - Cache dosyalarını değiştirmez.
# - Sadece validation generator'ın hızını ve yapısını test eder.
# ============================================================

import time
import numpy as np


print("VALIDATION PIPELINE BENCHMARK")
print("=" * 55)


# ============================================================
# 1. GENERATOR KONTROLÜ
# ============================================================

print(
    "Validation batches:",
    len(validation_generator)
)


# ============================================================
# 2. İLK BATCH KONTROLÜ
#
# Validation verisinin modele doğru formatta gelip
# gelmediğini kontrol ediyoruz.
# ============================================================

X_val_test, Y_val_test = validation_generator[0]


print("\nVALIDATION BATCH CHECK")
print("=" * 55)

print(
    "X shape:",
    X_val_test.shape
)

print(
    "Y shape:",
    Y_val_test.shape
)

print(
    "X dtype:",
    X_val_test.dtype
)

print(
    "Y dtype:",
    Y_val_test.dtype
)

print(
    "Mask classes:",
    np.unique(Y_val_test)
)

print(
    "NaN detected:",
    np.isnan(X_val_test).any()
)

print(
    "Inf detected:",
    np.isinf(X_val_test).any()
)


# ============================================================
# 3. WARM-UP
#
# İlk disk erişimi normalden farklı sürebilir.
# Bu nedenle ilk erişimi benchmark dışında bırakıyoruz.
# ============================================================

_ = validation_generator[0]

print("\nWarm-up completed. ✅")


# ============================================================
# 4. 10 VALIDATION BATCH'İNİN HIZINI ÖLÇ
# ============================================================

validation_times = []

test_batches = min(
    10,
    len(validation_generator)
)


print("\nSPEED TEST")
print("=" * 55)


for i in range(test_batches):

    start_time = time.time()

    X_test, Y_test = validation_generator[i]

    elapsed = time.time() - start_time

    validation_times.append(
        elapsed
    )

    print(
        f"Batch {i+1:02d}: "
        f"{elapsed:.4f} sec"
    )


# ============================================================
# 5. ORTALAMA VE MEDYAN
# ============================================================

average_val_time = np.mean(
    validation_times
)

median_val_time = np.median(
    validation_times
)


print("\n" + "=" * 55)
print("VALIDATION BENCHMARK SUMMARY")
print("=" * 55)

print(
    f"Average loading : "
    f"{average_val_time:.4f} sec/batch"
)

print(
    f"Median loading  : "
    f"{median_val_time:.4f} sec/batch"
)

print(
    f"Total batches   : "
    f"{len(validation_generator)}"
)


# ============================================================
# 6. KABA EPOCH VALIDATION SÜRESİ TAHMİNİ
#
# DİKKAT:
# Bu sadece DATA LOADING süresinin tahminidir.
# Model forward-pass süresi dahil değildir.
# ============================================================

estimated_seconds = (
    average_val_time
    * len(validation_generator)
)

estimated_minutes = (
    estimated_seconds / 60
)

estimated_hours = (
    estimated_minutes / 60
)


print("\nESTIMATED VALIDATION DATA-LOADING TIME")
print("=" * 55)

print(
    f"Seconds : {estimated_seconds:.1f}"
)

print(
    f"Minutes : {estimated_minutes:.1f}"
)

print(
    f"Hours   : {estimated_hours:.2f}"
)


# ============================================================
# 7. PERFORMANS DEĞERLENDİRMESİ
# ============================================================

print("\nPIPELINE STATUS")
print("=" * 55)


if average_val_time < 1:

    print(
        "✅ Validation loading performance is suitable."
    )

elif average_val_time < 3:

    print(
        "⚠️ Validation loading is usable, "
        "but optimization may help."
    )

else:

    print(
        "❌ Validation loading is a bottleneck."
    )

    print(
        "Validation cache optimization is recommended "
        "before full training."
    )


print("\n96. hücre tamamlandı. ✅")

## Validation Pipeline Benchmark

The validation pipeline was benchmarked before full training.

- **Validation batches:** 3,643
- **Average loading time:** 0.0094 s/batch
- **Median loading time:** 0.0091 s/batch
- **Estimated total data-loading time:** ~34 seconds
- No NaN or infinite values were detected.

**Result:** Validation data loading is sufficiently fast and does not require additional optimization.

In [ ]:
# ============================================================
# 97. HÜCRE — FINAL PRE-TRAINING CHECK
#
# AMAÇ:
# Gerçek U-Net eğitimini başlatmadan önce bütün training
# pipeline'ının son kez kontrol edilmesi.
#
# KONTROL EDİLECEKLER:
# 1) Model hazır mı?
# 2) Training generator hazır mı?
# 3) Validation generator hazır mı?
# 4) Training ve validation shape'leri uyumlu mu?
# 5) Tümörlü maskeler gerçekten mevcut mu?
# 6) NaN / Inf var mı?
# 7) Loss ve metric isimleri doğru mu?
# 8) Callback'ler hazır mı?
#
# BU HÜCRE:
# - Modeli EĞİTMEZ.
# - Model ağırlıklarını değiştirmez.
# - Cache dosyalarını değiştirmez.
# ============================================================

import numpy as np
import tensorflow as tf


print("FINAL PRE-TRAINING CHECK")
print("=" * 60)


# ============================================================
# 1. MODEL KONTROLÜ
# ============================================================

print("\n1) MODEL")
print("-" * 60)

print("Input shape :", model.input_shape)
print("Output shape:", model.output_shape)

print(
    "Parameters  :",
    f"{model.count_params():,}"
)

print(
    "Optimizer   :",
    type(model.optimizer).__name__
)

print(
    "Learning rate:",
    float(
        tf.keras.backend.get_value(
            model.optimizer.learning_rate
        )
    )
)


# ============================================================
# 2. TRAINING GENERATOR KONTROLÜ
# ============================================================

print("\n2) TRAINING GENERATOR")
print("-" * 60)

X_train_check, Y_train_check = (
    augmented_train_generator[0]
)

print(
    "Total batches:",
    len(augmented_train_generator)
)

print(
    "X shape:",
    X_train_check.shape
)

print(
    "Y shape:",
    Y_train_check.shape
)

print(
    "Mask classes:",
    np.unique(Y_train_check)
)

print(
    "NaN:",
    np.isnan(X_train_check).any()
)

print(
    "Inf:",
    np.isinf(X_train_check).any()
)


# ============================================================
# 3. VALIDATION GENERATOR KONTROLÜ
# ============================================================

print("\n3) VALIDATION GENERATOR")
print("-" * 60)

X_val_check, Y_val_check = (
    validation_generator[0]
)

print(
    "Total batches:",
    len(validation_generator)
)

print(
    "X shape:",
    X_val_check.shape
)

print(
    "Y shape:",
    Y_val_check.shape
)

print(
    "First batch mask classes:",
    np.unique(Y_val_check)
)

print(
    "NaN:",
    np.isnan(X_val_check).any()
)

print(
    "Inf:",
    np.isinf(X_val_check).any()
)


# ============================================================
# 4. VALIDATION'DA TÜMÖRLÜ BATCH ARA
#
# İlk validation batch'i tamamen background olabilir.
# Bu hata değildir.
#
# Burada ilk 100 batch içinde en az bir tane tumor pixel
# bulunan batch olup olmadığını kontrol ediyoruz.
# ============================================================

print("\n4) VALIDATION TUMOR CHECK")
print("-" * 60)

tumor_batch_found = False
tumor_batch_index = None
tumor_pixels = 0

max_batches_to_check = min(
    100,
    len(validation_generator)
)


for i in range(max_batches_to_check):

    _, Y_temp = validation_generator[i]

    current_tumor_pixels = int(
        np.sum(Y_temp > 0.5)
    )

    if current_tumor_pixels > 0:

        tumor_batch_found = True
        tumor_batch_index = i
        tumor_pixels = current_tumor_pixels

        break


if tumor_batch_found:

    print("Tumor batch found: ✅")
    print(
        "Batch index:",
        tumor_batch_index
    )
    print(
        "Tumor pixels:",
        tumor_pixels
    )

else:

    print(
        "⚠️ No tumor batch found in first",
        max_batches_to_check,
        "validation batches."
    )


# ============================================================
# 5. INPUT / OUTPUT UYUMU
# ============================================================

print("\n5) SHAPE COMPATIBILITY")
print("-" * 60)

train_input_ok = (
    X_train_check.shape[1:]
    == model.input_shape[1:]
)

val_input_ok = (
    X_val_check.shape[1:]
    == model.input_shape[1:]
)

train_output_ok = (
    Y_train_check.shape[1:]
    == model.output_shape[1:]
)

val_output_ok = (
    Y_val_check.shape[1:]
    == model.output_shape[1:]
)


print(
    "Train input → Model:",
    "✅" if train_input_ok else "❌"
)

print(
    "Val input → Model  :",
    "✅" if val_input_ok else "❌"
)

print(
    "Train mask → Output:",
    "✅" if train_output_ok else "❌"
)

print(
    "Val mask → Output  :",
    "✅" if val_output_ok else "❌"
)


# ============================================================
# 6. CALLBACK KONTROLÜ
# ============================================================

print("\n6) CALLBACKS")
print("-" * 60)

print(
    "Callbacks ready:",
    len(training_callbacks)
)

for callback in training_callbacks:

    print(
        " -",
        type(callback).__name__
    )


# ============================================================
# 7. LOSS / METRIC KONTROLÜ
# ============================================================

print("\n7) TRAINING CONFIGURATION")
print("-" * 60)

print(
    "Loss:",
    getattr(
        model.loss,
        "__name__",
        str(model.loss)
    )
)

print("Metrics expected:")
print(" - dice_coefficient")
print(" - iou_metric")


# ============================================================
# 8. FINAL STATUS
# ============================================================

all_shapes_ok = all([
    train_input_ok,
    val_input_ok,
    train_output_ok,
    val_output_ok
])

data_ok = (
    not np.isnan(X_train_check).any()
    and
    not np.isinf(X_train_check).any()
    and
    not np.isnan(X_val_check).any()
    and
    not np.isinf(X_val_check).any()
)


print("\n" + "=" * 60)
print("FINAL STATUS")
print("=" * 60)

print(
    "Shape compatibility :",
    "✅" if all_shapes_ok else "❌"
)

print(
    "Numerical integrity :",
    "✅" if data_ok else "❌"
)

print(
    "Validation tumor    :",
    "✅" if tumor_batch_found else "⚠️ CHECK"
)

print(
    "Callbacks           :",
    "✅" if len(training_callbacks) == 4 else "⚠️ CHECK"
)


if (
    all_shapes_ok
    and data_ok
    and tumor_batch_found
    and len(training_callbacks) == 4
):

    print("\n🚀 PIPELINE READY FOR FULL TRAINING")

else:

    print(
        "\n⚠️ CHECK WARNINGS BEFORE FULL TRAINING"
    )


print("\n97. hücre tamamlandı. ✅")

In [ ]:
# ============================================================
# 98. HÜCRE — REAL TRAINING SPEED BENCHMARK
#
# AMAÇ:
# Full training'e başlamadan önce gerçek training hızını ölçmek.
#
# NEDEN?
# Toplam 13,758 training batch var.
# Bilgisayarın GPU/CPU durumuna göre 1 epoch'un süresi
# ciddi şekilde değişebilir.
#
# BU HÜCRE:
# 1) GPU var mı kontrol eder.
# 2) Modelin mevcut ağırlıklarını RAM'de saklar.
# 3) 20 gerçek training step çalıştırır.
# 4) Ortalama step süresini hesaplar.
# 5) Full epoch süresini tahmin eder.
# 6) Testten sonra eski model ağırlıklarını geri yükler.
#
# ÖNEMLİ:
# Bu bir BENCHMARK'tır.
# Kalıcı training değildir.
# ============================================================

import time
import numpy as np
import tensorflow as tf


print("REAL TRAINING SPEED BENCHMARK")
print("=" * 60)


# ============================================================
# 1. HARDWARE KONTROLÜ
# ============================================================

gpus = tf.config.list_physical_devices("GPU")

if len(gpus) > 0:

    print("GPU detected: ✅")

    for gpu in gpus:
        print(" -", gpu.name)

else:

    print("GPU detected: ❌")
    print("Training is currently running on CPU.")


# ============================================================
# 2. MODEL AĞIRLIKLARINI YEDEKLE
#
# Benchmark sırasında model gerçekten gradient update yapacak.
# Bu nedenle mevcut ağırlıkları RAM'e kaydediyoruz.
#
# Test sonunda geri yükleyeceğiz.
# ============================================================

original_weights = model.get_weights()

print("\nModel weights backed up. ✅")


# ============================================================
# 3. BENCHMARK AYARLARI
# ============================================================

BENCHMARK_STEPS = 20

step_times = []

benchmark_losses = []


print("\nTRAINING STEP TEST")
print("=" * 60)

print(
    "Benchmark steps:",
    BENCHMARK_STEPS
)


# ============================================================
# 4. GERÇEK TRAINING STEP'LERİ
#
# train_on_batch:
# - forward pass
# - loss calculation
# - backpropagation
# - optimizer update
#
# yaptığı için gerçek training maliyetini ölçer.
# ============================================================

for step in range(BENCHMARK_STEPS):

    # Training generator'dan augmented batch al.
    X_batch, Y_batch = (
        augmented_train_generator[step]
    )

    start_time = time.time()

    results = model.train_on_batch(
        X_batch,
        Y_batch,
        return_dict=True
    )

    elapsed = time.time() - start_time

    step_times.append(elapsed)

    benchmark_losses.append(
        float(results["loss"])
    )

    print(
        f"Step {step + 1:02d}/{BENCHMARK_STEPS} | "
        f"{elapsed:.3f} sec | "
        f"loss: {float(results['loss']):.4f}"
    )


# ============================================================
# 5. MODELİ BENCHMARK ÖNCESİ DURUMUNA GERİ DÖNDÜR
# ============================================================

model.set_weights(original_weights)

print("\nOriginal model weights restored. ✅")


# ============================================================
# 6. HIZ HESAPLARI
#
# İlk birkaç step TensorFlow warm-up / graph tracing nedeniyle
# normalden yavaş olabilir.
#
# Bu yüzden mümkünse ilk 3 step'i ortalamadan çıkarıyoruz.
# ============================================================

if len(step_times) > 3:

    stable_times = step_times[3:]

else:

    stable_times = step_times


average_step_time = np.mean(
    stable_times
)

median_step_time = np.median(
    stable_times
)


# ============================================================
# 7. FULL EPOCH SÜRESİ TAHMİNİ
#
# Bu tahmin yalnızca training step süresini kullanır.
# Validation ve callback süresi ayrıca eklenecektir.
# ============================================================

total_train_batches = len(
    augmented_train_generator
)

estimated_epoch_seconds = (
    average_step_time
    * total_train_batches
)

estimated_epoch_minutes = (
    estimated_epoch_seconds / 60
)

estimated_epoch_hours = (
    estimated_epoch_minutes / 60
)


# ============================================================
# 8. SONUÇLAR
# ============================================================

print("\n" + "=" * 60)
print("TRAINING SPEED SUMMARY")
print("=" * 60)

print(
    f"Average stable step : "
    f"{average_step_time:.3f} sec"
)

print(
    f"Median stable step  : "
    f"{median_step_time:.3f} sec"
)

print(
    f"Training batches    : "
    f"{total_train_batches}"
)

print(
    "\nEstimated FULL epoch:"
)

print(
    f"  {estimated_epoch_seconds:.0f} seconds"
)

print(
    f"  {estimated_epoch_minutes:.1f} minutes"
)

print(
    f"  {estimated_epoch_hours:.2f} hours"
)


# ============================================================
# 9. ÖRNEK TOPLAM TRAINING TAHMİNLERİ
#
# EarlyStopping nedeniyle gerçek training daha erken
# durabilir. Bunlar sadece kaba tahminlerdir.
# ============================================================

print("\nROUGH TRAINING ESTIMATES")
print("=" * 60)

for epochs in [5, 10, 20]:

    hours = (
        estimated_epoch_hours
        * epochs
    )

    print(
        f"{epochs:02d} epochs ≈ "
        f"{hours:.1f} hours"
    )


# ============================================================
# 10. FINAL STATUS
# ============================================================

print("\n" + "=" * 60)

if len(gpus) > 0:

    print(
        "✅ GPU available for full training."
    )

else:

    print(
        "⚠️ No GPU detected."
    )

    print(
        "Do NOT start long full training "
        "until the estimated duration is reviewed."
    )


print("\n98. hücre tamamlandı. ✅")

## Training Speed Benchmark

A real GPU training benchmark was performed before full model training.

- **GPU:** Available
- **Average stable training step:** 0.064 s
- **Training batches:** 13,758
- **Estimated full epoch:** ~14.8 minutes
- **Estimated 10 epochs:** ~2.5 hours
- **Estimated 20 epochs:** ~4.9 hours

The benchmark weights were restored afterward, so the test did not affect the final model.

**Result:** The pipeline is computationally ready for full GPU training.

In [ ]:
# ============================================================
# 100. HÜCRE — COPY TRAINING CACHE TO LOCAL COLAB DISK
#
# AMAÇ:
# Training sırasında Google Drive I/O darboğazını kaldırmak.
#
# ŞİMDİKİ DURUM:
# Google Drive → NPZ → Generator → GPU
#
# YENİ DURUM:
# Google Drive → (bir kez kopyala) → /content → Generator → GPU
#
# NEDEN?
# Google Drive çok sayıda küçük/orta dosyaya sürekli erişimde
# yavaş kalabiliyor. Colab'ın /content diski çok daha hızlıdır.
#
# ÖNEMLİ:
# - Mevcut 2.63 GB cache yeniden oluşturulmaz.
# - Drive'daki cache silinmez/değiştirilmez.
# - Sadece geçici local kopya oluşturulur.
# - Colab runtime kapanırsa /content içindeki kopya silinir.
# ============================================================

import os
import shutil
import time


# ------------------------------------------------------------
# 1. KAYNAK VE HEDEF
# ------------------------------------------------------------

drive_cache_dir = os.path.join(
    project_dir,
    "cache_float16_train"
)

local_cache_dir = (
    "/content/cache_float16_train"
)


print("LOCAL CACHE SETUP")
print("=" * 60)

print(
    "Source:",
    drive_cache_dir
)

print(
    "Target:",
    local_cache_dir
)


# ------------------------------------------------------------
# 2. DRIVE CACHE KONTROLÜ
# ------------------------------------------------------------

if not os.path.exists(drive_cache_dir):

    raise FileNotFoundError(
        f"Drive cache not found: {drive_cache_dir}"
    )


drive_files = [
    f
    for f in os.listdir(drive_cache_dir)
    if f.endswith(".npz")
]


print(
    "\nDrive cache files:",
    len(drive_files)
)


# ------------------------------------------------------------
# 3. LOCAL KLASÖRÜ OLUŞTUR
# ------------------------------------------------------------

os.makedirs(
    local_cache_dir,
    exist_ok=True
)


# ------------------------------------------------------------
# 4. CACHE DOSYALARINI KOPYALA
#
# Daha önce kopyalanmış ve aynı boyutta olan dosyaları
# tekrar kopyalamıyoruz.
# ------------------------------------------------------------

start_time = time.time()

copied = 0
skipped = 0


for i, filename in enumerate(
    drive_files,
    start=1
):

    source = os.path.join(
        drive_cache_dir,
        filename
    )

    target = os.path.join(
        local_cache_dir,
        filename
    )


    # Dosya zaten local'de ve boyutu aynıysa atla.
    if (
        os.path.exists(target)
        and
        os.path.getsize(target)
        == os.path.getsize(source)
    ):

        skipped += 1

    else:

        shutil.copy2(
            source,
            target
        )

        copied += 1


    # Her 50 dosyada ilerleme göster.
    if (
        i % 50 == 0
        or i == len(drive_files)
    ):

        print(
            f"{i:03d}/{len(drive_files)} | "
            f"copied: {copied} | "
            f"skipped: {skipped}"
        )


# ------------------------------------------------------------
# 5. SONUÇ KONTROLÜ
# ------------------------------------------------------------

elapsed = time.time() - start_time


local_files = [
    f
    for f in os.listdir(local_cache_dir)
    if f.endswith(".npz")
]


local_size_gb = sum(
    os.path.getsize(
        os.path.join(
            local_cache_dir,
            f
        )
    )
    for f in local_files
) / (1024 ** 3)


print("\n" + "=" * 60)
print("LOCAL CACHE SUMMARY")
print("=" * 60)

print(
    "Drive files :",
    len(drive_files)
)

print(
    "Local files :",
    len(local_files)
)

print(
    "Copied      :",
    copied
)

print(
    "Skipped     :",
    skipped
)

print(
    f"Local size  : {local_size_gb:.2f} GB"
)

print(
    f"Copy time   : {elapsed / 60:.1f} minutes"
)


# ------------------------------------------------------------
# 6. GÜVENLİK KONTROLÜ
# ------------------------------------------------------------

if len(local_files) == len(drive_files):

    print(
        "\n✅ LOCAL TRAINING CACHE READY"
    )

else:

    print(
        "\n❌ File count mismatch."
    )


print(
    "\n100. hücre tamamlandı. ✅"
)

In [ ]:
# ============================================================
# 101. HÜCRE — LOCAL CACHE GENERATOR + SPEED TEST
#
# AMAÇ:
# 100. hücrede Google Drive'dan Colab'ın local diskine
# kopyaladığımız 2.63 GB training cache'i kullanmak.
#
# ESKİ PIPELINE:
# Google Drive
#      ↓
# Patient-Grouped Generator
#      ↓
# Augmentation
#      ↓
# GPU
#
# YENİ PIPELINE:
# /content LOCAL DISK
#      ↓
# Patient-Grouped Generator
#      ↓
# Augmentation
#      ↓
# GPU
#
# NEDEN?
# Google Drive erişimi full training sırasında GPU'nun
# veri beklemesine neden oldu.
#
# BU HÜCRE:
# - Modeli eğitmez.
# - Model ağırlıklarını değiştirmez.
# - Cache'i değiştirmez.
# - Local generator oluşturur ve hızını ölçer.
# ============================================================

import time
import numpy as np


print("LOCAL CACHE GENERATOR SETUP")
print("=" * 60)


# ============================================================
# 1. LOCAL CACHE DİZİNİ
# ============================================================

LOCAL_CACHE_DIR = "/content/cache_float16_train"

print(
    "Local cache:",
    LOCAL_CACHE_DIR
)


# ============================================================
# 2. LOCAL PATIENT-GROUPED GENERATOR
#
# 92'de oluşturduğumuz aynı generator class'ını kullanıyoruz.
# Tek fark:
#
# cache_dir artık Google Drive değil,
# Colab'ın local /content diski.
# ============================================================

local_grouped_train_generator = (
    PatientGroupedCachedGenerator(
        dataframe=clean_train_df,
        cache_dir=LOCAL_CACHE_DIR,
        batch_size=BATCH_SIZE,
        shuffle=True
    )
)


print(
    "\nLocal grouped generator created. ✅"
)

print(
    "Training patients:",
    len(
        local_grouped_train_generator.patient_ids
    )
)

print(
    "Total batches:",
    len(
        local_grouped_train_generator
    )
)


# ============================================================
# 3. AUGMENTATION WRAPPER
#
# 93'te oluşturduğumuz augmentation sistemi aynen korunuyor.
# Ancak artık veriler Drive yerine local diskten geliyor.
# ============================================================

local_augmented_train_generator = (
    AugmentedTrainingGenerator(
        local_grouped_train_generator
    )
)


print(
    "Local augmented generator created. ✅"
)


# ============================================================
# 4. İLK BATCH KONTROLÜ
# ============================================================

X_local, Y_local = (
    local_augmented_train_generator[0]
)


print("\nLOCAL BATCH CHECK")
print("=" * 60)

print(
    "X shape:",
    X_local.shape
)

print(
    "Y shape:",
    Y_local.shape
)

print(
    "X dtype:",
    X_local.dtype
)

print(
    "Y dtype:",
    Y_local.dtype
)

print(
    "Mask classes:",
    np.unique(Y_local)
)

print(
    "NaN:",
    np.isnan(X_local).any()
)

print(
    "Inf:",
    np.isinf(X_local).any()
)


# ============================================================
# 5. WARM-UP
#
# İlk dosya erişimini benchmark dışında bırakıyoruz.
# ============================================================

_ = local_augmented_train_generator[0]

print(
    "\nWarm-up completed. ✅"
)


# ============================================================
# 6. 20 BATCH DATA-LOADING TEST
#
# Burada model kullanılmıyor.
# Sadece local disk → generator → augmentation süresini
# ölçüyoruz.
# ============================================================

TEST_BATCHES = 20

loading_times = []


print("\nLOCAL DATA SPEED TEST")
print("=" * 60)


for i in range(TEST_BATCHES):

    start_time = time.time()

    X_test, Y_test = (
        local_augmented_train_generator[i]
    )

    elapsed = (
        time.time()
        - start_time
    )

    loading_times.append(
        elapsed
    )

    print(
        f"Batch {i+1:02d}: "
        f"{elapsed:.4f} sec"
    )


# ============================================================
# 7. SONUÇLAR
# ============================================================

average_local_time = np.mean(
    loading_times
)

median_local_time = np.median(
    loading_times
)


# Daha önce ölçtüğümüz Drive patient-grouped sonucu
drive_grouped_time = 0.7136


speedup_vs_drive = (
    drive_grouped_time
    / average_local_time
)


print("\n" + "=" * 60)
print("LOCAL CACHE BENCHMARK")
print("=" * 60)

print(
    f"Drive patient-grouped : "
    f"{drive_grouped_time:.4f} sec/batch"
)

print(
    f"Local cache generator : "
    f"{average_local_time:.4f} sec/batch"
)

print(
    f"Local median          : "
    f"{median_local_time:.4f} sec/batch"
)

print(
    f"Speedup vs Drive      : "
    f"{speedup_vs_drive:.1f}x"
)


# ============================================================
# 8. KABA DATA-LOADING EPOCH TAHMİNİ
#
# Bu yalnızca generator/data-loading süresidir.
# GPU training süresi ayrıca vardır.
# ============================================================

total_batches = len(
    local_augmented_train_generator
)

estimated_loading_seconds = (
    average_local_time
    * total_batches
)

estimated_loading_minutes = (
    estimated_loading_seconds
    / 60
)


print(
    "\nEstimated data-loading time / epoch:"
)

print(
    f"{estimated_loading_seconds:.1f} seconds"
)

print(
    f"{estimated_loading_minutes:.1f} minutes"
)


# ============================================================
# 9. FINAL STATUS
# ============================================================

print("\n" + "=" * 60)

if average_local_time < 0.10:

    print(
        "🚀 EXCELLENT — local cache is ready for training."
    )

elif average_local_time < 0.30:

    print(
        "✅ GOOD — local cache provides strong performance."
    )

else:

    print(
        "⚠️ Local loading is still slower than expected."
    )


print(
    "\n101. hücre tamamlandı. ✅"
)

In [ ]:
# ============================================================
# 102. HÜCRE — MASTER RECOVERY / LOCAL TRAINING SETUP
#
# AMAÇ:
# Colab runtime sorunlarından sonra training'e dönmeyi
# mümkün olduğunca kolaylaştırmak.
#
# KONTROL EDER:
# 1) project_dir mevcut mu?
# 2) Gerekli dataframe mevcut mu?
# 3) Generator class'ları mevcut mu?
# 4) Drive training cache mevcut mu?
# 5) Local /content cache eksikse yeniden kopyalar.
# 6) Local patient-grouped generator oluşturur.
# 7) Augmentation generator oluşturur.
# 8) Son batch ile pipeline'ı doğrular.
#
# NOT:
# Runtime tamamen sıfırlanırsa temel SESSION RESTORE
# hücrelerini yine bir kez çalıştırmak gerekir.
# Fakat local-cache/training setup bundan sonra bu hücreyle
# tek seferde yeniden kurulabilir.
# ============================================================

import os
import shutil
import numpy as np


print("MASTER TRAINING RECOVERY")
print("=" * 60)


# ============================================================
# 1. RAM'DEKİ TEMEL NESNELERİ KONTROL ET
# ============================================================

required_objects = [
    "project_dir",
    "clean_train_df",
    "PatientGroupedCachedGenerator",
    "AugmentedTrainingGenerator",
    "model",
    "validation_generator"
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]


if missing_objects:

    print("❌ Missing RAM objects:")

    for name in missing_objects:
        print(" -", name)

    raise RuntimeError(
        "\nRun SESSION RESTORE first. "
        "Then run cells 92 and 93 if their classes "
        "are still missing, and rerun this cell."
    )


print("Core RAM objects: ✅")


# ============================================================
# 2. CACHE PATHS
# ============================================================

DRIVE_CACHE_DIR = os.path.join(
    project_dir,
    "cache_float16_train"
)

LOCAL_CACHE_DIR = (
    "/content/cache_float16_train"
)


if not os.path.isdir(DRIVE_CACHE_DIR):

    raise FileNotFoundError(
        f"Drive cache not found: {DRIVE_CACHE_DIR}"
    )


drive_files = sorted([
    f
    for f in os.listdir(DRIVE_CACHE_DIR)
    if f.endswith(".npz")
])


print(
    "Drive cache files:",
    len(drive_files)
)


if len(drive_files) == 0:

    raise RuntimeError(
        "Drive cache folder contains no NPZ files."
    )


# ============================================================
# 3. LOCAL CACHE KLASÖRÜ
# ============================================================

os.makedirs(
    LOCAL_CACHE_DIR,
    exist_ok=True
)


# ============================================================
# 4. EKSİK / BOZUK LOCAL DOSYALARI TAMAMLA
#
# Dosya local'de yoksa veya boyutu Drive ile aynı değilse
# tekrar kopyalanır.
# ============================================================

copied = 0
valid = 0


for i, filename in enumerate(
    drive_files,
    start=1
):

    source = os.path.join(
        DRIVE_CACHE_DIR,
        filename
    )

    target = os.path.join(
        LOCAL_CACHE_DIR,
        filename
    )


    target_is_valid = (
        os.path.exists(target)
        and
        os.path.getsize(target)
        == os.path.getsize(source)
    )


    if target_is_valid:

        valid += 1

    else:

        shutil.copy2(
            source,
            target
        )

        copied += 1


    if (
        i % 100 == 0
        or i == len(drive_files)
    ):

        print(
            f"{i}/{len(drive_files)} checked | "
            f"copied: {copied}"
        )


# ============================================================
# 5. LOCAL CACHE SON KONTROL
# ============================================================

local_files = [
    f
    for f in os.listdir(LOCAL_CACHE_DIR)
    if f.endswith(".npz")
]


missing_local_files = [
    f
    for f in drive_files
    if not os.path.exists(
        os.path.join(
            LOCAL_CACHE_DIR,
            f
        )
    )
]


if missing_local_files:

    raise RuntimeError(
        f"{len(missing_local_files)} cache files "
        "are still missing locally."
    )


print("\nLocal cache: ✅")
print(
    "Available NPZ files:",
    len(local_files)
)

print(
    "Copied this run:",
    copied
)


# ============================================================
# 6. LOCAL PATIENT-GROUPED GENERATOR
# ============================================================

BATCH_SIZE = 8


local_grouped_train_generator = (
    PatientGroupedCachedGenerator(
        dataframe=clean_train_df,
        cache_dir=LOCAL_CACHE_DIR,
        batch_size=BATCH_SIZE,
        shuffle=True
    )
)


# ============================================================
# 7. AUGMENTATION WRAPPER
# ============================================================

local_augmented_train_generator = (
    AugmentedTrainingGenerator(
        local_grouped_train_generator
    )
)


print(
    "\nLocal training generator: ✅"
)

print(
    "Training batches:",
    len(local_augmented_train_generator)
)


# ============================================================
# 8. GERÇEK BATCH SANITY CHECK
# ============================================================

X_master, Y_master = (
    local_augmented_train_generator[0]
)


shape_ok = (
    X_master.shape[1:] == (128, 128, 4)
    and
    Y_master.shape[1:] == (128, 128, 1)
)

finite_ok = (
    np.isfinite(X_master).all()
    and
    np.isfinite(Y_master).all()
)


print("\nPIPELINE CHECK")
print("=" * 60)

print(
    "X shape:",
    X_master.shape
)

print(
    "Y shape:",
    Y_master.shape
)

print(
    "Mask classes:",
    np.unique(Y_master)
)

print(
    "Shapes:",
    "✅" if shape_ok else "❌"
)

print(
    "Finite values:",
    "✅" if finite_ok else "❌"
)


# ============================================================
# 9. FINAL STATUS
# ============================================================

if shape_ok and finite_ok:

    print("\n" + "=" * 60)
    print("🚀 LOCAL TRAINING PIPELINE READY")
    print("=" * 60)

    print(
        "\nUse this generator for training:"
    )

    print(
        "local_augmented_train_generator"
    )

else:

    raise RuntimeError(
        "Pipeline sanity check failed."
    )


print(
    "\n102. hücre tamamlandı. ✅"
)

In [ ]:
# ============================================================
# 103. HÜCRE — LOCAL FULL U-NET TRAINING
#
# AMAÇ:
# Optimize edilmiş LOCAL cache üzerinden gerçek U-Net
# eğitimini başlatmak.
#
# PIPELINE:
#
# /content/cache_float16_train
#          ↓
# Patient-Grouped Generator
#          ↓
# Dynamic Augmentation
#          ↓
# U-Net
#          ↓
# BCE + Dice Loss
#          ↓
# Dice + IoU
#
# GÜVENLİK:
# - En iyi model Drive'a kaydedilir.
# - Her epoch sonunda son model Drive'a kaydedilir.
# - Training log CSV olarak Drive'a yazılır.
# - EarlyStopping aktiftir.
# - ReduceLROnPlateau aktiftir.
# ============================================================

import os
import time
import tensorflow as tf


# ============================================================
# 1. TRAINING AYARLARI
# ============================================================

MAX_EPOCHS = 20

LOCAL_TRAIN_GENERATOR = (
    local_augmented_train_generator
)


print("LOCAL FULL U-NET TRAINING")
print("=" * 60)

print(
    "GPU:",
    tf.config.list_physical_devices("GPU")
)

print(
    "Training batches   :",
    len(LOCAL_TRAIN_GENERATOR)
)

print(
    "Validation batches :",
    len(validation_generator)
)

print(
    "Batch size         :",
    BATCH_SIZE
)

print(
    "Maximum epochs     :",
    MAX_EPOCHS
)


# ============================================================
# 2. CHECKPOINT DOSYALARI
# ============================================================

model_dir = os.path.join(
    project_dir,
    "models"
)

results_dir = os.path.join(
    project_dir,
    "results"
)

os.makedirs(
    model_dir,
    exist_ok=True
)

os.makedirs(
    results_dir,
    exist_ok=True
)


best_local_model_path = os.path.join(
    model_dir,
    "best_unet_local.keras"
)

latest_local_model_path = os.path.join(
    model_dir,
    "latest_unet_local.keras"
)

local_training_log_path = os.path.join(
    results_dir,
    "local_training_log.csv"
)


# ============================================================
# 3. CALLBACKS
# ============================================================

# Validation Dice en yüksek olduğunda en iyi modeli kaydet.
best_checkpoint = (
    tf.keras.callbacks.ModelCheckpoint(
        filepath=best_local_model_path,
        monitor="val_dice_coefficient",
        mode="max",
        save_best_only=True,
        verbose=1
    )
)


# Her epoch sonunda mevcut modeli de kaydet.
# Böylece Colab koparsa son tamamlanan epoch elimizde kalır.
latest_checkpoint = (
    tf.keras.callbacks.ModelCheckpoint(
        filepath=latest_local_model_path,
        save_best_only=False,
        verbose=0
    )
)


# Validation Dice 5 epoch gelişmezse dur.
early_stop = (
    tf.keras.callbacks.EarlyStopping(
        monitor="val_dice_coefficient",
        mode="max",
        patience=5,
        restore_best_weights=True,
        verbose=1
    )
)


# Validation Dice gelişmezse learning rate'i azalt.
reduce_lr = (
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_dice_coefficient",
        mode="max",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
)


# Epoch sonuçlarını Drive'a kaydet.
csv_logger = (
    tf.keras.callbacks.CSVLogger(
        local_training_log_path,
        append=False
    )
)


local_training_callbacks = [
    best_checkpoint,
    latest_checkpoint,
    early_stop,
    reduce_lr,
    csv_logger
]


# ============================================================
# 4. TRAINING ÖNCESİ SON KONTROL
# ============================================================

X_final_check, Y_final_check = (
    LOCAL_TRAIN_GENERATOR[0]
)

assert X_final_check.shape[1:] == (
    128, 128, 4
)

assert Y_final_check.shape[1:] == (
    128, 128, 1
)


print("\nFINAL CHECK")
print("-" * 60)

print(
    "X:",
    X_final_check.shape
)

print(
    "Y:",
    Y_final_check.shape
)

print(
    "Learning rate:",
    float(
        tf.keras.backend.get_value(
            model.optimizer.learning_rate
        )
    )
)

print(
    "Best checkpoint:",
    best_local_model_path
)

print(
    "Latest checkpoint:",
    latest_local_model_path
)

print(
    "Training log:",
    local_training_log_path
)


# ============================================================
# 5. GERÇEK TRAINING
# ============================================================

print("\n" + "=" * 60)
print("🚀 STARTING LOCAL FULL TRAINING")
print("=" * 60)


training_start = time.time()


history_local = model.fit(

    LOCAL_TRAIN_GENERATOR,

    validation_data=validation_generator,

    epochs=MAX_EPOCHS,

    callbacks=local_training_callbacks,

    verbose=1
)


# ============================================================
# 6. TRAINING SÜRESİ
# ============================================================

training_elapsed = (
    time.time()
    - training_start
)

training_minutes = (
    training_elapsed / 60
)

training_hours = (
    training_minutes / 60
)


# ============================================================
# 7. TRAINING SONUCU
# ============================================================

completed_epochs = len(
    history_local.history["loss"]
)


print("\n" + "=" * 60)
print("FULL LOCAL TRAINING COMPLETE ✅")
print("=" * 60)

print(
    "Completed epochs:",
    completed_epochs
)

print(
    f"Total time      : "
    f"{training_minutes:.1f} min"
)

print(
    f"Total time      : "
    f"{training_hours:.2f} h"
)


# ============================================================
# 8. EN İYİ VALIDATION DICE
# ============================================================

if "val_dice_coefficient" in history_local.history:

    best_val_dice = max(
        history_local.history[
            "val_dice_coefficient"
        ]
    )

    best_epoch = (
        history_local.history[
            "val_dice_coefficient"
        ].index(best_val_dice)
        + 1
    )

    print(
        "\nBest validation Dice:",
        f"{best_val_dice:.4f}"
    )

    print(
        "Best epoch:",
        best_epoch
    )


print(
    "\nBest model:"
)

print(
    best_local_model_path
)

print(
    "\nLatest completed model:"
)

print(
    latest_local_model_path
)

print(
    "\nTraining log:"
)

print(
    local_training_log_path
)


print(
    "\n103. hücre tamamlandı. ✅"
)

In [ ]:
lofimport tensorflow as tf

print(tf.config.list_physical_devices("GPU"))

In [ ]:
import tensorflow as tf

print(tf.config.list_physical_devices("GPU"))

In [ ]:

from google.colab import drive
import os

drive.mount('/content/drive')

project_dir = '/content/drive/MyDrive/NeuroRobust'

paths = {
    "Best model": f"{project_dir}/models/best_unet_local.keras",
    "Latest model": f"{project_dir}/models/latest_unet_local.keras",
    "Training log": f"{project_dir}/results/local_training_log.csv",
    "Training cache": f"{project_dir}/cache_float16_train",
}

print("NEUROROBUST RECOVERY CHECK")
print("-" * 40)

for name, path in paths.items():
    if os.path.exists(path):
        print(f"✅ {name}: FOUND")
    else:
        print(f"❌ {name}: NOT FOUND")


In [ ]:

import os
import tensorflow as tf
import pandas as pd

project_dir = "/content/drive/MyDrive/NeuroRobust"

latest_path = f"{project_dir}/models/latest_unet_local.keras"
best_path = f"{project_dir}/models/best_unet_local.keras"
log_path = f"{project_dir}/results/local_training_log.csv"

print("TensorFlow version:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices("GPU"))

print("\nLoading latest U-Net model...")

try:
    model = tf.keras.models.load_model(
        latest_path,
        compile=False
    )

    print("\n✅ MODEL LOADED SUCCESSFULLY!")
    print("Input shape:", model.input_shape)
    print("Output shape:", model.output_shape)
    print("Total parameters:", model.count_params())

    model.summary()

except Exception as e:
    print("\n❌ MODEL LOADING ERROR:")
    print(type(e).__name__, str(e))

print("\nTRAINING HISTORY")

if os.path.exists(log_path):
    history_df = pd.read_csv(log_path)
    display(history_df.tail(10))

    print("Completed epochs:", len(history_df))
    print("Best Validation Dice:",
          history_df["val_dice_coefficient"].max())


In [ ]:

# 104. HÜCRE — VALIDATION PREDICTION CHECK

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import nibabel as nib
import tensorflow as tf

project_dir = "/content/drive/MyDrive/NeuroRobust"
data_dir = f"{project_dir}/dataset/BraTS2021_Training_Data"

# 1. En iyi modeli yükle
best_model = tf.keras.models.load_model(
    f"{project_dir}/models/best_unet_local.keras",
    compile=False
)

# 2. Validation grubundan tümörlü bir örnek seç
val_df = pd.read_csv(
    f"{project_dir}/validation_slice_index.csv"
)

tumor_rows = val_df[val_df["has_tumor"] == 1]
row = tumor_rows.sort_values(
    "tumor_pixels", ascending=False
).iloc[len(tumor_rows) // 4]

patient_id = row["patient_id"]
slice_idx = int(row["slice_index"])

print("Patient:", patient_id)
print("Slice:", slice_idx)

# 3. Eğitimdeki gibi 3D MRI normalizasyonu
def prepare_volume(volume):
    volume = volume.astype(np.float32)
    brain = volume != 0
    output = np.zeros_like(volume)

    if brain.any():
        mean = volume[brain].mean()
        std = volume[brain].std()
        if std > 0:
            output[brain] = (volume[brain] - mean) / std

    return output

modalities = ["t1", "t1ce", "t2", "flair"]
channels = []

for modality in modalities:
    path = (
        f"{data_dir}/{patient_id}/"
        f"{patient_id}_{modality}.nii.gz"
    )

    volume = nib.load(path).get_fdata(dtype=np.float32)
    normalized = prepare_volume(volume)

    slice_image = normalized[:, :, slice_idx]
    resized = cv2.resize(
        slice_image, (128, 128),
        interpolation=cv2.INTER_LINEAR
    )
    channels.append(resized)

X = np.stack(channels, axis=-1).astype(np.float32)

# 4. Gerçek tümör maskesi
mask_path = (
    f"{data_dir}/{patient_id}/"
    f"{patient_id}_seg.nii.gz"
)

seg = nib.load(mask_path).get_fdata(dtype=np.float32)
ground_truth = (seg[:, :, slice_idx] > 0).astype(np.uint8)
ground_truth = cv2.resize(
    ground_truth, (128, 128),
    interpolation=cv2.INTER_NEAREST
)

# 5. Tahmin
probability = best_model.predict(
    X[np.newaxis, ...], verbose=0
)[0, :, :, 0]

prediction = (probability > 0.5).astype(np.uint8)

intersection = np.sum(prediction * ground_truth)
denominator = np.sum(prediction) + np.sum(ground_truth)
dice = (2 * intersection + 1e-6) / (denominator + 1e-6)

print("Ground truth pixels:", int(ground_truth.sum()))
print("Predicted pixels:", int(prediction.sum()))
print(f"Slice Dice (threshold=0.5): {dice:.4f}")

# 6. Görselleştirme
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

images = [
    X[:, :, 3],
    ground_truth,
    probability,
    prediction
]

titles = [
    "FLAIR MRI",
    "Ground Truth",
    "Predicted Probability",
    "Predicted Mask"
]

for ax, img, title in zip(axes, images, titles):
    ax.imshow(img, cmap="gray")
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

# 105. HÜCRE — SOFT DICE vs BINARY DICE
# Sadece değerlendirme yapar. Eğitim başlatmaz.

import numpy as np

y_true = ground_truth.astype(np.float32)
y_prob = probability.astype(np.float32)
y_binary = (y_prob > 0.5).astype(np.float32)

eps = 1e-6

# Eğitimdeki Soft Dice mantığı
soft_dice = (
    2 * np.sum(y_true * y_prob) + eps
) / (
    np.sum(y_true) + np.sum(y_prob) + eps
)

# Eşiklenmiş maske ile Binary Dice
binary_dice = (
    2 * np.sum(y_true * y_binary) + eps
) / (
    np.sum(y_true) + np.sum(y_binary) + eps
)

# Binary IoU
intersection = np.sum(y_true * y_binary)
union = np.sum(y_true) + np.sum(y_binary) - intersection
binary_iou = (intersection + eps) / (union + eps)

print("NEUROROBUST — METRIC COMPARISON")
print("=" * 45)
print(f"Patient: {patient_id}")
print(f"Slice: {slice_idx}")
print(f"Soft Dice:   {soft_dice:.4f}")
print(f"Binary Dice: {binary_dice:.4f}")
print(f"Binary IoU:  {binary_iou:.4f}")
print(f"Mean predicted probability: {y_prob.mean():.6f}")


In [ ]:

# ==========================================================
# 106. HÜCRE — MULTI-PATIENT VALIDATION DIAGNOSTIC
# 10 hasta, her hastadan en fazla 4 kesit
# Model eğitilmez. Checkpoint dosyaları değiştirilmez.
# ==========================================================

import os
import gc
import cv2
import nibabel as nib
import numpy as np
import pandas as pd

from IPython.display import display

project_dir = "/content/drive/MyDrive/NeuroRobust"

data_dir = (
    f"{project_dir}/dataset/BraTS2021_Training_Data"
)

val_df = pd.read_csv(
    f"{project_dir}/validation_slice_index.csv"
)

# Daha önce yüklediğimiz en iyi model
assert "best_model" in globals(), (
    "Önce 104. hücrede best_model'i yüklemelisin."
)

rng = np.random.default_rng(42)

# 1. Validation grubundan 10 farklı hasta
patient_ids = val_df["patient_id"].unique()

selected_patients = rng.choice(
    patient_ids,
    size=min(10, len(patient_ids)),
    replace=False
)

print("Selected patients:", len(selected_patients))
print("Preparing validation samples...")

results = []

# 2. Hastaları tek tek işle
for patient_no, pid in enumerate(selected_patients, 1):

    patient_df = val_df[
        val_df["patient_id"] == pid
    ]

    tumor_df = patient_df[
        patient_df["has_tumor"] == 1
    ]

    background_df = patient_df[
        patient_df["has_tumor"] == 0
    ]

    # Her sınıftan en fazla 2 kesit
    sample_parts = []

    for subset in [tumor_df, background_df]:
        if len(subset) > 0:
            sample_parts.append(
                subset.sample(
                    n=min(2, len(subset)),
                    random_state=42
                )
            )

    if not sample_parts:
        continue

    samples = pd.concat(sample_parts)

    # 3. Dört MRI modalitesini yükle
    volumes = []

    for modality in ["t1", "t1ce", "t2", "flair"]:

        path = (
            f"{data_dir}/{pid}/"
            f"{pid}_{modality}.nii.gz"
        )

        volume = nib.load(path).get_fdata(
            dtype=np.float32
        )

        # Eğitimde kullanılan 3D Z-score normalizasyonu
        brain = volume > 0
        normalized = np.zeros_like(
            volume, dtype=np.float32
        )

        if np.any(brain):
            mean = volume[brain].mean()
            std = volume[brain].std()

            if std > 0:
                normalized[brain] = (
                    volume[brain] - mean
                ) / std

        volumes.append(normalized)

        del volume

    seg_path = (
        f"{data_dir}/{pid}/{pid}_seg.nii.gz"
    )

    segmentation = nib.load(
        seg_path
    ).get_fdata(dtype=np.float32)

    X_batch = []
    Y_batch = []
    slice_numbers = []

    # 4. Seçilen kesitleri hazırla
    for _, row in samples.iterrows():

        idx = int(row["slice_index"])

        channels = []

        for volume in volumes:

            image = cv2.resize(
                volume[:, :, idx],
                (128, 128),
                interpolation=cv2.INTER_LINEAR
            )

            channels.append(image)

        X_batch.append(
            np.stack(channels, axis=-1)
        )

        mask = (
            segmentation[:, :, idx] > 0
        ).astype(np.float32)

        mask = cv2.resize(
            mask,
            (128, 128),
            interpolation=cv2.INTER_NEAREST
        )

        Y_batch.append(mask)
        slice_numbers.append(idx)

    X_batch = np.asarray(
        X_batch, dtype=np.float32
    )

    Y_batch = np.asarray(
        Y_batch, dtype=np.float32
    )

    # 5. Modelin tahminlerini hesapla
    predictions = best_model.predict(
        X_batch,
        verbose=0
    )[:, :, :, 0]

    eps = 1e-6

    for j, idx in enumerate(slice_numbers):

        truth = Y_batch[j]
        prob = predictions[j]
        binary = (prob > 0.5).astype(np.float32)

        intersection_soft = np.sum(
            truth * prob
        )

        soft_dice = (
            2 * intersection_soft + eps
        ) / (
            np.sum(truth) + np.sum(prob) + eps
        )

        intersection = np.sum(
            truth * binary
        )

        binary_dice = (
            2 * intersection + eps
        ) / (
            np.sum(truth) + np.sum(binary) + eps
        )

        union = (
            np.sum(truth)
            + np.sum(binary)
            - intersection
        )

        binary_iou = (
            intersection + eps
        ) / (union + eps)

        results.append({
            "patient_id": pid,
            "slice_index": idx,
            "has_tumor": int(truth.sum() > 0),
            "true_pixels": int(truth.sum()),
            "pred_pixels": int(binary.sum()),
            "soft_dice": float(soft_dice),
            "binary_dice": float(binary_dice),
            "binary_iou": float(binary_iou)
        })

    print(
        f"Patient {patient_no}/"
        f"{len(selected_patients)} completed"
    )

    del volumes, segmentation, X_batch
    del Y_batch, predictions

    gc.collect()

# 6. Sonuç tablosu
results_df = pd.DataFrame(results)

print("\n" + "=" * 55)
print("NEUROROBUST — MULTI-PATIENT RESULTS")
print("=" * 55)

print("Patients evaluated:",
      results_df["patient_id"].nunique())

print("Slices evaluated:", len(results_df))

print("\nOVERALL SAMPLE MEANS")
print(
    results_df[
        ["soft_dice", "binary_dice", "binary_iou"]
    ].mean().round(4)
)

print("\nTUMOR vs BACKGROUND")
summary = results_df.groupby(
    "has_tumor"
)[
    ["soft_dice", "binary_dice", "binary_iou"]
].agg(["mean", "count"])

display(summary.round(4))

print("\nLOWEST BINARY DICE EXAMPLES")
display(
    results_df.sort_values(
        "binary_dice"
    ).head(10)
)

# 7. Sonuçları kaydet
save_path = (
    f"{project_dir}/results/"
    "validation_diagnostic_10patients.csv"
)

results_df.to_csv(
    save_path, index=False
)

print("\nResults saved:", save_path)



### 106. Multi-Patient Validation Analysis

The trained U-Net model was evaluated on 40 MRI slices from 10 randomly selected validation patients, including 20 tumor-containing and 20 background-only slices.

#### Overall Results

| Metric | Mean Score |
|---|---:|
| Soft Dice | 0.3637 |
| Binary Dice | 0.7437 |
| Binary IoU | 0.7042 |

#### Tumor vs. Background Performance

| Slice Type | Soft Dice | Binary Dice | Binary IoU |
|---|---:|---:|---:|
| Background (20 slices) | 0.0001 | 0.7500 | 0.7500 |
| Tumor (20 slices) | 0.7272 | 0.7374 | 0.6583 |

#### Key Findings

- The model achieved a mean Binary Dice score of **0.7374 on tumor-containing slices**, demonstrating promising segmentation performance.
- On background-only slices, the Binary Dice score was **0.7500**, indicating that 15 out of 20 slices were correctly predicted as empty.
- Soft Dice was close to zero on background-only slices because the current formula assigns a very low score when the ground-truth mask is empty but the model predicts small nonzero probabilities.
- Some of the lowest-performing tumor slices contained very small tumor regions (3–15 pixels), making segmentation more challenging.
- The difference between overall Soft Dice (0.3637) and Binary Dice (0.7437) highlights the importance of metric definitions and empty-mask handling.

#### Conclusion

The results suggest that the low overall Soft Dice score does not fully represent the model's segmentation quality. The model performs reasonably well on tumor-containing slices but produces false positives on some background-only slices and struggles with certain small tumors.

Further evaluation is needed using the complete validation dataset, consistent preprocessing, and clearly defined Dice metrics before drawing conclusions about generalization performance.


In [ ]:

# =====================================================
# 107 — SEGMENTATION ERROR VISUALIZATION
# En kötü 3 tümörlü + 3 tümörsüz kesit
# Eğitim başlatmaz, model dosyalarını değiştirmez.
# =====================================================

import os
import numpy as np
import matplotlib.pyplot as plt
import nibabel as nib
import cv2

# 106. hücrenin sonuçlarını kullan
assert "results_df" in globals()
assert "best_model" in globals()

worst_tumor = (
    results_df[results_df["has_tumor"] == 1]
    .nsmallest(3, "binary_dice")
)

worst_background = (
    results_df[results_df["has_tumor"] == 0]
    .nsmallest(3, "binary_dice")
)

worst_samples = list(
    worst_tumor.to_dict("records")
    + worst_background.to_dict("records")
)

def load_example(patient_id, slice_idx):

    modalities = ["t1", "t1ce", "t2", "flair"]
    channels = []

    for modality in modalities:
        path = (
            f"{data_dir}/{patient_id}/"
            f"{patient_id}_{modality}.nii.gz"
        )

        volume = nib.load(path).get_fdata(
            dtype=np.float32
        )

        # Aynı normalizasyon: 106. hücre
        brain = volume > 0
        normalized = np.zeros_like(volume)

        if np.any(brain):
            mean = volume[brain].mean()
            std = volume[brain].std()

            if std > 0:
                normalized[brain] = (
                    volume[brain] - mean
                ) / std

        image = normalized[:, :, slice_idx]

        image = cv2.resize(
            image, (128, 128),
            interpolation=cv2.INTER_LINEAR
        )

        channels.append(image)

    X = np.stack(
        channels, axis=-1
    ).astype(np.float32)

    mask_path = (
        f"{data_dir}/{patient_id}/"
        f"{patient_id}_seg.nii.gz"
    )

    segmentation = nib.load(
        mask_path
    ).get_fdata(dtype=np.float32)

    truth = (
        segmentation[:, :, slice_idx] > 0
    ).astype(np.uint8)

    truth = cv2.resize(
        truth, (128, 128),
        interpolation=cv2.INTER_NEAREST
    )

    return X, truth


fig, axes = plt.subplots(
    len(worst_samples), 4,
    figsize=(15, 20)
)

for i, row in enumerate(worst_samples):

    pid = row["patient_id"]
    idx = int(row["slice_index"])

    X, truth = load_example(pid, idx)

    prob = best_model.predict(
        X[None, ...],
        verbose=0
    )[0, :, :, 0]

    pred = (prob > 0.5).astype(np.uint8)

    # Overlay:
    # Yeşil = doğru tümör
    # Kırmızı = yanlış pozitif
    # Mavi = kaçırılan tümör

    overlay = np.zeros((128, 128, 3))

    overlay[:, :, 1] = (
        (truth == 1) & (pred == 1)
    )

    overlay[:, :, 0] = (
        (truth == 0) & (pred == 1)
    )

    overlay[:, :, 2] = (
        (truth == 1) & (pred == 0)
    )

    images = [
        X[:, :, 3],
        truth,
        pred,
        overlay
    ]

    titles = [
        "FLAIR MRI",
        "Ground Truth",
        "Prediction",
        "Error Overlay"
    ]

    for j in range(4):

        ax = axes[i, j]

        if j == 3:
            ax.imshow(images[j])
        else:
            ax.imshow(images[j], cmap="gray")

        if i == 0:
            ax.set_title(titles[j], fontsize=12)

        ax.axis("off")

    axes[i, 0].set_ylabel(
        f"{pid}\nSlice {idx}\n"
        f"Dice: {row['binary_dice']:.3f}",
        fontsize=9
    )

plt.suptitle(
    "NeuroRobust — Worst Segmentation Examples",
    fontsize=16,
    y=0.995
)

plt.tight_layout()

save_path = (
    f"{project_dir}/results/"
    "segmentation_error_analysis.png"
)

plt.savefig(
    save_path,
    dpi=160,
    bbox_inches="tight"
)

plt.show()

print("Saved:", save_path)
print("Green = True Positive")
print("Red = False Positive")
print("Blue = False Negative")



### 107. Segmentation Error Analysis

This experiment visualizes six low-performing MRI slices selected from the multi-patient validation analysis: three tumor-containing slices and three background-only slices.

#### Visualization

Four views are displayed for each sample:

1. **FLAIR MRI:** The original brain MRI slice.
2. **Ground Truth:** The reference tumor segmentation mask.
3. **Prediction:** The binary mask predicted by the U-Net model using a threshold of 0.5.
4. **Error Overlay:** A visualization of segmentation errors.

The error overlay uses the following colors:
- **Green:** True Positive (correctly detected tumor pixels)
- **Red:** False Positive (background pixels incorrectly classified as tumor)
- **Blue:** False Negative (tumor pixels missed by the model)

#### Key Findings

**1. Small Tumor Detection**

The model struggles with some very small tumor regions. Several selected tumor-containing slices contain only a few tumor pixels.

The visualizations show missed tumor pixels and, in some cases, predictions that do not overlap with the ground-truth mask.

**2. False Positives on Background Slices**

Some background-only MRI slices contain small predicted tumor regions despite having no tumor in the reference mask.

This indicates that the model sometimes incorrectly identifies normal tissue as tumor.

**3. Segmentation Performance**

The previous multi-patient evaluation showed:
- Mean Binary Dice on tumor-containing slices: **0.7374**
- Mean Binary Dice on background-only slices: **0.7500**
- Correctly predicted empty masks: **15 out of 20**

The selected error examples highlight weaknesses that are not fully visible from average metrics alone.

#### Conclusion

The U-Net model shows promising segmentation performance on some tumor-containing MRI slices. However, the error analysis reveals difficulties with small tumor regions and false-positive predictions on background-only slices.

Future improvements should focus on:
- Verifying consistent training and inference preprocessing
- Evaluating performance across different tumor sizes
- Investigating threshold selection
- Reducing false-positive predictions
- Testing the model on the complete validation dataset

These findings are based on a limited diagnostic sample and should not be interpreted as final model performance.


In [ ]:

# ==========================================================
# 108 — TUMOR SIZE vs SEGMENTATION PERFORMANCE
# Uses saved results from cell 106
# No model training or checkpoint modifications
# ==========================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

project_dir = "/content/drive/MyDrive/NeuroRobust"
results_dir = f"{project_dir}/results"

# Load previous diagnostic results
csv_path = (
    f"{results_dir}/"
    "validation_diagnostic_10patients.csv"
)

df = pd.read_csv(csv_path)

# Only evaluate slices containing tumors
tumor_df = df[df["true_pixels"] > 0].copy()

assert not tumor_df.empty, "No tumor-containing slices found."

# 1. Group by tumor size
bins = [0, 10, 100, np.inf]

labels = [
    "Small (1-10)",
    "Medium (11-100)",
    "Large (101+)"
]

tumor_df["tumor_size"] = pd.cut(
    tumor_df["true_pixels"],
    bins=bins,
    labels=labels,
    include_lowest=False
)

# 2. Summary statistics
summary = (
    tumor_df.groupby(
        "tumor_size",
        observed=False
    )
    .agg(
        slices=("binary_dice", "size"),
        mean_dice=("binary_dice", "mean"),
        median_dice=("binary_dice", "median"),
        mean_iou=("binary_iou", "mean"),
        mean_soft_dice=("soft_dice", "mean")
    )
    .reset_index()
)

print("=" * 55)
print("NEUROROBUST — TUMOR SIZE ANALYSIS")
print("=" * 55)

print("\nTumor-containing slices:", len(tumor_df))
print("Unique patients:", tumor_df["patient_id"].nunique())

print("\nPERFORMANCE BY TUMOR SIZE")
display(summary.round(4))

# 3. Scatter plot: true tumor size vs Binary Dice
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(
    tumor_df["true_pixels"],
    tumor_df["binary_dice"],
    alpha=0.75,
    s=65
)

axes[0].set_xscale("log")
axes[0].set_xlabel("True Tumor Area (pixels, log scale)")
axes[0].set_ylabel("Binary Dice")
axes[0].set_title("Tumor Size vs Dice Score")
axes[0].set_ylim(-0.05, 1.05)
axes[0].grid(alpha=0.25)

# 4. Average Dice by size group
valid_summary = summary[summary["slices"] > 0]

axes[1].bar(
    valid_summary["tumor_size"].astype(str),
    valid_summary["mean_dice"]
)

axes[1].set_xlabel("Tumor Size Group")
axes[1].set_ylabel("Mean Binary Dice")
axes[1].set_title("Average Dice by Tumor Size")
axes[1].set_ylim(0, 1)
axes[1].tick_params(axis="x", rotation=15)
axes[1].grid(axis="y", alpha=0.25)

for i, row in enumerate(valid_summary.itertuples()):
    if pd.notna(row.mean_dice):
        axes[1].text(
            i,
            min(row.mean_dice + 0.03, 0.97),
            f"{row.mean_dice:.3f}\n(n={row.slices})",
            ha="center",
            fontsize=9
        )

plt.tight_layout()

plot_path = (
    f"{results_dir}/"
    "tumor_size_vs_dice.png"
)

plt.savefig(
    plot_path,
    dpi=160,
    bbox_inches="tight"
)

plt.show()

# 5. Analyze false positives on empty slices
background_df = df[df["true_pixels"] == 0].copy()

if not background_df.empty:

    fp_slices = (
        background_df["pred_pixels"] > 0
    ).sum()

    print("\nBACKGROUND FALSE POSITIVE ANALYSIS")
    print("Background slices:", len(background_df))
    print("Slices with false positives:", int(fp_slices))

    print(
        "False positive slice rate:",
        f"{fp_slices / len(background_df):.2%}"
    )

    display(
        background_df[
            ["patient_id", "slice_index", "pred_pixels"]
        ].sort_values(
            "pred_pixels", ascending=False
        ).head(10)
    )

# 6. Save summaries
summary.to_csv(
    f"{results_dir}/tumor_size_analysis.csv",
    index=False
)

print("\nAnalysis completed!")
print("Plot saved:", plot_path)
print("Summary saved to Google Drive.")



### 108. Tumor Size vs. Segmentation Performance

This experiment investigates how tumor size affects the segmentation performance of the trained U-Net model.

The analysis uses 20 tumor-containing MRI slices from the previous 10-patient validation sample. Tumor areas were measured in pixels after resizing the masks to 128 × 128.

#### Performance by Tumor Size

| Tumor Size | Slices | Mean Binary Dice | Mean Binary IoU |
|---|---:|---:|---:|
| Small (1–10 pixels) | 5 | 0.3896 | 0.2672 |
| Medium (11–100 pixels) | 3 | 0.6141 | 0.5699 |
| Large (101+ pixels) | 12 | 0.9132 | 0.8434 |

#### Key Findings

- **Large tumors:** The model achieved a mean Binary Dice of 0.9132, indicating strong segmentation performance on larger tumor regions in this sample.
- **Medium tumors:** The mean Binary Dice decreased to 0.6141, showing more variable segmentation results.
- **Small tumors:** The lowest mean Binary Dice (0.3896) was observed for tumors containing only 1–10 pixels.
- **Tumor size relationship:** The scatter plot suggests that segmentation performance improves as tumor area increases.
- **Small lesion sensitivity:** Very small tumor regions are particularly challenging because missing only a few pixels can significantly reduce the Dice score.

#### Background False Positive Analysis

The model was also evaluated on 20 background-only MRI slices.

- **Background slices evaluated:** 20
- **Slices with false-positive predictions:** 5
- **False-positive slice rate:** 25%
- **Correctly predicted empty slices:** 15 (75%)

The false-positive predictions ranged from 1 to 28 pixels, showing that the model occasionally identifies small background regions as tumors.

#### Conclusion

The results reveal a clear performance difference between small and large tumor regions in the evaluated sample.

The U-Net model performs well on larger tumors but struggles with very small tumor regions and produces occasional false positives on background-only slices.

Future work will focus on investigating preprocessing consistency, improving small-tumor detection, reducing false positives, and evaluating the model on the complete validation dataset.

**Limitation:** These findings are based on only 20 tumor-containing and 20 background-only slices from 10 patients. The results are preliminary and should not be interpreted as the model's final generalization performance.


In [ ]:

# =====================================================
# 109 — TRAINING PREPROCESSING CONSISTENCY CHECK
# Compare cached training data with original MRI files
# No training, no file modifications
# =====================================================

import os
import numpy as np
import nibabel as nib
import cv2

project_dir = "/content/drive/MyDrive/NeuroRobust"

data_dir = os.path.join(
    project_dir, "dataset", "BraTS2021_Training_Data"
)

cache_dir = os.path.join(
    project_dir, "cache_float16_train"
)

modalities = ["t1", "t1ce", "t2", "flair"]

# 1. Select one cached training patient
cache_files = sorted(
    f for f in os.listdir(cache_dir)
    if f.endswith(".npz")
)

assert cache_files, "No cached patients found."

cache_file = cache_files[0]
pid = cache_file.replace(".npz", "")

with np.load(os.path.join(cache_dir, cache_file)) as data:
    available_slices = data["slice_indices"].copy()

    # Select a slice from the middle of the cache
    position = len(available_slices) // 2
    slice_idx = int(available_slices[position])

    cached_x = data["X"][position].astype(np.float32)
    cached_y = data["Y"][position, :, :, 0].copy()

print("Patient:", pid)
print("Slice:", slice_idx)

# 2. Recreate the original training normalization
def normalize_like_training(volume):
    brain_mask = volume > 0

    normalized = np.zeros_like(
        volume, dtype=np.float32
    )

    if np.any(brain_mask):
        mean = volume[brain_mask].mean()
        std = volume[brain_mask].std()

        if std > 0:
            normalized[brain_mask] = (
                volume[brain_mask] - mean
            ) / std

    return normalized

# 3. Load original MRI volumes and preprocess
direct_channels = []

for modality in modalities:
    path = os.path.join(
        data_dir, pid,
        f"{pid}_{modality}.nii.gz"
    )

    volume = nib.load(path).get_fdata().astype(np.float32)

    normalized = normalize_like_training(volume)

    image = cv2.resize(
        normalized[:, :, slice_idx],
        (128, 128),
        interpolation=cv2.INTER_LINEAR
    )

    direct_channels.append(image)

direct_x = np.stack(
    direct_channels, axis=-1
).astype(np.float32)

# 4. Compare the ground-truth masks
seg_path = os.path.join(
    data_dir, pid, f"{pid}_seg.nii.gz"
)

seg = nib.load(seg_path).get_fdata()

direct_y = (
    seg[:, :, slice_idx] > 0
).astype(np.uint8)

direct_y = cv2.resize(
    direct_y,
    (128, 128),
    interpolation=cv2.INTER_NEAREST
)

# 5. Compare original and cached inputs
difference = np.abs(cached_x - direct_x)

mae = np.mean(difference)
max_diff = np.max(difference)

mask_equal = np.array_equal(
    cached_y, direct_y
)

# Compare after Float16 quantization
quantized_direct = direct_x.astype(
    np.float16
).astype(np.float32)

quantized_diff = np.abs(
    cached_x - quantized_direct
)

print("\n" + "=" * 55)
print("NEUROROBUST — PREPROCESSING CONSISTENCY")
print("=" * 55)

print("Cached X shape:", cached_x.shape)
print("Direct X shape:", direct_x.shape)

print("\nCHANNEL COMPARISON")

for i, name in enumerate(modalities):
    channel_mae = np.mean(difference[:, :, i])
    channel_max = np.max(difference[:, :, i])

    print(
        f"{name:5s} | "
        f"MAE: {channel_mae:.8f} | "
        f"Max difference: {channel_max:.8f}"
    )

print("\nOVERALL COMPARISON")

print(f"Mean Absolute Error: {mae:.8f}")
print(f"Maximum Difference: {max_diff:.8f}")

print(
    "Max difference after Float16 quantization:",
    f"{quantized_diff.max():.8f}"
)

print("Ground-truth masks identical:", mask_equal)

print("\nFINAL STATUS")

if (
    mae < 0.002
    and max_diff < 0.05
    and mask_equal
):
    print("✅ Preprocessing appears consistent.")
else:
    print("⚠️ Differences detected. Investigate before training.")

print("\n109. hücre tamamlandı.")



### 109. Preprocessing Consistency Check

The cached training data was compared with MRI data preprocessed directly from the original files.

**Results:**
- Input shapes matched: `(128, 128, 4)`
- Mean Absolute Error: `0.00003794`
- Maximum Difference: `0.00358486`
- Difference after Float16 quantization: `0.00000000`
- Ground-truth masks identical: `True`

**Conclusion:** The tested sample shows consistent preprocessing between the cached and directly processed training data. Small numerical differences are explained by Float16 precision. No significant preprocessing mismatch was detected in this sample.


In [ ]:

# ==========================================================
# 110 — VALIDATION SPLIT & DICE METRIC AUDIT
# No training | No checkpoint modifications
# ==========================================================

import os
import json
import numpy as np
import pandas as pd

project_dir = "/content/drive/MyDrive/NeuroRobust"

# 1. Load original patient split
with open(
    f"{project_dir}/patient_split.json", "r"
) as f:
    split = json.load(f)

train_ids = set(split["train"])
val_ids = set(split["validation"])
test_ids = set(split["test"])

print("NEUROROBUST — VALIDATION AUDIT")
print("=" * 55)

print("\n1. PATIENT SPLIT CHECK")
print("Train patients:", len(train_ids))
print("Validation patients:", len(val_ids))
print("Test patients:", len(test_ids))

print("Train / Validation overlap:",
      len(train_ids & val_ids))
print("Train / Test overlap:",
      len(train_ids & test_ids))
print("Validation / Test overlap:",
      len(val_ids & test_ids))

# 2. Verify validation index
val_index = pd.read_csv(
    f"{project_dir}/validation_slice_index.csv"
)

indexed_ids = set(val_index["patient_id"])

print("\n2. VALIDATION INDEX CHECK")
print("Total validation slices:", len(val_index))
print("Indexed validation patients:", len(indexed_ids))
print("Patients outside validation split:",
      len(indexed_ids - val_ids))

print("Tumor-containing slices:",
      int((val_index["has_tumor"] == 1).sum()))

print("Background-only slices:",
      int((val_index["has_tumor"] == 0).sum()))

# 3. Compare macro and pooled Soft Dice
def soft_dice_per_slice(y_true, y_prob, eps=1e-6):

    intersection = np.sum(
        y_true * y_prob, axis=(1, 2)
    )

    denominator = (
        np.sum(y_true, axis=(1, 2))
        + np.sum(y_prob, axis=(1, 2))
    )

    return (
        2 * intersection + eps
    ) / (denominator + eps)


def soft_dice_pooled(y_true, y_prob, eps=1e-6):

    intersection = np.sum(y_true * y_prob)

    return (
        2 * intersection + eps
    ) / (
        np.sum(y_true)
        + np.sum(y_prob)
        + eps
    )


# Use the previously evaluated MRI slice
assert "ground_truth" in globals()
assert "probability" in globals()

# Create an illustrative two-slice batch:
# A = actual tumor-containing validation sample
# B = empty ground truth with tiny probabilities
truth_batch = np.stack([
    ground_truth.astype(np.float32),
    np.zeros_like(ground_truth, dtype=np.float32)
])

prob_batch = np.stack([
    probability.astype(np.float32),
    np.full_like(
        probability, 0.001, dtype=np.float32
    )
])

per_slice = soft_dice_per_slice(
    truth_batch, prob_batch
)

macro_soft_dice = np.mean(per_slice)
pooled_soft_dice = soft_dice_pooled(
    truth_batch, prob_batch
)

binary_batch = (
    prob_batch > 0.5
).astype(np.float32)

binary_scores = soft_dice_per_slice(
    truth_batch, binary_batch
)

print("\n3. METRIC DEFINITION COMPARISON")
print("Illustrative batch: 1 tumor + 1 empty slice")
print("Tumor slice Soft Dice:",
      round(float(per_slice[0]), 4))
print("Empty slice Soft Dice:",
      round(float(per_slice[1]), 4))
print("Mean per-slice Soft Dice:",
      round(float(macro_soft_dice), 4))
print("Pooled Soft Dice:",
      round(float(pooled_soft_dice), 4))
print("Mean per-slice Binary Dice:",
      round(float(binary_scores.mean()), 4))

# 4. Review actual previous diagnostic results
diagnostic_path = (
    f"{project_dir}/results/"
    "validation_diagnostic_10patients.csv"
)

diagnostic = pd.read_csv(diagnostic_path)

print("\n4. PREVIOUS 40-SLICE DIAGNOSTIC")
print("Mean per-slice Soft Dice:",
      round(diagnostic["soft_dice"].mean(), 4))
print("Mean per-slice Binary Dice:",
      round(diagnostic["binary_dice"].mean(), 4))

print("\nFINAL CHECK")
if (
    len(train_ids & val_ids) == 0
    and len(train_ids & test_ids) == 0
    and len(val_ids & test_ids) == 0
    and len(indexed_ids - val_ids) == 0
):
    print("✅ Patient split and validation index consistent.")
else:
    print("⚠️ Split mismatch detected. Investigate.")

print("✅ Metric calculation comparison completed.")
print("No training was performed.")


In [ ]:

# ==========================================================
# 111 — VALIDATION EVALUATION (RESUMABLE)
# First test: 5, 20, 50, 100, 188 patients
# Later: set MAX_PATIENTS = None for all 188
# ==========================================================

import os
import gc
import cv2
import nibabel as nib
import numpy as np
import pandas as pd

project_dir = "/content/drive/MyDrive/NeuroRobust"
data_dir = f"{project_dir}/dataset/BraTS2021_Training_Data"
results_dir = f"{project_dir}/results"

os.makedirs(results_dir, exist_ok=True)

assert "best_model" in globals(), "Load best_model first."

# SAFETY: Start with 5 patients
MAX_PATIENTS = None
BATCH_SIZE_EVAL = 8

progress_path = (
    f"{results_dir}/validation_best_progress.csv"
)

val_df = pd.read_csv(
    f"{project_dir}/validation_slice_index.csv"
).reset_index(drop=True)

patient_ids = val_df["patient_id"].drop_duplicates().tolist()

if MAX_PATIENTS is not None:
    patient_ids = patient_ids[:MAX_PATIENTS]

# Resume previously completed patients
if os.path.exists(progress_path):
    results = pd.read_csv(progress_path)
else:
    results = pd.DataFrame()

def save_progress(df):
    temp_path = progress_path + ".tmp"
    df.to_csv(temp_path, index=False)
    os.replace(temp_path, progress_path)

def normalize_volume_111(volume):
    mask = volume != 0
    result = np.zeros_like(volume, dtype=np.float32)

    if mask.any():
        mean = volume[mask].mean()
        std = volume[mask].std()
        if std > 0:
            result[mask] = (volume[mask] - mean) / std

    return result

eps = 1e-6

print("NEUROROBUST — VALIDATION EVALUATION")
print("=" * 55)

for number, pid in enumerate(patient_ids, 1):

    patient_rows = val_df[
        val_df["patient_id"] == pid
    ]

    expected_indices = set(patient_rows.index)

    # Skip a patient only when all its slices are saved
    if not results.empty:
        saved_indices = set(
            results.loc[
                results["patient_id"] == pid,
                "row_index"
            ].astype(int)
        )

        if saved_indices == expected_indices:
            print(f"{number}/{len(patient_ids)}: {pid} already saved")
            continue

        # Remove any incomplete records for this patient
        results = results[results["patient_id"] != pid].copy()

    patient_dir = f"{data_dir}/{pid}"

    volumes = []

    for modality in ["t1", "t1ce", "t2", "flair"]:
        path = f"{patient_dir}/{pid}_{modality}.nii.gz"

        volume = nib.load(path).get_fdata(
            dtype=np.float32
        )

        volumes.append(normalize_volume_111(volume))
        del volume

    mask_volume = nib.load(
        f"{patient_dir}/{pid}_seg.nii.gz"
    ).get_fdata(dtype=np.float32)

    patient_results = []

    # Process slices in small batches
    for start in range(0, len(patient_rows), BATCH_SIZE_EVAL):

        batch_rows = patient_rows.iloc[
            start:start + BATCH_SIZE_EVAL
        ]

        X_batch = []
        Y_batch = []

        for row in batch_rows.itertuples():
            idx = int(row.slice_index)

            channels = [
                cv2.resize(
                    v[:, :, idx],
                    (128, 128),
                    interpolation=cv2.INTER_LINEAR
                )
                for v in volumes
            ]

            X_batch.append(
                np.stack(channels, axis=-1)
            )

            mask = (
                mask_volume[:, :, idx] > 0
            ).astype(np.float32)

            mask = cv2.resize(
                mask,
                (128, 128),
                interpolation=cv2.INTER_NEAREST
            )

            Y_batch.append(mask)

        X_batch = np.asarray(X_batch, dtype=np.float32)
        Y_batch = np.asarray(Y_batch, dtype=np.float32)

        probabilities = best_model.predict(
            X_batch, verbose=0
        )[:, :, :, 0]

        binary = probabilities > 0.5

        for j, row in enumerate(batch_rows.itertuples()):

            truth = Y_batch[j]
            prob = probabilities[j]
            pred = binary[j]

            gt_pixels = float(truth.sum())
            pred_pixels = float(pred.sum())

            soft_intersection = float(
                np.sum(truth * prob)
            )

            prob_sum = float(prob.sum())

            binary_intersection = float(
                np.sum(truth * pred)
            )

            soft_dice = (
                2 * soft_intersection + eps
            ) / (gt_pixels + prob_sum + eps)

            binary_dice = (
                2 * binary_intersection + eps
            ) / (gt_pixels + pred_pixels + eps)

            binary_iou = (
                binary_intersection + eps
            ) / (
                gt_pixels + pred_pixels
                - binary_intersection + eps
            )

            patient_results.append({
                "row_index": int(row.Index),
                "patient_id": pid,
                "slice_index": int(row.slice_index),
                "gt_pixels": gt_pixels,
                "pred_pixels": pred_pixels,
                "soft_intersection": soft_intersection,
                "prob_sum": prob_sum,
                "soft_dice": soft_dice,
                "binary_dice": binary_dice,
                "binary_iou": binary_iou
            })

    results = pd.concat(
        [results, pd.DataFrame(patient_results)],
        ignore_index=True
    )

    results = results.drop_duplicates(
        subset=["row_index"], keep="last"
    )

    # Save after every completed patient
    save_progress(results)

    print(
        f"{number}/{len(patient_ids)} completed — "
        f"{pid} | Saved slices: {len(results)}"
    )

    del volumes, mask_volume
    gc.collect()

# ==========================================================
# RESULTS
# ==========================================================

results = results.sort_values("row_index").copy()

# Summarize only the requested group of patients
current = results[
    results["patient_id"].isin(patient_ids)
].copy()

tumor = current[current["gt_pixels"] > 0]
background = current[current["gt_pixels"] == 0]

print("\n" + "=" * 55)
print("VALIDATION SUMMARY")
print("=" * 55)

print("Patients evaluated:",
      current["patient_id"].nunique())

print("Slices evaluated:", len(current))

print("\nTUMOR-CONTAINING SLICES")
print("Mean Soft Dice:", round(tumor["soft_dice"].mean(), 4))
print("Mean Binary Dice:", round(tumor["binary_dice"].mean(), 4))
print("Mean Binary IoU:", round(tumor["binary_iou"].mean(), 4))

print("\nBACKGROUND-ONLY SLICES")
print("Background slices:", len(background))
print("False-positive slices:",
      int((background["pred_pixels"] > 0).sum()))

print("False-positive slice rate:",
      round((background["pred_pixels"] > 0).mean() * 100, 2),
      "%"
)

print("\nALL SLICES")
print("Mean per-slice Soft Dice:",
      round(current["soft_dice"].mean(), 4))

print("Mean per-slice Binary Dice:",
      round(current["binary_dice"].mean(), 4))

# Pooled soft Dice across all evaluated pixels
pooled_soft = (
    2 * current["soft_intersection"].sum() + eps
) / (
    current["gt_pixels"].sum()
    + current["prob_sum"].sum()
    + eps
)

print("Pooled Soft Dice:", round(pooled_soft, 4))

print("\nSaved results:", progress_path)
print("111. hücre tamamlandı.")



## 111. Full Validation Evaluation — 188 Patients

### Objective

This experiment evaluates the trained U-Net model on the complete patient-level validation set of the BraTS2021 dataset.

The best saved checkpoint was used for inference. Patients were processed individually to reduce memory usage, and results were saved to Google Drive after each patient.

### Dataset

- **Validation patients:** 188
- **MRI slices evaluated:** 29,140
- **Tumor-containing slices after resizing:** 12,390
- **Background-only slices after resizing:** 16,750
- **Input modalities:** T1, T1ce, T2, FLAIR
- **Image resolution:** 128 × 128
- **Prediction threshold:** 0.5

The training, validation, and test patient groups were previously verified to have no overlapping patient IDs.

### Final Validation Results

| Metric | Result |
|---|---:|
| Tumor-slice Mean Soft Dice | 0.8372 |
| Tumor-slice Mean Binary Dice | **0.8414** |
| Tumor-slice Mean Binary IoU | **0.7665** |
| Overall Mean Soft Dice | 0.3560 |
| Overall Mean Binary Dice | 0.7749 |
| Pooled Soft Dice | 0.9094 |
| Background False-Positive Rate | 27.44% |

### Evaluation Progress

| Patients | Tumor Binary Dice | Pooled Soft Dice | Background FP Rate |
|---|---:|---:|---:|
| 5 | 0.8807 | 0.9319 | 31.93% |
| 20 | 0.8484 | 0.9221 | 26.68% |
| 50 | 0.8436 | 0.9172 | 24.89% |
| 100 | 0.8446 | 0.9093 | 25.95% |
| **188** | **0.8414** | **0.9094** | **27.44%** |

### Key Findings

**1. Tumor Segmentation Performance**

The model achieved a mean Binary Dice score of 0.8414 on tumor-containing slices, indicating promising segmentation performance across the validation set.

**2. Performance Stability**

The tumor-slice Dice score remained relatively stable as the number of evaluated patients increased from 50 to 188.

**3. Background False Positives**

The model produced false-positive predictions on 4,596 of 16,750 background-only slices (27.44%). Reducing these errors remains an important improvement target.

**4. Metric Differences**

The overall per-slice Soft Dice (0.3560) was substantially lower than the pooled Soft Dice (0.9094). This difference highlights the effect of aggregation methods and empty ground-truth masks on segmentation metrics.

These metrics describe different aspects of model performance and should not be used interchangeably.

**5. Small Tumor Detection**

Earlier diagnostic experiments showed reduced segmentation performance on very small tumor regions. Further analysis is required to assess this issue across the full validation dataset.

### Limitations and Next Steps

- Investigate the difference of 55 tumor-containing slices between the original slice index and the resized ground-truth masks.
- Evaluate false-positive regions and small tumor detection.
- Consider threshold optimization using validation data.
- Report patient-level segmentation metrics.
- Evaluate the final selected model on the untouched test set.

### Conclusion

The U-Net model achieved a mean Binary Dice of **0.8414 on tumor-containing slices** across all 188 validation patients.

The results demonstrate promising segmentation performance while identifying two important challenges: false positives on background-only slices and reduced sensitivity to very small tumor regions.

The full validation evaluation is complete. These results provide a baseline for further model improvement and final testing.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:

# ======================================================
# 112 — TUMOR MASK DISCREPANCY ANALYSIS
# Original slice index vs resized validation masks
# Read-only: No training or model modifications
# ======================================================

import os
import pandas as pd
import numpy as np

project_dir = "/content/drive/MyDrive/NeuroRobust"

index_path = (
    f"{project_dir}/validation_slice_index.csv"
)

results_path = (
    f"{project_dir}/results/"
    "validation_best_progress.csv"
)

# 1. Load original and evaluated data
val_index = pd.read_csv(index_path).reset_index(drop=True)
val_index["row_index"] = val_index.index

results = pd.read_csv(results_path)

assert results["row_index"].is_unique, (
    "Duplicate evaluation rows found."
)

# 2. Match the same MRI slices by row index
df = val_index.merge(
    results[
        ["row_index", "patient_id",
         "slice_index", "gt_pixels"]
    ],
    on="row_index",
    how="left",
    suffixes=("_index", "_eval"),
    validate="one_to_one"
)

assert df["gt_pixels"].notna().all(), (
    "Some validation slices are missing."
)

assert (
    df["patient_id_index"].astype(str).values
    == df["patient_id_eval"].astype(str).values
).all(), "Patient ID mismatch!"

assert (
    df["slice_index_index"].values
    == df["slice_index_eval"].values
).all(), "Slice index mismatch!"

# 3. Original vs resized tumor status
df["original_tumor"] = (
    df["has_tumor"].astype(int) == 1
)

df["resized_tumor"] = (
    df["gt_pixels"] > 0
)

lost = df[
    df["original_tumor"]
    & ~df["resized_tumor"]
].copy()

unexpected = df[
    ~df["original_tumor"]
    & df["resized_tumor"]
].copy()

print("=" * 55)
print("112 — TUMOR MASK DISCREPANCY")
print("=" * 55)

print("\nOriginal tumor slices:",
      int(df["original_tumor"].sum()))

print("Resized tumor slices:",
      int(df["resized_tumor"].sum()))

print("Lost tumor slices:", len(lost))
print("Unexpected new tumor slices:", len(unexpected))

print("\nPatients affected:",
      lost["patient_id_index"].nunique())

# 4. Examine original tumor sizes
if len(lost) > 0:

    print("\nORIGINAL TUMOR PIXELS — LOST SLICES")

    display(
        lost["tumor_pixels"].describe()
    )

    print("\nEXAMPLES")
    display(
        lost[
            ["patient_id_index",
             "slice_index_index",
             "tumor_pixels",
             "gt_pixels"]
        ].sort_values("tumor_pixels").head(15)
    )

    print("\nLOST SLICES BY PATIENT")
    display(
        lost.groupby("patient_id_index")
        .size()
        .sort_values(ascending=False)
        .head(10)
    )

# 5. Final diagnostic
print("\nFINAL RESULT")

if len(lost) == 55 and len(unexpected) == 0:
    print("✅ All 55 missing tumor slices identified.")
    print("Their resized ground-truth masks are empty.")
elif len(lost) == 0 and len(unexpected) == 0:
    print("✅ No original/resized tumor status mismatch.")
else:
    print("⚠️ Review the mismatch counts and examples.")

print("\nNo checkpoint was modified.")



### 112. Tumor Mask Discrepancy Analysis

The original validation slice index was compared with the resized ground-truth masks used during model evaluation.

**Results:**
- Original tumor-containing slices: **12,445**
- Tumor-containing slices after resizing: **12,390**
- Slices with lost tumor regions: **55 (0.44%)**
- Unexpected new tumor slices: **0**

**Key Finding:** All 55 missing tumor-containing slices were identified. Very small tumor regions, including single-pixel examples, became empty after resizing the masks to 128 × 128.

**Conclusion:** Mask downsampling can remove very small tumor regions and affect small-lesion evaluation. Future experiments should investigate alternative mask resizing methods and higher image resolutions.

The original MRI files and saved model checkpoints were not modified.


In [ ]:

# =====================================================
# 113 — SMALL TUMOR MASK RESIZING EXPERIMENT
# Compare Nearest Neighbor vs Area-based preservation
# No model training or checkpoint modifications
# =====================================================

import os
import cv2
import nibabel as nib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

project_dir = "/content/drive/MyDrive/NeuroRobust"
data_dir = f"{project_dir}/dataset/BraTS2021_Training_Data"

# Load previously saved validation records
val_df = pd.read_csv(
    f"{project_dir}/validation_slice_index.csv"
).reset_index(drop=True)

val_df["row_index"] = val_df.index

evaluated = pd.read_csv(
    f"{project_dir}/results/validation_best_progress.csv"
)

assert evaluated["row_index"].is_unique

# Match original and resized slice information
merged = val_df.merge(
    evaluated[["row_index", "gt_pixels"]],
    on="row_index",
    how="inner",
    validate="one_to_one"
)

assert len(merged) == len(val_df), (
    "Missing validation evaluation results."
)

# Identify tumor slices lost during resizing
lost = merged[
    (merged["has_tumor"] == 1)
    & (merged["gt_pixels"] == 0)
].copy()

print("Lost tumor-containing slices:", len(lost))

# Select three examples with the smallest tumor areas
examples = lost.sort_values(
    "tumor_pixels"
).head(3)

assert len(examples) > 0, "No lost slices found."

fig, axes = plt.subplots(
    len(examples), 3,
    figsize=(12, 4 * len(examples)),
    squeeze=False
)

for i, (_, row) in enumerate(examples.iterrows()):

    pid = row["patient_id"]
    idx = int(row["slice_index"])

    seg_path = (
        f"{data_dir}/{pid}/{pid}_seg.nii.gz"
    )

    seg = nib.load(seg_path).get_fdata(
        dtype=np.float32
    )

    original = (
        seg[:, :, idx] > 0
    ).astype(np.uint8)

    # Method 1: Current Nearest Neighbor
    nearest = cv2.resize(
        original,
        (128, 128),
        interpolation=cv2.INTER_NEAREST
    )

    # Method 2: Area-based preservation
    area_map = cv2.resize(
        original.astype(np.float32),
        (128, 128),
        interpolation=cv2.INTER_AREA
    )

    preserved = (
        area_map > 0
    ).astype(np.uint8)

    # Center zoom around the original tumor
    coords = np.argwhere(original > 0)
    cy, cx = np.mean(coords, axis=0).astype(int)

    # Original: 40 x 40 pixel region
    r = 20
    original_crop = original[
        max(0, cy-r):min(original.shape[0], cy+r),
        max(0, cx-r):min(original.shape[1], cx+r)
    ]

    # Corresponding center in resized masks
    sy = int(cy * 128 / original.shape[0])
    sx = int(cx * 128 / original.shape[1])

    r2 = 12

    def crop_128(img):
        return img[
            max(0, sy-r2):min(128, sy+r2),
            max(0, sx-r2):min(128, sx+r2)
        ]

    images = [
        original_crop,
        crop_128(nearest),
        crop_128(preserved)
    ]

    titles = [
        "Original Mask (Zoom)",
        "Nearest Neighbor (Zoom)",
        "Area-Preserving (Zoom)"
    ]

    for j in range(3):
        axes[i, j].imshow(
            images[j],
            cmap="gray",
            vmin=0,
            vmax=1,
            interpolation="nearest"
        )
        axes[i, j].set_title(titles[j])
        axes[i, j].axis("off")

    print(
        f"{pid} | Slice {idx} | "
        f"Original: {original.sum()} | "
        f"Nearest: {nearest.sum()} | "
        f"Area-based: {preserved.sum()}"
    )

plt.suptitle(
    "NeuroRobust — Small Tumor Mask Preservation",
    fontsize=15
)

plt.tight_layout()

save_path = (
    f"{project_dir}/results/"
    "small_tumor_resizing_comparison.png"
)

plt.savefig(
    save_path,
    dpi=160,
    bbox_inches="tight"
)

plt.show()

print("\nFigure saved:", save_path)
print("113. hücre tamamlandı.")


### Small Tumor Mask Preservation Analysis

This experiment confirmed that the missing tumor-containing slices are caused by the mask resizing step rather than by the model itself.

Among the 55 lost tumor-containing slices, the inspected examples showed that the original ground-truth mask contained only **1 pixel**. When resized with **nearest-neighbor interpolation**, these tiny tumor regions disappeared completely (**1 → 0 pixels**). In contrast, the **area-preserving alternative** was able to retain them (**1 → 1 pixel**).

This result shows that the discrepancy comes from preprocessing, especially for extremely small tumor regions. Therefore, the loss of these slices should be interpreted as a **mask resizing artifact**, not as an actual segmentation failure of the U-Net model.

Overall, the analysis suggests that future improvements should focus on using a more robust mask resizing strategy to better preserve very small tumor regions.

In [ ]:

# ==========================================================
# 114 — SMALL TUMOR PRESERVATION EXPERIMENT
# Analyze all 55 lost slices + 30 control tumor slices
# No training, no checkpoint changes
# ==========================================================

import os
import gc
import cv2
import nibabel as nib
import numpy as np
import pandas as pd

from IPython.display import display

project_dir = "/content/drive/MyDrive/NeuroRobust"
data_dir = f"{project_dir}/dataset/BraTS2021_Training_Data"
results_dir = f"{project_dir}/results"

os.makedirs(results_dir, exist_ok=True)

# 1. Load validation records
val_df = pd.read_csv(
    f"{project_dir}/validation_slice_index.csv"
).reset_index(drop=True)

val_df["row_index"] = val_df.index

evaluation = pd.read_csv(
    f"{results_dir}/validation_best_progress.csv"
)

assert evaluation["row_index"].is_unique

merged = val_df.merge(
    evaluation[["row_index", "gt_pixels"]],
    on="row_index",
    how="inner",
    validate="one_to_one"
)

assert len(merged) == len(val_df), (
    "Validation records are incomplete."
)

# 2. Identify the lost tumor slices
lost = merged[
    (merged["has_tumor"] == 1)
    & (merged["gt_pixels"] == 0)
].copy()

assert len(lost) == 55, (
    f"Expected 55 lost slices, found {len(lost)}"
)

# 3. Select additional tumor-containing controls
controls = merged[
    (merged["has_tumor"] == 1)
    & (merged["gt_pixels"] > 0)
].copy()

# Select up to 10 from each original tumor-size range
control_groups = [
    controls[controls["tumor_pixels"] <= 10],
    controls[
        (controls["tumor_pixels"] > 10)
        & (controls["tumor_pixels"] <= 100)
    ],
    controls[controls["tumor_pixels"] > 100]
]

control_samples = [
    group.sample(
        n=min(10, len(group)),
        random_state=42
    )
    for group in control_groups
    if len(group) > 0
]

control_df = pd.concat(
    control_samples,
    ignore_index=True
)

lost["sample_type"] = "Lost tumor"
control_df["sample_type"] = "Control"

samples = pd.concat(
    [lost, control_df],
    ignore_index=True
)

print("NEUROROBUST — MASK PRESERVATION EXPERIMENT")
print("=" * 55)
print("Lost slices:", len(lost))
print("Control slices:", len(control_df))
print("Total slices:", len(samples))

# 4. Compare resizing methods
records = []

# Load each patient's segmentation only once
for pid, patient_rows in samples.groupby(
    "patient_id", sort=False
):

    path = (
        f"{data_dir}/{pid}/{pid}_seg.nii.gz"
    )

    seg = nib.load(path).get_fdata(
        dtype=np.float32
    )

    for _, row in patient_rows.iterrows():

        idx = int(row["slice_index"])

        original = (
            seg[:, :, idx] > 0
        ).astype(np.uint8)

        nearest = cv2.resize(
            original,
            (128, 128),
            interpolation=cv2.INTER_NEAREST
        )

        area_fraction = cv2.resize(
            original.astype(np.float32),
            (128, 128),
            interpolation=cv2.INTER_AREA
        )

        # Any positive contribution is treated as tumor
        any_overlap = (
            area_fraction > 0
        ).astype(np.uint8)

        original_pixels = int(original.sum())
        nearest_pixels = int(nearest.sum())
        overlap_pixels = int(any_overlap.sum())

        # Expected area if physical area were preserved
        scale_factor = (
            128 * 128 / original.size
        )

        expected_pixels = (
            original_pixels * scale_factor
        )

        records.append({
            "patient_id": pid,
            "slice_index": idx,
            "sample_type": row["sample_type"],
            "original_pixels": original_pixels,
            "expected_scaled_pixels": expected_pixels,
            "nearest_pixels": nearest_pixels,
            "any_overlap_pixels": overlap_pixels,
            "recovered": (
                nearest_pixels == 0
                and overlap_pixels > 0
            ),
            "extra_vs_nearest": (
                overlap_pixels - nearest_pixels
            ),
            "area_inflation_factor": (
                overlap_pixels / expected_pixels
                if expected_pixels > 0 else np.nan
            )
        })

    del seg
    gc.collect()

results_114 = pd.DataFrame(records)

# 5. Summarize lost slices
lost_results = results_114[
    results_114["sample_type"] == "Lost tumor"
].copy()

recovered_count = int(
    lost_results["recovered"].sum()
)

print("\n" + "=" * 55)
print("A. LOST TUMOR RECOVERY")
print("=" * 55)

print("Originally lost:", len(lost_results))
print("Recovered by any-overlap:", recovered_count)
print(
    "Recovery rate:",
    f"{recovered_count / len(lost_results):.2%}"
)

print("\nOriginal pixel statistics:")
display(
    lost_results["original_pixels"].describe()
)

print("\nNew mask pixel statistics:")
display(
    lost_results["any_overlap_pixels"].describe()
)

# 6. Compare control slices
control_results = results_114[
    results_114["sample_type"] == "Control"
].copy()

print("\n" + "=" * 55)
print("B. CONTROL SLICE AREA COMPARISON")
print("=" * 55)

print("Control slices:", len(control_results))

print(
    "Mean Nearest mask pixels:",
    round(control_results["nearest_pixels"].mean(), 2)
)

print(
    "Mean Any-overlap mask pixels:",
    round(control_results["any_overlap_pixels"].mean(), 2)
)

print(
    "Median additional pixels:",
    round(control_results["extra_vs_nearest"].median(), 2)
)

print(
    "Median inflation vs expected scaled area:",
    round(
        control_results["area_inflation_factor"].median(),
        3
    )
)

# 7. Show example comparisons
print("\nC. EXAMPLE RESULTS")

display(
    results_114[
        ["patient_id",
         "slice_index",
         "sample_type",
         "original_pixels",
         "expected_scaled_pixels",
         "nearest_pixels",
         "any_overlap_pixels",
         "area_inflation_factor"]
    ].head(15).round(3)
)

# 8. Save without modifying old evaluation data
save_path = (
    f"{results_dir}/"
    "small_tumor_preservation_experiment.csv"
)

results_114.to_csv(
    save_path,
    index=False
)

print("\n" + "=" * 55)
print("FINAL STATUS")
print("=" * 55)

print("Experiment completed.")
print("Results saved:", save_path)
print("No model or original dataset was modified.")



### 114. Small Tumor Preservation Experiment

We compared Nearest Neighbor interpolation with an area-based any-overlap method on 55 lost tumor slices and 30 additional control slices.

**Results:**
- Recovered tumor-containing slices: **55/55 (100%)**
- Mean Nearest Neighbor mask area (controls): **224.13 pixels**
- Mean Any-overlap mask area (controls): **260.40 pixels**
- Median additional mask area (controls): **17 pixels**
- Median area inflation relative to expected scaled area: **1.844×**

**Conclusion:** The any-overlap method successfully preserved all 55 previously lost tumor-containing slices. However, it also increased mask areas in the control sample, indicating a trade-off between small-lesion preservation and segmentation-label accuracy.

Further threshold experiments are needed before adopting this method for model training or evaluation. The original dataset and model checkpoints remain unchanged.


In [ ]:

# ======================================================
# 115 — AREA-BASED MASK THRESHOLD EXPERIMENT
# Compare preservation vs area inflation
# No training or checkpoint modification
# ======================================================

import os
import cv2
import nibabel as nib
import numpy as np
import pandas as pd
from IPython.display import display

project_dir = "/content/drive/MyDrive/NeuroRobust"
data_dir = f"{project_dir}/dataset/BraTS2021_Training_Data"
results_dir = f"{project_dir}/results"

samples = pd.read_csv(
    f"{results_dir}/small_tumor_preservation_experiment.csv"
)

thresholds = [0.0, 0.05, 0.10, 0.25, 0.50]
records = []

for pid, group in samples.groupby("patient_id"):

    seg_path = f"{data_dir}/{pid}/{pid}_seg.nii.gz"
    seg = nib.load(seg_path).get_fdata(dtype=np.float32)

    for row in group.itertuples():

        idx = int(row.slice_index)
        original = (seg[:, :, idx] > 0).astype(np.float32)

        area_map = cv2.resize(
            original,
            (128, 128),
            interpolation=cv2.INTER_AREA
        )

        nearest = cv2.resize(
            original,
            (128, 128),
            interpolation=cv2.INTER_NEAREST
        )

        expected_area = (
            original.sum() * 128 * 128 / original.size
        )

        candidates = {
            "Nearest": nearest > 0.5
        }

        for t in thresholds:
            candidates[f"Area > {t:.2f}"] = area_map > t

        for method, mask in candidates.items():
            pixels = int(mask.sum())

            records.append({
                "patient_id": pid,
                "slice_index": idx,
                "sample_type": row.sample_type,
                "method": method,
                "mask_pixels": pixels,
                "preserved": pixels > 0,
                "area_ratio": (
                    pixels / expected_area
                    if expected_area > 0 else np.nan
                ),
                "area_relative_error": (
                    abs(pixels - expected_area) / expected_area
                    if expected_area > 0 else np.nan
                )
            })

    del seg

df115 = pd.DataFrame(records)

lost_summary = (
    df115[df115["sample_type"] == "Lost tumor"]
    .groupby("method")["preserved"]
    .agg(["sum", "count"])
)

control_summary = (
    df115[df115["sample_type"] == "Control"]
    .groupby("method")
    .agg(
        median_area_ratio=("area_ratio", "median"),
        median_relative_error=("area_relative_error", "median"),
        mean_mask_pixels=("mask_pixels", "mean")
    )
)

summary115 = lost_summary.join(control_summary)

summary115["recovery_rate"] = (
    summary115["sum"] / summary115["count"]
)

summary115 = summary115.rename(
    columns={"sum": "recovered_slices"}
).drop(columns=["count"])

order = (
    ["Nearest"]
    + [f"Area > {t:.2f}" for t in thresholds]
)

summary115 = summary115.reindex(order)

print("=" * 60)
print("115 — MASK THRESHOLD COMPARISON")
print("=" * 60)

display(summary115.round(4))

print("\nInterpretation:")
print("- Recovery rate: fraction of 55 lost slices preserved")
print("- Area ratio: mask pixels / expected scaled pixels")
print("- Relative error: area difference from expected scale")
print("- Lower area error is preferable, but not sufficient")

# Save separately; original results are unchanged
df115.to_csv(
    f"{results_dir}/mask_threshold_details.csv",
    index=False
)

summary115.to_csv(
    f"{results_dir}/mask_threshold_summary.csv"
)

print("\nResults saved successfully.")
print("No model training was performed.")


In [ ]:

# ==========================================================
# 116 — SELECTIVE SMALL-TUMOR RECOVERY
# Nearest Neighbor + one-pixel rescue for lost tumors
# No training, no checkpoint modifications
# ==========================================================

import os
import cv2
import nibabel as nib
import numpy as np
import pandas as pd

from IPython.display import display

project_dir = "/content/drive/MyDrive/NeuroRobust"
data_dir = f"{project_dir}/dataset/BraTS2021_Training_Data"
results_dir = f"{project_dir}/results"

samples = pd.read_csv(
    f"{results_dir}/small_tumor_preservation_experiment.csv"
)

records = []

for pid, group in samples.groupby("patient_id"):

    seg_path = f"{data_dir}/{pid}/{pid}_seg.nii.gz"
    seg = nib.load(seg_path).get_fdata(dtype=np.float32)

    for row in group.itertuples():

        idx = int(row.slice_index)

        original = (
            seg[:, :, idx] > 0
        ).astype(np.float32)

        # Baseline
        nearest = cv2.resize(
            original,
            (128, 128),
            interpolation=cv2.INTER_NEAREST
        ).astype(np.uint8)

        # Fractional pixel coverage
        area_map = cv2.resize(
            original,
            (128, 128),
            interpolation=cv2.INTER_AREA
        )

        area_010 = (area_map > 0.10).astype(np.uint8)

        # Proposed selective recovery
        hybrid = nearest.copy()
        rescue_applied = False

        if original.sum() > 0 and hybrid.sum() == 0:

            # Find the output pixel with maximum tumor coverage
            flat_index = int(np.argmax(area_map))
            y, x = np.unravel_index(
                flat_index, area_map.shape
            )

            if area_map[y, x] > 0:
                hybrid[y, x] = 1
                rescue_applied = True

        expected = (
            original.sum() * 128 * 128 / original.size
        )

        records.append({
            "patient_id": pid,
            "slice_index": idx,
            "sample_type": row.sample_type,
            "original_pixels": int(original.sum()),
            "expected_pixels": float(expected),
            "nearest_pixels": int(nearest.sum()),
            "area_010_pixels": int(area_010.sum()),
            "hybrid_pixels": int(hybrid.sum()),
            "rescued": rescue_applied,
            "changed_pixels_vs_nearest": int(
                np.count_nonzero(hybrid != nearest)
            ),
            "area_010_abs_error": abs(
                int(area_010.sum()) - expected
            ),
            "hybrid_abs_error": abs(
                int(hybrid.sum()) - expected
            )
        })

results116 = pd.DataFrame(records)

lost = results116[
    results116["sample_type"] == "Lost tumor"
]

control = results116[
    results116["sample_type"] == "Control"
]

print("=" * 55)
print("116 — SELECTIVE RECOVERY EXPERIMENT")
print("=" * 55)

print("\nA. LOST TUMOR RECOVERY")
print("Lost slices:", len(lost))

print(
    "Area > 0.10 recovered:",
    int((lost["area_010_pixels"] > 0).sum())
)

print(
    "Selective recovery:",
    int((lost["hybrid_pixels"] > 0).sum())
)

print(
    "Recovery operations:",
    int(lost["rescued"].sum())
)

print("\nB. CONTROL MASK PRESERVATION")

print("Control slices:", len(control))

unchanged = int(
    (control["changed_pixels_vs_nearest"] == 0).sum()
)

print("Unchanged control masks:", unchanged)

print(
    "Mean Nearest pixels:",
    round(control["nearest_pixels"].mean(), 2)
)

print(
    "Mean Area > 0.10 pixels:",
    round(control["area_010_pixels"].mean(), 2)
)

print(
    "Mean Selective pixels:",
    round(control["hybrid_pixels"].mean(), 2)
)

print("\nC. EXAMPLE COMPARISON")

display(
    results116[
        [
            "patient_id",
            "sample_type",
            "original_pixels",
            "nearest_pixels",
            "area_010_pixels",
            "hybrid_pixels",
            "changed_pixels_vs_nearest"
        ]
    ].head(12)
)

save_path = (
    f"{results_dir}/"
    "selective_recovery_experiment.csv"
)

results116.to_csv(save_path, index=False)

print("\nResults saved:", save_path)
print("Original masks and model checkpoints unchanged.")



### 115–116. Small Tumor Preservation and Selective Recovery

Two experiments were conducted to investigate the trade-off between preserving very small tumor regions and avoiding artificial mask expansion.

#### Experiment 1: Area-Based Threshold Comparison

Different thresholds were tested using area-based mask resizing.

- Nearest Neighbor: 0/55 lost tumor slices recovered.
- Area > 0.10: 55/55 slices recovered, but control masks became larger.
- Area > 0.25: 39/55 slices recovered.
- Area > 0.50: 10/55 slices recovered.

#### Experiment 2: Selective Small-Tumor Recovery

A selective recovery strategy was tested. Nearest Neighbor was used by default, and one tumor pixel was added only when a non-empty original tumor mask became completely empty after resizing.

**Results:**
- Lost tumor slices recovered: **55/55 (100%)**
- Control masks unchanged: **30/30 (100%)**
- Mean control mask area: **224.13 pixels**, identical to Nearest Neighbor
- Recovery operations performed: **55**

#### Conclusion

Selective Recovery preserved all previously lost tumor-containing slices without modifying the 30 sampled control masks.

This approach avoids the widespread mask expansion observed with area-based thresholding.

However, these experiments evaluate label preservation rather than model segmentation accuracy. Further validation is required to assess spatial accuracy and the effect of the proposed preprocessing method on model performance.

The original dataset and model checkpoints were not modified.


In [ ]:

# ==========================================================
# 117 — CONNECTED COMPONENT FILTERING EXPERIMENT
# 20 validation patients
# Goal: reduce false positives without losing small tumors
# No training or model checkpoint modifications
# ==========================================================

import os
import gc
import cv2
import numpy as np
import pandas as pd
import nibabel as nib
import tensorflow as tf

from google.colab import drive
from IPython.display import display

# 1. Drive connection
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

project_dir = "/content/drive/MyDrive/NeuroRobust"
data_dir = f"{project_dir}/dataset/BraTS2021_Training_Data"
results_dir = f"{project_dir}/results"

os.makedirs(results_dir, exist_ok=True)

# 2. Load the best checkpoint if needed
if "best_model" not in globals():
    best_model = tf.keras.models.load_model(
        f"{project_dir}/models/best_unet_local.keras",
        compile=False
    )

print("Best U-Net checkpoint ready.")

# 3. Experiment settings
MAX_PATIENTS = None
BATCH_SIZE = 8

# Minimum connected component area (pixels)
MIN_COMPONENT_SIZES = [0, 2, 5, 10, 20, 50]

progress_path = (
    f"{results_dir}/"
    "connected_component_experiment_progress.csv"
)

val_df = pd.read_csv(
    f"{project_dir}/validation_slice_index.csv"
).reset_index(drop=True)

val_df["row_index"] = val_df.index

patient_ids = (
    val_df["patient_id"]
    .drop_duplicates()
    .tolist()[:MAX_PATIENTS]
)

# Resume support
if os.path.exists(progress_path):
    results117 = pd.read_csv(progress_path)
else:
    results117 = pd.DataFrame()

# 4. MRI normalization: same as cell 111
def normalize_volume_117(volume):
    brain = volume != 0
    output = np.zeros_like(volume, dtype=np.float32)

    if np.any(brain):
        mean = volume[brain].mean()
        std = volume[brain].std()

        if std > 0:
            output[brain] = (
                volume[brain] - mean
            ) / std

    return output

# 5. Remove small connected components
def filter_components(mask, min_size):

    if min_size <= 1:
        return mask.astype(np.uint8)

    mask = mask.astype(np.uint8)

    number, labels, stats, _ = (
        cv2.connectedComponentsWithStats(
            mask, connectivity=8
        )
    )

    keep = np.zeros(number, dtype=bool)

    if number > 1:
        keep[1:] = (
            stats[1:, cv2.CC_STAT_AREA] >= min_size
        )

    return keep[labels].astype(np.uint8)

eps = 1e-6

print("\n117 — CONNECTED COMPONENT EVALUATION")
print("=" * 55)

# 6. Evaluate patients
for number, pid in enumerate(patient_ids, 1):

    patient_rows = val_df[
        val_df["patient_id"] == pid
    ]

    expected_count = (
        len(patient_rows) * len(MIN_COMPONENT_SIZES)
    )

    if not results117.empty:

        existing = results117[
            results117["patient_id"] == pid
        ]

        if (
            len(existing) == expected_count
            and not existing.duplicated(
                ["row_index", "min_size"]
            ).any()
        ):
            print(
                f"{number}/{len(patient_ids)} "
                f"{pid} already saved"
            )
            continue

        # Reprocess incomplete patients
        results117 = results117[
            results117["patient_id"] != pid
        ].copy()

    patient_dir = f"{data_dir}/{pid}"

    volumes = []

    for modality in ["t1", "t1ce", "t2", "flair"]:

        path = (
            f"{patient_dir}/{pid}_{modality}.nii.gz"
        )

        volume = nib.load(path).get_fdata(
            dtype=np.float32
        )

        volumes.append(
            normalize_volume_117(volume)
        )

        del volume

    seg = nib.load(
        f"{patient_dir}/{pid}_seg.nii.gz"
    ).get_fdata(dtype=np.float32)

    patient_results = []

    for start in range(0, len(patient_rows), BATCH_SIZE):

        batch_rows = patient_rows.iloc[
            start:start + BATCH_SIZE
        ]

        X_batch = []
        Y_batch = []

        for row in batch_rows.itertuples():

            idx = int(row.slice_index)

            channels = [
                cv2.resize(
                    volume[:, :, idx],
                    (128, 128),
                    interpolation=cv2.INTER_LINEAR
                )
                for volume in volumes
            ]

            X_batch.append(
                np.stack(channels, axis=-1)
            )

            truth = (
                seg[:, :, idx] > 0
            ).astype(np.uint8)

            truth = cv2.resize(
                truth,
                (128, 128),
                interpolation=cv2.INTER_NEAREST
            )

            Y_batch.append(truth)

        X_batch = np.asarray(
            X_batch, dtype=np.float32
        )

        Y_batch = np.asarray(
            Y_batch, dtype=np.uint8
        )

        probabilities = best_model.predict(
            X_batch, verbose=0
        )[:, :, :, 0]

        # Same 0.5 threshold as baseline
        raw_predictions = (
            probabilities > 0.5
        ).astype(np.uint8)

        for j, row in enumerate(batch_rows.itertuples()):

            truth = Y_batch[j]
            raw_prediction = raw_predictions[j]

            gt_pixels = int(truth.sum())

            # Test each component-size threshold
            for min_size in MIN_COMPONENT_SIZES:

                filtered = filter_components(
                    raw_prediction, min_size
                )

                pred_pixels = int(filtered.sum())
                intersection = int(
                    np.sum(
                        truth.astype(bool)
                        & filtered.astype(bool)
                    )
                )

                dice = (
                    2 * intersection + eps
                ) / (
                    gt_pixels + pred_pixels + eps
                )

                union = (
                    gt_pixels + pred_pixels
                    - intersection
                )

                iou = (
                    intersection + eps
                ) / (union + eps)

                patient_results.append({
                    "row_index": int(row.row_index),
                    "patient_id": pid,
                    "slice_index": int(row.slice_index),
                    "min_size": min_size,
                    "gt_pixels": gt_pixels,
                    "pred_pixels": pred_pixels,
                    "intersection": intersection,
                    "binary_dice": dice,
                    "binary_iou": iou
                })

    results117 = pd.concat(
        [results117, pd.DataFrame(patient_results)],
        ignore_index=True
    )

    # Save after each completed patient
    temp_path = progress_path + ".tmp"

    results117.to_csv(
        temp_path, index=False
    )

    os.replace(temp_path, progress_path)

    print(
        f"{number}/{len(patient_ids)} completed — {pid}"
    )

    del volumes, seg
    gc.collect()

# ==========================================================
# 7. Compare filtering methods
# ==========================================================

current = results117[
    results117["patient_id"].isin(patient_ids)
].copy()

summary_rows = []

for min_size in MIN_COMPONENT_SIZES:

    subset = current[
        current["min_size"] == min_size
    ]

    tumor = subset[
        subset["gt_pixels"] > 0
    ]

    background = subset[
        subset["gt_pixels"] == 0
    ]

    small = tumor[
        tumor["gt_pixels"] <= 10
    ]

    summary_rows.append({
        "Min Component": min_size,
        "Tumor Dice": tumor["binary_dice"].mean(),
        "Small Tumor Dice": small["binary_dice"].mean(),
        "Small Tumors Missed": (
            (small["intersection"] == 0).mean()
            if len(small) > 0 else np.nan
        ),
        "Background FP Rate": (
            (background["pred_pixels"] > 0).mean()
        ),
        "Mean FP Pixels": (
            background["pred_pixels"].mean()
        ),
        "All-Slice Dice": subset["binary_dice"].mean()
    })

summary117 = pd.DataFrame(summary_rows)

print("\n" + "=" * 65)
print("117 — CONNECTED COMPONENT FILTERING RESULTS")
print("=" * 65)

print("Patients:", current["patient_id"].nunique())

print(
    "Slices:",
    current[current["min_size"] == 0].shape[0]
)

display(summary117.round(4))

# 8. Cross-check baseline with cell 111
baseline_path = (
    f"{results_dir}/validation_best_progress.csv"
)

if os.path.exists(baseline_path):

    previous = pd.read_csv(baseline_path)

    baseline = current[
        current["min_size"] == 0
    ]

    comparison = baseline.merge(
        previous[["row_index", "binary_dice"]],
        on="row_index",
        suffixes=("_117", "_111")
    )

    if len(comparison) == len(baseline):

        max_difference = (
            comparison["binary_dice_117"]
            - comparison["binary_dice_111"]
        ).abs().max()

        print(
            "\nBaseline max Dice difference vs cell 111:",
            round(float(max_difference), 8)
        )

        if max_difference > 0.001:
            print(
                "WARNING: Baseline mismatch. "
                "Check preprocessing before conclusions."
            )

# 9. Save summary
summary_path = (
    f"{results_dir}/"
    "connected_component_experiment_summary.csv"
)

summary117.to_csv(
    summary_path, index=False
)

print("\nExperiment completed.")
print("Results saved to Google Drive.")
print("No model training was performed.")



### 117. Connected Component Filtering Analysis

Connected Component Filtering was evaluated on 3,100 MRI slices from 20 validation patients to reduce false-positive predictions.

**Key Results:**
- Baseline Tumor Dice: **0.8484**
- Baseline Background FP Rate: **26.68%**
- 2-pixel filter: Tumor Dice **0.8483**, FP Rate **24.10%**
- 5-pixel filter: Tumor Dice **0.8450**, FP Rate **17.73%**
- 20-pixel filter: FP Rate **5.16%**, but Small Tumor Dice dropped to **0.0338**

**Findings:**

Larger component thresholds substantially reduced false positives but also removed genuine small tumor regions.

The 2-pixel filter showed a promising preliminary trade-off, reducing false positives while maintaining tumor segmentation performance.

The baseline predictions were consistent with the previous validation experiment (maximum Dice difference: 0.0).

**Conclusion:**

Connected Component Filtering can reduce false positives, but aggressive filtering significantly harms small-tumor detection. Further validation is required before selecting a final post-processing strategy.

No model weights were modified during this experiment.


In [ ]:

from google.colab import drive
import os

drive.mount('/content/drive')

project_dir = "/content/drive/MyDrive/NeuroRobust"

print("Project found:", os.path.exists(project_dir))
print("Model found:", os.path.exists(
    f"{project_dir}/models/best_unet_local.keras"
))
print("Experiment 117 found:", os.path.exists(
    f"{project_dir}/results/connected_component_experiment_summary.csv"
))



## 117–118. Connected Component Filtering — Full Validation Analysis

### Objective

This experiment investigated whether Connected Component Filtering could reduce false-positive tumor predictions without significantly affecting small-tumor segmentation.

Six minimum component sizes (0, 2, 5, 10, 20, and 50 pixels) were evaluated using the best U-Net checkpoint.

### Dataset

- Validation patients: **188**
- MRI slices: **29,140**
- Input resolution: **128 × 128**
- Prediction threshold: **0.5**

### Final Results

| Minimum Component | Tumor Dice | Small Tumor Dice | Background FP Rate |
|---|---:|---:|---:|
| 0 (Baseline) | 0.8414 | 0.2572 | 27.44% |
| 2 | 0.8411 | 0.2519 | 24.58% |
| 5 | 0.8395 | 0.2245 | 17.50% |
| 10 | 0.8329 | 0.1300 | 10.60% |
| 20 | 0.8151 | 0.0277 | 5.18% |
| 50 | 0.7687 | 0.0033 | 1.31% |

### Key Findings

- Larger component thresholds substantially reduced background false positives.
- Aggressive filtering also removed genuine small tumor predictions.
- The 2-pixel filter reduced the false-positive slice rate from 27.44% to 24.58%, while tumor Dice changed only slightly (0.8414 to 0.8411).
- Small Tumor Dice decreased from 0.2572 to 0.2519 with the 2-pixel filter.
- The 5-pixel filter achieved a larger false-positive reduction but caused greater small-tumor performance loss.
- Baseline predictions matched the previous validation experiment (maximum Dice difference: 0.0).

### Conclusion

Connected Component Filtering demonstrates a trade-off between false-positive reduction and small-tumor sensitivity.

The 2-pixel threshold is a conservative candidate because it reduces false positives while largely maintaining overall tumor segmentation performance.

However, it still causes a small reduction in small-tumor sensitivity.

The final filtering threshold must be fixed before evaluation on the independent test dataset.

No model weights were modified during this experiment.


In [ ]:
# ============================================================
# 119 — INDEPENDENT TEST PREPARATION
# ============================================================

from google.colab import drive
from pathlib import Path
import pandas as pd

drive.mount('/content/drive')

project_dir = Path("/content/drive/MyDrive/NeuroRobust")

print("=" * 55)
print("119 — INDEPENDENT TEST PREPARATION")
print("=" * 55)

# 1. Proje ve model kontrolü
model_path = project_dir / "models/best_unet_local.keras"

print("\nPROJECT STATUS")
print("Project found:", project_dir.exists())
print("Model found:", model_path.exists())

# 2. Hasta ayrım dosyalarını araştır
print("\nPATIENT SPLIT FILES")

folders = [
    project_dir,
    project_dir / "data",
    project_dir / "splits",
    project_dir / "results"
]

split_files = []

for folder in folders:
    if not folder.exists():
        continue

    for file in folder.iterdir():
        if file.is_file() and file.suffix.lower() in [".csv", ".json"]:
            if any(
                key in file.name.lower()
                for key in ["split", "train", "val", "test", "patient"]
            ):
                split_files.append(file)

for file in sorted(set(split_files)):
    print(file.relative_to(project_dir))

    if file.suffix.lower() == ".csv":
        try:
            df = pd.read_csv(file, nrows=3)
            print("Columns:", list(df.columns))
        except Exception as e:
            print("Read error:", str(e))

# 3. Mevcut oturumdaki veri ayrımı değişkenleri
print("\nAVAILABLE DATASET VARIABLES")

for name in [
    "train_patients",
    "val_patients",
    "test_patients",
    "train_patient_ids",
    "val_patient_ids",
    "test_patient_ids",
    "train_df",
    "val_df",
    "test_df"
]:
    if name in globals():
        obj = globals()[name]
        try:
            print(name, ":", len(obj))
        except TypeError:
            print(name, ": available")

print("\n119 — PREPARATION CHECK COMPLETED")

In [ ]:

# ==========================================================
# 120 — INDEPENDENT TEST EVALUATION
# Locked settings: threshold 0.5, component size 2
# First 5 patients; resumable and checkpoint-safe
# ==========================================================

import os
import json
import gc
import cv2
import nibabel as nib
import numpy as np
import pandas as pd
import tensorflow as tf

from IPython.display import display

project_dir = "/content/drive/MyDrive/NeuroRobust"
data_dir = f"{project_dir}/dataset/BraTS2021_Training_Data"
results_dir = f"{project_dir}/results"

# Start safely with 5 patients
MAX_TEST_PATIENTS = None
BATCH_SIZE = 8

# Fixed before testing — DO NOT optimize on test data
PRED_THRESHOLD = 0.5
COMPONENT_SIZES = [0, 2]

progress_path = (
    f"{results_dir}/independent_test_progress.csv"
)

# 1. Read patient-level split
with open(f"{project_dir}/patient_split.json") as f:
    split = json.load(f)

train_ids = set(split["train"])
val_ids = set(split["validation"])
test_ids = list(split["test"])

assert len(test_ids) == len(set(test_ids))
assert not (set(test_ids) & train_ids)
assert not (set(test_ids) & val_ids)
assert not (train_ids & val_ids)

print("INDEPENDENT TEST — PATIENT SPLIT")
print("Train:", len(train_ids))
print("Validation:", len(val_ids))
print("Test:", len(test_ids))
print("Patient overlap: 0")

# 2. Verify that ground-truth MRI files exist
selected_ids = (
    test_ids if MAX_TEST_PATIENTS is None
    else test_ids[:MAX_TEST_PATIENTS]
)

for pid in selected_ids:
    required = [
        f"{data_dir}/{pid}/{pid}_{m}.nii.gz"
        for m in ["t1", "t1ce", "t2", "flair", "seg"]
    ]

    missing = [p for p in required if not os.path.isfile(p)]
    assert not missing, f"Missing files for {pid}: {missing}"

print("MRI files verified for", len(selected_ids), "patients")

# 3. Load original best checkpoint
if "best_model" not in globals():
    best_model = tf.keras.models.load_model(
        f"{project_dir}/models/best_unet_local.keras",
        compile=False
    )

assert best_model.input_shape[-1] == 4
assert best_model.output_shape[1:] == (128, 128, 1)

# 4. Use the same preprocessing as full validation
def normalize_test_volume(volume):
    brain = volume != 0
    result = np.zeros_like(volume, dtype=np.float32)

    if np.any(brain):
        mean = volume[brain].mean()
        std = volume[brain].std()

        if std > 0:
            result[brain] = (volume[brain] - mean) / std

    return result

def component_filter(mask, min_size):
    mask = mask.astype(np.uint8)

    if min_size <= 1:
        return mask

    n, labels, stats, _ = cv2.connectedComponentsWithStats(
        mask, connectivity=8
    )

    keep = np.zeros(n, dtype=bool)
    keep[1:] = stats[1:, cv2.CC_STAT_AREA] >= min_size

    return keep[labels].astype(np.uint8)

# 5. Resume saved test results
if os.path.exists(progress_path):
    saved = pd.read_csv(progress_path)
else:
    saved = pd.DataFrame()

eps = 1e-6

print("\n120 — INDEPENDENT TEST RUN")
print("=" * 55)

for number, pid in enumerate(selected_ids, 1):
    patient_dir = f"{data_dir}/{pid}"

    seg_path = f"{patient_dir}/{pid}_seg.nii.gz"
    slice_count = nib.load(seg_path).shape[2]

    expected_keys = {
        (s, m)
        for s in range(slice_count)
        for m in COMPONENT_SIZES
    }

    if not saved.empty:
        old = saved[saved["patient_id"] == pid]

        existing_keys = set(
            zip(old["slice_index"].astype(int),
                old["min_size"].astype(int))
        )

        if existing_keys == expected_keys and len(old) == len(expected_keys):
            print(f"{number}/{len(selected_ids)} already saved — {pid}")
            continue

        saved = saved[saved["patient_id"] != pid].copy()

    volumes = []

    for modality in ["t1", "t1ce", "t2", "flair"]:
        path = f"{patient_dir}/{pid}_{modality}.nii.gz"

        volume = nib.load(path).get_fdata(dtype=np.float32)
        volumes.append(normalize_test_volume(volume))
        del volume

    seg = nib.load(seg_path).get_fdata(dtype=np.float32)

    patient_records = []

    for start in range(0, slice_count, BATCH_SIZE):
        indices = list(
            range(start, min(start + BATCH_SIZE, slice_count))
        )

        X_batch = []
        Y_batch = []

        for idx in indices:
            channels = [
                cv2.resize(
                    vol[:, :, idx],
                    (128, 128),
                    interpolation=cv2.INTER_LINEAR
                )
                for vol in volumes
            ]

            X_batch.append(
                np.stack(channels, axis=-1)
            )

            truth = (seg[:, :, idx] > 0).astype(np.uint8)

            truth = cv2.resize(
                truth, (128, 128),
                interpolation=cv2.INTER_NEAREST
            )

            Y_batch.append(truth)

        X_batch = np.asarray(X_batch, dtype=np.float32)
        Y_batch = np.asarray(Y_batch, dtype=np.uint8)

        probabilities = best_model.predict(
            X_batch, verbose=0
        )[:, :, :, 0]

        for j, idx in enumerate(indices):
            truth = Y_batch[j]
            raw = (
                probabilities[j] > PRED_THRESHOLD
            ).astype(np.uint8)

            gt_pixels = int(truth.sum())

            for min_size in COMPONENT_SIZES:
                pred = component_filter(raw, min_size)

                pred_pixels = int(pred.sum())

                intersection = int(
                    np.logical_and(truth, pred).sum()
                )

                dice = (
                    2 * intersection + eps
                ) / (gt_pixels + pred_pixels + eps)

                iou = (
                    intersection + eps
                ) / (
                    gt_pixels + pred_pixels - intersection + eps
                )

                patient_records.append({
                    "patient_id": pid,
                    "slice_index": idx,
                    "min_size": min_size,
                    "gt_pixels": gt_pixels,
                    "pred_pixels": pred_pixels,
                    "intersection": intersection,
                    "binary_dice": dice,
                    "binary_iou": iou
                })

    saved = pd.concat(
        [saved, pd.DataFrame(patient_records)],
        ignore_index=True
    )

    # Atomic save after each patient
    temp_path = progress_path + ".tmp"
    saved.to_csv(temp_path, index=False)
    os.replace(temp_path, progress_path)

    print(f"{number}/{len(selected_ids)} completed — {pid}")

    del volumes, seg
    gc.collect()

# 6. Report results only for the requested test patients
current = saved[
    saved["patient_id"].isin(selected_ids)
].copy()

summary_rows = []

for min_size in COMPONENT_SIZES:
    subset = current[current["min_size"] == min_size]

    tumor = subset[subset["gt_pixels"] > 0]
    background = subset[subset["gt_pixels"] == 0]
    small = tumor[tumor["gt_pixels"] <= 10]

    # Each patient contributes one volume-level Dice
    patient_stats = subset.groupby("patient_id").agg(
        gt=("gt_pixels", "sum"),
        pred=("pred_pixels", "sum"),
        tp=("intersection", "sum")
    )

    patient_dice = (
        2 * patient_stats["tp"] + eps
    ) / (
        patient_stats["gt"] + patient_stats["pred"] + eps
    )

    summary_rows.append({
        "Min Component": min_size,
        "Tumor Slice Dice": tumor["binary_dice"].mean(),
        "Tumor Slice IoU": tumor["binary_iou"].mean(),
        "Small Tumor Dice": small["binary_dice"].mean(),
        "Small Tumors Missed": (
            (small["intersection"] == 0).mean()
            if len(small) else np.nan
        ),
        "Background FP Rate": (
            (background["pred_pixels"] > 0).mean()
        ),
        "Mean Patient Volume Dice": patient_dice.mean()
    })

summary120 = pd.DataFrame(summary_rows)

print("\n" + "=" * 65)
print("120 — INDEPENDENT TEST RESULTS")
print("=" * 65)
print("Patients:", current["patient_id"].nunique())
print("Slices:", len(current[current["min_size"] == 0]))

display(summary120.round(4))

print("\nProgress saved:", progress_path)
print("No training or checkpoint changes.")



## 120. Independent Test Evaluation — Final Results

### Objective

The best U-Net checkpoint was evaluated on an independent test dataset to assess segmentation performance on previously unseen patients.

Two methods were compared:
1. Baseline U-Net without post-processing
2. U-Net with 2-pixel Connected Component Filtering

The prediction threshold (0.5) and post-processing configuration were selected before independent testing.

### Dataset and Evaluation

- **Independent test patients:** 188
- **MRI slices evaluated:** 29,140
- **MRI modalities:** T1, T1ce, T2, FLAIR
- **Input resolution:** 128 × 128
- **Prediction threshold:** 0.5
- **Patient overlap between train, validation, and test:** 0

### Final Test Results

| Metric | Baseline U-Net | 2-Pixel Filtering |
|---|---:|---:|
| Tumor Slice Dice | 0.8434 | 0.8432 |
| Tumor Slice IoU | 0.7711 | 0.7709 |
| Small Tumor Dice | 0.2526 | 0.2480 |
| Small Tumors Missed | 43.33% | 44.68% |
| Background False-Positive Rate | 29.33% | 26.45% |
| Mean Patient Volume Dice | 0.8891 | 0.8894 |

### Validation vs. Independent Test

The baseline U-Net achieved:

| Metric | Validation | Independent Test |
|---|---:|---:|
| Tumor Slice Dice | 0.8414 | 0.8434 |
| Tumor Slice IoU | 0.7665 | 0.7711 |
| Background FP Rate | 27.44% | 29.33% |

### Key Findings

**1. Generalization Performance**

The model achieved similar tumor-slice Dice scores on validation (0.8414) and independent test (0.8434) data, suggesting relatively stable segmentation performance across these patient groups.

**2. False-Positive Reduction**

The 2-pixel Connected Component Filter reduced the background false-positive slice rate from 29.33% to 26.45%.

**3. Small Tumor Sensitivity**

Small tumor detection remains challenging. Filtering slightly decreased Small Tumor Dice from 0.2526 to 0.2480 and increased the proportion of completely missed small tumors.

**4. Patient-Level Performance**

The baseline model achieved a mean patient-level volumetric Dice score of 0.8891.

**5. Trade-Off Analysis**

Connected Component Filtering reduced false positives but introduced a small reduction in small-lesion sensitivity. Therefore, aggressive filtering should be avoided without additional analysis.

### Limitations

- The segmentation model operates on resized 128 × 128 MRI slices.
- Very small tumor regions may be lost during mask downsampling.
- Small tumor sensitivity remains limited.
- Results are based on binary whole-tumor segmentation rather than separate BraTS tumor subregions.
- Independent clinical validation on external datasets has not been performed.

### Conclusion

The NeuroRobust U-Net achieved a **mean tumor-slice Dice score of 0.8434** and a **mean patient-level volumetric Dice score of 0.8891** on 188 independent test patients.

The results support promising performance on this dataset, while highlighting important limitations involving small tumor detection and false positives.

The independent test evaluation is complete. No model weights were modified during testing.


In [ ]:

# ==========================================================
# 121 — FINAL RESULTS VISUALIZATION
# NeuroRobust | BraTS2021
# Full validation vs independent test
# No training or checkpoint modification
# ==========================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import drive
from IPython.display import display

# 1. Connect Google Drive
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

project_dir = "/content/drive/MyDrive/NeuroRobust"
results_dir = f"{project_dir}/results"
figures_dir = f"{results_dir}/final_figures"

os.makedirs(figures_dir, exist_ok=True)

# 2. Load the completed experiments
validation_path = (
    f"{results_dir}/"
    "connected_component_experiment_progress.csv"
)

test_path = (
    f"{results_dir}/independent_test_progress.csv"
)

validation = pd.read_csv(validation_path)
test = pd.read_csv(test_path)

METHODS = [0, 2]

# 3. Verify both datasets are complete
def verify_results(df, dataset_name):

    required_columns = [
        "patient_id", "slice_index", "min_size",
        "gt_pixels", "pred_pixels",
        "binary_dice", "binary_iou"
    ]

    missing_columns = [
        c for c in required_columns
        if c not in df.columns
    ]

    assert not missing_columns, (
        f"{dataset_name}: Missing columns: "
        f"{missing_columns}"
    )

    assert not df.duplicated(
        ["patient_id", "slice_index", "min_size"]
    ).any(), f"{dataset_name}: Duplicate records found."

    subsets = []

    for method in METHODS:

        part = df[df["min_size"] == method].copy()

        assert part["patient_id"].nunique() == 188, (
            f"{dataset_name}: Expected 188 patients "
            f"for method {method}"
        )

        assert len(part) == 29140, (
            f"{dataset_name}: Expected 29140 slices "
            f"for method {method}, found {len(part)}"
        )

        subsets.append(part)

    # Ensure the same slices occur in both methods
    keys0 = set(zip(
        subsets[0]["patient_id"],
        subsets[0]["slice_index"]
    ))

    keys2 = set(zip(
        subsets[1]["patient_id"],
        subsets[1]["slice_index"]
    ))

    assert keys0 == keys2, (
        f"{dataset_name}: Different evaluated slices."
    )

    print(
        f"{dataset_name}: 188 patients, "
        "29140 slices per method — VERIFIED"
    )

verify_results(validation, "Validation")
verify_results(test, "Independent Test")

# 4. Calculate comparable metrics
def calculate_metrics(df, dataset_name):

    records = []

    for method in METHODS:

        subset = df[df["min_size"] == method]

        tumor = subset[
            subset["gt_pixels"] > 0
        ]

        background = subset[
            subset["gt_pixels"] == 0
        ]

        small = tumor[
            tumor["gt_pixels"] <= 10
        ]

        records.append({
            "Dataset": dataset_name,
            "Method": (
                "Baseline U-Net"
                if method == 0
                else "2-pixel Filter"
            ),
            "Tumor Dice": tumor["binary_dice"].mean(),
            "Tumor IoU": tumor["binary_iou"].mean(),
            "Small Tumor Dice": (
                small["binary_dice"].mean()
            ),
            "Background FP Rate": (
                (background["pred_pixels"] > 0).mean()
            ),
            "Tumor Slices": len(tumor),
            "Background Slices": len(background),
            "Small Tumor Slices": len(small)
        })

    return pd.DataFrame(records)

summary = pd.concat([
    calculate_metrics(validation, "Validation"),
    calculate_metrics(test, "Independent Test")
], ignore_index=True)

print("\n" + "=" * 60)
print("121 — FINAL COMPARISON SUMMARY")
print("=" * 60)

display(summary.round(4))

# 5. Professional, consistent figure formatting
plt.rcParams.update({
    "figure.figsize": (9, 5),
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.labelsize": 11,
    "savefig.dpi": 200
})

datasets = ["Validation", "Independent Test"]
methods = ["Baseline U-Net", "2-pixel Filter"]

def make_comparison_chart(
    metric,
    title,
    ylabel,
    filename,
    percentage=False,
    ylim=None
):

    pivot = summary.pivot(
        index="Dataset",
        columns="Method",
        values=metric
    ).reindex(index=datasets, columns=methods)

    values = pivot.copy()

    if percentage:
        values = values * 100

    x = np.arange(len(datasets))
    width = 0.34

    fig, ax = plt.subplots(figsize=(9, 5))

    for i, method in enumerate(methods):

        positions = x + (i - 0.5) * width
        bars = ax.bar(
            positions,
            values[method].values,
            width,
            label=method
        )

        for bar in bars:
            height = bar.get_height()

            label = (
                f"{height:.2f}%"
                if percentage
                else f"{height:.4f}"
            )

            ax.annotate(
                label,
                xy=(
                    bar.get_x() + bar.get_width() / 2,
                    height
                ),
                xytext=(0, 4),
                textcoords="offset points",
                ha="center",
                fontsize=10
            )

    ax.set_xticks(x)
    ax.set_xticklabels(datasets)
    ax.set_title(title)
    ax.set_ylabel(ylabel)

    if ylim is not None:
        ax.set_ylim(*ylim)
    else:
        ax.set_ylim(
            0,
            values.max().max() * 1.16
        )

    ax.legend()
    ax.grid(axis="y", alpha=0.2)
    ax.set_axisbelow(True)

    plt.tight_layout()

    output = os.path.join(
        figures_dir, filename
    )

    fig.savefig(
        output,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

    print("Saved:", output)

# ==========================================================
# GRAPH 1 — TUMOR DICE COMPARISON
# ==========================================================

make_comparison_chart(
    metric="Tumor Dice",
    title="NeuroRobust — Validation vs Independent Test",
    ylabel="Mean Binary Dice (Tumor-Containing Slices)",
    filename="01_validation_vs_test_dice.png",
    ylim=(0, 1)
)

# ==========================================================
# GRAPH 2 — FALSE POSITIVE REDUCTION
# ==========================================================

make_comparison_chart(
    metric="Background FP Rate",
    title="NeuroRobust — Background False-Positive Rate",
    ylabel="False-Positive Slice Rate (%)",
    filename="02_false_positive_reduction.png",
    percentage=True,
    ylim=(0, 40)
)

# ==========================================================
# GRAPH 3 — SMALL TUMOR PERFORMANCE
# ==========================================================

make_comparison_chart(
    metric="Small Tumor Dice",
    title="NeuroRobust — Small Tumor Segmentation",
    ylabel="Mean Binary Dice (1–10 Tumor Pixels)",
    filename="03_small_tumor_dice.png",
    ylim=(0, 0.5)
)

# ==========================================================
# 6. SAVE FINAL SUMMARY
# ==========================================================

summary_path = os.path.join(
    results_dir,
    "final_validation_test_comparison.csv"
)

summary.to_csv(
    summary_path,
    index=False
)

print("\n" + "=" * 60)
print("FINAL VISUALIZATION STATUS")
print("=" * 60)

print("Three figures created successfully.")
print("Comparison table saved:", summary_path)
print("Figures directory:", figures_dir)
print("No model training was performed.")



### 121. Final Results Visualization

Three comparative figures were generated using the complete validation and independent test results, each containing 188 patients and 29,140 MRI slices.

**Key Findings:**
- Validation Tumor Dice: **0.8414**
- Independent Test Tumor Dice: **0.8434**
- Test background false-positive rate: **29.33% → 26.45%** with 2-pixel filtering
- Test Small Tumor Dice: **0.2526 → 0.2480** after filtering

**Conclusion:** The model demonstrated consistent tumor segmentation performance across validation and independent test datasets. Connected Component Filtering reduced false positives but slightly decreased small-tumor sensitivity.

The three figures were saved for the final research report and GitHub documentation. These results highlight both the model's strengths and its remaining limitations.


In [ ]:

# ==========================================================
# 122 — FINAL QUALITATIVE SEGMENTATION ANALYSIS
# NeuroRobust | BraTS2021
#
# Uses the completed independent test results.
# Creates publication-style MRI comparison panels.
#
# No model training.
# No checkpoint or original dataset modifications.
# ==========================================================

import os
import gc
import cv2
import json
import numpy as np
import pandas as pd
import nibabel as nib
import tensorflow as tf
import matplotlib.pyplot as plt

from google.colab import drive
from IPython.display import display

# ----------------------------------------------------------
# 1. DRIVE AND FILES
# ----------------------------------------------------------

if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

project_dir = "/content/drive/MyDrive/NeuroRobust"
data_dir = f"{project_dir}/dataset/BraTS2021_Training_Data"
results_dir = f"{project_dir}/results"
figures_dir = f"{results_dir}/final_figures"

os.makedirs(figures_dir, exist_ok=True)

test_path = f"{results_dir}/independent_test_progress.csv"
model_path = f"{project_dir}/models/best_unet_local.keras"

assert os.path.isfile(test_path), "Test results not found."
assert os.path.isfile(model_path), "Checkpoint not found."

test_df = pd.read_csv(test_path)

required = [
    "patient_id", "slice_index", "min_size",
    "gt_pixels", "pred_pixels",
    "intersection", "binary_dice"
]

assert all(c in test_df.columns for c in required)

baseline = test_df[
    test_df["min_size"] == 0
].copy().reset_index(drop=True)

filtered = test_df[
    test_df["min_size"] == 2
].copy().reset_index(drop=True)

assert len(baseline) == 29140
assert len(filtered) == 29140
assert baseline["patient_id"].nunique() == 188

print("INDEPENDENT TEST RESULTS VERIFIED")
print("Patients:", baseline["patient_id"].nunique())
print("Slices:", len(baseline))

# ----------------------------------------------------------
# 2. SELECT EXAMPLES FROM INDEPENDENT TEST
# ----------------------------------------------------------
# Selection is for visualization only.
# No threshold or model parameters are adjusted.

selected = []
used = set()

def add_example(label, candidates, sort_columns=None,
                ascending=True):

    available = candidates.copy()

    available["key"] = list(
        zip(
            available["patient_id"],
            available["slice_index"]
        )
    )

    available = available[
        ~available["key"].isin(used)
    ]

    if available.empty:
        print("No example found for:", label)
        return

    if sort_columns is not None:
        available = available.sort_values(
            sort_columns,
            ascending=ascending
        )

    row = available.iloc[0].copy()
    key = (row["patient_id"], int(row["slice_index"]))

    used.add(key)
    selected.append((label, row))


tumor = baseline[baseline["gt_pixels"] > 0].copy()
background = baseline[baseline["gt_pixels"] == 0].copy()

# A. Strong segmentation on a large tumor
add_example(
    "Strong tumor segmentation",
    tumor[tumor["gt_pixels"] >= 100],
    ["binary_dice", "gt_pixels"],
    [False, False]
)

# B. A typical tumor-containing slice
median_dice = tumor["binary_dice"].median()

typical = tumor[tumor["gt_pixels"] >= 20].copy()

typical["distance_to_median"] = (
    typical["binary_dice"] - median_dice
).abs()

add_example(
    "Typical tumor segmentation",
    typical,
    ["distance_to_median"],
    True
)

# C. A small tumor successfully detected
small_detected = tumor[
    (tumor["gt_pixels"] <= 10)
    & (tumor["intersection"] > 0)
]

add_example(
    "Small tumor detected",
    small_detected,
    ["binary_dice"],
    False
)

# D. A completely missed small tumor
small_missed = tumor[
    (tumor["gt_pixels"] <= 10)
    & (tumor["intersection"] == 0)
]

add_example(
    "Small tumor missed",
    small_missed,
    ["gt_pixels"],
    True
)

# E. Background false positive
false_positive = background[
    background["pred_pixels"] > 0
]

add_example(
    "Background false positive",
    false_positive,
    ["pred_pixels"],
    False
)

# F. Correctly predicted empty mask
clean_background = background[
    background["pred_pixels"] == 0
].copy()

clean_background["middle_distance"] = (
    clean_background["slice_index"] - 77
).abs()

add_example(
    "Correct empty prediction",
    clean_background,
    ["middle_distance"],
    True
)

assert len(selected) >= 4, (
    "Not enough examples for visualization."
)

print("\nSELECTED INDEPENDENT TEST EXAMPLES")

manifest = pd.DataFrame([
    {
        "example": label,
        "patient_id": row["patient_id"],
        "slice_index": int(row["slice_index"]),
        "true_pixels": int(row["gt_pixels"]),
        "pred_pixels": int(row["pred_pixels"]),
        "baseline_dice": float(row["binary_dice"])
    }
    for label, row in selected
])

display(manifest)

# ----------------------------------------------------------
# 3. LOAD BEST SAVED MODEL
# ----------------------------------------------------------

best_model = tf.keras.models.load_model(
    model_path,
    compile=False
)

assert best_model.input_shape[-1] == 4
assert best_model.output_shape[1:] == (128, 128, 1)

print("\nBest U-Net checkpoint loaded.")

# ----------------------------------------------------------
# 4. PREPROCESSING — SAME AS VALIDATION AND TEST
# ----------------------------------------------------------

MODALITIES = ["t1", "t1ce", "t2", "flair"]

def normalize_volume(volume):

    brain = volume != 0
    output = np.zeros_like(
        volume, dtype=np.float32
    )

    if brain.any():

        mean = volume[brain].mean()
        std = volume[brain].std()

        if std > 0:
            output[brain] = (
                volume[brain] - mean
            ) / std

    return output


def load_test_slice(pid, idx):

    channels = []

    for modality in MODALITIES:

        path = (
            f"{data_dir}/{pid}/"
            f"{pid}_{modality}.nii.gz"
        )

        volume = nib.load(path).get_fdata(
            dtype=np.float32
        )

        normalized = normalize_volume(volume)

        image = cv2.resize(
            normalized[:, :, idx],
            (128, 128),
            interpolation=cv2.INTER_LINEAR
        )

        channels.append(image)

        del volume, normalized

    X = np.stack(
        channels, axis=-1
    ).astype(np.float32)

    seg_path = (
        f"{data_dir}/{pid}/"
        f"{pid}_seg.nii.gz"
    )

    seg = nib.load(seg_path).get_fdata(
        dtype=np.float32
    )

    truth = (
        seg[:, :, idx] > 0
    ).astype(np.uint8)

    truth = cv2.resize(
        truth,
        (128, 128),
        interpolation=cv2.INTER_NEAREST
    )

    return X, truth


def filter_2px(mask):

    mask = mask.astype(np.uint8)

    number, labels, stats, _ = (
        cv2.connectedComponentsWithStats(
            mask, connectivity=8
        )
    )

    keep = np.zeros(number, dtype=bool)

    if number > 1:
        keep[1:] = (
            stats[1:, cv2.CC_STAT_AREA] >= 2
        )

    return keep[labels].astype(np.uint8)


def create_error_overlay(truth, pred):

    overlay = np.zeros(
        (128, 128, 3), dtype=np.float32
    )

    # Green: True Positive
    overlay[:, :, 1] = (
        (truth == 1) & (pred == 1)
    )

    # Red: False Positive
    overlay[:, :, 0] = (
        (truth == 0) & (pred == 1)
    )

    # Blue: False Negative
    overlay[:, :, 2] = (
        (truth == 1) & (pred == 0)
    )

    return overlay


def dice_score(truth, pred):

    eps = 1e-6

    return (
        2 * np.logical_and(truth, pred).sum() + eps
    ) / (
        truth.sum() + pred.sum() + eps
    )


# ----------------------------------------------------------
# 5. RUN INFERENCE ON SELECTED TEST CASES
# ----------------------------------------------------------

examples = []

for number, (label, row) in enumerate(selected, 1):

    pid = row["patient_id"]
    idx = int(row["slice_index"])

    X, truth = load_test_slice(pid, idx)

    probability = best_model.predict(
        X[None, ...],
        verbose=0
    )[0, :, :, 0]

    raw = (
        probability > 0.5
    ).astype(np.uint8)

    filtered_mask = filter_2px(raw)

    # Confirm the visualization reproduces saved test scores
    raw_dice = dice_score(truth, raw)

    assert abs(
        raw_dice - float(row["binary_dice"])
    ) < 1e-4, (
        f"Prediction mismatch for {pid}, slice {idx}"
    )

    examples.append({
        "label": label,
        "patient": pid,
        "slice": idx,
        "flair": X[:, :, 3],
        "truth": truth,
        "probability": probability,
        "raw": raw,
        "filtered": filtered_mask,
        "overlay": create_error_overlay(
            truth, raw
        ),
        "dice": raw_dice,
        "filtered_dice": dice_score(
            truth, filtered_mask
        )
    })

    print(
        f"{number}/{len(selected)} | "
        f"{label} | {pid} | "
        f"Slice {idx} | Dice {raw_dice:.4f}"
    )

    gc.collect()

# ----------------------------------------------------------
# 6. PUBLICATION-STYLE COMPARISON PANEL
# ----------------------------------------------------------

columns = [
    "FLAIR MRI",
    "Ground Truth",
    "Probability Map",
    "Baseline U-Net",
    "2-pixel Filter",
    "Error Map"
]

fig, axes = plt.subplots(
    len(examples), 6,
    figsize=(21, 3.9 * len(examples)),
    squeeze=False
)

for i, example in enumerate(examples):

    images = [
        example["flair"],
        example["truth"],
        example["probability"],
        example["raw"],
        example["filtered"],
        example["overlay"]
    ]

    for j, image in enumerate(images):

        ax = axes[i, j]

        if j == 5:
            ax.imshow(image, interpolation="nearest")

        elif j == 2:
            ax.imshow(
                image,
                cmap="inferno",
                vmin=0,
                vmax=1
            )

        elif j == 0:
            # Robust MRI display contrast
            brain = image[image != 0]

            if brain.size > 0:
                low, high = np.percentile(
                    brain, [2, 98]
                )

                ax.imshow(
                    image,
                    cmap="gray",
                    vmin=low,
                    vmax=high
                )
            else:
                ax.imshow(image, cmap="gray")

        else:
            ax.imshow(
                image,
                cmap="gray",
                vmin=0,
                vmax=1,
                interpolation="nearest"
            )

        if i == 0:
            ax.set_title(
                columns[j],
                fontsize=11,
                fontweight="bold",
                pad=12
            )

        ax.axis("off")

    axes[i, 0].set_ylabel(
        f"{example['label']}\n"
        f"{example['patient']}\n"
        f"Slice {example['slice']}\n"
        f"Dice {example['dice']:.3f}",
        fontsize=10,
        rotation=0,
        labelpad=105,
        va="center"
    )

fig.suptitle(
    "NeuroRobust — Independent Test Segmentation Examples",
    fontsize=20,
    fontweight="bold",
    y=0.998
)

fig.text(
    0.5,
    0.004,
    "Error map: Green = True Positive | "
    "Red = False Positive | Blue = False Negative\n"
    "Illustrative cases selected from the independent test set; "
    "not representative of average performance.",
    ha="center",
    fontsize=11
)

fig.tight_layout(
    rect=[0.06, 0.035, 1, 0.975],
    h_pad=2
)

panel_path = (
    f"{figures_dir}/"
    "04_independent_test_segmentation_examples.png"
)

fig.savefig(
    panel_path,
    dpi=180,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()
plt.close(fig)

# ----------------------------------------------------------
# 7. SMALL TUMOR ZOOM PANEL
# ----------------------------------------------------------

small_examples = [
    e for e in examples
    if e["label"] in [
        "Small tumor detected",
        "Small tumor missed"
    ]
]

if small_examples:

    fig, axes = plt.subplots(
        len(small_examples), 3,
        figsize=(12, 4 * len(small_examples)),
        squeeze=False
    )

    for i, example in enumerate(small_examples):

        truth = example["truth"]
        coords = np.argwhere(truth > 0)

        if len(coords) == 0:
            continue

        cy, cx = np.mean(
            coords, axis=0
        ).astype(int)

        radius = 15

        y1 = max(0, cy - radius)
        y2 = min(128, cy + radius)
        x1 = max(0, cx - radius)
        x2 = min(128, cx + radius)

        views = [
            truth[y1:y2, x1:x2],
            example["raw"][y1:y2, x1:x2],
            example["overlay"][y1:y2, x1:x2]
        ]

        titles = [
            "Ground Truth — Zoom",
            "Prediction — Zoom",
            "Error Map — Zoom"
        ]

        for j, view in enumerate(views):

            ax = axes[i, j]

            if j == 2:
                ax.imshow(
                    view,
                    interpolation="nearest"
                )
            else:
                ax.imshow(
                    view,
                    cmap="gray",
                    vmin=0,
                    vmax=1,
                    interpolation="nearest"
                )

            ax.set_title(titles[j])
            ax.axis("off")

        axes[i, 0].set_ylabel(
            example["label"],
            rotation=0,
            labelpad=105,
            va="center"
        )

    fig.suptitle(
        "NeuroRobust — Small Tumor Error Analysis",
        fontsize=16,
        fontweight="bold"
    )

    fig.tight_layout(
        rect=[0.08, 0, 1, 0.96]
    )

    zoom_path = (
        f"{figures_dir}/"
        "05_small_tumor_zoom_analysis.png"
    )

    fig.savefig(
        zoom_path,
        dpi=200,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.show()
    plt.close(fig)

    print("Small tumor zoom saved:", zoom_path)

# ----------------------------------------------------------
# 8. SAVE EXAMPLE MANIFEST
# ----------------------------------------------------------

manifest_path = (
    f"{results_dir}/"
    "qualitative_test_examples.csv"
)

manifest.to_csv(
    manifest_path,
    index=False
)

print("\n" + "=" * 60)
print("122 — FINAL VISUALIZATION STATUS")
print("=" * 60)

print("Independent test examples:", len(examples))
print("Main comparison figure:", panel_path)
print("Example manifest:", manifest_path)
print("Original model and dataset unchanged.")
print("122. hücre tamamlandı.")



### 122. Qualitative Segmentation Analysis

Six representative examples were selected from the independent test set to visualize the strengths and limitations of the U-Net model.

**Visualizations:**
- FLAIR MRI images
- Ground-truth tumor masks
- Predicted probability maps
- Baseline U-Net predictions
- 2-pixel filtered predictions
- Error maps (green: TP, red: FP, blue: FN)

**Key Findings:**
- Strong tumor segmentation example: Dice **0.9911**
- Typical tumor segmentation example: Dice **0.9260**
- Small tumor examples demonstrated both successful detection and complete misses.
- A background-only example revealed a large false-positive region.
- Empty ground-truth and prediction masks correctly produced a Dice score of 1.0.

**Conclusion:**

The qualitative results demonstrate strong segmentation performance on selected tumor-containing slices, while highlighting limitations in small-tumor detection and false-positive predictions.

These examples were selected for illustration and do not represent average model performance. The quantitative results are reported separately using all 188 independent test patients.
